# Customer Support AI Agent — TWCS

## Hiver SDE Intern Take-Home Assignment

This notebook contains the complete research, experimentation, evaluation,
and analysis pipeline for building a historically grounded AI customer-support
agent using the Customer Support on Twitter (TWCS) dataset.

### Objective

Build an AI support agent for one selected customer-support brand that can:

1. Identify the customer's intent.
2. Retrieve relevant historical support resolutions.
3. Generate a response grounded in historical evidence.
4. Decide whether the request should be automatically handled or escalated.

### Methodology

The project follows this progression:

Dataset
→ Data Quality
→ Conversation Reconstruction
→ Brand Selection
→ Intent Discovery
→ Baselines
→ Intent Classification
→ Historical Resolution Retrieval
→ Response Generation
→ Escalation Decision
→ End-to-End Agent
→ Evaluation
→ Failure Analysis

### Important Data Principle

The dataset is conversational rather than a collection of independent
classification examples. Conversation relationships will therefore be
reconstructed using the tweet relationship fields, and evaluation splits
will be performed at the conversation level to reduce leakage.

### Important Historical-Data Limitation

TWCS contains historical customer-support interactions. Historical responses
are treated as evidence of how similar issues were handled in the dataset,
not as proof of the brand's current policies or procedures.

> The goal is to build a trustworthy support-agent prototype grounded in
> historical evidence, not to claim that it represents an official brand
> support system.

In [1]:
from pathlib import Path
import sys
import platform

print("Python version :", sys.version.split()[0])
print("Platform       :", platform.platform())

Python version : 3.11.9
Platform       : Windows-10-10.0.26200-SP0


In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
GOLDEN_DATA_DIR = DATA_DIR / "golden"

print("\nProject root   :", PROJECT_ROOT)
print("Raw data       :", RAW_DATA_DIR)
print("Processed data :", PROCESSED_DATA_DIR)
print("Golden data    :", GOLDEN_DATA_DIR)


Project root   : d:\BECAME_DEVELOPER\hiver-sde-ai-agent
Raw data       : d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\raw
Processed data : d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed
Golden data    : d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\golden


In [3]:
# Create directories if they do not already exist
for directory in [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    GOLDEN_DATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\nDirectory setup complete.")


Directory setup complete.


In [4]:
import pandas as pd
import numpy as np

from collections import Counter
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

pandas : 2.2.3
numpy  : 2.1.3


In [5]:
# Discover files available in the raw dataset directory

raw_files = sorted(
    path for path in RAW_DATA_DIR.iterdir()
    if path.is_file()
)

print(f"Files found: {len(raw_files)}\n")

for path in raw_files:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.name:<30} {size_mb:>10.2f} MB")

Files found: 1

twcs.csv                           492.58 MB


In [6]:
# Inspect a small sample of the TWCS dataset

TWCS_PATH = RAW_DATA_DIR / "twcs.csv"

sample_df = pd.read_csv(
    TWCS_PATH,
    nrows=10
)

print("Dataset file:", TWCS_PATH)
print("\nShape of sample:", sample_df.shape)
print("\nColumns:")
print(sample_df.columns.tolist())

print("\nFirst 10 rows:")
display(sample_df)

Dataset file: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\raw\twcs.csv

Shape of sample: (10, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

First 10 rows:


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further as...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messages and no one is responding as usual,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your ...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,"@115712 Can you please send us a private message, so that I can gain further details about your account?","5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,"@115713 This is saddening to hear. Please shoot us a DM, so that we can look into this for you. -KC",NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,"@115713 We understand your concerns and we'd like for you to please send us a Direct Message, so that we can further...",12,16.0


In [7]:
# Inspect data types and missing values in the sample

print("DATA TYPES")
print("=" * 50)
display(sample_df.dtypes.to_frame("dtype"))

print("\nMISSING VALUES")
print("=" * 50)

missing_summary = pd.DataFrame({
    "missing_count": sample_df.isna().sum(),
    "missing_percent": sample_df.isna().mean().mul(100).round(2)
})

display(missing_summary)

DATA TYPES


,dtype
tweet_id,int64
author_id,object
inbound,bool
created_at,object
text,object
response_tweet_id,object
in_response_to_tweet_id,float64



MISSING VALUES


,missing_count,missing_percent
tweet_id,0,0.0
author_id,0,0.0
inbound,0,0.0
created_at,0,0.0
text,0,0.0
response_tweet_id,2,20.0
in_response_to_tweet_id,1,10.0


In [8]:
# Load the complete TWCS dataset with memory-conscious dtypes
TWCS_DTYPES = {
    "tweet_id": "int64",
    "author_id": "string",
    "inbound": "bool",
    "created_at": "string",
    "text": "string",
    "response_tweet_id": "string",
    "in_response_to_tweet_id": "Int64",
}

twcs = pd.read_csv(
    TWCS_PATH,
    dtype=TWCS_DTYPES,
    low_memory=False
)

print("Dataset loaded successfully!")
print(f"Rows    : {len(twcs):,}")
print(f"Columns : {len(twcs.columns)}")
print(f"Memory  : {twcs.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

Dataset loaded successfully!
Rows    : 2,811,774
Columns : 7
Memory  : 1.17 GB


In [9]:
# Dataset shape
print("DATASET SHAPE")
print("=" * 60)
print(f"Rows    : {len(twcs):,}")
print(f"Columns : {len(twcs.columns)}")

DATASET SHAPE
Rows    : 2,811,774
Columns : 7


In [10]:
# duplicate tweet_ids
duplicate_tweet_ids = twcs["tweet_id"].duplicated().sum()

print("\nDUPLICATE TWEET IDs")
print("=" * 60)
print(f"Duplicate tweet_id rows : {duplicate_tweet_ids:,}")


DUPLICATE TWEET IDs
Duplicate tweet_id rows : 0


In [11]:
# duplicate complete rows
duplicate_rows = twcs.duplicated().sum()

print("\nDUPLICATE COMPLETE ROWS")
print("=" * 60)
print(f"Duplicate rows : {duplicate_rows:,}")


DUPLICATE COMPLETE ROWS
Duplicate rows : 0


In [12]:
# missing values
missing_summary = pd.DataFrame({
    "missing_count": twcs.isna().sum(),
    "missing_percent": (
        twcs.isna().mean() * 100
    ).round(2)
})

print("\nMISSING VALUES")
print("=" * 60)
display(missing_summary)


MISSING VALUES


,missing_count,missing_percent
tweet_id,0,0.00
author_id,0,0.00
inbound,0,0.00
created_at,0,0.00
text,0,0.00
response_tweet_id,1040629,37.01
in_response_to_tweet_id,794335,28.25


In [13]:
# empty/whitespace-only text
empty_text = (
    twcs["text"]
    .str.strip()
    .eq("")
    .sum()
)

print("\nEMPTY TEXT")
print("=" * 60)
print(f"Empty/whitespace-only text : {empty_text:,}")


EMPTY TEXT
Empty/whitespace-only text : 0


In [14]:
# inbound/outbound distribution
direction_summary = pd.DataFrame({
    "count": twcs["inbound"].value_counts(),
    "percentage": (
        twcs["inbound"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
})

direction_summary.index = [
    "Customer → Company" if value else "Company → Customer"
    for value in direction_summary.index
]

print("\nMESSAGE DIRECTION")
print("=" * 60)
display(direction_summary)


MESSAGE DIRECTION


,count,percentage
Customer → Company,1537843,54.69
Company → Customer,1273931,45.31


In [15]:
# timestamp validation
twcs["created_at_parsed"] = pd.to_datetime(
    twcs["created_at"],
    errors="coerce",
    utc=True
)

invalid_timestamps = twcs["created_at_parsed"].isna().sum()

print("TIMESTAMP VALIDATION")
print("=" * 60)

print(f"Invalid timestamps : {invalid_timestamps:,}")

print("\nTIMESTAMP RANGE")
print("=" * 60)

print("Earliest:", twcs["created_at_parsed"].min())
print("Latest  :", twcs["created_at_parsed"].max())

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8164\987411096.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  twcs["created_at_parsed"] = pd.to_datetime(


TIMESTAMP VALIDATION
Invalid timestamps : 0

TIMESTAMP RANGE
Earliest: 2008-05-08 20:13:59+00:00
Latest  : 2017-12-03 23:14:01+00:00


In [16]:
print("YEAR DISTRIBUTION")
print("=" * 60)

year_distribution = (
    twcs["created_at_parsed"]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("year")
    .reset_index(name="tweet_count")
)

year_distribution["percentage"] = (
    year_distribution["tweet_count"] / len(twcs) * 100
).round(2)

display(year_distribution)


YEAR DISTRIBUTION


,year,tweet_count,percentage
0,2008,2,0.00
1,2010,7,0.00
2,2011,11,0.00
3,2012,56,0.00
4,2013,86,0.00
5,2014,199,0.01
6,2015,427,0.02
7,2016,1512,0.05
8,2017,2809474,99.92


The 2008–2016 records are unusual but legitimate timestamp values in the file; won't delete them simply because they're old. Will preserve the dataset and investigate only if they become relevant to our analysis.

Agent is grounded in historical support interactions, overwhelmingly from 2017. It should not claim to represent current brand policies.

Discover candidate support brands

In [17]:
# Company/support messages only
support_messages = twcs.loc[
    ~twcs["inbound"],
    ["author_id", "tweet_id", "text"]
].copy()

brand_volume = (
    support_messages
    .groupby("author_id")
    .agg(
        support_tweets=("tweet_id", "count"),
        unique_responses=("text", "nunique")
    )
    .sort_values("support_tweets", ascending=False)
)

brand_volume["response_uniqueness_pct"] = (
    brand_volume["unique_responses"]
    / brand_volume["support_tweets"]
    * 100
).round(2)

print(f"Unique support accounts: {len(brand_volume):,}")
print("\nTOP 30 SUPPORT ACCOUNTS")
print("=" * 80)

display(brand_volume.head(30))

Unique support accounts: 108

TOP 30 SUPPORT ACCOUNTS


,support_tweets,unique_responses,response_uniqueness_pct
author_id,,,
AmazonHelp,169840,169805,99.98
AppleSupport,106860,106770,99.92
Uber_Support,56270,56198,99.87
SpotifyCares,43265,43261,99.99
Delta,42253,42243,99.98
Tesco,38573,38562,99.97
AmericanAir,36764,36763,100.00
TMobileHelp,34317,34303,99.96
comcastcares,33031,33018,99.96


The response_uniqueness_pct is ~99.7–100% for essentially every major brand.

That tells almost every support tweet is textually unique.

In [18]:
brand_customer = (
    twcs.loc[
        twcs["inbound"],
        ["author_id", "tweet_id"]
    ]
    .groupby("author_id")
    .agg(
        customer_tweets=("tweet_id", "count")
    )
)

print(f"Unique customer accounts: {len(brand_customer):,}")
print("\nTOP 30 CUSTOMER ACCOUNTS")
print("=" * 80)

display(brand_customer.head(30).sort_values("customer_tweets", ascending=False))

Unique customer accounts: 702,669

TOP 30 CUSTOMER ACCOUNTS


,customer_tweets
author_id,
10304,10
10766,5
11242,5
10026,3
10297,2
10221,2
10424,2
10637,2
10423,2


In [19]:
# INVALID (not using this brand_summary)

brand_summary = brand_volume.join( 
    brand_customer,
    how="left"
)

brand_summary["customer_tweets"] = (
    brand_summary["customer_tweets"]
    .fillna(0)
    .astype("int64")
)

brand_summary["total_tweets"] = (
    brand_summary["support_tweets"] + brand_summary["customer_tweets"]
)

brand_summary["support_share_pct"] = (
    brand_summary["support_tweets"] / brand_summary["total_tweets"] * 100
).round(2) 

In [20]:
print("TOP 30 SUPPORT BRANDS — CUSTOMER + SUPPORT VOLUME")
print("=" * 100)

display(
    brand_summary[
        [
            "support_tweets",
            "customer_tweets",
            "total_tweets",
            "support_share_pct"
        ]
    ].head(30)
)

TOP 30 SUPPORT BRANDS — CUSTOMER + SUPPORT VOLUME


,support_tweets,customer_tweets,total_tweets,support_share_pct
author_id,,,,
AmazonHelp,169840,0,169840,100.0
AppleSupport,106860,0,106860,100.0
Uber_Support,56270,0,56270,100.0
SpotifyCares,43265,0,43265,100.0
Delta,42253,0,42253,100.0
Tesco,38573,0,38573,100.0
AmericanAir,36764,0,36764,100.0
TMobileHelp,34317,0,34317,100.0
comcastcares,33031,0,33031,100.0


customer_tweets are 0 -> cannot connect customers to brands through author_id directly

should not use this brand_summary result in analysis

In [21]:
# conversation linking
support_tweet_lookup = (
    twcs.loc[
        ~twcs["inbound"],
        ["tweet_id", "author_id"]
    ]
    .rename(columns={"author_id": "support_account"})
    .set_index("tweet_id")
)

print("Support tweets indexed :", f"{len(support_tweet_lookup):,}")
print("Unique support accounts:", f"{support_tweet_lookup['support_account'].nunique():,}")

print("\nSample lookup:")
display(support_tweet_lookup.head(10))

Support tweets indexed : 1,273,931
Unique support accounts: 108

Sample lookup:


,support_account
tweet_id,
1,sprintcare
4,sprintcare
6,sprintcare
11,sprintcare
15,sprintcare
17,sprintcare
19,sprintcare
21,Ask_Spectrum
25,Ask_Spectrum


In [22]:
# Customer → Support References
customer_tweets = twcs.loc[
    twcs["inbound"],
    ["tweet_id", "author_id", "response_tweet_id", "in_response_to_tweet_id"]
].copy()

# Customer tweets that contain at least one response_tweet_id
customer_with_responses = customer_tweets[
    customer_tweets["response_tweet_id"].notna()
].copy()

print("CUSTOMER → SUPPORT LINKAGE")
print("=" * 70)

print(
    f"Customer tweets                         : {len(customer_tweets):,}"
)

print(
    f"Customer tweets with response_tweet_id  : "
    f"{len(customer_with_responses):,}"
)

print(
    f"Percentage with response_tweet_id       : "
    f"{len(customer_with_responses) / len(customer_tweets) * 100:.2f}%"
)

CUSTOMER → SUPPORT LINKAGE
Customer tweets                         : 1,537,843
Customer tweets with response_tweet_id  : 1,303,829
Percentage with response_tweet_id       : 84.78%


In [23]:
# check whether response_tweet_id points to support tweets
def contains_support_reference(value):
    if pd.isna(value):
        return False

    tweet_ids = str(value).split(",")

    return any(
        int(tweet_id.strip()) in support_tweet_lookup.index
        for tweet_id in tweet_ids
        if tweet_id.strip().isdigit()
    )


customer_with_responses["references_support_tweet"] = (
    customer_with_responses["response_tweet_id"]
    .apply(contains_support_reference)
)

support_reference_count = (
    customer_with_responses["references_support_tweet"].sum()
)

print(
    f"\nCustomer tweets referencing a support tweet : "
    f"{support_reference_count:,}"
)

print(
    f"Percentage of linked customer tweets          : "
    f"{support_reference_count / len(customer_with_responses) * 100:.2f}%"
)


Customer tweets referencing a support tweet : 1,164,365
Percentage of linked customer tweets          : 89.30%


In [24]:
# Support → Customer References
support_tweets = twcs.loc[
    ~twcs["inbound"],
    ["tweet_id", "author_id", "response_tweet_id", "in_response_to_tweet_id"]
].copy()

support_with_parent = support_tweets[
    support_tweets["in_response_to_tweet_id"].notna()
].copy()

print("SUPPORT → CUSTOMER LINKAGE")
print("=" * 70)

print(
    f"Support tweets                         : "
    f"{len(support_tweets):,}"
)

print(
    f"Support tweets with in_response_to_tweet_id : "
    f"{len(support_with_parent):,}"
)

print(
    f"Percentage with parent tweet           : "
    f"{len(support_with_parent) / len(support_tweets) * 100:.2f}%"
)

SUPPORT → CUSTOMER LINKAGE
Support tweets                         : 1,273,931
Support tweets with in_response_to_tweet_id : 1,266,942
Percentage with parent tweet           : 99.45%


In [25]:
customer_tweet_ids = set(
    customer_tweets["tweet_id"]
)

support_with_parent["parent_is_customer"] = (
    support_with_parent["in_response_to_tweet_id"]
    .isin(customer_tweet_ids)
)

customer_parent_count = (
    support_with_parent["parent_is_customer"].sum()
)

print(
    f"\nSupport tweets replying to customer tweets : "
    f"{customer_parent_count:,}"
)

print(
    f"Percentage of support tweets with customer parent : "
    f"{customer_parent_count / len(support_tweets) * 100:.2f}%"
)


Support tweets replying to customer tweets : 1,261,888
Percentage of support tweets with customer parent : 99.05%


99.05% number is especially valuable: almost every support tweet has a customer tweet as its immediate parent.

Correctly calculate customer volume by brand

In [26]:
# customer volume linked to support brands

def get_support_accounts(response_ids):
    """Return support accounts referenced by a customer's response_tweet_id."""
    
    if pd.isna(response_ids):
        return []
    
    accounts = []
    
    for tweet_id in str(response_ids).split(","):
        tweet_id = tweet_id.strip()
        
        if not tweet_id.isdigit():
            continue
        
        tweet_id = int(tweet_id)
        
        if tweet_id in support_tweet_lookup.index:
            accounts.append(
                support_tweet_lookup.loc[tweet_id, "support_account"]
            )
    
    return list(set(accounts))

In [27]:
customer_brand_links = customer_tweets[
    ["tweet_id", "response_tweet_id"]
].copy()

customer_brand_links["support_accounts"] = (
    customer_brand_links["response_tweet_id"]
    .apply(get_support_accounts)
)

# Keep only customer tweets that can be linked to at least one support account
customer_brand_links = customer_brand_links[
    customer_brand_links["support_accounts"].str.len() > 0
].copy()

# One row per customer tweet ↔ support brand relationship
customer_brand_links = customer_brand_links.explode(
    "support_accounts"
)

customer_brand_links = customer_brand_links.rename(
    columns={"support_accounts": "support_account"}
)

print("CUSTOMER → BRAND LINKAGE")
print("=" * 70)

print(
    f"Customer tweets linked to support brand : "
    f"{customer_brand_links['tweet_id'].nunique():,}"
)

print(
    f"Customer-brand relationships            : "
    f"{len(customer_brand_links):,}"
)

print(
    f"Unique support brands represented       : "
    f"{customer_brand_links['support_account'].nunique():,}"
)


CUSTOMER → BRAND LINKAGE
Customer tweets linked to support brand : 1,164,365
Customer-brand relationships            : 1,166,579
Unique support brands represented       : 108


1,166,579 > 1,164,365

That's because some customer tweets reference more than one support tweet, so a single customer tweet can legitimately create multiple customer→brand relationships.

That's exactly why we said earlier that tweet-level linkage ≠ conversation-level analysis.

In [28]:
# customer value per brand
brand_customer_volume = (
    customer_brand_links
    .groupby("support_account")
    .agg(
        customer_tweets=("tweet_id", "nunique")
    )
    .sort_values("customer_tweets", ascending=False)
)

print("\nTOP 30 BRANDS BY LINKED CUSTOMER VOLUME")
print("=" * 80)

display(brand_customer_volume.head(30))


TOP 30 BRANDS BY LINKED CUSTOMER VOLUME


,customer_tweets
support_account,
AmazonHelp,154976
AppleSupport,106623
Uber_Support,55182
SpotifyCares,41585
AmericanAir,36457
Delta,36134
TMobileHelp,33837
comcastcares,30369
SouthwestAir,28285


each tweet can point to its immediate parent using: in_response_to_tweet_id

A tweet can also have multiple responses through: response_tweet_id

Therefore, the dataset can be represented as a forest of conversation trees.

Conversation Quality

In [29]:
tweet_ids = twcs["tweet_id"]

parent_ids = twcs["in_response_to_tweet_id"]

has_parent = parent_ids.notna()

existing_parent = parent_ids.isin(
    tweet_ids
)

missing_parent_reference = (
    has_parent & ~existing_parent
)

print("CONVERSATION GRAPH VALIDATION")
print("=" * 75)

print(f"Total tweets                         : {len(twcs):,}")
print(f"Tweets with a parent                 : {has_parent.sum():,}")
print(
    f"Tweets without a parent              : "
    f"{(~has_parent).sum():,}"
)

print(
    f"\nValid parent references              : "
    f"{(has_parent & existing_parent).sum():,}"
)

print(
    f"Missing parent references             : "
    f"{missing_parent_reference.sum():,}"
)

print(
    f"Parent-reference validity             : "
    f"{(has_parent & existing_parent).sum() / has_parent.sum() * 100:.2f}%"
)

CONVERSATION GRAPH VALIDATION
Total tweets                         : 2,811,774
Tweets with a parent                 : 2,017,439
Tweets without a parent              : 794,335

Valid parent references              : 2,013,577
Missing parent references             : 3,862
Parent-reference validity             : 99.81%


In [30]:
# ceck any tweet points to itself
self_references = (
    has_parent &
    (parent_ids == tweet_ids)
).sum()

print(
    f"\nSelf-referencing tweets              : "
    f"{self_references:,}"
)


Self-referencing tweets              : 0


A non-null in_response_to_tweet_id should point to another tweet in TWCS.

Tweet ID Ordering

In [31]:
valid_parent_mask = (
    twcs["in_response_to_tweet_id"].notna()
    & twcs["in_response_to_tweet_id"].isin(twcs["tweet_id"])
)

parent_order_check = twcs.loc[
    valid_parent_mask,
    ["tweet_id", "in_response_to_tweet_id"]
].copy()

parent_order_check["parent_id"] = (
    parent_order_check["in_response_to_tweet_id"]
    .astype("int64")
)

parent_order_check["parent_before_child"] = (
    parent_order_check["parent_id"]
    < parent_order_check["tweet_id"]
)

parent_before_count = (
    parent_order_check["parent_before_child"].sum()
)

total_valid_parents = len(parent_order_check)

print("PARENT / CHILD ID ORDERING")
print("=" * 75)

print(f"Valid parent-child relationships : {total_valid_parents:,}")
print(
    f"Parent ID < child ID             : "
    f"{parent_before_count:,}"
)

print(
    f"Parent-before-child percentage   : "
    f"{parent_before_count / total_valid_parents * 100:.4f}%"
)

print(
    f"Parent ID >= child ID            : "
    f"{total_valid_parents - parent_before_count:,}"
)

PARENT / CHILD ID ORDERING
Valid parent-child relationships : 2,013,577
Parent ID < child ID             : 895,760
Parent-before-child percentage   : 44.4860%
Parent ID >= child ID            : 1,117,817


In [32]:
# Show any exceptions, if they exist
exceptions = parent_order_check[
    ~parent_order_check["parent_before_child"]
]

print("\nEXAMPLES OF EXCEPTIONS")
print("=" * 75)

display(exceptions.head(10))


EXAMPLES OF EXCEPTIONS


,tweet_id,in_response_to_tweet_id,parent_id,parent_before_child
0,1,3,3,False
2,3,4,4,False
3,4,5,5,False
4,5,6,6,False
5,6,8,8,False
7,11,12,12,False
8,12,15,15,False
9,15,16,16,False
10,16,17,17,False
11,17,18,18,False


tweet IDs are absolutely not a safe chronological ordering mechanism.
Therefore, will not use tweet IDs to infer conversation order.

Then can go through created_at
- Are parent tweets actually earlier than their child tweets according to created_at?

Use the actual timestamp for conversation ordering

In [33]:
# Build a minimal tweet_id -> timestamp lookup.
tweet_time_lookup = (
    twcs[
        ["tweet_id", "created_at_parsed"]
    ]
    .set_index("tweet_id")["created_at_parsed"]
)

temporal_check = twcs.loc[
    valid_parent_mask,
    ["tweet_id", "in_response_to_tweet_id", "created_at_parsed"]
].copy()

temporal_check["parent_id"] = (
    temporal_check["in_response_to_tweet_id"]
    .astype("int64")
)

temporal_check["parent_created_at"] = (
    temporal_check["parent_id"]
    .map(tweet_time_lookup)
)

temporal_check["parent_before_child"] = (
    temporal_check["parent_created_at"]
    < temporal_check["created_at_parsed"]
)

valid_temporal_rows = temporal_check[
    temporal_check["parent_created_at"].notna()
]

temporal_valid_count = (
    valid_temporal_rows["parent_before_child"].sum()
)

temporal_total = len(valid_temporal_rows)

print("TEMPORAL PARENT / CHILD ORDERING")
print("=" * 75)

print(
    f"Valid parent-child pairs with timestamps : "
    f"{temporal_total:,}"
)

print(
    f"Parent timestamp < child timestamp       : "
    f"{temporal_valid_count:,}"
)

print(
    f"Parent-before-child percentage            : "
    f"{temporal_valid_count / temporal_total * 100:.4f}%"
)

print(
    f"Non-chronological relationships            : "
    f"{temporal_total - temporal_valid_count:,}"
)

print("\nEXAMPLES OF NON-CHRONOLOGICAL RELATIONSHIPS")
print("=" * 75)

temporal_exceptions = valid_temporal_rows[
    ~valid_temporal_rows["parent_before_child"]
]

display(
    temporal_exceptions[
        [
            "tweet_id",
            "parent_id",
            "parent_created_at",
            "created_at_parsed"
        ]
    ].head(10)
)

TEMPORAL PARENT / CHILD ORDERING
Valid parent-child pairs with timestamps : 2,013,577
Parent timestamp < child timestamp       : 2,013,521
Parent-before-child percentage            : 99.9972%
Non-chronological relationships            : 56

EXAMPLES OF NON-CHRONOLOGICAL RELATIONSHIPS


,tweet_id,parent_id,parent_created_at,created_at_parsed
187812,221474,221473,2017-10-04 17:18:43+00:00,2017-10-04 17:18:43+00:00
331656,379549,379550,2017-10-09 06:34:14+00:00,2017-10-09 06:34:14+00:00
331657,379550,379551,2017-10-09 06:34:14+00:00,2017-10-09 06:34:14+00:00
331658,379551,379552,2017-10-09 06:34:14+00:00,2017-10-09 06:34:14+00:00
337322,385533,385529,2017-10-17 12:29:53+00:00,2017-10-17 12:29:53+00:00
337325,385535,385534,2017-10-24 07:33:56+00:00,2017-10-24 07:33:56+00:00
337329,385540,385538,2017-10-24 15:53:08+00:00,2017-10-24 15:53:08+00:00
337332,385543,385542,2017-10-25 09:30:36+00:00,2017-10-25 09:30:36+00:00
337334,385545,385544,2017-10-25 09:30:37+00:00,2017-10-25 09:30:37+00:00
337342,385550,385549,2017-10-17 13:59:44+00:00,2017-10-17 13:59:44+00:00


TWCS's explicit parent-child relationships are temporally consistent. We can use in_response_to_tweet_id to reconstruct conversation chains and created_at to order the turns.

binary < check classified equal timestamps as exceptions, should correct that interpretation rather than call them non-chronological.

In [34]:
valid_parent_mask.head(10)

0     True
1     True
2     True
3     True
4     True
5     True
6    False
7     True
8     True
9     True
Name: in_response_to_tweet_id, dtype: boolean

In [35]:
# Identify conversation roots
is_root = ~valid_parent_mask

root_tweets = twcs.loc[
    is_root,
    ["tweet_id", "inbound", "created_at_parsed", "text"]
].copy()

print("CONVERSATION ROOTS")
print("=" * 75)

print(f"Total tweets              : {len(twcs):,}")
print(f"Valid parent-linked tweets: {valid_parent_mask.sum():,}")
print(f"Root tweets               : {is_root.sum():,}")

print(
    f"Root percentage           : "
    f"{is_root.mean() * 100:.2f}%"
)

print("\nROOT MESSAGE DIRECTION")
print("=" * 75)

root_direction = (
    root_tweets["inbound"]
    .map({
        True: "Customer → Company",
        False: "Company → Customer"
    })
    .value_counts()
)

display(root_direction.to_frame("root_count"))

print("\nSAMPLE ROOT TWEETS")
print("=" * 75)

display(
    root_tweets.head(10)
)

CONVERSATION ROOTS
Total tweets              : 2,811,774
Valid parent-linked tweets: 2,013,577
Root tweets               : 798,197
Root percentage           : 28.39%

ROOT MESSAGE DIRECTION


,root_count
inbound,
Customer → Company,789547
Company → Customer,8650



SAMPLE ROOT TWEETS


,tweet_id,inbound,created_at_parsed,text
6,8,True,2017-10-31 21:45:10+00:00,@sprintcare is the worst customer service
12,18,True,2017-10-31 19:56:01+00:00,"@115714 y’all lie about your “great” connection. 5 bars LTE, still won’t load something. Smh."
14,20,True,2017-10-31 22:03:34+00:00,"@115714 whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have never i..."
23,29,True,2017-10-31 22:01:35+00:00,actually that's a broken link you sent me and incorrect information https://t.co/V4yfrHR8VI
25,31,True,2017-10-31 22:06:54+00:00,"Yo @Ask_Spectrum, your customer service reps are super nice— but imma start trippin if y’all don’t get my service go..."
27,33,True,2017-10-31 22:06:56+00:00,My picture on @Ask_Spectrum pretty much every day. Why should I pay $171 per month? https://t.co/U6ptkQa5Ik
31,36,True,2017-10-31 22:10:46+00:00,somebody from @VerizonSupport please help meeeeee 😩😩😩😩 I'm having the worst luck with your customer service
33,39,True,2017-10-31 22:12:16+00:00,@VerizonSupport My friend is without internet we need to play videogames together please our skills diminish every m...
43,49,True,2017-10-31 21:42:09+00:00,"@115722 tried to pay a bill for 60 days. No service, rude CS, and several transfers. Look up my equipment # and give..."
53,59,True,2017-10-31 19:54:51+00:00,@115722 is the worst ISP I’ve ever had


In [36]:
# reconstruct conversation chains
# Number of tweets
n_rows = len(twcs)

tweet_id_array = twcs["tweet_id"].to_numpy(dtype=np.int64)

# Parent tweet ID.
# -1 means there is no parent.
parent_id_array = (
    twcs["in_response_to_tweet_id"]
    .fillna(-1)
    .to_numpy(dtype=np.int64)
)

# Map tweet_id -> row position
tweet_id_index = pd.Index(tweet_id_array)

parent_row = tweet_id_index.get_indexer(parent_id_array)

no_parent = parent_id_array == -1
valid_parent = parent_row >= 0
unresolved_parent = (~no_parent) & (~valid_parent)

print("PARENT RELATIONSHIP CATEGORIES")
print("=" * 75)

print(f"No parent          : {no_parent.sum():,}")
print(f"Valid parent       : {valid_parent.sum():,}")
print(f"Unresolved parent  : {unresolved_parent.sum():,}")

PARENT RELATIONSHIP CATEGORIES
No parent          : 794,335
Valid parent       : 2,013,577
Unresolved parent  : 3,862


In [37]:
# Initialize every tweet as its own root.

root = np.arange(n_rows, dtype=np.int64)

# Tweets with a valid parent initially point to their parent.
root[valid_parent] = parent_row[valid_parent]

iterations = 0

while True:
    previous_root = root.copy()

    # For every tweet, jump to the root currently assigned
    # to its current ancestor.
    root = root[root]

    iterations += 1

    if np.array_equal(root, previous_root):
        break

# Convert root row positions → root tweet IDs
twcs["conversation_id"] = tweet_id_array[root]

In [38]:
# Conversation root type

twcs["conversation_root_type"] = np.where(
    no_parent,
    "no_parent",
    np.where(
        unresolved_parent,
        "unresolved_parent",
        "child"
    )
)


In [39]:
print("\nCONVERSATION ROOT RECONSTRUCTION")
print("=" * 75)

print(f"Pointer-jumping iterations : {iterations:,}")

print(
    f"Unique conversation IDs    : "
    f"{twcs['conversation_id'].nunique():,}"
)

print("\nROOT TYPES")
print("=" * 75)

display(
    twcs["conversation_root_type"]
    .value_counts()
    .to_frame("tweet_count")
)


CONVERSATION ROOT RECONSTRUCTION
Pointer-jumping iterations : 11
Unique conversation IDs    : 798,197

ROOT TYPES


,tweet_count
conversation_root_type,
child,2013577
no_parent,794335
unresolved_parent,3862


In [40]:
# Step 19 — Validate Reconstructed Conversation
TEST_TWEET_ID = 1

test_conversation_id = twcs.loc[
    twcs["tweet_id"] == TEST_TWEET_ID,
    "conversation_id"
].iloc[0]

test_conversation = (
    twcs.loc[
        twcs["conversation_id"] == test_conversation_id,
        [
            "tweet_id",
            "author_id",
            "inbound",
            "created_at_parsed",
            "in_response_to_tweet_id",
            "conversation_id",
            "text"
        ]
    ]
    .sort_values("created_at_parsed")
    .reset_index(drop=True)
)

print("CONVERSATION VALIDATION")
print("=" * 90)

print(f"Test tweet ID       : {TEST_TWEET_ID}")
print(f"Conversation ID     : {test_conversation_id}")
print(f"Number of turns     : {len(test_conversation)}")

display(test_conversation)

CONVERSATION VALIDATION
Test tweet ID       : 1
Conversation ID     : 8
Number of turns     : 10


,tweet_id,author_id,inbound,created_at_parsed,in_response_to_tweet_id,conversation_id,text
0,8,115712,True,2017-10-31 21:45:10+00:00,<NA>,8,@sprintcare is the worst customer service
1,10,sprintcare,False,2017-10-31 21:45:59+00:00,8,8,@115712 Hello! We never like our customers to feel like they are not valued.
2,9,sprintcare,False,2017-10-31 21:46:14+00:00,8,8,@115712 I would love the chance to review the account and provide assistance.
3,6,sprintcare,False,2017-10-31 21:46:24+00:00,8,8,"@115712 Can you please send us a private message, so that I can gain further details about your account?"
4,7,115712,True,2017-10-31 21:47:48+00:00,6,8,@sprintcare the only way I can get a response is to tweet apparently
5,5,115712,True,2017-10-31 21:49:35+00:00,6,8,@sprintcare I did.
6,4,sprintcare,False,2017-10-31 21:54:49+00:00,5,8,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your ...
7,3,115712,True,2017-10-31 22:08:27+00:00,4,8,@sprintcare I have sent several private messages and no one is responding as usual
8,1,sprintcare,False,2017-10-31 22:10:47+00:00,3,8,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further as...
9,2,115712,True,2017-10-31 22:11:45+00:00,1,8,@sprintcare and how do you propose we do that


- Tweets in the same parent chain received the same conversation_id.
- The resulting conversation is chronologically ordered.
- Customer/support roles make sense within the conversation.

Conversation Structure Validation

In [41]:
conversation_stats = (
    twcs
    .groupby("conversation_id")
    .agg(
        conversation_size=("tweet_id", "size"), # can also use .count()
        customer_tweets=("inbound", "sum"),
        support_tweets=("inbound", lambda x: (~x).sum()),
        unique_authors=("author_id", "nunique"),
        start_time=("created_at_parsed", "min"),
        end_time=("created_at_parsed", "max"),
    )
)

conversation_stats["duration_minutes"] = (
    (
        conversation_stats["end_time"]
        - conversation_stats["start_time"]
    )
    .dt.total_seconds()
    / 60
)

conversation_stats["is_multi_turn"] = (
    conversation_stats["conversation_size"] >= 2
)

conversation_stats["has_customer_and_support"] = (
    (conversation_stats["customer_tweets"] > 0)
    & (conversation_stats["support_tweets"] > 0)
)

print("CONVERSATION STRUCTURE")
print("=" * 80)

print(
    f"Unique conversations              : "
    f"{len(conversation_stats):,}"
)

print(
    f"Multi-tweet conversations         : "
    f"{conversation_stats['is_multi_turn'].sum():,}"
)

print(
    f"Multi-tweet percentage             : "
    f"{conversation_stats['is_multi_turn'].mean() * 100:.2f}%"
)

print(
    f"Customer + support conversations   : "
    f"{conversation_stats['has_customer_and_support'].sum():,}"
)

print(
    f"Customer + support percentage      : "
    f"{conversation_stats['has_customer_and_support'].mean() * 100:.2f}%"
)

print("\nCONVERSATION SIZE DISTRIBUTION")
print("=" * 80)

display(
    conversation_stats["conversation_size"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_frame("conversation_size")
)

CONVERSATION STRUCTURE
Unique conversations              : 798,197
Multi-tweet conversations         : 798,197
Multi-tweet percentage             : 100.00%
Customer + support conversations   : 798,110
Customer + support percentage      : 99.99%

CONVERSATION SIZE DISTRIBUTION


,conversation_size
count,798197.000000
mean,3.522657
std,5.152779
min,2.000000
25%,2.000000
50%,2.000000
75%,4.000000
90%,6.000000
95%,8.000000
99%,15.000000


For each reconstructed conversation, which support brand does it belong to?

Map Conversations to Support Brands

In [42]:
# For each reconstructed conversation, collect the unique
# support accounts that participated in it.

conversation_brand_map = (
    twcs.loc[
        ~twcs["inbound"],
        ["conversation_id", "author_id"]
    ]
    .drop_duplicates()
    .groupby("conversation_id")
    .agg(
        support_brands=("author_id", list),
        support_brand_count=("author_id", "nunique")
    )
)

# Classify each conversation
conversation_brand_map["brand_mapping_type"] = np.select(
    [
        conversation_brand_map["support_brand_count"] == 1,
        conversation_brand_map["support_brand_count"] > 1
    ],
    [
        "single_brand",
        "multiple_brands"
    ],
    default="no_support_brand"
)

# Counts
support_conversation_count = len(conversation_brand_map)

single_brand_count = (
    conversation_brand_map["brand_mapping_type"]
    .eq("single_brand")
    .sum()
)

multi_brand_count = (
    conversation_brand_map["brand_mapping_type"]
    .eq("multiple_brands")
    .sum()
)

print("CONVERSATION → BRAND MAPPING")
print("=" * 80)

print(f"Total conversations          : {len(conversation_stats):,}")
print(f"Conversations with support   : {support_conversation_count:,}")
print(f"Single-brand conversations   : {single_brand_count:,}")
print(f"Multi-brand conversations    : {multi_brand_count:,}")

print("\nMAPPING DISTRIBUTION")
print("=" * 80)

mapping_distribution = (
    conversation_brand_map["brand_mapping_type"]
    .value_counts()
    .to_frame("conversation_count")
)

mapping_distribution["percentage"] = (
    mapping_distribution["conversation_count"]
    / len(conversation_stats)
    * 100
).round(2)

display(mapping_distribution)

print("\nEXAMPLES OF MULTI-BRAND CONVERSATIONS")
print("=" * 80)

multi_brand_examples = conversation_brand_map.loc[
    conversation_brand_map["brand_mapping_type"] == "multiple_brands"
].head(10)

display(multi_brand_examples)

CONVERSATION → BRAND MAPPING
Total conversations          : 798,197
Conversations with support   : 798,197
Single-brand conversations   : 795,131
Multi-brand conversations    : 3,066

MAPPING DISTRIBUTION


,conversation_count,percentage
brand_mapping_type,,
single_brand,795131,99.62
multiple_brands,3066,0.38



EXAMPLES OF MULTI-BRAND CONVERSATIONS


,support_brands,support_brand_count,brand_mapping_type
conversation_id,,,
1299,"[sprintcare, TMobileHelp]",2,multiple_brands
1998,"[TMobileHelp, sprintcare]",2,multiple_brands
2659,"[AppleSupport, sprintcare]",2,multiple_brands
7442,"[marksandspencer, Tesco, sainsburys, AldiUK, Morrisons]",5,multiple_brands
7511,"[SouthwestAir, AmericanAir]",2,multiple_brands
8036,"[comcastcares, sprintcare]",2,multiple_brands
8063,"[comcastcares, VerizonSupport]",2,multiple_brands
8210,"[SouthwestAir, AmericanAir]",2,multiple_brands
8608,"[Uber_Support, AskLyft]",2,multiple_brands


Conversation Quality by Brand

In [43]:
# keepin only single brand conversations
single_brand_conversations = conversation_brand_map.loc[
    conversation_brand_map["brand_mapping_type"] == "single_brand"
].copy()

# Extract the single support brand
single_brand_conversations["support_account"] = (
    single_brand_conversations["support_brands"]
    .str[0]
)

In [44]:
# add conv statistics
brand_conversation_data = (
    single_brand_conversations[
        ["support_account"]
    ]
    .join(
        conversation_stats[
            [
                "conversation_size",
                "customer_tweets",
                "support_tweets",
                "duration_minutes"
            ]
        ],
        how="inner"
    )
)

In [45]:
# define meaninggful multi-turn signals
brand_conversation_data["has_multiple_customer_turns"] = (
    brand_conversation_data["customer_tweets"] >= 2
)

brand_conversation_data["has_multiple_support_turns"] = (
    brand_conversation_data["support_tweets"] >= 2
)

brand_conversation_data["has_3_plus_tweets"] = (
    brand_conversation_data["conversation_size"] >= 3
)

brand_conversation_data["has_4_plus_tweets"] = (
    brand_conversation_data["conversation_size"] >= 4
)

In [46]:
# aggregate by brand
brand_conversation_quality = (
    brand_conversation_data
    .groupby("support_account")
    .agg(
        conversations=("conversation_size", "size"),

        mean_size=("conversation_size", "mean"),
        median_size=("conversation_size", "median"),

        pct_3_plus=("has_3_plus_tweets", "mean"),
        pct_4_plus=("has_4_plus_tweets", "mean"),

        pct_multiple_customer_turns=(
            "has_multiple_customer_turns",
            "mean"
        ),

        pct_multiple_support_turns=(
            "has_multiple_support_turns",
            "mean"
        ),

        median_duration_minutes=(
            "duration_minutes",
            "median"
        )
    )
)

In [47]:
# Convert proportions to percentages
percentage_columns = [
    "pct_3_plus",
    "pct_4_plus",
    "pct_multiple_customer_turns",
    "pct_multiple_support_turns"
]

brand_conversation_quality[percentage_columns] *= 100

brand_conversation_quality[percentage_columns] = (
    brand_conversation_quality[percentage_columns]
    .round(2)
)

brand_conversation_quality[
    ["mean_size", "median_size", "median_duration_minutes"]
] = (
    brand_conversation_quality[
        ["mean_size", "median_size", "median_duration_minutes"]
    ]
    .round(2)
)

In [48]:
# rank by conversation ability
brand_conversation_quality = (
    brand_conversation_quality
    .sort_values(
        "conversations",
        ascending=False
    )
)


print("BRAND-LEVEL CONVERSATION QUALITY")
print("=" * 110)

display(
    brand_conversation_quality.head(30)
)

BRAND-LEVEL CONVERSATION QUALITY


,conversations,mean_size,median_size,pct_3_plus,pct_4_plus,pct_multiple_customer_turns,pct_multiple_support_turns,median_duration_minutes
support_account,,,,,,,,
AmazonHelp,82246,4.52,3.0,61.95,47.96,57.07,49.63,27.14
AppleSupport,80552,2.95,2.0,34.73,24.82,34.60,22.21,135.54
Uber_Support,41848,3.05,2.0,35.88,24.61,35.04,22.40,18.16
SpotifyCares,28224,3.23,2.0,37.00,28.63,35.08,29.17,89.28
AmericanAir,25941,3.27,2.0,42.91,30.38,42.37,27.13,21.73
Delta,25907,3.32,2.0,42.02,29.83,35.47,33.33,23.33
comcastcares,23802,2.94,2.0,35.85,21.05,29.22,24.01,65.03
TMobileHelp,22447,3.38,2.0,36.77,27.26,36.07,23.13,5.82
SouthwestAir,21389,2.96,2.0,32.65,20.59,31.78,19.56,12.12


customer problem - 
support investigation - 
customer clarification - 
support resolution

Inspect Real Support Responses

In [49]:
# inspect real multi-turn conversations
candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Tesco",
    "XboxSupport"
]

# Candidate conversations:
# - single brand
# - at least 4 tweets
# - contains both customer and support messages

candidate_conversations = (
    brand_conversation_data.loc[
        brand_conversation_data["support_account"].isin(candidate_brands)
        & (brand_conversation_data["conversation_size"] >= 4)
        & (brand_conversation_data["customer_tweets"] >= 2)
        & (brand_conversation_data["support_tweets"] >= 2)
    ]
    .groupby("support_account", group_keys=False)
    .head(3)
    .index
)

In [50]:
inspection_df = twcs.loc[
    twcs["conversation_id"].isin(candidate_conversations),
    [
        "conversation_id",
        "tweet_id",
        "author_id",
        "inbound",
        "created_at_parsed",
        "text"
    ]
].copy()

inspection_df = inspection_df.sort_values(
    ["conversation_id", "created_at_parsed"]
)

In [51]:
for brand in candidate_brands:

    print("\n")
    print("=" * 100)
    print(f"{brand}")
    print("=" * 100)

    brand_conversation_ids = (
        brand_conversation_data.loc[
            brand_conversation_data["support_account"] == brand
        ]
        .loc[
            lambda x: x["conversation_size"] >= 4
        ]
        .loc[
            lambda x: x["customer_tweets"] >= 2
        ]
        .loc[
            lambda x: x["support_tweets"] >= 2
        ]
        .index
        .tolist()
    )

    # Inspect up to 3 conversations
    for conversation_id in brand_conversation_ids[:3]:

        conversation = inspection_df.loc[
            inspection_df["conversation_id"] == conversation_id
        ]

        print(f"\nConversation ID: {conversation_id}")
        print("-" * 100)

        for _, row in conversation.iterrows():

            speaker = (
                "CUSTOMER"
                if row["inbound"]
                else "SUPPORT"
            )

            print(
                f"[{speaker}] "
                f"{row['created_at_parsed']} | "
                f"{row['text']}"
            )



AmazonHelp

Conversation ID: 272
----------------------------------------------------------------------------------------------------
[CUSTOMER] 2017-11-22 09:14:39+00:00 | amazonのfireTVstickが見れない😢
[SUPPORT] 2017-11-22 09:23:01+00:00 | @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
[CUSTOMER] 2017-11-22 09:24:30+00:00 | @AmazonHelp ありがとうございます。
今、電話で主人が対応していただいてます。
[CUSTOMER] 2017-11-22 09:30:36+00:00 | @AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。
[SUPPORT] 2017-11-22 09:40:27+00:00 | @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
[CUSTOMER] 2017-11-22 09:44:04+00:00 | @AmazonHelp こちらこそありがとうございました。
[SUPPORT] 2017-11-22 10:06:26+00:00 | @115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET

Conversation ID: 617
----------------------------------------------------------------------------------------------------
[CUSTOMER] 2017-10-31 22:16:32+00:00 | Way to drop the 

A long conversation ≠ a resolved conversation.

Quantify Resolution Richness

In [52]:
# Candidate brands selected for deeper comparison
resolution_candidates = [
    "AmazonHelp",
    "AppleSupport",
    "Tesco",
    "XboxSupport"
]

resolution_support = twcs.loc[
    (~twcs["inbound"])
    & (twcs["author_id"].isin(resolution_candidates))
].copy()

resolution_support["text_lower"] = (
    resolution_support["text"]
    .str.lower()
    .fillna("")
)

In [53]:
# routin signals
routing_pattern = (
    r"\bdm\b"
    r"|direct message"
    r"|private message"
    r"|private messaging"
    r"|reach us"
    r"|contact us"
    r"|call us"
    r"|phone"
    r"|chat"
)

resolution_support["has_routing_signal"] = (
    resolution_support["text_lower"]
    .str.contains(routing_pattern, regex=True, na=False)
)


In [54]:
# info gathhering signals
question_signal = (
    resolution_support["text"]
    .str.contains(r"\?", regex=True, na=False)
)

diagnostic_patterns = (
    r"which "
    r"|what "
    r"|when "
    r"|where "
    r"|how "
    r"|have you "
    r"|did you "
    r"|can you "
    r"|could you "
    r"|please.*send"
)

resolution_support["has_diagnostic_signal"] = (
    question_signal
    | resolution_support["text_lower"]
        .str.contains(
            diagnostic_patterns,
            regex=True,
            na=False
        )
)

In [55]:
# action signals
action_patterns = (
    r"try "
    r"|restart"
    r"|reset"
    r"|check "
    r"|select "
    r"|click "
    r"|go to "
    r"|follow "
    r"|install "
    r"|update "
    r"|unplug"
    r"|power"
    r"|clear "
)

resolution_support["has_action_signal"] = (
    resolution_support["text_lower"]
    .str.contains(
        action_patterns,
        regex=True,
        na=False
    )
)

In [56]:
# investigation signals
investigation_patterns = (
    r"look into"
    r"|investigate"
    r"|review"
    r"|internal"
    r"|manager"
    r"|log"
    r"|account"
    r"|details"
    r"|information"
)

resolution_support["has_investigation_signal"] = (
    resolution_support["text_lower"]
    .str.contains(
        investigation_patterns,
        regex=True,
        na=False
    )
)

actionable > investigation > diagnostic > routing > closure

In [57]:
# assign response-level resolution category
resolution_support["resolution_category"] = np.select(
    [
        resolution_support["has_action_signal"],
        resolution_support["has_investigation_signal"],
        resolution_support["has_diagnostic_signal"],
        resolution_support["has_routing_signal"]
    ],
    [
        "ACTIONABLE",
        "CASE_INVESTIGATION",
        "DIAGNOSTIC",
        "ROUTING"
    ],
    default="OTHER"
)

In [58]:
resolution_summary = (
    resolution_support
    .groupby(["author_id", "resolution_category"])
    .size()
    .unstack(fill_value=0)
)

resolution_summary["total_support_responses"] = (
    resolution_summary.sum(axis=1)
)

resolution_categories = [
    "ACTIONABLE",
    "CASE_INVESTIGATION",
    "DIAGNOSTIC",
    "ROUTING",
    "OTHER"
]

for category in resolution_categories:
    if category not in resolution_summary.columns:
        resolution_summary[category] = 0

resolution_summary["substantive_rate_pct"] = (
    (
        resolution_summary["ACTIONABLE"]
        + resolution_summary["CASE_INVESTIGATION"]
        + resolution_summary["DIAGNOSTIC"]
    )
    / resolution_summary["total_support_responses"]
    * 100
).round(2)

resolution_summary = resolution_summary.sort_values(
    "substantive_rate_pct",
    ascending=False
)

print("RESPONSE-LEVEL RESOLUTION SIGNALS")
print("=" * 100)

display(
    resolution_summary[
        [
            "total_support_responses",
            "ACTIONABLE",
            "CASE_INVESTIGATION",
            "DIAGNOSTIC",
            "ROUTING",
            "OTHER",
            "substantive_rate_pct"
        ]
    ]
)

RESPONSE-LEVEL RESOLUTION SIGNALS


resolution_category,total_support_responses,ACTIONABLE,CASE_INVESTIGATION,DIAGNOSTIC,ROUTING,OTHER,substantive_rate_pct
author_id,,,,,,,
AppleSupport,106860,20870,20755,35487,19130,10618,72.16
XboxSupport,24557,6386,2911,7391,820,7049,67.96
Tesco,38573,2478,10890,11165,1477,12563,63.60
AmazonHelp,169840,14805,46804,38576,6090,63565,58.99


will explicitly call it something like:

“response-level substantive-resolution signal rate”

Validate Resolution Heuristic (REJECTED)

In [59]:
validation_categories = [
    "ACTIONABLE",
    "CASE_INVESTIGATION",
    "DIAGNOSTIC",
    "ROUTING",
    "OTHER"
]

validation_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Tesco",
    "XboxSupport"
]

for category in validation_categories:

    print("\n")
    print("=" * 110)
    print(f"CATEGORY: {category}")
    print("=" * 110)

    category_examples = (
        resolution_support.loc[
            (resolution_support["author_id"].isin(validation_brands))
            & (
                resolution_support["resolution_category"]
                == category
            ),
            [
                "author_id",
                "text",
                "has_action_signal",
                "has_investigation_signal",
                "has_diagnostic_signal",
                "has_routing_signal"
            ]
        ]
        .sample(
            n=min(
                5,
                (
                    resolution_support[
                        resolution_support["resolution_category"]
                        == category
                    ]
                    .shape[0]
                )
            ),
            random_state=42
        )
    )

    display(category_examples)



CATEGORY: ACTIONABLE


,author_id,text,has_action_signal,has_investigation_signal,has_diagnostic_signal,has_routing_signal
708220,AmazonHelp,@183225 You must have received a correspondence from our team. Kindly check the same here: https://t.co/ubzNHWZvL2 a...,True,False,False,False
281594,AmazonHelp,@192776 Have you had a chance to try uninstalling and reinstalling the skill for the Sonos speaker? ^GR,True,False,True,False
2129483,AmazonHelp,@665365 Thank you for taking time to register your email address. Kindly fill in the form through the link mentioned...,True,False,False,False
98422,AppleSupport,"@143867 We'll do all we can to help. Follow up with us in DM. Tell us how you restored the device, and the type of d...",True,False,True,True
435923,AmazonHelp,@233281 Try reaching us here: https://t.co/hApLpMlfHN ^AP,True,False,False,False




CATEGORY: CASE_INVESTIGATION


,author_id,text,has_action_signal,has_investigation_signal,has_diagnostic_signal,has_routing_signal
64192,AmazonHelp,@133917 Apologies for the inconvenience you've experienced regarding the issue. Could you kindly elaborate on it fur...,False,True,True,False
1826152,AppleSupport,@586666 We know how important having auto correct work properly is. Send us a DM and we'll look into this with you. ...,False,True,True,True
2344230,AmazonHelp,@715702 I’m sorry about the issue you’re facing. Let us look into it. Please share your details (1/2)^HR,False,True,False,False
409712,Tesco,"@226113 Hi there, thanks for getting back to me. I'll ask one of my colleagues to contact the store to investigate t...",False,True,True,False
852112,AppleSupport,@344943 We're happy to help you with this. DM us the details of what you are experiencing here: https://t.co/GDrqU22YpT,False,True,True,True




CATEGORY: DIAGNOSTIC


,author_id,text,has_action_signal,has_investigation_signal,has_diagnostic_signal,has_routing_signal
1767548,AppleSupport,@572114 We're happy to help. Have you checked your restriction settings? Are you sign in with the correct Apple ID? ...,False,False,True,True
1077707,AppleSupport,@400212 We'll be happy to see how we can help. Which iPhone and iOS version are you using?,False,False,True,True
970087,AmazonHelp,@373964 I'm so sorry for the frustration! Who is the carrier for the order? ^DD,False,False,True,False
1546620,AppleSupport,"@196053 Let us help. For a full breakdown of what to do with a possible phishing attempt, please visit: https://t.co...",False,False,True,False
774051,AppleSupport,@325213 We're here to help. DM us what iOS 11 version you're on. https://t.co/GDrqU22YpT,False,False,True,True




CATEGORY: ROUTING


,author_id,text,has_action_signal,has_investigation_signal,has_diagnostic_signal,has_routing_signal
851803,XboxSupport,"@141881 Hi there, if you'd like to request a refund, please contact our chat support team here: https://t.co/lUV7XYl...",False,False,False,True
629913,AppleSupport,@288835 Let's meet up in DM to discuss options related to isolation further at this time. https://t.co/GDrqU22YpT,False,False,False,True
1210040,AmazonHelp,@431402 I'm sorry for the inconvenience on the return of your order. Call us here: https://t.co/vlvfJr4nN9 &amp; we'...,False,False,False,True
1957249,AppleSupport,@623074 Thanks for reaching out. We have a workaround for this here: https://t.co/xXaXeeSRt9 DM us if you still need...,False,False,False,True
618168,AppleSupport,@250275 Updating is a great first step. Let's meet in DM for a closer look at the issue. https://t.co/GDrqU22YpT,False,False,False,True




CATEGORY: OTHER


,author_id,text,has_action_signal,has_investigation_signal,has_diagnostic_signal,has_routing_signal
322209,AmazonHelp,"@203474 I get your concern, please connect with our support team here: https://t.co/TdDksLo6Mf. We'll assist you on ...",False,False,False,False
515226,AmazonHelp,@256800 I understand you want to return your order. You can place a return request here: https://t.co/dgreAZQ5qf. Do...,False,False,False,False
1531427,Tesco,@361095 The correct charge will apply in around 3 days and the £1 will be refunded. This is used to validate your c...,False,False,False,False
2272441,AmazonHelp,"@699165 Late packages have 36 hours past the given delivery date to be delivered. If you don't have it by then, plea...",False,False,False,False
1742310,Tesco,"@388632 Hi Caroline, sorry to see your pumpkin was damaged and mouldy inside. I'd like to refund this for you and pa...",False,False,False,False


- ACTIONABLE is not reliably actionable
- DIAGNOSTIC and ROUTING overlap

current substantive_rate_pct is not trustworthy.

first resolution heuristic was misleading, validated it against real examples and rejected it instead of using a bad metric.

Create conversation-level resolution evidence

In [60]:
resolution_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Tesco",
    "XboxSupport"
]


# Select single-brand conversations
resolution_conversations = (
    brand_conversation_data.loc[
        brand_conversation_data["support_account"].isin(resolution_brands)
    ]
    .copy()
)

print("RESOLUTION EPISODE POOL")
print("=" * 80)

print(
    f"Total candidate conversations : "
    f"{len(resolution_conversations):,}"
)

display(
    resolution_conversations
    .groupby("support_account")
    .size()
    .sort_values(ascending=False)
    .to_frame("conversation_count")
)

RESOLUTION EPISODE POOL
Total candidate conversations : 192,718


,conversation_count
support_account,
AmazonHelp,82246
AppleSupport,80552
Tesco,16566
XboxSupport,13354


In [61]:
# Build chronological conversation records
resolution_tweets = twcs.loc[
    twcs["conversation_id"].isin(
        resolution_conversations.index
    ),
    [
        "conversation_id",
        "tweet_id",
        "author_id",
        "inbound",
        "created_at_parsed",
        "text"
    ]
].copy()

resolution_tweets = resolution_tweets.sort_values(
    [
        "conversation_id",
        "created_at_parsed",
        "tweet_id"
    ]
)

In [62]:
# helping functions
def first_customer_message(group):
    rows = group[group["inbound"]]

    if rows.empty:
        return None

    return rows.iloc[0]["text"]


def last_customer_message(group):
    rows = group[group["inbound"]]

    if rows.empty:
        return None

    return rows.iloc[-1]["text"]


def first_support_message(group):
    rows = group[~group["inbound"]]

    if rows.empty:
        return None

    return rows.iloc[0]["text"]


def last_support_message(group):
    rows = group[~group["inbound"]]

    if rows.empty:
        return None

    return rows.iloc[-1]["text"]

def conversation_text(group):
    messages = []

    for _, row in group.iterrows():

        speaker = (
            "CUSTOMER"
            if row["inbound"]
            else "SUPPORT"
        )

        messages.append(
            f"[{speaker}] {row['text']}"
        )

    return "\n".join(messages)


In [63]:
resolution_episodes = (
    resolution_tweets
    .groupby("conversation_id")
    .apply(
        lambda group: pd.Series({
            "first_customer_message": first_customer_message(group),

            "last_customer_message": last_customer_message(group),

            "first_support_message": first_support_message(group),

            "last_support_message": last_support_message(group),

            "conversation_text": conversation_text(group),

            "customer_turns": int(group["inbound"].sum()),

            "support_turns": int((~group["inbound"]).sum()),

            "conversation_size": len(group),

            "start_time": group["created_at_parsed"].min(),

            "end_time": group["created_at_parsed"].max()
        }),
        include_groups=False
    )
)

resolution_episodes["duration_minutes"] = (
    (
        resolution_episodes["end_time"]
        - resolution_episodes["start_time"]
    )
    .dt.total_seconds()
    / 60
)

In [64]:
# Add brand
resolution_episodes = resolution_episodes.join(
    resolution_conversations[["support_account"]],
    how="left"
)

print("\nRESOLUTION EPISODES CREATED")
print("=" * 80)

print(
    f"Resolution episodes : "
    f"{len(resolution_episodes):,}"
)

print(
    f"Columns              : "
    f"{len(resolution_episodes.columns)}"
)

display(
    resolution_episodes.head(3)
)


RESOLUTION EPISODES CREATED
Resolution episodes : 192,718
Columns              : 12


,first_customer_message,last_customer_message,first_support_message,last_support_message,conversation_text,customer_turns,support_turns,conversation_size,start_time,end_time,duration_minutes,support_account
conversation_id,,,,,,,,,,,,
272,amazonのfireTVstickが見れない😢,@AmazonHelp こちらこそありがとうございました。,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pb...,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET,[CUSTOMER] amazonのfireTVstickが見れない😢\n[SUPPORT] @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシ...,4,3,7,2017-11-22 09:14:39+00:00,2017-11-22 10:06:26+00:00,51.783333,AmazonHelp
277,"@XboxSupport can I change me sons Xbox live account to his Hotmail account, currently linked to my Hotmail account","@XboxSupport can I change me sons Xbox live account to his Hotmail account, currently linked to my Hotmail account","@115771 Hi, you can change your Microsoft account email through the steps here: https://t.co/DkEhOHbOYy . If the ema...","@115771 Hi, you can change your Microsoft account email through the steps here: https://t.co/DkEhOHbOYy . If the ema...","[CUSTOMER] @XboxSupport can I change me sons Xbox live account to his Hotmail account, currently linked to my Hotmai...",1,1,2,2017-12-02 09:17:57+00:00,2017-12-02 18:06:03+00:00,528.100000,XboxSupport
318,@XboxSupport @115786 @115787 @15913 can anyone let me know when our pre orders are going to be shipped for xbox one x??,"@XboxSupport @115786 Kinda need to make sure I have 680$ cdn in my bank, so would be nice to know.. I dun just have ...",@115785 Hi there! We'd recommend reaching out to the chat team here: https://t.co/UrUi80i54l ^JA,"@115785 @115786 Did you pre-order this console via the Microsoft store? If so, then let's have you reach out here: h...",[CUSTOMER] @XboxSupport @115786 @115787 @15913 can anyone let me know when our pre orders are going to be shipped fo...,7,3,10,2017-10-31 20:03:55+00:00,2017-11-01 01:13:10+00:00,309.250000,XboxSupport


- Customer/support turns are separated.
- Conversations are chronologically ordered.
- The episodes retain the support brand.

Stratified Resolution Screening Sample

In [65]:
resolution_episodes["size_bucket"] = pd.cut(
    resolution_episodes["conversation_size"],
    bins=[0, 2, 4, 8, np.inf],
    labels=[
        "2_tweets",
        "3_to_4_tweets",
        "5_to_8_tweets",
        "9_plus_tweets"
    ],
    include_lowest=True
)

In [66]:
# Sample from each brand × conversation-size bucket
sample_per_bucket = 8

sampled_groups = []

for (brand, bucket), group in resolution_episodes.groupby(
    ["support_account", "size_bucket"],
    observed=True
):

    n_sample = min(sample_per_bucket, len(group))

    if n_sample == 0:
        continue

    sampled = group.sample(
        n=n_sample,
        random_state=42
    ).copy()

    # The conversation_id is the INDEX of resolution_episodes.
    # Explicitly convert that index into a column.
    sampled["conversation_id"] = sampled.index

    # Explicitly preserve grouping information.
    sampled["support_account"] = brand
    sampled["size_bucket"] = bucket

    sampled_groups.append(sampled)

In [67]:
# combine sampled conversations
resolution_screening_sample = pd.concat(
    sampled_groups,
    axis=0,
    ignore_index=True
)

In [68]:
# select inspection columns
screening_columns = [
    "conversation_id",
    "support_account",
    "size_bucket",
    "conversation_size",
    "customer_turns",
    "support_turns",
    "duration_minutes",
    "first_customer_message",
    "last_customer_message",
    "first_support_message",
    "last_support_message",
    "conversation_text"
]

resolution_screening_sample = (
    resolution_screening_sample[
        screening_columns
    ]
    .sort_values(
        [
            "support_account",
            "size_bucket",
            "conversation_id"
        ]
    )
    .reset_index(drop=True)
)

In [69]:
# report sample
print("RESOLUTION SCREENING SAMPLE")
print("=" * 90)

print(
    f"Sampled conversations : "
    f"{len(resolution_screening_sample):,}"
)

print(
    f"Brands                : "
    f"{resolution_screening_sample['support_account'].nunique()}"
)

print("\nSAMPLE SIZE BY BRAND")
print("=" * 90)

display(
    resolution_screening_sample
    .groupby("support_account")
    .size()
    .to_frame("sample_count")
)

print("\nSAMPLE SIZE BY BRAND × CONVERSATION SIZE")
print("=" * 90)

display(
    resolution_screening_sample
    .groupby(
        ["support_account", "size_bucket"],
        observed=True
    )
    .size()
    .to_frame("sample_count")
)

RESOLUTION SCREENING SAMPLE
Sampled conversations : 128
Brands                : 4

SAMPLE SIZE BY BRAND


,sample_count
support_account,
AmazonHelp,32
AppleSupport,32
Tesco,32
XboxSupport,32



SAMPLE SIZE BY BRAND × CONVERSATION SIZE


sample_count
support_account size_bucket                
AmazonHelp      2_tweets                  8
                3_to_4_tweets             8
                5_to_8_tweets             8
                9_plus_tweets             8
AppleSupport    2_tweets                  8
                3_to_4_tweets             8
                5_to_8_tweets             8
                9_plus_tweets             8
Tesco           2_tweets                  8
                3_to_4_tweets             8
                5_to_8_tweets             8
                9_plus_tweets             8
XboxSupport     2_tweets                  8
                3_to_4_tweets             8
                5_to_8_tweets             8
                9_plus_tweets             8

Display Resolution Annotation Batch

In [70]:
# Take the first 20 conversations for manual screening.
annotation_batch = (
    resolution_screening_sample
    .head(20)
    .copy()
)

print("RESOLUTION ANNOTATION BATCH")
print("=" * 100)

for _, row in annotation_batch.iterrows():

    print("\n" + "=" * 100)

    print(
        f"Conversation ID : {row['conversation_id']}"
    )

    print(
        f"Brand           : {row['support_account']}"
    )

    print(
        f"Size            : {row['conversation_size']} tweets"
    )

    print(
        f"Customer turns  : {row['customer_turns']}"
    )

    print(
        f"Support turns   : {row['support_turns']}"
    )

    print(
        f"Duration        : {row['duration_minutes']:.1f} minutes"
    )

    print("-" * 100)

    print(row["conversation_text"])

RESOLUTION ANNOTATION BATCH

Conversation ID : 227908
Brand           : AmazonHelp
Size            : 2 tweets
Customer turns  : 1
Support turns   : 1
Duration        : 29.9 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @115850 My issue is contained in the below pic. Please look into it and reply. https://t.co/j9pvkXhaa5
[SUPPORT] @170338 Allow us to take a closer look. Please call us here: https://t.co/vlvfJr4nN9 for live assistance. ^HK

Conversation ID : 236650
Brand           : AmazonHelp
Size            : 2 tweets
Customer turns  : 1
Support turns   : 1
Duration        : 9.3 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] Le plus marrant c'est que j'ai vu le livreur avec le colis en bas et j'ai demandé, il m'a dit "Non le colis est pas pour vous". 
@120533 https://t.co/F5H41AXRll
[SUPPORT] @172357 Bonjour Loïck, je suis désolée pour cela.

In [71]:
annotation_batch = (
    resolution_screening_sample
    .groupby("support_account", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=5,
            random_state=42
        ),
        include_groups=True
    )
    .reset_index(drop=True)
)

print("BALANCED RESOLUTION ANNOTATION BATCH")
print("=" * 100)

print(
    f"Total conversations : {len(annotation_batch):,}"
)

print("\nCONVERSATIONS PER BRAND")
print("=" * 100)

display(
    annotation_batch
    .groupby("support_account")
    .size()
    .to_frame("count")
)

BALANCED RESOLUTION ANNOTATION BATCH
Total conversations : 20

CONVERSATIONS PER BRAND


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8164\3238689582.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,count
support_account,
AmazonHelp,5
AppleSupport,5
Tesco,5
XboxSupport,5


In [72]:
for _, row in annotation_batch.iterrows():

    print("\n" + "=" * 100)

    print(
        f"Conversation ID : {row['conversation_id']}"
    )

    print(
        f"Brand           : {row['support_account']}"
    )

    print(
        f"Size            : {row['conversation_size']} tweets"
    )

    print(
        f"Customer turns  : {row['customer_turns']}"
    )

    print(
        f"Support turns   : {row['support_turns']}"
    )

    print(
        f"Duration        : {row['duration_minutes']:.1f} minutes"
    )

    print("-" * 100)

    print(row["conversation_text"])


Conversation ID : 1666514
Brand           : AmazonHelp
Size            : 12 tweets
Customer turns  : 6
Support turns   : 6
Duration        : 3091.5 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @115833 when can I set recurring reminders on Alexa?
[SUPPORT] @161011 Our support team would be happy to help you out with your query! Please contact us from here: https://t.co/vlvfJr4nN9. ^MP
[CUSTOMER] @AmazonHelp Getting the following error while trying to get to the URL. https://t.co/RGEjkbxQg8
[SUPPORT] @161011 I've checked and seen the link is working fine. Kindly try accessing it here:  https://t.co/2t6DQoUmNZ again. ^AP
[CUSTOMER] @AmazonHelp Done, thanks!
[SUPPORT] @161011 You're welcome. Please keep us posted for further concerns. ^BS
[CUSTOMER] @AmazonHelp Looks like recurring reminders are not possible for now :( That's quite disappointing. Google calendar integration?
[SUPPORT] @161011 I'm sorry I couldn't c

AmazonHelp has substantial operational support, but many public responses route customers elsewhere.

AppleSupport has strong diagnostic/troubleshooting behavior and some confirmed outcomes.

Tesco shows particularly strong case-action behavior.

XboxSupport has concrete technical troubleshooting and investigation.

NON_SUPPORT, ROUTING_ONLY, DIAGNOSTIC, ACTIONABLE_GUIDANCE, CASE_ACTION, CONFIRMED_OUTCOME, UNRESOLVED, AMBIGUOUS

Create human annotation sheet for resolution screening

In [73]:
from pathlib import Path

# Project root = parent of the notebooks directory
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

annotation_cols = [
    "conversation_id",
    "support_account",
    "size_bucket",
    "conversation_size",
    "customer_turns",
    "support_turns",
    "duration_minutes",
    "conversation_text",
]

resolution_screening_annotations = (
    resolution_screening_sample[annotation_cols]
    .copy()
)

# Human annotation fields
resolution_screening_annotations["resolution_label"] = ""
resolution_screening_annotations["annotation_notes"] = ""
resolution_screening_annotations["annotator"] = "human"

# Save
annotation_path = (
    PROCESSED_DIR / "resolution_screening_annotations.csv"
)

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Project root:", PROJECT_ROOT)
print("Saved:", annotation_path)
print("Rows:", len(resolution_screening_annotations))
print("Columns:", resolution_screening_annotations.columns.tolist())

Project root: d:\BECAME_DEVELOPER\hiver-sde-ai-agent
Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\resolution_screening_annotations.csv
Rows: 128
Columns: ['conversation_id', 'support_account', 'size_bucket', 'conversation_size', 'customer_turns', 'support_turns', 'duration_minutes', 'conversation_text', 'resolution_label', 'annotation_notes', 'annotator']


In [74]:
from pathlib import Path
import pandas as pd

annotation_path = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "resolution_screening_annotations.csv"
)

resolution_screening_annotations = pd.read_csv(annotation_path)

# Show the first 20 conversations for manual review
batch_1 = resolution_screening_annotations.iloc[:20].copy()

print("Batch size:", len(batch_1))
print("Total conversations:", len(resolution_screening_annotations))

for i, row in batch_1.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 20
Total conversations: 128

ROW: 0
Conversation ID: 227908
Brand: AmazonHelp
Size: 2 tweets
Customer turns: 1
Support turns: 1
Duration: 29.883333333333333 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @115850 My issue is contained in the below pic. Please look into it and reply. https://t.co/j9pvkXhaa5
[SUPPORT] @170338 Allow us to take a closer look. Please call us here: https://t.co/vlvfJr4nN9 for live assistance. ^HK

ROW: 1
Conversation ID: 236650
Brand: AmazonHelp
Size: 2 tweets
Customer turns: 1
Support turns: 1
Duration: 9.3 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] Le plus marrant c'est que j'ai vu le livreur avec le colis en bas et j'ai demandé, il m'a dit "Non le colis est pas pour vous". 
@120533 https://t.co/F5H41AXRll
[SUPPORT] @172357 Bonjour Loïck, je suis désolée pour cela. L'avez-vous signalé à notre SAV? 

In [75]:
# Save Batch 1 human annotations

batch_1_labels = {
    0: ("ROUTING_ONLY", "Support only redirects customer to phone assistance."),
    1: ("DIAGNOSTIC", "Support asks whether the issue was reported; no resolution."),
    2: ("CASE_ACTION", "Support says the delivery complaint will be forwarded as internal feedback."),
    3: ("ROUTING_ONLY", "Support redirects customer to another support channel."),
    4: ("ACTIONABLE_GUIDANCE", "Provides delivery-date policy and a way to check the promised date."),
    5: ("ROUTING_ONLY", "Customer is directed to customer service to handle the damaged delivery."),
    6: ("ROUTING_ONLY", "Support redirects customer to phone/chat without answering the question."),
    7: ("NON_SUPPORT", "Social/positive engagement rather than a support issue."),
    8: ("ROUTING_ONLY", "Support gathers one detail but ultimately routes customer to phone/chat."),
    9: ("CONFIRMED_OUTCOME", "Customer explicitly reports that the video quality improved."),
    10: ("ROUTING_ONLY", "Support directs customer to the support team; no resolution."),
    11: ("ACTIONABLE_GUIDANCE", "Provides concrete return/replacement options."),
    12: ("ROUTING_ONLY", "Customer cannot reach support and is redirected to phone/email."),
    13: ("DIAGNOSTIC", "Support checks whether the promised delivery estimate was missed."),
    14: ("AMBIGUOUS", "Support explains the price change but no clear support outcome/action occurs."),
    15: ("DIAGNOSTIC", "Support gathers the carrier information for the delivery investigation."),
    16: ("CONFIRMED_OUTCOME", "Customer explicitly confirms that the order arrived."),
    17: ("UNRESOLVED", "Delivery remains delayed; support only asks customer to wait."),
    18: ("DIAGNOSTIC", "Support gathers tracking information and routes to phone/chat; unresolved."),
    19: ("CASE_ACTION", "Support says the feedback will be shared internally."),
}

for row_idx, (label, note) in batch_1_labels.items():
    resolution_screening_annotations.loc[
        row_idx, "resolution_label"
    ] = label

    resolution_screening_annotations.loc[
        row_idx, "annotation_notes"
    ] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Batch 1 annotations saved.")
print()
print(
    resolution_screening_annotations
    .iloc[:20]["resolution_label"]
    .value_counts()
)

Batch 1 annotations saved.

resolution_label
ROUTING_ONLY           7
DIAGNOSTIC             4
CASE_ACTION            2
ACTIONABLE_GUIDANCE    2
CONFIRMED_OUTCOME      2
NON_SUPPORT            1
AMBIGUOUS              1
UNRESOLVED             1
Name: count, dtype: int64


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8164\3192402335.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ROUTING_ONLY' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  resolution_screening_annotations.loc[
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8164\3192402335.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Support only redirects customer to phone assistance.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  resolution_screening_annotations.loc[


In [76]:
#Display Batch 2 for manual annotation

batch_2 = resolution_screening_annotations.iloc[20:40].copy()

print("Batch size:", len(batch_2))
print("Rows:", batch_2.index.tolist())

for i, row in batch_2.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 20
Rows: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]

ROW: 20
Conversation ID: 930342
Brand: AmazonHelp
Size: 6 tweets
Customer turns: 3
Support turns: 3
Duration: 2039.6833333333332 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @AmazonHelp Bonjour, a défaut de pouvoir annuler ma commande, puis je avoir des informations sur pourquoi celle-ci met autant de temps à être envoyée?? On m'annonce depuis vendredi un envoi immédiat. Ce qui n'est pas le cas. J'ai précommandé l'article depuis longtemps....
[SUPPORT] @340706 Bonjour, je suis désolée pour cela. Que dit le suivi SVP ? ^SM
[CUSTOMER] @AmazonHelp Simplement "envoi aujourd'hui", livraison prévue le 21/22 Novembre. Mais si le colis n'est pas envoyé les dates prévues ne peuvent pas être respecté. Aucun mail d'explications pour ce retard également.
[SUPPORT] @340706 Quelle était la date de livraison indiquée lors de 

In [77]:
# Save Batch 2 human annotations

batch_2_labels = {
    20: ("CASE_ACTION", "Support investigates delivery issue and records customer complaint as internal feedback."),
    21: ("CASE_ACTION", "Support collects carrier information and initiates Transportation-team review."),
    22: ("CASE_ACTION", "Support requests details so the delivery issue can be investigated."),
    23: ("DIAGNOSTIC", "Support gathers error/context information and routes for closer investigation."),
    24: ("NON_SUPPORT", "Social engagement about books; no support issue."),
    25: ("ACTIONABLE_GUIDANCE", "Detailed instructions explain delivery/payment/receiving process and what to do if absent."),
    26: ("CASE_ACTION", "Support gathers information and offers team investigation of repeated delivery failures."),
    27: ("DIAGNOSTIC", "Support gathers carrier and delivery information but does not resolve the issue."),
    28: ("UNRESOLVED", "Packages remain undelivered; no resolution is shown."),
    29: ("ACTIONABLE_GUIDANCE", "Provides a working link and explains that the requested recurring-reminder feature is unavailable."),
    30: ("CASE_ACTION", "Support attempts to obtain details and route the case for investigation; no outcome shown."),
    31: ("ACTIONABLE_GUIDANCE", "Provides concrete Prime-cancellation and reorder information."),
    32: ("ROUTING_ONLY", "Immediately directs customer to DM for investigation."),
    33: ("ROUTING_ONLY", "Only moves the issue to DM."),
    34: ("ACTIONABLE_GUIDANCE", "Provides a concrete workaround for the software issue."),
    35: ("ROUTING_ONLY", "Directs customer to DM without substantive troubleshooting."),
    36: ("ACTIONABLE_GUIDANCE", "Provides a specific address-editing path and fallback DM option."),
    37: ("ROUTING_ONLY", "Only requests device/iOS information through DM."),
    38: ("ACTIONABLE_GUIDANCE", "Directly answers that Apple IDs cannot be merged and provides more information."),
    39: ("ROUTING_ONLY", "Requests more details through DM without substantive help."),
}

for row_idx, (label, note) in batch_2_labels.items():
    resolution_screening_annotations.loc[
        row_idx, "resolution_label"
    ] = label

    resolution_screening_annotations.loc[
        row_idx, "annotation_notes"
    ] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Batch 2 annotations saved.")
print()
print(
    resolution_screening_annotations
    .iloc[20:40]["resolution_label"]
    .value_counts()
)

Batch 2 annotations saved.

resolution_label
ACTIONABLE_GUIDANCE    6
CASE_ACTION            5
ROUTING_ONLY           5
DIAGNOSTIC             2
NON_SUPPORT            1
UNRESOLVED             1
Name: count, dtype: int64


In [78]:
# Display Batch 3 for manual annotation

batch_3 = resolution_screening_annotations.iloc[40:60].copy()

print("Batch size:", len(batch_3))
print("Rows:", batch_3.index.tolist())

for i, row in batch_3.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 20
Rows: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]

ROW: 40
Conversation ID: 1248350
Brand: AppleSupport
Size: 4 tweets
Customer turns: 3
Support turns: 1
Duration: 320.76666666666665 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] you do it’s starts freezing and not working and closing apps while you’re in them and all kinds crazy stuff.. Why is that @115858 ? Smh 🤔🤔
[CUSTOMER] @412883 @115858 It won’t let me update it and I’m getting charged for 4G data use 😡
[SUPPORT] @412881 Thanks for reaching out. We're here to help. DM us so we can look into this further with you. https://t.co/GDrqU22YpT
[CUSTOMER] @AppleSupport @412881 It’s happening to mine as well.

ROW: 41
Conversation ID: 1360112
Brand: AppleSupport
Size: 3 tweets
Customer turns: 2
Support turns: 1
Duration: 134.83333333333334 minutes
---------------------------------------------------------------------

In [79]:
# Save Batch 3 human annotations

batch_3_labels = {
    40: ("ROUTING_ONLY", "Support only asks customer to move the problem to DM."),
    41: ("ACTIONABLE_GUIDANCE", "Directly answers the wireless-charging question with relevant product information."),
    42: ("ACTIONABLE_GUIDANCE", "Provides a concrete workaround for the software issue."),
    43: ("UNRESOLVED", "Customer tried the workaround and says it did not work; support then routes to DM."),
    44: ("DIAGNOSTIC", "Support asks whether notifications work normally or only the icon is affected."),
    45: ("DIAGNOSTIC", "Support gathers information about which apps/features trigger the freezing."),
    46: ("ACTIONABLE_GUIDANCE", "Explains the app/developer responsibility and how the behavior works."),
    47: ("ROUTING_ONLY", "Support only offers to investigate through DM."),
    48: ("ROUTING_ONLY", "Support requests the existing case number and moves the issue to DM."),
    49: ("CONFIRMED_OUTCOME", "Customer says the suggested Wi-Fi change worked and support confirms resolution."),
    50: ("ACTIONABLE_GUIDANCE", "Provides a concrete troubleshooting article/next step; no successful outcome shown."),
    51: ("DIAGNOSTIC", "Support gathers frequency/details and requests device/version information."),
    52: ("DIAGNOSTIC", "Support gathers more information about the Voice Control problem."),
    53: ("DIAGNOSTIC", "Support gathers device/iOS/app context before continuing in DM."),
    54: ("ROUTING_ONLY", "Support only asks customer to continue the discussion in DM."),
    55: ("CONFIRMED_OUTCOME", "Support explains the behavior and customer confirms it makes sense."),
    56: ("DIAGNOSTIC", "Support gathers iOS/version/timing information; no resolution shown."),
    57: ("DIAGNOSTIC", "Support attempts troubleshooting, gathers additional information, then routes to DM."),
    58: ("UNRESOLVED", "Suggested troubleshooting had already been tried and no resolution follows."),
    59: ("ACTIONABLE_GUIDANCE", "Support identifies the app developer as the appropriate support path based on the issue."),
}

for row_idx, (label, note) in batch_3_labels.items():
    resolution_screening_annotations.loc[
        row_idx, "resolution_label"
    ] = label

    resolution_screening_annotations.loc[
        row_idx, "annotation_notes"
    ] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Batch 3 annotations saved.")
print()
print(
    resolution_screening_annotations
    .iloc[40:60]["resolution_label"]
    .value_counts()
)

Batch 3 annotations saved.

resolution_label
DIAGNOSTIC             7
ACTIONABLE_GUIDANCE    5
ROUTING_ONLY           4
UNRESOLVED             2
CONFIRMED_OUTCOME      2
Name: count, dtype: int64


In [80]:
# Display Batch 4 for manual annotation

batch_4 = resolution_screening_annotations.iloc[60:80].copy()

print("Batch size:", len(batch_4))
print("Rows:", batch_4.index.tolist())

for i, row in batch_4.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 20
Rows: [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]

ROW: 60
Conversation ID: 1752360
Brand: AppleSupport
Size: 10 tweets
Customer turns: 6
Support turns: 4
Duration: 6398.45 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @AppleSupport got this today and looks like a phishing attempt. Can you verify? https://t.co/A6Osq0jZCD
[SUPPORT] @116714 That is definitely not from us. This article shows how to report a phishing attempt: https://t.co/V2QRdtocGo
[CUSTOMER] @AppleSupport Ok. Thanks.
[SUPPORT] @116714 You're welcome. If you need help in the future, feel free to let us know. Have a wonderful day.
[CUSTOMER] @AppleSupport Any way to fix it so I can read story titles ? https://t.co/zdIlcA6byp
[CUSTOMER] @AppleSupport Here is the link:
https://t.co/ckmkD2rO3p
[SUPPORT] @116714 We're here to help. The link shared took us to an Apple News story without an issue. Do the 

In [81]:
# Save Batch 4 human annotations

batch_4_labels = {
    60: ("CONFIRMED_OUTCOME", "Customer explicitly confirms the Apple News workaround worked."),
    61: ("ACTIONABLE_GUIDANCE", "Support provides a concrete workaround; customer accepts it as the current solution."),
    62: ("ROUTING_ONLY", "Support redirects to an English-language support resource and asks for more details."),
    63: ("ROUTING_ONLY", "Support only redirects the customer to an English-language support channel."),
    64: ("CASE_ACTION", "Support requests the address so the delivery complaint can be fed back to the store."),
    65: ("NON_SUPPORT", "Casual food/suggestion conversation rather than a support issue."),
    66: ("DIAGNOSTIC", "Support requests order/customer details to investigate the complaint."),
    67: ("NON_SUPPORT", "Positive product feedback/social engagement."),
    68: ("DIAGNOSTIC", "Support requests account details to investigate missing vouchers."),
    69: ("ROUTING_ONLY", "Support only confirms a DM response; no substantive assistance is visible."),
    70: ("DIAGNOSTIC", "Support asks the customer to elaborate so the complaint can be investigated."),
    71: ("NON_SUPPORT", "Positive/general shopping engagement."),
    72: ("CASE_ACTION", "Support checks internally, identifies a supplier issue, and offers to check other stores."),
    73: ("CASE_ACTION", "Support offers replacement/refund and collects information needed to execute it."),
    74: ("DIAGNOSTIC", "Support requests a summary and receipt image to investigate the receipt issue."),
    75: ("CASE_ACTION", "Support explicitly says it will log feedback and arrange a refund."),
    76: ("DIAGNOSTIC", "Support requests customer details for investigation; no outcome is shown."),
    77: ("ACTIONABLE_GUIDANCE", "Support gives concrete advice to watch for relevant offers."),
    78: ("UNRESOLVED", "The proposed workaround does not solve the customer's actual fuel-station problem."),
    79: ("CASE_ACTION", "Support initiates a refund/supplier process and requests required information."),
}

for row_idx, (label, note) in batch_4_labels.items():
    resolution_screening_annotations.loc[
        row_idx, "resolution_label"
    ] = label

    resolution_screening_annotations.loc[
        row_idx, "annotation_notes"
    ] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Batch 4 annotations saved.")
print()
print(
    resolution_screening_annotations
    .iloc[60:80]["resolution_label"]
    .value_counts()
)

Batch 4 annotations saved.

resolution_label
CASE_ACTION            5
DIAGNOSTIC             5
ROUTING_ONLY           3
NON_SUPPORT            3
ACTIONABLE_GUIDANCE    2
CONFIRMED_OUTCOME      1
UNRESOLVED             1
Name: count, dtype: int64


In [82]:
# Display Batch 5 for manual annotation

batch_5 = resolution_screening_annotations.iloc[80:100].copy()

print("Batch size:", len(batch_5))
print("Rows:", batch_5.index.tolist())

for i, row in batch_5.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 20
Rows: [80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]

ROW: 80
Conversation ID: 95884
Brand: Tesco
Size: 5 tweets
Customer turns: 2
Support turns: 3
Duration: 19.2 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] When you're on in massive line @Tesco  express with one person service and two stuff members standing on the shop floor laughing at the guy serving and one of them refuses to get on the till even when asked by another stuff member. Great customer service!
[SUPPORT] @136878 Hi Laila, Sorry to learn that this has happened and I can understand how you'd be unhappy to see this. If you could let me know which store this was at, what time this happened, the colleagues names or descriptions of the colleagues. 1/2
[SUPPORT] @136878 Once I have this information I will let the stores management aware of this and they will take the appropriate action. Thanks - Lee 2/2
[C

In [83]:
# Save Batch 5 annotations

batch_5_labels = {
    80: ("CASE_ACTION", "Support will pass the complaint to store management after collecting details."),
    81: ("CONFIRMED_OUTCOME", "Customer returned the products and support confirms supplier quality feedback will occur; customer acknowledges completion."),
    82: ("CASE_ACTION", "Support collects store/customer details to pass packaging feedback to head office."),
    83: ("CASE_ACTION", "Support is collecting evidence/details to formally report the product issue to the supplier."),
    84: ("ACTIONABLE_GUIDANCE", "Support explains the Partner model and tells the customer how to filter for Tesco products/promotions."),
    85: ("CASE_ACTION", "Support collects details to escalate the hygiene complaint to store management."),
    86: ("CASE_ACTION", "Support investigates with the store/supplier and provides the reason for the product range decision."),
    87: ("CASE_ACTION", "Support agrees to log and pass the security complaint/feedback to store management."),
    88: ("CASE_ACTION", "Support collects details to log the customer's request for additional charging facilities and provides availability information."),
    89: ("ACTIONABLE_GUIDANCE", "Support checks inventory and gives the customer the aisle/location and stock information."),
    90: ("ACTIONABLE_GUIDANCE", "Support provides the store-closing information and explains the timing during extreme weather."),
    91: ("UNRESOLVED", "Troubleshooting is offered, but the customer rejects it and the actual website problem remains unresolved."),
    92: ("UNRESOLVED", "The customer explicitly reports the problem is still occurring; the visible conversation only moves into DM/support investigation."),
    93: ("ACTIONABLE_GUIDANCE", "Support explains the return policy/process and tells the customer how electrical returns are handled."),
    94: ("CASE_ACTION", "Support initiates complaint logging/supplier feedback and reimbursement handling, then continues collecting required details."),
    95: ("CASE_ACTION", "Support collects product/customer details for supplier investigation while also suggesting a possible size adjustment."),
    96: ("DIAGNOSTIC", "Support asks for more details before investigating the purchase/game entitlement issue."),
    97: ("ACTIONABLE_GUIDANCE", "Support provides a direct resource for managing/cancelling the subscription."),
    98: ("DIAGNOSTIC", "Support needs the Gamertag and additional details before investigating the missing update."),
    99: ("ACTIONABLE_GUIDANCE", "Support provides a concrete troubleshooting guide for the wireless-network problem."),
}

for row_idx, (label, note) in batch_5_labels.items():
    resolution_screening_annotations.loc[row_idx, "resolution_label"] = label
    resolution_screening_annotations.loc[row_idx, "annotation_notes"] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Batch 5 annotations saved.")
print(resolution_screening_annotations.iloc[80:100]["resolution_label"].value_counts())

Batch 5 annotations saved.
resolution_label
CASE_ACTION            9
ACTIONABLE_GUIDANCE    6
UNRESOLVED             2
DIAGNOSTIC             2
CONFIRMED_OUTCOME      1
Name: count, dtype: int64


In [84]:
# Display Batch 6 for manual annotation

batch_6 = resolution_screening_annotations.iloc[100:120].copy()

print("Batch size:", len(batch_6))
print("Rows:", batch_6.index.tolist())

for i, row in batch_6.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 20
Rows: [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119]

ROW: 100
Conversation ID: 2285965
Brand: XboxSupport
Size: 2 tweets
Customer turns: 1
Support turns: 1
Duration: 607.5333333333333 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @XboxSupport need to have your guys fix the messaging apps "talk to text" feature. It's not good. Thank you.
[SUPPORT] @664268 Hey, could you direct message us and explain in detail what issue you are experiencing?
 ^RM

ROW: 101
Conversation ID: 2328826
Brand: XboxSupport
Size: 2 tweets
Customer turns: 1
Support turns: 1
Duration: 139.75 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] @XboxSupport I have the disc for Dishonored 2. It installed on 1 Xbox, but not on my other.  The disc had no damage.  What can I do?
[SUPPORT] @485337 Hello, thanks 

In [85]:
# Save Batch 6 annotations

batch_6_labels = {
    100: ("DIAGNOSTIC", "Support asks for detailed information before investigating the messaging issue."),
    101: ("DIAGNOSTIC", "Support asks whether an installation error occurs before troubleshooting."),
    102: ("ACTIONABLE_GUIDANCE", "Support provides a concrete account-check step through the supplied link."),
    103: ("DIAGNOSTIC", "Support asks what happens and requests Gamertag/details for investigation."),
    104: ("ACTIONABLE_GUIDANCE", "Support directs the customer to the formal suspension/case-review process."),
    105: ("DIAGNOSTIC", "Support requests Gamertag and comparison with other games to diagnose the server issue."),
    106: ("DIAGNOSTIC", "Support requests Gamertag and service-request information to investigate the complaint."),
    107: ("ACTIONABLE_GUIDANCE", "Support explains that it cannot handle enforcement and directs the customer to case review."),
    108: ("DIAGNOSTIC", "Support requests Gamertag and additional details before investigating the login issue."),
    109: ("UNRESOLVED", "Xbox acknowledges an outage, but the customer's refund request remains unanswered in the visible conversation."),
    110: ("DIAGNOSTIC", "Support requests Gamertag and a detailed issue summary; no troubleshooting outcome is reached."),
    111: ("UNRESOLVED", "The issue was temporarily resolved through chat but explicitly recurs afterward."),
    112: ("ACTIONABLE_GUIDANCE", "Support directs the customer to submit a Case Review for enforcement information."),
    113: ("DIAGNOSTIC", "Support requests Gamertag and clarification so the issue can be investigated."),
    114: ("DIAGNOSTIC", "Support asks for clarification of the 'teredo' issue; no solution is provided."),
    115: ("DIAGNOSTIC", "Support progressively gathers error/network information and requests diagnostic details."),
    116: ("UNRESOLVED", "Customer explicitly says the live-chat team has not solved the issue."),
    117: ("DIAGNOSTIC", "Support is still gathering information and testing settings; no resolution is shown."),
    118: ("DIAGNOSTIC", "Customer introduces a technical issue and support requests network/account details for investigation."),
    119: ("DIAGNOSTIC", "Support is actively gathering information about the unexpected console behavior; no outcome yet."),
}

for row_idx, (label, note) in batch_6_labels.items():
    resolution_screening_annotations.loc[row_idx, "resolution_label"] = label
    resolution_screening_annotations.loc[row_idx, "annotation_notes"] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Batch 6 annotations saved.")
print(resolution_screening_annotations.iloc[100:120]["resolution_label"].value_counts())

Batch 6 annotations saved.
resolution_label
DIAGNOSTIC             13
ACTIONABLE_GUIDANCE     4
UNRESOLVED              3
Name: count, dtype: int64


In [86]:
# Display final annotation batch

batch_7 = resolution_screening_annotations.iloc[120:128].copy()

print("Batch size:", len(batch_7))
print("Rows:", batch_7.index.tolist())

for i, row in batch_7.iterrows():
    print("\n" + "=" * 100)
    print(f"ROW: {i}")
    print(f"Conversation ID: {row['conversation_id']}")
    print(f"Brand: {row['support_account']}")
    print(f"Size: {row['conversation_size']} tweets")
    print(f"Customer turns: {row['customer_turns']}")
    print(f"Support turns: {row['support_turns']}")
    print(f"Duration: {row['duration_minutes']} minutes")
    print("-" * 100)
    print(row["conversation_text"])

Batch size: 8
Rows: [120, 121, 122, 123, 124, 125, 126, 127]

ROW: 120
Conversation ID: 881763
Brand: XboxSupport
Size: 13 tweets
Customer turns: 8
Support turns: 5
Duration: 1324.65 minutes
----------------------------------------------------------------------------------------------------
[CUSTOMER] Lost over 100,000 Gamerscore after the most recent @115786 update. @15913, @116543, @115787 is this a known issue?
[CUSTOMER] @115786 @15913 @116543 @115787 Someone on Reddit has the same issue I'm having. https://t.co/gKtOM60Ki8
[SUPPORT] @329367 @115786 @15913 @116543 @115787 Definitely odd. To be sure is this consistent on the Xbox App and the https://t.co/6DxzuMtRWi?? Can you also dm your GT and 1/2 ^IS
[SUPPORT] @329367 @115786 @15913 @116543 @115787 the specific achievements that you notice are missing? That would be very helpful. https://t.co/nPX1yNG0Tv 2/2 ^IS
[CUSTOMER] @XboxSupport Yup. The issue is on my Xbox, App, and website. Clicking on an individual game shows the achieveme

In [87]:
# Save final batch annotations

batch_7_labels = {
    120: ("DIAGNOSTIC", "Support is collecting Gamertag, missing achievements, and troubleshooting history; no resolution yet."),
    121: ("UNRESOLVED", "Xbox says the issue is being investigated, but the customer continues reporting that it is not fixed."),
    122: ("ACTIONABLE_GUIDANCE", "Across the support interactions, concrete guidance is provided: feedback route, account explanation, enforcement review, and a troubleshooting guide."),
    123: ("ACTIONABLE_GUIDANCE", "Support directly explains the purchasing requirement and provides the relevant information/link."),
    124: ("CONFIRMED_OUTCOME", "The customer explicitly says the issue has been fixed and that everything is good."),
    125: ("ACTIONABLE_GUIDANCE", "Support gives concrete instructions for changing the Microsoft account email/alias and explains the login implication."),
    126: ("ACTIONABLE_GUIDANCE", "Support gives a concrete next step through chat support and direct messaging, including what information to provide."),
    127: ("DIAGNOSTIC", "Multiple unrelated issues appear; support requests clarification and account information, with no demonstrated resolution."),
}

for row_idx, (label, note) in batch_7_labels.items():
    resolution_screening_annotations.loc[row_idx, "resolution_label"] = label
    resolution_screening_annotations.loc[row_idx, "annotation_notes"] = note

resolution_screening_annotations.to_csv(
    annotation_path,
    index=False,
    encoding="utf-8"
)

print("Final batch annotations saved.")
print("\nFinal batch counts:")
print(resolution_screening_annotations.iloc[120:128]["resolution_label"].value_counts())

Final batch annotations saved.

Final batch counts:
resolution_label
ACTIONABLE_GUIDANCE    4
DIAGNOSTIC             2
UNRESOLVED             1
CONFIRMED_OUTCOME      1
Name: count, dtype: int64


In [88]:
print("\nTotal annotation counts:")
print(resolution_screening_annotations["resolution_label"].value_counts())

print("\nTotal rows:", len(resolution_screening_annotations))
print("Annotated rows:", resolution_screening_annotations["resolution_label"].notna().sum())
print("Remaining unannotated:", resolution_screening_annotations["resolution_label"].isna().sum())


Total annotation counts:
resolution_label
DIAGNOSTIC             35
ACTIONABLE_GUIDANCE    29
CASE_ACTION            21
ROUTING_ONLY           19
UNRESOLVED             11
CONFIRMED_OUTCOME       7
NON_SUPPORT             5
AMBIGUOUS               1
Name: count, dtype: int64

Total rows: 128
Annotated rows: 128
Remaining unannotated: 0


useful evidence that “support response exists” ≠ “issue was resolved.”

In [89]:
# Final verification of resolution screening annotations

import pandas as pd

# Reload from disk to verify that the saved artifact is correct
resolution_screening_annotations = pd.read_csv(
    annotation_path
)

print("File:", annotation_path)
print("Rows:", len(resolution_screening_annotations))
print("Columns:", resolution_screening_annotations.columns.tolist())

print("\nMissing resolution labels:")
print(
    resolution_screening_annotations["resolution_label"]
    .isna()
    .sum()
)

print("\nOverall resolution-label distribution:")
label_counts = (
    resolution_screening_annotations["resolution_label"]
    .value_counts()
)

label_summary = pd.DataFrame({
    "count": label_counts,
    "percentage": (label_counts / len(resolution_screening_annotations) * 100).round(2)
})

print(label_summary)

print("\nBrand × resolution label:")
brand_resolution = pd.crosstab(
    resolution_screening_annotations["support_account"],
    resolution_screening_annotations["resolution_label"]
)

print(brand_resolution)

print("\nBrand totals:")
print(
    resolution_screening_annotations["support_account"]
    .value_counts()
)

print("\nAnnotation verification complete.")

File: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\resolution_screening_annotations.csv
Rows: 128
Columns: ['conversation_id', 'support_account', 'size_bucket', 'conversation_size', 'customer_turns', 'support_turns', 'duration_minutes', 'conversation_text', 'resolution_label', 'annotation_notes', 'annotator']

Missing resolution labels:
0

Overall resolution-label distribution:
                     count  percentage
resolution_label                      
DIAGNOSTIC              35       27.34
ACTIONABLE_GUIDANCE     29       22.66
CASE_ACTION             21       16.41
ROUTING_ONLY            19       14.84
UNRESOLVED              11        8.59
CONFIRMED_OUTCOME        7        5.47
NON_SUPPORT              5        3.91
AMBIGUOUS                1        0.78

Brand × resolution label:
resolution_label  ACTIONABLE_GUIDANCE  AMBIGUOUS  CASE_ACTION  \
support_account                                                 
AmazonHelp                          5          1            7  

AMAZONHELP is best overall fit for the assignment because it gives us the strongest combination of:

- Very high data volume — 82K conversations
- Deep conversations — highest mean conversation size among the four
- Strong customer back-and-forth
- Meaningful case-action examples
- Actionable guidance + confirmed outcomes
- Routing and unresolved cases, which are valuable for building and evaluating escalation
- Enough breadth for a useful intent taxonomy
- Enough data to support retrieval and train/test splits without running out of examples

Selected brand: AmazonHelp

AmazonHelp was selected using a data-driven comparison of conversation volume, conversation depth, multi-turn interaction, resolution evidence, and escalation opportunities. It had the largest candidate conversation pool (82,246), the highest mean conversation size (4.52 tweets), and strong evidence of both substantive support actions and unresolved/routing cases. Tesco showed stronger case-action density in the manual screening sample, but its substantially smaller conversation pool made AmazonHelp the better overall choice for building and evaluating the complete agent.

Build the AmazonHelp conversation dataset

In [90]:
amazon_conversations = resolution_episodes[
    resolution_episodes["support_account"] == "AmazonHelp"
].copy()

print("AmazonHelp conversations:", len(amazon_conversations))
print(
    "Columns:",
    amazon_conversations.columns.tolist()
)

print("\nConversation size:")
print(
    amazon_conversations["conversation_size"]
    .describe()
)

print("\nCustomer turns:")
print(
    amazon_conversations["customer_turns"]
    .describe()
)

print("\nSupport turns:")
print(
    amazon_conversations["support_turns"]
    .describe()
)

print("\nSize buckets:")
print(
    amazon_conversations["size_bucket"]
    .value_counts()
)

print("\nResolution screening examples for AmazonHelp:")
print(
    resolution_screening_annotations[
        resolution_screening_annotations["support_account"] == "AmazonHelp"
    ][
        ["conversation_id", "conversation_size",
         "customer_turns", "support_turns",
         "resolution_label"]
    ].to_string(index=False)
)

AmazonHelp conversations: 82246
Columns: ['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']

Conversation size:
count    82246.000000
mean         4.515928
std          5.107119
min          2.000000
25%          2.000000
50%          3.000000
75%          5.000000
max        448.000000
Name: conversation_size, dtype: float64

Customer turns:
count    82246.000000
mean         2.458186
std          3.280729
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        447.000000
Name: customer_turns, dtype: float64

Support turns:
count    82246.000000
mean         2.057741
std          2.139243
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        162.000000
Name: support_turns, dtype: float64

Size b

- 31,296 short 2-tweet conversations
- 25,490 with 3–4 tweets
- 17,850 with 5–8 tweets
- 7,610 with 9+ tweets
- Mean conversation size: 4.52 tweets
- Median: 3 tweets
- Mean customer turns: 2.46
- Mean support turns: 2.06

In [91]:
# Inspect AmazonHelp customer queries for intent discovery

import pandas as pd

# Balanced sample across conversation-size buckets
bucket_samples = []

for bucket in [
    "2_tweets",
    "3_to_4_tweets",
    "5_to_8_tweets",
    "9_plus_tweets"
]:
    subset = amazon_conversations[
        amazon_conversations["size_bucket"] == bucket
    ].copy()

    sample_n = min(250, len(subset))

    bucket_samples.append(
        subset.sample(
            n=sample_n,
            random_state=42
        )
    )

intent_discovery_sample = pd.concat(
    bucket_samples,
    ignore_index=True
)

print("Total sampled conversations:", len(intent_discovery_sample))
print(
    intent_discovery_sample["size_bucket"]
    .value_counts()
)

print("\n" + "=" * 100)
print("CUSTOMER OPENING MESSAGES")
print("=" * 100)

for i, (_, row) in enumerate(
    intent_discovery_sample.iterrows()
):
    print(
        f"\n[{i}] "
        f"bucket={row['size_bucket']} | "
        f"conversation_size={row['conversation_size']}"
    )
    print(row["first_customer_message"])

Total sampled conversations: 1000
size_bucket
2_tweets         250
3_to_4_tweets    250
5_to_8_tweets    250
9_plus_tweets    250
Name: count, dtype: int64

CUSTOMER OPENING MESSAGES

[0] bucket=2_tweets | conversation_size=2
Amazon Prime Musicで音楽聴きながら作業して捗る

[1] bucket=2_tweets | conversation_size=2
@115830 Hi there, having a bit of problem. I received an email saying that my delivery has been delivered, but because it was sent to my work address and after office hours.. not too sure where the delivery man has left it? it's no where to be seen.

[2] bucket=2_tweets | conversation_size=2
@115850 just gt a call frm delivery person regarding where my address is. Yes at 7:58 in the mrning. What a great way to wakeup! #scarcasm

[3] bucket=2_tweets | conversation_size=2
Vous êtes sérieux @120533 de livrer ça dans cet état ?? https://t.co/4PLAp7wGD9

[4] bucket=2_tweets | conversation_size=2
@115830 hi, is it possible to redirect an order to an amazon locker after it's been despatched? Than

The sample has many delivery variants:

- late delivery
- Prime one-day/two-day failure
- "out for delivery" but not delivered
- false attempted delivery
- wrong address
- neighbor delivery
- delivery instructions ignored
- carrier problems
- package missing

splitting all of these into separate classes would create artificially fine-grained intents and make the golden set harder to label consistently.

Create the 150-example taxonomy audit

In [92]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# use the existing 1000 example intent discovery sample
audit_source = intent_discovery_sample.copy()

print("Source rows:", len(audit_source))
print(audit_source["size_bucket"].value_counts().sort_index())

Source rows: 1000
size_bucket
2_tweets         250
3_to_4_tweets    250
5_to_8_tweets    250
9_plus_tweets    250
Name: count, dtype: int64


In [93]:
# Create the audit sample
audit_parts = []

bucket_counts = {
    "2_tweets": 38,
    "3_to_4_tweets": 37,
    "5_to_8_tweets": 37,
    "9_plus_tweets": 38,
}

for bucket, n in bucket_counts.items():
    part = (
        audit_source[audit_source["size_bucket"] == bucket]
        .sample(n=n, random_state=42)
        .copy()
    )
    audit_parts.append(part)

taxonomy_audit = (
    pd.concat(audit_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Add blank annotation columns
taxonomy_audit["intent_label"] = ""
taxonomy_audit["annotation_notes"] = ""
taxonomy_audit["annotator"] = ""

# Save
audit_path = PROCESSED_DIR / "taxonomy_audit_150.csv"
taxonomy_audit.to_csv(audit_path, index=False)

print("Taxonomy audit created.")
print("Rows:", len(taxonomy_audit))
print("Saved to:", audit_path)

print("\nSize-bucket distribution:")
print(taxonomy_audit["size_bucket"].value_counts().sort_index())

Taxonomy audit created.
Rows: 150
Saved to: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\taxonomy_audit_150.csv

Size-bucket distribution:
size_bucket
2_tweets         38
3_to_4_tweets    37
5_to_8_tweets    37
9_plus_tweets    38
Name: count, dtype: int64


In [94]:
print(taxonomy_audit.columns.tolist())


['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'intent_label', 'annotation_notes', 'annotator']


In [95]:
print(taxonomy_audit.head(2).to_string())

                                                                                                                     first_customer_message                                           last_customer_message                                                                                                            first_support_message                                                                                                             last_support_message                                                                                                                                                                                                                                                                                                               conversation_text  customer_turns  support_turns  conversation_size                start_time                  end_time  duration_minutes support_account    size_bucket intent_label annotation_notes annotator
0  @AmazonHelp please see t

In [96]:
# Show first 30 taxonomy-audit examples for manual annotation
batch_1 = taxonomy_audit.iloc[:30][
    [
        "first_customer_message",
        "last_customer_message",
        "conversation_text",
        "customer_turns",
        "support_turns",
        "conversation_size",
        "size_bucket",
        "intent_label",
        "annotation_notes",
    ]
].copy()

pd.set_option("display.max_colwidth", 500)

display(batch_1)

,first_customer_message,last_customer_message,conversation_text,customer_turns,support_turns,conversation_size,size_bucket,intent_label,annotation_notes
0,@AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP,@AmazonHelp Thank you,[CUSTOMER] @AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP\n[SUPPORT] @347202 I'm sorry for the condition of the parcel. Please contact us directly to explore options: https://t.co/JzP7hlA23B ^JF\n[CUSTOMER] @AmazonHelp Thank you,2,1,3,3_to_4_tweets,,
1,@120533 Intéressants vos emballages... https://t.co/kMqmBDEt52,@120533 Intéressants vos emballages... https://t.co/kMqmBDEt52,"[CUSTOMER] @120533 Intéressants vos emballages... https://t.co/kMqmBDEt52\n[SUPPORT] @733790 Navrée pour cela, je vous prie de nous laisser un commentaire sur l'emballage via ce lien: https://t.co/6m981cpbIF. ^MA",1,1,2,2_tweets,,
2,"You guys should really Google the word ""Guaranteed"" before giving me an arrival date. Just terrible this year. #Amazon @115821 #late #guess?",@AmazonHelp Per Amazon arriving today. Per UPS arriving tomorrow with 1 of 6 guaranteed delivered today.,"[CUSTOMER] You guys should really Google the word ""Guaranteed"" before giving me an arrival date. Just terrible this year. #Amazon @115821 #late #guess?\n[SUPPORT] @314885 Oh no, Jason! Have we missed a delivery date on a current order? Please let us know! ^MW\n[CUSTOMER] @AmazonHelp Consistently. I would really prefer actual dates to “guaranteed” ones. No matter the cost.\n[SUPPORT] @314885 I understand your frustration. Are you noticing this trend with a certain carrier: https://t.co/q4L...",5,5,10,9_plus_tweets,,
3,I had an issue with delivery. Ordered on 3rd. Product is still not delivered. No help from Amazon side. Just asking me to wait @115850,"@AmazonHelp Yes, I got many emails regarding this issue and finally they want me to wait till tomorrow. Hope this will be resolved tomorrow.","[CUSTOMER] I had an issue with delivery. Ordered on 3rd. Product is still not delivered. No help from Amazon side. Just asking me to wait @115850\n[SUPPORT] @183115 Could you please confirm if you have received a corresponding email from our team by checking your message centre. 1/2 ^AH\n[SUPPORT] @183115 Click the below link to visit your message centre. https://t.co/DTSNmGldJf. 2/2 ^AH\n[CUSTOMER] @AmazonHelp Yes, I got many emails regarding this issue and finally they want me to wait ti...",2,3,5,5_to_8_tweets,,
4,"@115850 Hello Amazon team, I have ordered an item and got an email that it was delivered to me today. I am at home and didn't receive any product. When I try to call the agent number (8179260086) it is switched off. I am a Prime customer and your service is ridiculous. https://t.co/cibAXpH6jG","@AmazonHelp You have sent an email that product was Delivered and in Application you have updated status that, it was handed directly to me and the delivery receipt was signed by me, which is completely false. This seems to be the signature forgery and fraud. @120781 @115850 https://t.co/LVYjdo0ky4","[CUSTOMER] @115850 Hello Amazon team, I have ordered an item and got an email that it was delivered to me today. I am at home and didn't receive any product. When I try to call the agent number (8179260086) it is switched off. I am a Prime customer and your service is ridiculous. https://t.co/cibAXpH6jG\n[SUPPORT] @787339 Please don't provide your order details, we consider it to be personal information. Our page is visible to the public. 2/2^KK\n[SUPPORT] @787339 Sorry for the trouble. I wo...",2,3,5,5_to_8_tweets,,
5,"Dear @115828 @115821 My ""allegedly"" brand new copy of Tokyo Xanadu is missing something... https://t.co/APYKHYijZI","Dear @115828 @115821 My ""allegedly"" brand new copy of Tokyo Xanadu is missing something... https://t.co/APYKHYijZI","[CUSTOMER] Dear @115828 @115821 My ""alleg

In [97]:
for i, row in taxonomy_audit.iloc[:30].iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print(f"Size: {row['size_bucket']} | "
          f"Customer turns: {row['customer_turns']} | "
          f"Support turns: {row['support_turns']}")
    print("-" * 100)
    print("FIRST CUSTOMER:")
    print(row["first_customer_message"])
    print("\nLAST CUSTOMER:")
    print(row["last_customer_message"])
    print()

ROW 0
Size: 3_to_4_tweets | Customer turns: 2 | Support turns: 1
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP

LAST CUSTOMER:
@AmazonHelp Thank you

ROW 1
Size: 2_tweets | Customer turns: 1 | Support turns: 1
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@120533 Intéressants vos emballages... https://t.co/kMqmBDEt52

LAST CUSTOMER:
@120533 Intéressants vos emballages... https://t.co/kMqmBDEt52

ROW 2
Size: 9_plus_tweets | Customer turns: 5 | Support turns: 5
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
You guys should really Google the word "Guaranteed" before giving me an arrival date. Just terrible this year. #Amazon @115821 #late #guess?

Enter Batch 1 into the dataframe

In [98]:
batch_1_labels = {
    0: "PRODUCT_PROBLEM",
    1: "OTHER_NON_SUPPORT",
    2: "DELIVERY_DELAY",
    3: "DELIVERY_DELAY",
    4: "DELIVERY_MISSING_OR_MISDELIVERED",
    5: "PRODUCT_PROBLEM",
    6: "ORDER_STATUS_OR_CANCELLATION",
    7: "DELIVERY_DELAY",
    8: "PAYMENT_BILLING",
    9: "DELIVERY_DELAY",
    10: "ACCOUNT_ACCESS_SECURITY",
    11: "OTHER_NON_SUPPORT",
    12: "WEBSITE_APP_TECHNICAL",
    13: "PRIME_MEMBERSHIP",
    14: "OTHER_NON_SUPPORT",
    15: "PRODUCT_PROBLEM",
    16: "OTHER_NON_SUPPORT",
    17: "DELIVERY_MISSING_OR_MISDELIVERED",
    18: "GIFT_CARD_PROMOTION",
    19: "ORDER_STATUS_OR_CANCELLATION",
    20: "PRODUCT_PROBLEM",
    21: "OTHER_NON_SUPPORT",
    22: "OTHER_NON_SUPPORT",
    23: "PRODUCT_PROBLEM",
    24: "DELIVERY_DELAY",
    25: "PRODUCT_AVAILABILITY_INFORMATION",
    26: "DELIVERY_DELAY",
    27: "DELIVERY_MISSING_OR_MISDELIVERED",
    28: "PRODUCT_PROBLEM",
    29: "DELIVERY_DELAY",
}

for row_idx, label in batch_1_labels.items():
    taxonomy_audit.loc[row_idx, "intent_label"] = label

# Add notes for the taxonomy-gap example
taxonomy_audit.loc[14, "annotation_notes"] = (
    "Does not fit current taxonomy cleanly; possible seller-support intent."
)

taxonomy_audit.loc[21, "annotation_notes"] = (
    "General customer-service complaint without a specific underlying issue."
)

taxonomy_audit.loc[16, "annotation_notes"] = (
    "Advertisement/content feedback rather than a defined support issue."
)

# Save
taxonomy_audit.to_csv(audit_path, index=False)

print("Batch 1 saved.")
print("Annotated:", (taxonomy_audit["intent_label"].str.strip() != "").sum())
print("Remaining:", (taxonomy_audit["intent_label"].str.strip() == "").sum())

Batch 1 saved.
Annotated: 30
Remaining: 120


In [99]:
for i, row in taxonomy_audit.iloc[30:60].iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print(
        f"Size: {row['size_bucket']} | "
        f"Customer turns: {row['customer_turns']} | "
        f"Support turns: {row['support_turns']}"
    )
    print("-" * 100)
    print("FIRST CUSTOMER:")
    print(row["first_customer_message"])
    print("\nLAST CUSTOMER:")
    print(row["last_customer_message"])
    print()

ROW 30
Size: 2_tweets | Customer turns: 1 | Support turns: 1
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@AmazonHelp . IRe: Your Amazon.in Order  # 402-4987357-3643550 dear Amazon stop making fools .

LAST CUSTOMER:
@AmazonHelp . IRe: Your Amazon.in Order  # 402-4987357-3643550 dear Amazon stop making fools .

ROW 31
Size: 2_tweets | Customer turns: 1 | Support turns: 1
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@40172 when is your next season coming up?

LAST CUSTOMER:
@40172 when is your next season coming up?

ROW 32
Size: 3_to_4_tweets | Customer turns: 2 | Support turns: 1
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@115850   Ok den tell me till when I have to wait..first tym I dint got any call or msg regarding the delivery person..day is about to end..I paid to g

In [100]:
batch_2_labels = {
    30: "OTHER_NON_SUPPORT",
    31: "OTHER_NON_SUPPORT",
    32: "DELIVERY_DELAY",
    33: "DELIVERY_MISSING_OR_MISDELIVERED",
    34: "DELIVERY_DELAY",
    35: "PRODUCT_PROBLEM",
    36: "RETURN_REPLACEMENT_REFUND",
    37: "PRODUCT_PROBLEM",
    38: "DIGITAL_CONTENT",
    39: "PRODUCT_PROBLEM",
    40: "OTHER_NON_SUPPORT",
    41: "DELIVERY_DELAY",
    42: "WEBSITE_APP_TECHNICAL",
    43: "DELIVERY_MISSING_OR_MISDELIVERED",
    44: "PRIME_MEMBERSHIP",
    45: "ORDER_STATUS_OR_CANCELLATION",
    46: "PAYMENT_BILLING",
    47: "DELIVERY_DELAY",
    48: "PRIME_MEMBERSHIP",
    49: "DELIVERY_DELAY",
    50: "DEVICE_TECHNICAL_SUPPORT",
    51: "ORDER_STATUS_OR_CANCELLATION",
    52: "PRIME_MEMBERSHIP",
    53: "DELIVERY_DELAY",
    54: "PRODUCT_PROBLEM",
    55: "DELIVERY_DELAY",
    56: "DELIVERY_DELAY",
    57: "OTHER_NON_SUPPORT",
    58: "DIGITAL_CONTENT",
    59: "OTHER_NON_SUPPORT",
}

for row_idx, label in batch_2_labels.items():
    taxonomy_audit.loc[row_idx, "intent_label"] = label

# Add useful taxonomy notes
taxonomy_audit.loc[40, "annotation_notes"] = (
    "Support/contact follow-up but underlying issue is not identifiable."
)

taxonomy_audit.loc[45, "annotation_notes"] = (
    "Order lifecycle problem; later becomes shipping eligibility issue."
)

taxonomy_audit.loc[44, "annotation_notes"] = (
    "Prime benefit/fee issue, therefore Prime Membership rather than delivery delay."
)

taxonomy_audit.loc[55, "annotation_notes"] = (
    "Initial issue is delayed delivery; eventual successful delivery does not change primary intent."
)

taxonomy_audit.to_csv(audit_path, index=False)

print("Batch 2 saved.")
print("Annotated:", (taxonomy_audit["intent_label"].str.strip() != "").sum())
print("Remaining:", (taxonomy_audit["intent_label"].str.strip() == "").sum())

Batch 2 saved.
Annotated: 60
Remaining: 90


In [101]:
for i, row in taxonomy_audit.iloc[60:90].iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print(
        f"Size: {row['size_bucket']} | "
        f"Customer turns: {row['customer_turns']} | "
        f"Support turns: {row['support_turns']}"
    )
    print("-" * 100)
    print("FIRST CUSTOMER:")
    print(row["first_customer_message"])
    print("\nLAST CUSTOMER:")
    print(row["last_customer_message"])
    print()

ROW 60
Size: 9_plus_tweets | Customer turns: 7 | Support turns: 5
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
Dear @115830 - what's the point of sending me an app validation code with 10 mins life if it takes 5 hours to get to me?

LAST CUSTOMER:
@AmazonHelp I've initiated a chat session now - thanks

ROW 61
Size: 2_tweets | Customer turns: 1 | Support turns: 1
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
I have ordered so many books from @115850 . But everytime I get the same old boring bookmarks. Could you please send me some really cool exciting bookmarks for my next order of book?.

LAST CUSTOMER:
I have ordered so many books from @115850 . But everytime I get the same old boring bookmarks. Could you please send me some really cool exciting bookmarks for my next order of book?.

ROW 62
Size: 2_tweets | Customer turns: 1 | Support turns: 1


In [102]:
batch_3_labels = {
    60: "ACCOUNT_ACCESS_SECURITY",
    61: "OTHER_NON_SUPPORT",
    62: "WEBSITE_APP_TECHNICAL",
    63: "DELIVERY_DELAY",
    64: "PAYMENT_BILLING",
    65: "PRODUCT_AVAILABILITY_INFORMATION",
    66: "PRODUCT_AVAILABILITY_INFORMATION",
    67: "DIGITAL_CONTENT",
    68: "PRODUCT_PROBLEM",
    69: "DELIVERY_MISSING_OR_MISDELIVERED",
    70: "PRODUCT_AVAILABILITY_INFORMATION",
    71: "OTHER_NON_SUPPORT",
    72: "DEVICE_TECHNICAL_SUPPORT",
    73: "RETURN_REPLACEMENT_REFUND",
    74: "PRIME_MEMBERSHIP",
    75: "DELIVERY_MISSING_OR_MISDELIVERED",
    76: "DIGITAL_CONTENT",
    77: "DELIVERY_DELAY",
    78: "DEVICE_TECHNICAL_SUPPORT",
    79: "ORDER_STATUS_OR_CANCELLATION",
    80: "DELIVERY_DELAY",
    81: "PRODUCT_PROBLEM",
    82: "DELIVERY_DELAY",
    83: "RETURN_REPLACEMENT_REFUND",
    84: "PRODUCT_AVAILABILITY_INFORMATION",
    85: "DELIVERY_DELAY",
    86: "DELIVERY_DELAY",
    87: "OTHER_NON_SUPPORT",
    88: "OTHER_NON_SUPPORT",
    89: "RETURN_REPLACEMENT_REFUND",
}

for row_idx, label in batch_3_labels.items():
    taxonomy_audit.loc[row_idx, "intent_label"] = label

# Record useful boundary notes
taxonomy_audit.loc[64, "annotation_notes"] = (
    "Transaction/charge confirmation issue; resolved by help line."
)

taxonomy_audit.loc[68, "annotation_notes"] = (
    "Repeated damaged delivery; shipper avoidance is secondary."
)

taxonomy_audit.loc[80, "annotation_notes"] = (
    "Prime mentioned, but actual problem is delivery time/estimate."
)

taxonomy_audit.loc[87, "annotation_notes"] = (
    "Packaging criticism without damage or concrete support request."
)

taxonomy_audit.loc[88, "annotation_notes"] = (
    "Environmental packaging criticism; no product defect."
)

taxonomy_audit.to_csv(audit_path, index=False)

print("Batch 3 saved.")
print("Annotated:", (taxonomy_audit["intent_label"].str.strip() != "").sum())
print("Remaining:", (taxonomy_audit["intent_label"].str.strip() == "").sum())

Batch 3 saved.
Annotated: 90
Remaining: 60


In [103]:
for i, row in taxonomy_audit.iloc[90:120].iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print(
        f"Size: {row['size_bucket']} | "
        f"Customer turns: {row['customer_turns']} | "
        f"Support turns: {row['support_turns']}"
    )
    print("-" * 100)
    print("FIRST CUSTOMER:")
    print(row["first_customer_message"])
    print("\nLAST CUSTOMER:")
    print(row["last_customer_message"])
    print()

ROW 90
Size: 5_to_8_tweets | Customer turns: 3 | Support turns: 3
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@AmazonHelp one of my orders say its due to be delivered 2nd November but still no sign of it? 🤔

LAST CUSTOMER:
@AmazonHelp It doesn’t say

ROW 91
Size: 9_plus_tweets | Customer turns: 4 | Support turns: 6
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@115850 why my order is cancelled...? Need help 9782077700 https://t.co/fryoj1BkZz

LAST CUSTOMER:
@AmazonHelp I have already done this.. plz call me directly.. My prepaid order has been cancelled

ROW 92
Size: 5_to_8_tweets | Customer turns: 3 | Support turns: 4
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@AmazonHelp https://t.co/9eUtPo7MRN

LAST CUSTOMER:
@AmazonHelp Still waiting for your response

ROW 93
Size: 9_

In [104]:
batch_4_labels = {
    90: "DELIVERY_DELAY",
    91: "ORDER_STATUS_OR_CANCELLATION",
    92: "OTHER_NON_SUPPORT",
    93: "PRODUCT_PROBLEM",
    94: "DELIVERY_DELAY",
    95: "DELIVERY_DELAY",
    96: "DIGITAL_CONTENT",
    97: "OTHER_NON_SUPPORT",
    98: "DELIVERY_DELAY",
    99: "DELIVERY_DELAY",
    100: "OTHER_NON_SUPPORT",
    101: "OTHER_NON_SUPPORT",
    102: "PAYMENT_BILLING",
    103: "OTHER_NON_SUPPORT",
    104: "OTHER_NON_SUPPORT",
    105: "WEBSITE_APP_TECHNICAL",
    106: "RETURN_REPLACEMENT_REFUND",
    107: "DELIVERY_DELAY",
    108: "DELIVERY_DELAY",
    109: "DELIVERY_DELAY",
    110: "PRODUCT_AVAILABILITY_INFORMATION",
    111: "PAYMENT_BILLING",
    112: "DELIVERY_DELAY",
    113: "PRODUCT_PROBLEM",
    114: "DELIVERY_DELAY",
    115: "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    116: "OTHER_NON_SUPPORT",
    117: "GIFT_CARD_PROMOTION",
    118: "DELIVERY_DELAY",
    119: "DELIVERY_DELAY",
}

for row_idx, label in batch_4_labels.items():
    taxonomy_audit.loc[row_idx, "intent_label"] = label

# Useful annotation notes
taxonomy_audit.loc[104, "annotation_notes"] = (
    "Legal/escalation complaint, but underlying support issue is insufficiently identifiable."
)

taxonomy_audit.loc[110, "annotation_notes"] = (
    "Digital product mentioned, but issue concerns advertised availability/release timing."
)

taxonomy_audit.loc[113, "annotation_notes"] = (
    "Return request is caused by poor product quality; primary intent is product problem."
)

taxonomy_audit.loc[115, "annotation_notes"] = (
    "Delivery-person behavior/access problem rather than delivery lateness."
)

taxonomy_audit.loc[116, "annotation_notes"] = (
    "DM routing request with no underlying issue visible."
)

taxonomy_audit.to_csv(audit_path, index=False)

print("Batch 4 saved.")
print("Annotated:", (taxonomy_audit["intent_label"].str.strip() != "").sum())
print("Remaining:", (taxonomy_audit["intent_label"].str.strip() == "").sum())

Batch 4 saved.
Annotated: 120
Remaining: 30


In [105]:
for i, row in taxonomy_audit.iloc[120:150].iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print(
        f"Size: {row['size_bucket']} | "
        f"Customer turns: {row['customer_turns']} | "
        f"Support turns: {row['support_turns']}"
    )
    print("-" * 100)
    print("FIRST CUSTOMER:")
    print(row["first_customer_message"])
    print("\nLAST CUSTOMER:")
    print(row["last_customer_message"])
    print()

ROW 120
Size: 5_to_8_tweets | Customer turns: 4 | Support turns: 4
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@AmazonHelp NO WHERE on the product page did it say the glasses couldn’t be returned. Now I’m stuck with a pair I can’t even use! 😡😡

LAST CUSTOMER:
@AmazonHelp Yes and they referred me back to Amazon

ROW 121
Size: 3_to_4_tweets | Customer turns: 2 | Support turns: 2
----------------------------------------------------------------------------------------------------
FIRST CUSTOMER:
@AmazonHelp llevo casi un mes para devolver un producto. Vinieron a buscarlo y aún no he tenido noticias. En la app no sale ni registrada la recogida del paquete, cuando sí lo está. Necesito una solución... porque esto no es normal 😤

LAST CUSTOMER:
@AmazonHelp Sí, está marcado sólo el primer paso de recogida de la devolución programada con fecha del 8 de noviembre. Y desde entonces no se ha actualizado, pero el paquete vinier

In [106]:
batch_5_labels = {
    120: "RETURN_REPLACEMENT_REFUND",
    121: "RETURN_REPLACEMENT_REFUND",
    122: "PRODUCT_PROBLEM",
    123: "DELIVERY_MISSING_OR_MISDELIVERED",
    124: "PRODUCT_PROBLEM",
    125: "ACCOUNT_ACCESS_SECURITY",
    126: "DELIVERY_MISSING_OR_MISDELIVERED",
    127: "OTHER_NON_SUPPORT",
    128: "ORDER_STATUS_OR_CANCELLATION",
    129: "OTHER_NON_SUPPORT",
    130: "OTHER_NON_SUPPORT",
    131: "DELIVERY_MISSING_OR_MISDELIVERED",
    132: "DELIVERY_MISSING_OR_MISDELIVERED",
    133: "DELIVERY_DELAY",
    134: "OTHER_NON_SUPPORT",
    135: "ACCOUNT_ACCESS_SECURITY",
    136: "DIGITAL_CONTENT",
    137: "DELIVERY_MISSING_OR_MISDELIVERED",
    138: "DELIVERY_DELAY",
    139: "DELIVERY_DELAY",
    140: "DELIVERY_DELAY",
    141: "OTHER_NON_SUPPORT",
    142: "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    143: "OTHER_NON_SUPPORT",
    144: "OTHER_NON_SUPPORT",
    145: "DELIVERY_DELAY",
    146: "RETURN_REPLACEMENT_REFUND",
    147: "DELIVERY_MISSING_OR_MISDELIVERED",
    148: "ACCOUNT_ACCESS_SECURITY",
    149: "PRODUCT_PROBLEM",
}

for row_idx, label in batch_5_labels.items():
    taxonomy_audit.loc[row_idx, "intent_label"] = label

# Add useful notes
taxonomy_audit.loc[128, "annotation_notes"] = (
    "Shipment notification followed by refund; interpreted as order-state/cancellation issue."
)

taxonomy_audit.loc[125, "annotation_notes"] = (
    "Account reportedly unlocked but ordering remains blocked."
)

taxonomy_audit.loc[148, "annotation_notes"] = (
    "Prime/order-history disappearance is treated as account-state problem."
)

taxonomy_audit.loc[149, "annotation_notes"] = (
    "Poor product performance/quality is the underlying issue."
)

taxonomy_audit.to_csv(audit_path, index=False)

print("Batch 5 saved.")
print("Annotated:", (taxonomy_audit["intent_label"].str.strip() != "").sum())
print("Remaining:", (taxonomy_audit["intent_label"].str.strip() == "").sum())

Batch 5 saved.
Annotated: 150
Remaining: 0


Taxonomy audit analysis

In [107]:
import pandas as pd

# 1. Basic validation
print("=" * 80)
print("TAXONOMY AUDIT VALIDATION")
print("=" * 80)

print("Total examples:", len(taxonomy_audit))
print(
    "Annotated:",
    (taxonomy_audit["intent_label"].str.strip() != "").sum()
)
print(
    "Missing:",
    (taxonomy_audit["intent_label"].str.strip() == "").sum()
)

# 2. Intent distribution
intent_counts = (
    taxonomy_audit["intent_label"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

intent_counts["percentage"] = (
    intent_counts["count"] / len(taxonomy_audit) * 100
).round(2)

print("\n" + "=" * 80)
print("INTENT DISTRIBUTION")
print("=" * 80)

display(intent_counts)

# 3. Distribution sorted alphabetically
print("\n" + "=" * 80)
print("INTENT DISTRIBUTION — ALPHABETICAL")
print("=" * 80)

display(
    intent_counts.sort_values("intent").reset_index(drop=True)
)

# 4. Rare intents
print("\n" + "=" * 80)
print("RARE INTENTS (< 5 examples)")
print("=" * 80)

rare_intents = intent_counts[intent_counts["count"] < 5]

if len(rare_intents) == 0:
    print("No intent has fewer than 5 examples.")
else:
    display(rare_intents)


# 5. OTHER_NON_SUPPORT rate
other_count = (
    taxonomy_audit["intent_label"]
    .eq("OTHER_NON_SUPPORT")
    .sum()
)

other_rate = other_count / len(taxonomy_audit) * 100

print("\n" + "=" * 80)
print("OTHER_NON_SUPPORT")
print("=" * 80)
print(f"Count: {other_count}")
print(f"Rate: {other_rate:.2f}%")

# 6. Intent distribution by conversation size
print("\n" + "=" * 80)
print("INTENT × CONVERSATION SIZE")
print("=" * 80)

intent_by_bucket = pd.crosstab(
    taxonomy_audit["intent_label"],
    taxonomy_audit["size_bucket"]
)

display(intent_by_bucket)

# 7. Potential taxonomy-gap examples
print("\n" + "=" * 80)
print("EXAMPLES WITH TAXONOMY NOTES")
print("=" * 80)

noted = taxonomy_audit[
    taxonomy_audit["annotation_notes"].fillna("").str.strip() != ""
][
    [
        "first_customer_message",
        "intent_label",
        "annotation_notes"
    ]
]

display(noted)

# 8. Coverage
print("\n" + "=" * 80)
print("TAXONOMY COVERAGE")
print("=" * 80)

print(
    f"Every example received exactly one operational label: "
    f"{len(taxonomy_audit) == taxonomy_audit['intent_label'].notna().sum()}"
)

print(
    f"Examples classified as OTHER_NON_SUPPORT: "
    f"{other_count}/{len(taxonomy_audit)} ({other_rate:.2f}%)"
)

TAXONOMY AUDIT VALIDATION
Total examples: 150
Annotated: 150
Missing: 0

INTENT DISTRIBUTION


,intent,count,percentage
0,DELIVERY_DELAY,38,25.33
1,OTHER_NON_SUPPORT,29,19.33
2,PRODUCT_PROBLEM,17,11.33
3,DELIVERY_MISSING_OR_MISDELIVERED,13,8.67
4,RETURN_REPLACEMENT_REFUND,8,5.33
5,ORDER_STATUS_OR_CANCELLATION,7,4.67
6,PRODUCT_AVAILABILITY_INFORMATION,6,4.00
7,DIGITAL_CONTENT,6,4.00
8,PAYMENT_BILLING,5,3.33
9,ACCOUNT_ACCESS_SECURITY,5,3.33



INTENT DISTRIBUTION — ALPHABETICAL


,intent,count,percentage
0,ACCOUNT_ACCESS_SECURITY,5,3.33
1,DELIVERY_ATTEMPT_OR_INSTRUCTIONS,2,1.33
2,DELIVERY_DELAY,38,25.33
3,DELIVERY_MISSING_OR_MISDELIVERED,13,8.67
4,DEVICE_TECHNICAL_SUPPORT,3,2.00
5,DIGITAL_CONTENT,6,4.00
6,GIFT_CARD_PROMOTION,2,1.33
7,ORDER_STATUS_OR_CANCELLATION,7,4.67
8,OTHER_NON_SUPPORT,29,19.33
9,PAYMENT_BILLING,5,3.33



RARE INTENTS (< 5 examples)


...


OTHER_NON_SUPPORT
Count: 29
Rate: 19.33%

INTENT × CONVERSATION SIZE


size_bucket,2_tweets,3_to_4_tweets,5_to_8_tweets,9_plus_tweets
intent_label,,,,
ACCOUNT_ACCESS_SECURITY,0,1,3,1
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0,0,0,2
DELIVERY_DELAY,7,9,12,10
DELIVERY_MISSING_OR_MISDELIVERED,4,1,6,2
DEVICE_TECHNICAL_SUPPORT,1,0,1,1
DIGITAL_CONTENT,2,1,0,3
GIFT_CARD_PROMOTION,0,1,0,1
ORDER_STATUS_OR_CANCELLATION,1,1,3,2
OTHER_NON_SUPPORT,14,7,2,6



EXAMPLES WITH TAXONOMY NOTES


,first_customer_message,intent_label,annotation_notes
14,Hey @115830 please pay my seller funds which has been held for over 115 days! __email__ https://t.co/Iua8OUsUfU,OTHER_NON_SUPPORT,Does not fit current taxonomy cleanly; possible seller-support intent.
16,@115850 Your sale advertisement today. Trivia: Find the mistake in the image. U need better testers to validate content. Can I Join you? https://t.co/xMojPqCjvL,OTHER_NON_SUPPORT,Advertisement/content feedback rather than a defined support issue.
21,@115850 worst customer service executive left between the chat you ppl are moving from bad to worse order number 404-1341814-7017162,OTHER_NON_SUPPORT,General customer-service complaint without a specific underlying issue.
40,@AmazonHelp plse help .check DM and help quickly,OTHER_NON_SUPPORT,Support/contact follow-up but underlying issue is not identifiable.
44,@115830 it's an absolute joke I pay for prime and now you are charging a delivery fee that was free with prime!!! Ragging 😡 #Amazon #AmazonPrime,PRIME_MEMBERSHIP,"Prime benefit/fee issue, therefore Prime Membership rather than delivery delay."
45,"@116875 hace 6 días compre un disco duro y es fecha que en el estatus de envío sigue diciendo: ""preparando envío""",ORDER_STATUS_OR_CANCELLATION,Order lifecycle problem; later becomes shipping eligibility issue.
55,@AmazonHelp had two packages scheduled to arrive by 8pm from ur driver. 8:47 pm and no packages. Should I assume they are not coming today?,DELIVERY_DELAY,Initial issue is delayed delivery; eventual successful delivery does not change primary intent.
64,@115830 #lookout #everyone #amazonprime #charges #noconfrimation 🛒🧐,PAYMENT_BILLING,Transaction/charge confirmation issue; resolved by help line.
68,@AmazonHelp I received ANOTHER damaged #AmazonSubscribeAndSave delivered by @118706 it was clearly marked as HEAVY &amp; delivered on its side.,PRODUCT_PROBLEM,Repeated damaged delivery; shipper avoidance is secondary.
80,"@115850 Amazon prime is not prime anymore, even the prime products take 4-5 days to deliver. :(",DELIVERY_DELAY,"Prime mentioned, but actual problem is delivery time/estimate."



TAXONOMY COVERAGE
Every example received exactly one operational label: True
Examples classified as OTHER_NON_SUPPORT: 29/150 (19.33%)


LOCK the AmazonHelp intent taxonomy

In [108]:
# Add the locked taxonomy to the project
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_INTENTS = [
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "ORDER_STATUS_OR_CANCELLATION",
    "RETURN_REPLACEMENT_REFUND",
    "PRODUCT_PROBLEM",
    "PAYMENT_BILLING",
    "PRIME_MEMBERSHIP",
    "ACCOUNT_ACCESS_SECURITY",
    "GIFT_CARD_PROMOTION",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "DIGITAL_CONTENT",
    "DEVICE_TECHNICAL_SUPPORT",
    "WEBSITE_APP_TECHNICAL",
    "OTHER_NON_SUPPORT",
]

INTENT_DEFINITIONS = {
    "DELIVERY_DELAY":
        "Shipment is late, delayed, or misses the expected delivery window.",

    "DELIVERY_MISSING_OR_MISDELIVERED":
        "Package is marked delivered but not received, or delivered to the wrong location.",

    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS":
        "Delivery attempt behavior, driver behavior, delivery instructions, or delivery location/slot issues.",

    "ORDER_STATUS_OR_CANCELLATION":
        "Order lifecycle status, cancellation, dispatch, or preorder issues.",

    "RETURN_REPLACEMENT_REFUND":
        "Return, replacement, or refund process/status.",

    "PRODUCT_PROBLEM":
        "Product is damaged, defective, wrong, incomplete, used, counterfeit, or otherwise has a product-quality problem.",

    "PAYMENT_BILLING":
        "Payment method, card, unexpected charge, billing, or Amazon Pay issue.",

    "PRIME_MEMBERSHIP":
        "Prime signup, renewal, cancellation, membership status, or Prime benefits.",

    "ACCOUNT_ACCESS_SECURITY":
        "Login, password, locked account, hacked account, authentication, or security issue.",

    "GIFT_CARD_PROMOTION":
        "Gift card, coupon, promotion, cashback, or promotional-credit issue.",

    "PRODUCT_AVAILABILITY_INFORMATION":
        "Product stock, availability, pricing, shipping eligibility, or general product information.",

    "DIGITAL_CONTENT":
        "Prime Video, Amazon Music, Kindle content, or other digital-content access/availability.",

    "DEVICE_TECHNICAL_SUPPORT":
        "Kindle, Fire TV, Fire Stick, Echo, Alexa, or other Amazon-device technical issue.",

    "WEBSITE_APP_TECHNICAL":
        "Amazon website, mobile app, checkout, UI, form, or technical site/app error.",

    "OTHER_NON_SUPPORT":
        "Non-support/social content, insufficient information, generic complaints, marketing, or cases that do not fit another operational intent.",
}

taxonomy_config = {
    "brand": "AmazonHelp",
    "taxonomy_version": "v1",
    "status": "locked",
    "audit_size": 150,
    "intents": FINAL_INTENTS,
    "definitions": INTENT_DEFINITIONS,
}

taxonomy_path = PROCESSED_DIR / "amazonhelp_intent_taxonomy_v1.json"

with open(taxonomy_path, "w", encoding="utf-8") as f:
    json.dump(taxonomy_config, f, indent=2, ensure_ascii=False)

print("=" * 80)
print("FINAL TAXONOMY LOCKED")
print("=" * 80)
print("Brand:", taxonomy_config["brand"])
print("Version:", taxonomy_config["taxonomy_version"])
print("Status:", taxonomy_config["status"])
print("Number of intents:", len(FINAL_INTENTS))
print("Audit examples:", taxonomy_config["audit_size"])
print("Saved to:", taxonomy_path)

FINAL TAXONOMY LOCKED
Brand: AmazonHelp
Version: v1
Status: locked
Number of intents: 15
Audit examples: 150
Saved to: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_taxonomy_v1.json


need to create train / validation / test splits at the conversation level, not tweet level.

one tweet from that conversation goes into training and another goes into test, the classifier can effectively see the same issue during training and testing. 
That would make test score misleadingly high.

One conversation = one split.

In [109]:
# INSPECT EXISTING CONVERSATION DATA

print("=" * 80)
print("TWCS")
print("=" * 80)

print("Rows:", len(twcs))
print("Has conversation_id:", "conversation_id" in twcs.columns)
print("Has support_account:", "support_account" in twcs.columns)

print("\nTWCS columns:")
print(list(twcs.columns))


print("\n" + "=" * 80)
print("RESOLUTION EPISODES")
print("=" * 80)

print("Rows:", len(resolution_episodes))
print("Columns:")
print(list(resolution_episodes.columns))

print("\nSupport accounts:")
if "support_account" in resolution_episodes.columns:
    print(
        resolution_episodes["support_account"]
        .value_counts()
        .head(15)
    )

print("\nAmazonHelp rows:")
if "support_account" in resolution_episodes.columns:
    print(
        "AmazonHelp:",
        (
            resolution_episodes["support_account"]
            .eq("AmazonHelp")
            .sum()
        )
    )

print("\n" + "=" * 80)
print("AMAZON CONVERSATIONS")
print("=" * 80)

print("Exists:", "amazon_conversations" in globals())

if "amazon_conversations" in globals():
    print("Rows:", len(amazon_conversations))
    print("Columns:")
    print(list(amazon_conversations.columns))

TWCS
Rows: 2811774
Has conversation_id: True
Has support_account: False

TWCS columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'created_at_parsed', 'conversation_id', 'conversation_root_type']

RESOLUTION EPISODES
Rows: 192718
Columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']

Support accounts:
support_account
AmazonHelp      82246
AppleSupport    80552
Tesco           16566
XboxSupport     13354
Name: count, dtype: int64

AmazonHelp rows:
AmazonHelp: 82246

AMAZON CONVERSATIONS
Exists: True
Rows: 82246
Columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_t

In [110]:
# INSPECT RESOLUTION EPISODE IDENTITY

print("=" * 80)
print("RESOLUTION EPISODE IDENTITY CHECK")
print("=" * 80)

print("resolution_episodes columns:")
print(list(resolution_episodes.columns))

print("\nAmazonHelp columns:")
print(list(amazon_conversations.columns))

# Check whether any hidden/index-like information survived
print("\nIndex information:")
print("resolution_episodes index:", resolution_episodes.index)
print("amazon_conversations index:", amazon_conversations.index)

# Check whether conversation_text is duplicated within AmazonHelp
amazon_text_counts = (
    amazon_conversations["conversation_text"]
    .value_counts()
)

duplicate_texts = amazon_text_counts[
    amazon_text_counts > 1
]

print("\n" + "=" * 80)
print("DUPLICATE CONVERSATION TEXT")
print("=" * 80)

print(
    "Unique conversation texts:",
    amazon_conversations["conversation_text"].nunique()
)

print(
    "Duplicated conversation texts:",
    len(duplicate_texts)
)

print(
    "AmazonHelp rows participating in duplicate texts:",
    int(duplicate_texts.sum())
)

if len(duplicate_texts) > 0:
    print("\nExample duplicated conversation text:")
    example_text = duplicate_texts.index[0]

    display(
        amazon_conversations[
            amazon_conversations["conversation_text"]
            == example_text
        ][
            [
                "conversation_text",
                "conversation_size",
                "customer_turns",
                "support_turns",
                "start_time",
                "end_time",
                "support_account"
            ]
        ]
    )

# Check whether start/end timestamps + size are unique enough
identity_cols = [
    "conversation_text",
    "conversation_size",
    "customer_turns",
    "support_turns",
    "start_time",
    "end_time"
]

print("\n" + "=" * 80)
print("COMPOSITE IDENTITY CHECK")
print("=" * 80)

print(
    "Unique composite records:",
    amazon_conversations[identity_cols].drop_duplicates().shape[0]
)

print(
    "Total AmazonHelp records:",
    len(amazon_conversations)
)

RESOLUTION EPISODE IDENTITY CHECK
resolution_episodes columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']

AmazonHelp columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']

Index information:
resolution_episodes index: Index([    272,     277,     318,     320,     323,     325,     617,     621,
           624,     630,
       ...
       2987738, 2987740, 2987742, 2987815, 2987818, 2987820, 2987822, 2987901,
       2987903, 2987939],
      dtype='int64', name='conversation_id', length=192718)
amazon_conversations index: Index([    272,     325,     617,     621,   

In [111]:
# RESTORE conversation_id FROM INDEX

import pandas as pd
from pathlib import Path

print("=" * 80)
print("RESTORING conversation_id")
print("=" * 80)

# ------------------------------------------------------------
# 1. Copy the existing AmazonHelp dataset
# ------------------------------------------------------------

amazon_conversations = amazon_conversations.copy()

# ------------------------------------------------------------
# 2. The index is already conversation_id
# ------------------------------------------------------------

print("Index name:", amazon_conversations.index.name)

if amazon_conversations.index.name != "conversation_id":
    raise ValueError(
        "Expected the AmazonHelp dataframe index to be "
        "conversation_id."
    )

# Convert the index into an explicit column
amazon_conversations = amazon_conversations.reset_index()

# ------------------------------------------------------------
# 3. Validate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VALIDATION")
print("=" * 80)

print("Rows:", f"{len(amazon_conversations):,}")
print(
    "Unique conversation IDs:",
    f"{amazon_conversations['conversation_id'].nunique():,}"
)

print(
    "Missing conversation IDs:",
    amazon_conversations["conversation_id"].isna().sum()
)

print(
    "Duplicate conversation IDs:",
    amazon_conversations["conversation_id"].duplicated().sum()
)

print(
    "Support accounts:",
    amazon_conversations["support_account"].unique()
)

# ------------------------------------------------------------
# 4. Assertions
# ------------------------------------------------------------

assert len(amazon_conversations) == 82_246
assert amazon_conversations["conversation_id"].notna().all()
assert amazon_conversations["conversation_id"].is_unique
assert amazon_conversations["support_account"].eq("AmazonHelp").all()

print("\n✓ 82,246 unique AmazonHelp conversations recovered.")
print("✓ Every conversation has a valid conversation_id.")
print("✓ No duplicate conversation IDs.")
print("✓ Every record belongs to AmazonHelp.")

# ------------------------------------------------------------
# 5. Save corrected dataset
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

amazon_path = (
    PROCESSED_DIR /
    "amazonhelp_conversations_with_ids.csv"
)

amazon_conversations.to_csv(
    amazon_path,
    index=False,
    encoding="utf-8"
)

print("\nSaved:")
print(amazon_path)

print("\nColumns:")
print(list(amazon_conversations.columns))

RESTORING conversation_id
Index name: conversation_id

VALIDATION
Rows: 82,246
Unique conversation IDs: 82,246
Missing conversation IDs: 0
Duplicate conversation IDs: 0
Support accounts: ['AmazonHelp']

✓ 82,246 unique AmazonHelp conversations recovered.
✓ Every conversation has a valid conversation_id.
✓ No duplicate conversation IDs.
✓ Every record belongs to AmazonHelp.

Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversations_with_ids.csv

Columns:
['conversation_id', 'first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']


Conversation-level Train / Validation / Test split

In [112]:
# LEAKAGE-SAFE CONVERSATION-LEVEL SPLIT

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

print("=" * 80)
print("CONVERSATION-LEVEL DATASET SPLIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Work from the corrected AmazonHelp dataset
# ------------------------------------------------------------

amazon = amazon_conversations.copy()

required_columns = [
    "conversation_id",
    "conversation_text",
    "support_account"
]

missing_columns = [
    col for col in required_columns
    if col not in amazon.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# 2. Basic identity checks
# ------------------------------------------------------------

assert amazon["conversation_id"].notna().all()
assert amazon["conversation_id"].is_unique
assert amazon["support_account"].eq("AmazonHelp").all()

print("Total conversations:", f"{len(amazon):,}")
print(
    "Unique conversation IDs:",
    f"{amazon['conversation_id'].nunique():,}"
)

# ------------------------------------------------------------
# 3. Split conversation IDs
# ------------------------------------------------------------

all_ids = amazon["conversation_id"].to_numpy()

train_ids, temp_ids = train_test_split(
    all_ids,
    test_size=0.20,
    random_state=RANDOM_STATE
)

validation_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 4. Assign split
# ------------------------------------------------------------

split_map = {}

for cid in train_ids:
    split_map[cid] = "train"

for cid in validation_ids:
    split_map[cid] = "validation"

for cid in test_ids:
    split_map[cid] = "test"

amazon["split"] = amazon["conversation_id"].map(split_map)

# ------------------------------------------------------------
# 5. Validate split assignment
# ------------------------------------------------------------

assert amazon["split"].notna().all()

train = amazon[amazon["split"] == "train"].copy()
validation = amazon[amazon["split"] == "validation"].copy()
test = amazon[amazon["split"] == "test"].copy()

print("\n" + "=" * 80)
print("SPLIT SIZES")
print("=" * 80)

print(
    f"Train:      {len(train):,} "
    f"({len(train) / len(amazon) * 100:.2f}%)"
)

print(
    f"Validation: {len(validation):,} "
    f"({len(validation) / len(amazon) * 100:.2f}%)"
)

print(
    f"Test:       {len(test):,} "
    f"({len(test) / len(amazon) * 100:.2f}%)"
)

# ------------------------------------------------------------
# 6. CRITICAL LEAKAGE CHECK
# ------------------------------------------------------------

train_ids_set = set(train["conversation_id"])
validation_ids_set = set(validation["conversation_id"])
test_ids_set = set(test["conversation_id"])

train_validation_overlap = (
    train_ids_set & validation_ids_set
)

train_test_overlap = (
    train_ids_set & test_ids_set
)

validation_test_overlap = (
    validation_ids_set & test_ids_set
)

print("\n" + "=" * 80)
print("CONVERSATION LEAKAGE CHECK")
print("=" * 80)

print(
    "Train ∩ Validation:",
    len(train_validation_overlap)
)

print(
    "Train ∩ Test:",
    len(train_test_overlap)
)

print(
    "Validation ∩ Test:",
    len(validation_test_overlap)
)

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

print("\n✓ ZERO conversation overlap across splits.")

# ------------------------------------------------------------
# 7. Check that all conversations are assigned exactly once
# ------------------------------------------------------------

split_counts = amazon["split"].value_counts()

assert split_counts.sum() == len(amazon)

print("\nAll conversations assigned exactly once: ✓")

# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

split_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits.csv"
)

amazon.to_csv(
    split_path,
    index=False,
    encoding="utf-8"
)

print("\nSaved:")
print(split_path)

# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP COMPLETE")
print("=" * 80)

print("Train conversations:     ", f"{len(train):,}")
print("Validation conversations:", f"{len(validation):,}")
print("Test conversations:      ", f"{len(test):,}")
print("Total:                   ", f"{len(amazon):,}")

CONVERSATION-LEVEL DATASET SPLIT
Total conversations: 82,246
Unique conversation IDs: 82,246

SPLIT SIZES
Train:      65,796 (80.00%)
Validation: 8,225 (10.00%)
Test:       8,225 (10.00%)

CONVERSATION LEAKAGE CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

✓ ZERO conversation overlap across splits.

All conversations assigned exactly once: ✓

Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits.csv

STEP COMPLETE
Train conversations:      65,796
Validation conversations: 8,225
Test conversations:       8,225
Total:                    82,246


In [113]:
# CHECK GOLDEN AUDIT vs DATA SPLITS

import pandas as pd

print("=" * 80)
print("GOLDEN AUDIT / SPLIT CHECK")
print("=" * 80)

# ------------------------------------------------------------
# Load the saved split
# ------------------------------------------------------------

split_df = pd.read_csv(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits.csv"
)

print("Split rows:", f"{len(split_df):,}")

# ------------------------------------------------------------
# Inspect audit columns
# ------------------------------------------------------------

print("\nAudit rows:", len(taxonomy_audit))
print("Audit columns:")
print(list(taxonomy_audit.columns))

# ------------------------------------------------------------
# We intentionally do NOT assume the audit has conversation_id.
# Check whether it does.
# ------------------------------------------------------------

print("\nAudit has conversation_id:",
      "conversation_id" in taxonomy_audit.columns)

# ------------------------------------------------------------
# Compare using conversation text ONLY as a diagnostic.
# We will NOT use it to assign IDs.
# ------------------------------------------------------------

audit_texts = set(
    taxonomy_audit["conversation_text"]
    .astype(str)
)

split_df["is_taxonomy_audit"] = (
    split_df["conversation_text"]
    .astype(str)
    .isin(audit_texts)
)

audit_matches = split_df[
    split_df["is_taxonomy_audit"]
].copy()

print("\n" + "=" * 80)
print("AUDIT MATCH CHECK")
print("=" * 80)

print(
    "Audit examples found in split:",
    len(audit_matches)
)

print(
    "Expected audit examples:",
    len(taxonomy_audit)
)

if len(audit_matches) == len(taxonomy_audit):
    print("✓ All audit examples are present in the split.")
else:
    print(
        "⚠ Match count differs. Do not proceed to labeling yet."
    )

# ------------------------------------------------------------
# Show split distribution of matched audit examples
# ------------------------------------------------------------

if len(audit_matches) > 0:
    print("\nAudit examples by split:")
    print(
        audit_matches["split"]
        .value_counts()
    )

# ------------------------------------------------------------
# Show whether any audit example is in TEST
# ------------------------------------------------------------

test_audit = audit_matches[
    audit_matches["split"] == "test"
]

print(
    "\nAudit examples currently in TEST:",
    len(test_audit)
)

print(
    "Audit examples currently outside TEST:",
    len(audit_matches) - len(test_audit)
)

GOLDEN AUDIT / SPLIT CHECK
Split rows: 82,246

Audit rows: 150
Audit columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'intent_label', 'annotation_notes', 'annotator']

Audit has conversation_id: False

AUDIT MATCH CHECK
Audit examples found in split: 150
Expected audit examples: 150
✓ All audit examples are present in the split.

Audit examples by split:
split
train         118
test           20
validation     12
Name: count, dtype: int64

Audit examples currently in TEST: 20
Audit examples currently outside TEST: 130


FREEZE TAXONOMY AUDIT OUT OF MODEL DEVELOPMENT

In [114]:
import pandas as pd
from pathlib import Path

print("=" * 80)
print("FREEZING TAXONOMY AUDIT EXAMPLES")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load the conversation-level split
# ------------------------------------------------------------

split_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed"
    r"\amazonhelp_conversation_splits.csv"
)

split_df = pd.read_csv(split_path)

# ------------------------------------------------------------
# 2. Identify taxonomy-audit conversations
# ------------------------------------------------------------

audit_texts = set(
    taxonomy_audit["conversation_text"]
    .astype(str)
)

split_df["is_taxonomy_audit"] = (
    split_df["conversation_text"]
    .astype(str)
    .isin(audit_texts)
)

# ------------------------------------------------------------
# 3. Validate exactly 150
# ------------------------------------------------------------

audit_count = split_df["is_taxonomy_audit"].sum()

print("Taxonomy audit examples found:", audit_count)

assert audit_count == 150

print("✓ Exactly 150 audit examples identified.")

# ------------------------------------------------------------
# 4. Examine their current split locations
# ------------------------------------------------------------

print("\nAudit examples by split:")

display(
    pd.crosstab(
        split_df["split"],
        split_df["is_taxonomy_audit"]
    )
)

# ------------------------------------------------------------
# 5. Create development-safe pools
# ------------------------------------------------------------

development_train = split_df[
    (split_df["split"] == "train") &
    (~split_df["is_taxonomy_audit"])
].copy()

development_validation = split_df[
    (split_df["split"] == "validation") &
    (~split_df["is_taxonomy_audit"])
].copy()

# Test remains completely untouched.
final_test = split_df[
    split_df["split"] == "test"
].copy()

print("\n" + "=" * 80)
print("DEVELOPMENT POOLS")
print("=" * 80)

print(
    "Training conversations available:",
    f"{len(development_train):,}"
)

print(
    "Validation conversations available:",
    f"{len(development_validation):,}"
)

print(
    "Test conversations:",
    f"{len(final_test):,}"
)

# ------------------------------------------------------------
# 6. Make sure no audit examples remain in development
# ------------------------------------------------------------

assert development_train["is_taxonomy_audit"].sum() == 0
assert development_validation["is_taxonomy_audit"].sum() == 0

print("\n✓ No taxonomy-audit examples in training.")
print("✓ No taxonomy-audit examples in validation.")

# ------------------------------------------------------------
# 7. Save frozen split metadata
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

frozen_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits_v2.csv"
)

split_df.to_csv(
    frozen_path,
    index=False,
    encoding="utf-8"
)

print("\nSaved:")
print(frozen_path)

FREEZING TAXONOMY AUDIT EXAMPLES
Taxonomy audit examples found: 150
✓ Exactly 150 audit examples identified.

Audit examples by split:


is_taxonomy_audit,False,True
split,,
test,8205,20
train,65678,118
validation,8213,12



DEVELOPMENT POOLS
Training conversations available: 65,678
Validation conversations available: 8,213
Test conversations: 8,225

✓ No taxonomy-audit examples in training.
✓ No taxonomy-audit examples in validation.

Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits_v2.csv


SILVER-LABELING HEURISTIC

In [115]:
import re
import pandas as pd

FINAL_INTENTS = [
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "ORDER_STATUS_OR_CANCELLATION",
    "RETURN_REPLACEMENT_REFUND",
    "PRODUCT_PROBLEM",
    "PAYMENT_BILLING",
    "PRIME_MEMBERSHIP",
    "ACCOUNT_ACCESS_SECURITY",
    "GIFT_CARD_PROMOTION",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "DIGITAL_CONTENT",
    "DEVICE_TECHNICAL_SUPPORT",
    "WEBSITE_APP_TECHNICAL",
    "OTHER_NON_SUPPORT",
]

PATTERNS = {
    "DELIVERY_MISSING_OR_MISDELIVERED": [
        r"\bdelivered\b",
        r"\bpackage\b.*\bmissing\b",
        r"\bparcel\b.*\bmissing\b",
        r"\bnot received\b",
        r"\bnever received\b",
        r"\bwrong address\b",
        r"\bwrong house\b",
        r"\bneighbor\b",
        r"\bneighbour\b",
        r"\bmisdelivered\b",
    ],

    "DELIVERY_DELAY": [
        r"\blate\b",
        r"\bdelayed\b",
        r"\bdelay\b",
        r"\boverdue\b",
        r"\bnot arrived\b",
        r"\bhasn't arrived\b",
        r"\bhasnt arrived\b",
        r"\bstill waiting\b",
        r"\bdelivery date\b",
        r"\bdelivery time\b",
    ],

    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS": [
        r"\bdelivery attempt\b",
        r"\bfailed attempt\b",
        r"\bdriver\b",
        r"\bcourier\b",
        r"\bdelivery instructions\b",
        r"\bleave .* door\b",
        r"\bleave .* porch\b",
        r"\bdelivery slot\b",
        r"\bdelivery person\b",
    ],

    "RETURN_REPLACEMENT_REFUND": [
        r"\breturn\b",
        r"\brefund\b",
        r"\breplacement\b",
        r"\breplace\b",
        r"\bexchange\b",
        r"\breturn label\b",
        r"\breturning\b",
    ],

    "PRODUCT_PROBLEM": [
        r"\bdamaged\b",
        r"\bbroken\b",
        r"\bdefective\b",
        r"\bfaulty\b",
        r"\bwrong item\b",
        r"\bwrong product\b",
        r"\bmissing parts\b",
        r"\bdoesn't work\b",
        r"\bdoesnt work\b",
        r"\bnot working\b",
        r"\bpoor quality\b",
        r"\bquality\b",
        r"\bscratched\b",
    ],

    "PAYMENT_BILLING": [
        r"\bcharged\b",
        r"\bcharge\b",
        r"\bbilling\b",
        r"\bpayment\b",
        r"\bcredit card\b",
        r"\bdebit card\b",
        r"\bpaypal\b",
        r"\bamazon pay\b",
    ],

    "PRIME_MEMBERSHIP": [
        r"\bprime\b",
        r"\bmembership\b",
        r"\bprime member\b",
        r"\bprime membership\b",
        r"\bprime trial\b",
        r"\bprime renewal\b",
    ],

    "ACCOUNT_ACCESS_SECURITY": [
        r"\blogin\b",
        r"\blog in\b",
        r"\bsign in\b",
        r"\bsign-in\b",
        r"\bpassword\b",
        r"\baccount locked\b",
        r"\blocked out\b",
        r"\bhacked\b",
        r"\bsecurity\b",
        r"\b2fa\b",
        r"\btwo factor\b",
        r"\bverification code\b",
    ],

    "GIFT_CARD_PROMOTION": [
        r"\bgift card\b",
        r"\bgiftcard\b",
        r"\bcoupon\b",
        r"\bpromo\b",
        r"\bpromotion\b",
        r"\bdiscount\b",
        r"\bcashback\b",
        r"\bvoucher\b",
    ],

    "PRODUCT_AVAILABILITY_INFORMATION": [
        r"\bin stock\b",
        r"\bout of stock\b",
        r"\bavailable\b",
        r"\bavailability\b",
        r"\bprice\b",
        r"\bcost\b",
        r"\bhow much\b",
    ],

    "DIGITAL_CONTENT": [
        r"\bprime video\b",
        r"\bamazon video\b",
        r"\bamazon music\b",
        r"\bkindle book\b",
        r"\be[- ]?book\b",
        r"\bstreaming\b",
    ],

    "DEVICE_TECHNICAL_SUPPORT": [
        r"\bkindle\b",
        r"\becho\b",
        r"\balexa\b",
        r"\bfire tv\b",
        r"\bfire stick\b",
        r"\bfirestick\b",
        r"\btablet\b",
        r"\bdevice\b",
    ],

    "WEBSITE_APP_TECHNICAL": [
        r"\bapp\b",
        r"\bwebsite\b",
        r"\bsite\b",
        r"\bcheckout\b",
        r"\berror\b",
        r"\berror message\b",
        r"\bbutton\b",
        r"\bloading\b",
    ],

    "ORDER_STATUS_OR_CANCELLATION": [
        r"\border status\b",
        r"\bcancel\b",
        r"\bcancelled\b",
        r"\bcanceled\b",
        r"\bdispatch\b",
        r"\bshipped\b",
        r"\bshipment\b",
        r"\border\b.*\bstatus\b",
    ],
}


def predict_intent(text):

    text = str(text).lower()

    scores = {}

    for intent, patterns in PATTERNS.items():

        score = sum(
            bool(re.search(pattern, text))
            for pattern in patterns
        )

        scores[intent] = score

    best_intent = max(scores, key=scores.get)
    best_score = scores[best_intent]

    if best_score == 0:
        return "OTHER_NON_SUPPORT"

    tied = [
        intent
        for intent, score in scores.items()
        if score == best_score
    ]

    if len(tied) > 1:
        return "OTHER_NON_SUPPORT"

    return best_intent


# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

audit_eval = taxonomy_audit.copy()

audit_eval["heuristic_prediction"] = (
    audit_eval["first_customer_message"]
    .fillna("")
    .map(predict_intent)
)

print("=" * 80)
print("HEURISTIC PREDICTIONS CREATED")
print("=" * 80)

print("Examples:", len(audit_eval))
print(
    "Predicted labels:",
    audit_eval["heuristic_prediction"].nunique()
)

display(
    audit_eval[
        [
            "first_customer_message",
            "intent_label",
            "heuristic_prediction"
        ]
    ].head(20)
)

HEURISTIC PREDICTIONS CREATED
Examples: 150
Predicted labels: 15


,first_customer_message,intent_label,heuristic_prediction
0,@AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP,PRODUCT_PROBLEM,OTHER_NON_SUPPORT
1,@120533 Intéressants vos emballages... https://t.co/kMqmBDEt52,OTHER_NON_SUPPORT,OTHER_NON_SUPPORT
2,"You guys should really Google the word ""Guaranteed"" before giving me an arrival date. Just terrible this year. #Amazon @115821 #late #guess?",DELIVERY_DELAY,DELIVERY_DELAY
3,I had an issue with delivery. Ordered on 3rd. Product is still not delivered. No help from Amazon side. Just asking me to wait @115850,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED
4,"@115850 Hello Amazon team, I have ordered an item and got an email that it was delivered to me today. I am at home and didn't receive any product. When I try to call the agent number (8179260086) it is switched off. I am a Prime customer and your service is ridiculous. https://t.co/cibAXpH6jG",DELIVERY_MISSING_OR_MISDELIVERED,OTHER_NON_SUPPORT
5,"Dear @115828 @115821 My ""allegedly"" brand new copy of Tokyo Xanadu is missing something... https://t.co/APYKHYijZI",PRODUCT_PROBLEM,OTHER_NON_SUPPORT
6,"Can’t believe @115830’s response to me asking why my Xbox One X hasn’t been despatched yet. “If you’re not happy with it, cancel”",ORDER_STATUS_OR_CANCELLATION,ORDER_STATUS_OR_CANCELLATION
7,@115828 @115821 You ruined my https://t.co/uBSwBeyAjG shipping is a joke. Thanks.For real.Thanks. Made cancelling prime easy choice,DELIVERY_DELAY,PRIME_MEMBERSHIP
8,@AmazonHelp When I was adding mny there was no clearification that money cant be credited back to bank a/c I hve lost my mney I don't buy anythng frm u,PAYMENT_BILLING,OTHER_NON_SUPPORT
9,An update : It’s still Monday. \n\n#DarkerAsToldByChristian #Darker #ChristianGreysPov #ChristianGrey https://t.co/0EZOIk2Tq9,DELIVERY_DELAY,OTHER_NON_SUPPORT


EVALUATE SILVER-LABELING HEURISTIC

In [116]:
print("=" * 80)
print("SILVER-LABELING HEURISTIC EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Overall accuracy
# ------------------------------------------------------------

accuracy = (
    audit_eval["heuristic_prediction"]
    == audit_eval["intent_label"]
).mean()

print(
    f"\nAccuracy on 150 human-labelled examples: "
    f"{accuracy * 100:.2f}%"
)

# ------------------------------------------------------------
# 2. Confusion table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONFUSION TABLE")
print("=" * 80)

confusion = pd.crosstab(
    audit_eval["intent_label"],
    audit_eval["heuristic_prediction"],
    rownames=["Human label"],
    colnames=["Heuristic prediction"]
)

display(confusion)

# ------------------------------------------------------------
# 3. Per-intent performance
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PER-INTENT ACCURACY")
print("=" * 80)

per_intent = (
    audit_eval
    .assign(
        correct=lambda x:
        x["intent_label"]
        == x["heuristic_prediction"]
    )
    .groupby("intent_label")["correct"]
    .agg(["count", "mean"])
    .reset_index()
)

per_intent["accuracy_percentage"] = (
    per_intent["mean"] * 100
).round(2)

per_intent = per_intent.drop(columns=["mean"])

display(
    per_intent.sort_values(
        "accuracy_percentage"
    )
)

# ------------------------------------------------------------
# 4. Inspect mistakes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("HEURISTIC ERRORS")
print("=" * 80)

errors = audit_eval[
    audit_eval["intent_label"]
    != audit_eval["heuristic_prediction"]
][
    [
        "first_customer_message",
        "intent_label",
        "heuristic_prediction",
        "annotation_notes"
    ]
].copy()

print("Number of errors:", len(errors))

display(errors)

SILVER-LABELING HEURISTIC EVALUATION

Accuracy on 150 human-labelled examples: 36.67%

CONFUSION TABLE


Heuristic prediction,ACCOUNT_ACCESS_SECURITY,DELIVERY_ATTEMPT_OR_INSTRUCTIONS,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED,DEVICE_TECHNICAL_SUPPORT,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,ORDER_STATUS_OR_CANCELLATION,OTHER_NON_SUPPORT,PAYMENT_BILLING,PRIME_MEMBERSHIP,PRODUCT_AVAILABILITY_INFORMATION,PRODUCT_PROBLEM,RETURN_REPLACEMENT_REFUND,WEBSITE_APP_TECHNICAL
Human label,,,,,,,,,,,,,,,
ACCOUNT_ACCESS_SECURITY,1,0,0,0,0,0,0,0,1,0,1,0,0,0,2
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
DELIVERY_DELAY,0,1,3,6,0,0,0,0,19,0,8,1,0,0,0
DELIVERY_MISSING_OR_MISDELIVERED,0,0,0,7,0,0,0,0,5,0,1,0,0,0,0
DEVICE_TECHNICAL_SUPPORT,0,0,0,0,2,0,0,0,1,0,0,0,0,0,0
DIGITAL_CONTENT,0,0,0,0,0,1,0,0,3,0,0,1,0,0,1
GIFT_CARD_PROMOTION,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0
ORDER_STATUS_OR_CANCELLATION,0,0,1,0,0,0,0,3,3,0,0,0,0,0,0
OTHER_NON_SUPPORT,0,0,0,0,0,0,0,0,27,0,1,0,0,0,1



PER-INTENT ACCURACY


,intent_label,count,accuracy_percentage
11,PRODUCT_AVAILABILITY_INFORMATION,6,0.00
2,DELIVERY_DELAY,38,7.89
12,PRODUCT_PROBLEM,17,11.76
5,DIGITAL_CONTENT,6,16.67
0,ACCOUNT_ACCESS_SECURITY,5,20.00
9,PAYMENT_BILLING,5,20.00
13,RETURN_REPLACEMENT_REFUND,8,25.00
14,WEBSITE_APP_TECHNICAL,4,25.00
7,ORDER_STATUS_OR_CANCELLATION,7,42.86
6,GIFT_CARD_PROMOTION,2,50.00



HEURISTIC ERRORS
Number of errors: 95


,first_customer_message,intent_label,heuristic_prediction,annotation_notes
0,@AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP,PRODUCT_PROBLEM,OTHER_NON_SUPPORT,
3,I had an issue with delivery. Ordered on 3rd. Product is still not delivered. No help from Amazon side. Just asking me to wait @115850,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED,
4,"@115850 Hello Amazon team, I have ordered an item and got an email that it was delivered to me today. I am at home and didn't receive any product. When I try to call the agent number (8179260086) it is switched off. I am a Prime customer and your service is ridiculous. https://t.co/cibAXpH6jG",DELIVERY_MISSING_OR_MISDELIVERED,OTHER_NON_SUPPORT,
5,"Dear @115828 @115821 My ""allegedly"" brand new copy of Tokyo Xanadu is missing something... https://t.co/APYKHYijZI",PRODUCT_PROBLEM,OTHER_NON_SUPPORT,
7,@115828 @115821 You ruined my https://t.co/uBSwBeyAjG shipping is a joke. Thanks.For real.Thanks. Made cancelling prime easy choice,DELIVERY_DELAY,PRIME_MEMBERSHIP,
...,...,...,...,...
140,@116316 was nutzt einem AmazonPrime wenn man die Pakete nie pünktlich bekommt?! Erst 3-5 Tage später?! Danke an #Hermes!,DELIVERY_DELAY,OTHER_NON_SUPPORT,
145,"Don’t know why I bother ordering anything with @115821 anymore, they can never deliver on time 🙄",DELIVERY_DELAY,OTHER_NON_SUPPORT,
146,"@116875 ¿Qué hago, Amazon? Hice una devolución y @178440 no tiene idea de mi paquete. Selló la guía (aún la tengo) y dice que no lo he enviado.",RETURN_REPLACEMENT_REFUND,OTHER_NON_SUPPORT,
148,"@AmazonHelp sumthing wrong in my account, lost prime membership, no order history, cust service not taking my calls. Manisha Chandnani",ACCOUNT_ACCESS_SECURITY,PRIME_MEMBERSHIP,Prime/order-history disappearance is treated as account-state problem.


Weak-labeling baseline: A transparent keyword heuristic achieved only 36.67% accuracy on 150 manually reviewed examples, demonstrating that lexical rules were insufficient for reliable intent labeling.

LLM Environment Check

In [117]:
import os
import importlib.util
import sys

print("=" * 80)
print("LLM ENVIRONMENT CHECK")
print("=" * 80)

print("Python:", sys.version)

LLM ENVIRONMENT CHECK
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [118]:
packages = [
    "groq",
    "anthropic",
    "mistralai",
    "openai",
    "langchain",
    "langgraph",
    "sentence_transformers",
    "faiss",
]

print("\nInstalled packages:")
for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package:25s} : {'YES' if installed else 'NO'}")


Installed packages:
groq                      : NO
anthropic                 : NO
mistralai                 : NO
openai                    : YES
langchain                 : YES
langgraph                 : NO
sentence_transformers     : YES
faiss                     : YES


In [119]:
env_keys = [
    "GROQ_API_KEY",
    "ANTHROPIC_API_KEY",
    "MISTRAL_API_KEY",
    "OPENAI_API_KEY",
]

print("\nAPI key availability:")
for key in env_keys:
    exists = bool(os.getenv(key))
    print(f"{key:25s} : {'SET' if exists else 'NOT SET'}")


API key availability:
GROQ_API_KEY              : NOT SET
ANTHROPIC_API_KEY         : NOT SET
MISTRAL_API_KEY           : NOT SET
OPENAI_API_KEY            : NOT SET


In [120]:
from pathlib import Path

project_root = Path.cwd().parent

env_path = project_root / ".env"

print("\nProject root:")
print(project_root)

print("\n.env file exists:", env_path.exists())

if env_path.exists():
    print(".env location:", env_path)


Project root:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent

.env file exists: False


Development Data Check

In [121]:
import pandas as pd
from pathlib import Path

print("=" * 80)
print("DEVELOPMENT DATA CHECK")
print("=" * 80)

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

split_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits_v2.csv"
)

if not split_path.exists():
    raise FileNotFoundError(
        f"Expected split file not found:\n{split_path}"
    )

split_df = pd.read_csv(split_path)

print("Loaded:", split_path)
print("Rows:", f"{len(split_df):,}")

DEVELOPMENT DATA CHECK
Loaded: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits_v2.csv
Rows: 82,246


In [122]:
train_pool = split_df[
    (split_df["split"] == "train") &
    (~split_df["is_taxonomy_audit"])
].copy()

validation_pool = split_df[
    (split_df["split"] == "validation") &
    (~split_df["is_taxonomy_audit"])
].copy()

test_pool = split_df[
    split_df["split"] == "test"
].copy()

print("\nDevelopment pools:")
print(
    "Train:",
    f"{len(train_pool):,}"
)

print(
    "Validation:",
    f"{len(validation_pool):,}"
)

print(
    "Test:",
    f"{len(test_pool):,}"
)


Development pools:
Train: 65,678
Validation: 8,213
Test: 8,225


In [123]:
required = [
    "conversation_id",
    "first_customer_message",
    "conversation_text",
    "support_account",
    "split",
    "is_taxonomy_audit",
]

missing = [
    c for c in required
    if c not in split_df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )


In [124]:
assert train_pool["is_taxonomy_audit"].sum() == 0
assert validation_pool["is_taxonomy_audit"].sum() == 0

print("\n✓ Training pool excludes taxonomy audit.")
print("✓ Validation pool excludes taxonomy audit.")
print("✓ Test pool remains untouched.")


✓ Training pool excludes taxonomy audit.
✓ Validation pool excludes taxonomy audit.
✓ Test pool remains untouched.


In [125]:
PILOT_SIZE = 100

pilot = train_pool.sample(
    n=PILOT_SIZE,
    random_state=42
)

print(
    f"\nPilot sample prepared: {len(pilot)} conversations"
)

print("\nPilot intent-label input examples:")

display(
    pilot[
        [
            "conversation_id",
            "first_customer_message",
            "conversation_size"
        ]
    ].head(10)
)


Pilot sample prepared: 100 conversations

Pilot intent-label input examples:


,conversation_id,first_customer_message,conversation_size
65169,2448288,@115821 tracking says my shipment was delivered but it hasn’t arrived. What do I️ do?,5
66550,2493763,mandy finally made an amazon account 😳 sorry bank account,2
40625,1339616,ordered phone on @115850 twice money got deducted from acc &amp; twice amazon says payment failed. on retrying they charging more. Disgusting.,9
58868,2254646,"My @32611 pre-order of the 3CD/Blu-Ray Deluxe ""Automatic…"" re-issue was not delivered by @115821 and they aren't sure when it will arrive. :(",3
6418,231318,@AmazonHelp why u increase price on big sale day nd after giving discount make it older rate making fool of people,3
31281,1001077,@115821 this guy is tossing packages on 51st street btwn 1st/2nd ave https://t.co/HXAvTTJB0Z,2
56781,2144476,そういえばAmazon prime会員なので、今更ながら始めてprimeビデオ？を観てみた。いぬやしきの第一話,2
58413,2226591,@AmazonHelp Lots of confusion whether i m the winner on not. Still not confirmed. No upadate yet.,3
61801,2319945,"per la seconda volta mi arriva un articolo diverso da quello ordinato: mai più @120540, sapevatelo.....",4
79211,2907786,"I don't live close enough for 2 hour Amazon delivery but I get same day, aka the coffee I ordered this morning will be here in a few hours. HURRYYYY!",12


In [126]:
import sys
import importlib.metadata as md

print("=" * 80)
print("DEPENDENCY VERSION CHECK")
print("=" * 80)

packages = [
    "sentence-transformers",
    "transformers",
    "torch",
    "huggingface-hub",
    "tokenizers",
]

for package in packages:
    try:
        print(f"{package:25s}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package:25s}: NOT INSTALLED")

print("\nPython:", sys.version)

DEPENDENCY VERSION CHECK
sentence-transformers    : 5.1.0
transformers             : 4.55.0
torch                    : 2.4.1
huggingface-hub          : 0.34.4
tokenizers               : 0.21.4

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [127]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 80)
print("ZERO-SHOT SEMANTIC INTENT BASELINE")
print("=" * 80)


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


ZERO-SHOT SEMANTIC INTENT BASELINE


In [128]:
INTENT_DEFINITIONS = {
    "DELIVERY_DELAY":
        "Shipment is late, delayed, or misses the expected delivery window.",

    "DELIVERY_MISSING_OR_MISDELIVERED":
        "Package is marked delivered but not received, or delivered to the wrong location.",

    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS":
        "Delivery attempt behavior, driver behavior, delivery instructions, or delivery location or slot issues.",

    "ORDER_STATUS_OR_CANCELLATION":
        "Order lifecycle status, cancellation, dispatch, or preorder issues.",

    "RETURN_REPLACEMENT_REFUND":
        "Return, replacement, or refund process or status.",

    "PRODUCT_PROBLEM":
        "Product is damaged, defective, wrong, incomplete, used, counterfeit, or has a product-quality problem.",

    "PAYMENT_BILLING":
        "Payment method, card, unexpected charge, billing, or Amazon Pay issue.",

    "PRIME_MEMBERSHIP":
        "Prime signup, renewal, cancellation, membership status, or Prime benefits.",

    "ACCOUNT_ACCESS_SECURITY":
        "Login, password, locked account, hacked account, authentication, or security issue.",

    "GIFT_CARD_PROMOTION":
        "Gift card, coupon, promotion, cashback, or promotional-credit issue.",

    "PRODUCT_AVAILABILITY_INFORMATION":
        "Product stock, availability, pricing, shipping eligibility, or general product information.",

    "DIGITAL_CONTENT":
        "Prime Video, Amazon Music, Kindle content, or other digital-content access or availability.",

    "DEVICE_TECHNICAL_SUPPORT":
        "Kindle, Fire TV, Fire Stick, Echo, Alexa, or other Amazon-device technical issue.",

    "WEBSITE_APP_TECHNICAL":
        "Amazon website, mobile app, checkout, UI, form, or technical site or app error.",

    "OTHER_NON_SUPPORT":
        "Non-support or social content, insufficient information, generic complaints, marketing, or cases that do not fit another operational intent.",
}

FINAL_INTENTS = list(INTENT_DEFINITIONS.keys())

In [129]:
import os

# Remove invalid SSL certificate overrides
os.environ.pop("SSL_CERT_FILE", None)
os.environ.pop("REQUESTS_CA_BUNDLE", None)
os.environ.pop("CURL_CA_BUNDLE", None)

print("SSL_CERT_FILE:", os.environ.get("SSL_CERT_FILE"))
print("REQUESTS_CA_BUNDLE:", os.environ.get("REQUESTS_CA_BUNDLE"))
print("CURL_CA_BUNDLE:", os.environ.get("CURL_CA_BUNDLE"))

print("\n✓ Invalid certificate overrides removed.")

SSL_CERT_FILE: None
REQUESTS_CA_BUNDLE: None
CURL_CA_BUNDLE: None

✓ Invalid certificate overrides removed.


In [130]:
MODEL_NAME = "all-MiniLM-L6-v2"

print("\nLoading embedding model:")
print(MODEL_NAME)

model = SentenceTransformer(MODEL_NAME)

print("✓ Model loaded.")


Loading embedding model:
all-MiniLM-L6-v2
✓ Model loaded.


In [131]:
intent_names = list(INTENT_DEFINITIONS.keys())
intent_texts = list(INTENT_DEFINITIONS.values())

intent_embeddings = model.encode(
    intent_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(
    "\nIntent embeddings:",
    intent_embeddings.shape
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Intent embeddings: (15, 384)


In [132]:
#  Embed 150 HUMAN-LABELLED audit examples
audit_texts = (
    taxonomy_audit["first_customer_message"]
    .fillna("")
    .astype(str)
    .tolist()
)

audit_embeddings = model.encode(
    audit_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("✓ 150 audit examples embedded.")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✓ 150 audit examples embedded.


In [133]:
#Predict using cosine similarity
similarities = audit_embeddings @ intent_embeddings.T

prediction_indices = similarities.argmax(axis=1)

predictions = [
    intent_names[i]
    for i in prediction_indices
]

confidence = similarities[
    np.arange(len(similarities)),
    prediction_indices
]

In [134]:
# Evaluation dataframe
semantic_eval = taxonomy_audit.copy()

semantic_eval["semantic_prediction"] = predictions
semantic_eval["semantic_similarity"] = confidence

In [135]:
# Overall accuracy
accuracy = accuracy_score(
    semantic_eval["intent_label"],
    semantic_eval["semantic_prediction"]
)

print("\n" + "=" * 80)
print("ZERO-SHOT SEMANTIC BASELINE")
print("=" * 80)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")


ZERO-SHOT SEMANTIC BASELINE
Accuracy: 0.3067
Accuracy: 30.67%


In [136]:
print("\n" + "=" * 80)
print("PER-INTENT PERFORMANCE")
print("=" * 80)

report = classification_report(
    semantic_eval["intent_label"],
    semantic_eval["semantic_prediction"],
    labels=intent_names,
    output_dict=True,
    zero_division=0
)

per_intent = pd.DataFrame(report).T

display(
    per_intent[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)


PER-INTENT PERFORMANCE


,precision,recall,f1-score,support
DELIVERY_DELAY,0.857,0.474,0.610,38.000
DELIVERY_MISSING_OR_MISDELIVERED,0.333,0.615,0.432,13.000
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0.000,0.000,0.000,2.000
ORDER_STATUS_OR_CANCELLATION,0.250,0.286,0.267,7.000
RETURN_REPLACEMENT_REFUND,0.750,0.375,0.500,8.000
PRODUCT_PROBLEM,0.333,0.235,0.276,17.000
PAYMENT_BILLING,0.079,0.600,0.140,5.000
PRIME_MEMBERSHIP,0.667,0.400,0.500,5.000
ACCOUNT_ACCESS_SECURITY,0.000,0.000,0.000,5.000
GIFT_CARD_PROMOTION,0.000,0.000,0.000,2.000


In [137]:
# Lowest-confidence examples
print("\n" + "=" * 80)
print("LOWEST-CONFIDENCE EXAMPLES")
print("=" * 80)

low_confidence = (
    semantic_eval
    .sort_values("semantic_similarity")
    [
        [
            "first_customer_message",
            "intent_label",
            "semantic_prediction",
            "semantic_similarity"
        ]
    ]
    .head(20)
)

display(low_confidence)


LOWEST-CONFIDENCE EXAMPLES


,first_customer_message,intent_label,semantic_prediction,semantic_similarity
62,@116313 アマゾンのサイトのフォントサイズですが、大きくなりましたか？それとも、当PC（ちなみにChrome使用）だけでしょうか？,WEBSITE_APP_TECHNICAL,ACCOUNT_ACCESS_SECURITY,0.055659
22,アマゾンプライムでアニメを見る日々が始まる……\nネットで探す必要がなくなった…\n喜びすぎてチキンになるわ… https://t.co/ow0gXbNsQ0,OTHER_NON_SUPPORT,WEBSITE_APP_TECHNICAL,0.058308
24,Désormais chez @120533 : 1 jour ouvré = 3 jours ouvrés https://t.co/IQwOxCXPWT,DELIVERY_DELAY,PRODUCT_PROBLEM,0.060634
125,@116928 segun esto me desbloquearon mi cuenta pero al hacer mi pedido no me deja!!! No entiendo entonces 😡😡,ACCOUNT_ACCESS_SECURITY,WEBSITE_APP_TECHNICAL,0.071616
74,知らん間にAmazonプライム会員にさせられてたので有効活用してやろうと思うんやけど送料無料以外に何が出来るんや,PRIME_MEMBERSHIP,PRODUCT_PROBLEM,0.075309
95,"@AmazonHelp Ich bin etwas verwirrt. Wann kommt das Paket denn? Gestern, heute, morgen, Montag? https://t.co/nYRbEmTteO",DELIVERY_DELAY,DELIVERY_DELAY,0.082451
141,EU TÔ TREMENDO. @117086 EU TE AMO MUITO. https://t.co/Br5xFHHmTz,OTHER_NON_SUPPORT,PRODUCT_PROBLEM,0.090902
58,マジでアマゾンミュージック様天才すぎっからみんなプライムはいれ,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,0.092713
52,げ、いつの間にかAmazonプライム会員になっていた。いつだよ,PRIME_MEMBERSHIP,PRODUCT_PROBLEM,0.101404
38,せっかくプライム会員なので30日間だけアマゾンミュージコお試しで〜。 https://t.co/MVdp7mPd3T,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,0.102542


In [138]:
# Save results
semantic_path = (
    PROCESSED_DIR /
    "semantic_intent_baseline_150.csv"
)

semantic_eval.to_csv(
    semantic_path,
    index=False
)

print("\n✓ Saved:")
print(semantic_path)


✓ Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\semantic_intent_baseline_150.csv


In [139]:
# ============================================================
# STEP 60D — SEMANTIC INTENT EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report

# ------------------------------------------------------------
# 1. Locked taxonomy
# ------------------------------------------------------------

INTENT_DEFINITIONS = {
    "DELIVERY_DELAY":
        "Shipment is late, delayed, or misses the expected delivery window.",

    "DELIVERY_MISSING_OR_MISDELIVERED":
        "Package is marked delivered but not received, or delivered to the wrong location.",

    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS":
        "Delivery attempt behavior, driver behavior, delivery instructions, or delivery location or slot issues.",

    "ORDER_STATUS_OR_CANCELLATION":
        "Order lifecycle status, cancellation, dispatch, or preorder issues.",

    "RETURN_REPLACEMENT_REFUND":
        "Return, replacement, or refund process or status.",

    "PRODUCT_PROBLEM":
        "Product is damaged, defective, wrong, incomplete, used, counterfeit, or has a product-quality problem.",

    "PAYMENT_BILLING":
        "Payment method, card, unexpected charge, billing, or Amazon Pay issue.",

    "PRIME_MEMBERSHIP":
        "Prime signup, renewal, cancellation, membership status, or Prime benefits.",

    "ACCOUNT_ACCESS_SECURITY":
        "Login, password, locked account, hacked account, authentication, or security issue.",

    "GIFT_CARD_PROMOTION":
        "Gift card, coupon, promotion, cashback, or promotional-credit issue.",

    "PRODUCT_AVAILABILITY_INFORMATION":
        "Product stock, availability, pricing, shipping eligibility, or general product information.",

    "DIGITAL_CONTENT":
        "Prime Video, Amazon Music, Kindle content, or other digital-content access or availability.",

    "DEVICE_TECHNICAL_SUPPORT":
        "Kindle, Fire TV, Fire Stick, Echo, Alexa, or other Amazon-device technical issue.",

    "WEBSITE_APP_TECHNICAL":
        "Amazon website, mobile app, checkout, UI, form, or technical site or app error.",

    "OTHER_NON_SUPPORT":
        "Non-support or social content, insufficient information, generic complaints, marketing, or cases that do not fit another operational intent.",
}

intent_names = list(INTENT_DEFINITIONS.keys())
intent_texts = list(INTENT_DEFINITIONS.values())

# ------------------------------------------------------------
# 2. Embed intent descriptions
# ------------------------------------------------------------

intent_embeddings = model.encode(
    intent_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("✓ Intent descriptions embedded.")

# ------------------------------------------------------------
# 3. Embed 150 HUMAN-LABELLED audit examples
# ------------------------------------------------------------

audit_texts = (
    taxonomy_audit["first_customer_message"]
    .fillna("")
    .astype(str)
    .tolist()
)

audit_embeddings = model.encode(
    audit_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("✓ 150 audit examples embedded.")

# ------------------------------------------------------------
# 4. Predict using cosine similarity
# ------------------------------------------------------------

similarities = audit_embeddings @ intent_embeddings.T

prediction_indices = similarities.argmax(axis=1)

predictions = [
    intent_names[i]
    for i in prediction_indices
]

confidence = similarities[
    np.arange(len(similarities)),
    prediction_indices
]

# ------------------------------------------------------------
# 5. Evaluation dataframe
# ------------------------------------------------------------

semantic_eval = taxonomy_audit.copy()

semantic_eval["semantic_prediction"] = predictions
semantic_eval["semantic_similarity"] = confidence

# ------------------------------------------------------------
# 6. Overall accuracy
# ------------------------------------------------------------

accuracy = accuracy_score(
    semantic_eval["intent_label"],
    semantic_eval["semantic_prediction"]
)

print("\n" + "=" * 80)
print("ZERO-SHOT SEMANTIC BASELINE")
print("=" * 80)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

# ------------------------------------------------------------
# 7. Classification report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PER-INTENT PERFORMANCE")
print("=" * 80)

report = classification_report(
    semantic_eval["intent_label"],
    semantic_eval["semantic_prediction"],
    labels=intent_names,
    output_dict=True,
    zero_division=0
)

per_intent = pd.DataFrame(report).T

display(
    per_intent[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

# ------------------------------------------------------------
# 8. Lowest-confidence examples
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOWEST-CONFIDENCE EXAMPLES")
print("=" * 80)

low_confidence = (
    semantic_eval
    .sort_values("semantic_similarity")
    [
        [
            "first_customer_message",
            "intent_label",
            "semantic_prediction",
            "semantic_similarity"
        ]
    ]
    .head(20)
)

display(low_confidence)

# ------------------------------------------------------------
# 9. Save result
# ------------------------------------------------------------

semantic_path = (
    PROCESSED_DIR /
    "semantic_intent_baseline_150.csv"
)

semantic_eval.to_csv(
    semantic_path,
    index=False
)

print("\n✓ Saved:")
print(semantic_path)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Intent descriptions embedded.


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✓ 150 audit examples embedded.

ZERO-SHOT SEMANTIC BASELINE
Accuracy: 0.3067
Accuracy: 30.67%

PER-INTENT PERFORMANCE


,precision,recall,f1-score,support
DELIVERY_DELAY,0.857,0.474,0.610,38.000
DELIVERY_MISSING_OR_MISDELIVERED,0.333,0.615,0.432,13.000
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0.000,0.000,0.000,2.000
ORDER_STATUS_OR_CANCELLATION,0.250,0.286,0.267,7.000
RETURN_REPLACEMENT_REFUND,0.750,0.375,0.500,8.000
PRODUCT_PROBLEM,0.333,0.235,0.276,17.000
PAYMENT_BILLING,0.079,0.600,0.140,5.000
PRIME_MEMBERSHIP,0.667,0.400,0.500,5.000
ACCOUNT_ACCESS_SECURITY,0.000,0.000,0.000,5.000
GIFT_CARD_PROMOTION,0.000,0.000,0.000,2.000



LOWEST-CONFIDENCE EXAMPLES


,first_customer_message,intent_label,semantic_prediction,semantic_similarity
62,@116313 アマゾンのサイトのフォントサイズですが、大きくなりましたか？それとも、当PC（ちなみにChrome使用）だけでしょうか？,WEBSITE_APP_TECHNICAL,ACCOUNT_ACCESS_SECURITY,0.055659
22,アマゾンプライムでアニメを見る日々が始まる……\nネットで探す必要がなくなった…\n喜びすぎてチキンになるわ… https://t.co/ow0gXbNsQ0,OTHER_NON_SUPPORT,WEBSITE_APP_TECHNICAL,0.058308
24,Désormais chez @120533 : 1 jour ouvré = 3 jours ouvrés https://t.co/IQwOxCXPWT,DELIVERY_DELAY,PRODUCT_PROBLEM,0.060634
125,@116928 segun esto me desbloquearon mi cuenta pero al hacer mi pedido no me deja!!! No entiendo entonces 😡😡,ACCOUNT_ACCESS_SECURITY,WEBSITE_APP_TECHNICAL,0.071616
74,知らん間にAmazonプライム会員にさせられてたので有効活用してやろうと思うんやけど送料無料以外に何が出来るんや,PRIME_MEMBERSHIP,PRODUCT_PROBLEM,0.075309
95,"@AmazonHelp Ich bin etwas verwirrt. Wann kommt das Paket denn? Gestern, heute, morgen, Montag? https://t.co/nYRbEmTteO",DELIVERY_DELAY,DELIVERY_DELAY,0.082451
141,EU TÔ TREMENDO. @117086 EU TE AMO MUITO. https://t.co/Br5xFHHmTz,OTHER_NON_SUPPORT,PRODUCT_PROBLEM,0.090902
58,マジでアマゾンミュージック様天才すぎっからみんなプライムはいれ,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,0.092713
52,げ、いつの間にかAmazonプライム会員になっていた。いつだよ,PRIME_MEMBERSHIP,PRODUCT_PROBLEM,0.101404
38,せっかくプライム会員なので30日間だけアマゾンミュージコお試しで〜。 https://t.co/MVdp7mPd3T,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,0.102542



✓ Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\semantic_intent_baseline_150.csv


Zero-shot semantic matching is insufficient for operational intent classification, achieving 30.7% accuracy and 27.6% macro-F1 on the 150-example human-labelled audit. This motivated a supervised classifier rather than relying on intent-description similarity alone.

TF-IDF + LOGISTIC REGRESSION BASELINE

In [140]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [141]:
print("=" * 80)
print("TF-IDF + LOGISTIC REGRESSION BASELINE")
print("=" * 80)

# Prepare data
X = (
    taxonomy_audit["first_customer_message"]
    .fillna("")
    .astype(str)
)

y = taxonomy_audit["intent_label"].astype(str)

print("\nExamples:", len(X))
print("Intents:", y.nunique())

print("\nClass distribution:")
print(y.value_counts())

TF-IDF + LOGISTIC REGRESSION BASELINE

Examples: 150
Intents: 15

Class distribution:
intent_label
DELIVERY_DELAY                      38
OTHER_NON_SUPPORT                   29
PRODUCT_PROBLEM                     17
DELIVERY_MISSING_OR_MISDELIVERED    13
RETURN_REPLACEMENT_REFUND            8
ORDER_STATUS_OR_CANCELLATION         7
PRODUCT_AVAILABILITY_INFORMATION     6
DIGITAL_CONTENT                      6
PAYMENT_BILLING                      5
ACCOUNT_ACCESS_SECURITY              5
PRIME_MEMBERSHIP                     5
WEBSITE_APP_TECHNICAL                4
DEVICE_TECHNICAL_SUPPORT             3
GIFT_CARD_PROMOTION                  2
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     2
Name: count, dtype: int64


In [142]:
# Pipeline
tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

In [143]:
# stratified cross-validation
min_class_count = y.value_counts().min()

n_splits = min(5, min_class_count)

print(f"\nUsing {n_splits}-fold stratified cross-validation.")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

predictions = cross_val_predict(
    tfidf_lr,
    X,
    y,
    cv=cv
)


Using 2-fold stratified cross-validation.


In [144]:
accuracy = accuracy_score(y, predictions)

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Cross-validation accuracy   : {accuracy:.4f}")
print(f"Cross-validation accuracy(%): {accuracy * 100:.2f}%")


RESULT
Cross-validation accuracy   : 0.3133
Cross-validation accuracy(%): 31.33%


In [145]:
print("\n" + "=" * 80)
print("PER-INTENT PERFORMANCE")
print("=" * 80)

report = classification_report(
    y,
    predictions,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

display(
    report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)


PER-INTENT PERFORMANCE


,precision,recall,f1-score,support
ACCOUNT_ACCESS_SECURITY,0.000,0.000,0.000,5.000
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0.000,0.000,0.000,2.000
DELIVERY_DELAY,0.306,0.395,0.345,38.000
DELIVERY_MISSING_OR_MISDELIVERED,0.148,0.308,0.200,13.000
DEVICE_TECHNICAL_SUPPORT,0.000,0.000,0.000,3.000
DIGITAL_CONTENT,0.000,0.000,0.000,6.000
GIFT_CARD_PROMOTION,0.000,0.000,0.000,2.000
ORDER_STATUS_OR_CANCELLATION,1.000,0.714,0.833,7.000
OTHER_NON_SUPPORT,0.348,0.552,0.427,29.000
PAYMENT_BILLING,0.000,0.000,0.000,5.000


In [146]:

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

labels = sorted(y.unique())

cm = confusion_matrix(
    y,
    predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

display(cm_df)


CONFUSION MATRIX


,ACCOUNT_ACCESS_SECURITY,DELIVERY_ATTEMPT_OR_INSTRUCTIONS,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED,DEVICE_TECHNICAL_SUPPORT,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,ORDER_STATUS_OR_CANCELLATION,OTHER_NON_SUPPORT,PAYMENT_BILLING,PRIME_MEMBERSHIP,PRODUCT_AVAILABILITY_INFORMATION,PRODUCT_PROBLEM,RETURN_REPLACEMENT_REFUND,WEBSITE_APP_TECHNICAL
ACCOUNT_ACCESS_SECURITY,0,0,3,1,0,0,0,0,0,0,0,0,0,1,0
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0
DELIVERY_DELAY,1,0,15,9,0,0,0,0,9,1,0,0,2,1,0
DELIVERY_MISSING_OR_MISDELIVERED,0,0,8,4,0,0,0,0,0,0,0,0,1,0,0
DEVICE_TECHNICAL_SUPPORT,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0
DIGITAL_CONTENT,0,0,1,1,0,0,0,0,3,0,0,0,1,0,0
GIFT_CARD_PROMOTION,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
ORDER_STATUS_OR_CANCELLATION,0,0,1,0,0,0,0,5,1,0,0,0,0,0,0
OTHER_NON_SUPPORT,0,0,5,3,0,1,0,0,16,2,0,1,1,0,0
PAYMENT_BILLING,0,0,2,1,0,0,0,0,2,0,0,0,0,0,0


In [147]:
errors = taxonomy_audit.copy()

errors["tfidf_prediction"] = predictions

errors = errors[
    errors["intent_label"] != errors["tfidf_prediction"]
][
    [
        "first_customer_message",
        "intent_label",
        "tfidf_prediction"
    ]
]

print("\n" + "=" * 80)
print("MISCLASSIFIED EXAMPLES")
print("=" * 80)

display(errors.head(30))


MISCLASSIFIED EXAMPLES


,first_customer_message,intent_label,tfidf_prediction
0,@AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP,PRODUCT_PROBLEM,OTHER_NON_SUPPORT
3,I had an issue with delivery. Ordered on 3rd. Product is still not delivered. No help from Amazon side. Just asking me to wait @115850,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED
5,"Dear @115828 @115821 My ""allegedly"" brand new copy of Tokyo Xanadu is missing something... https://t.co/APYKHYijZI",PRODUCT_PROBLEM,OTHER_NON_SUPPORT
7,@115828 @115821 You ruined my https://t.co/uBSwBeyAjG shipping is a joke. Thanks.For real.Thanks. Made cancelling prime easy choice,DELIVERY_DELAY,OTHER_NON_SUPPORT
8,@AmazonHelp When I was adding mny there was no clearification that money cant be credited back to bank a/c I hve lost my mney I don't buy anythng frm u,PAYMENT_BILLING,DELIVERY_DELAY
9,An update : It’s still Monday. \n\n#DarkerAsToldByChristian #Darker #ChristianGreysPov #ChristianGrey https://t.co/0EZOIk2Tq9,DELIVERY_DELAY,OTHER_NON_SUPPORT
10,@AmazonHelp There's no option of changing the Registered Mobile Number in your App. Need to change the number urgently. Pl help.,ACCOUNT_ACCESS_SECURITY,RETURN_REPLACEMENT_REFUND
12,@AmazonHelp I can’t unsubscribe to this. It just says “coming soon” on the page that’s loaded. I really don’t want daily @144771 emails. 🤢 https://t.co/XW16sJlszK,WEBSITE_APP_TECHNICAL,OTHER_NON_SUPPORT
13,Canceling my prime membership because @115821 has finally wasted all of my patience. Corporate America 🙄🙄🙄,PRIME_MEMBERSHIP,DELIVERY_DELAY
14,Hey @115830 please pay my seller funds which has been held for over 115 days! __email__ https://t.co/Iua8OUsUfU,OTHER_NON_SUPPORT,DELIVERY_DELAY


In [148]:
# train baseline model on 1150 examples
tfidf_lr.fit(X, y)

print("\n✓ TF-IDF + Logistic Regression baseline trained.")


✓ TF-IDF + Logistic Regression baseline trained.


In [149]:
# ============================================================
# STEP 61 — TF-IDF + LOGISTIC REGRESSION BASELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("=" * 80)
print("TF-IDF + LOGISTIC REGRESSION BASELINE")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------

X = (
    taxonomy_audit["first_customer_message"]
    .fillna("")
    .astype(str)
)

y = taxonomy_audit["intent_label"].astype(str)

print("\nExamples:", len(X))
print("Intents:", y.nunique())

print("\nClass distribution:")
print(y.value_counts())

# ------------------------------------------------------------
# 2. Pipeline
# ------------------------------------------------------------

tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

# ------------------------------------------------------------
# 3. Stratified cross-validation
# ------------------------------------------------------------

min_class_count = y.value_counts().min()

n_splits = min(5, min_class_count)

print(f"\nUsing {n_splits}-fold stratified cross-validation.")

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

predictions = cross_val_predict(
    tfidf_lr,
    X,
    y,
    cv=cv
)

# ------------------------------------------------------------
# 4. Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y, predictions)

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Cross-validation accuracy: {accuracy:.4f}")
print(f"Cross-validation accuracy: {accuracy * 100:.2f}%")

# ------------------------------------------------------------
# 5. Classification report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PER-INTENT PERFORMANCE")
print("=" * 80)

report = classification_report(
    y,
    predictions,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

display(
    report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

# ------------------------------------------------------------
# 6. Confusion matrix
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

labels = sorted(y.unique())

cm = confusion_matrix(
    y,
    predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

display(cm_df)

# ------------------------------------------------------------
# 7. Misclassified examples
# ------------------------------------------------------------

errors = taxonomy_audit.copy()

errors["tfidf_prediction"] = predictions

errors = errors[
    errors["intent_label"] != errors["tfidf_prediction"]
][
    [
        "first_customer_message",
        "intent_label",
        "tfidf_prediction"
    ]
]

print("\n" + "=" * 80)
print("MISCLASSIFIED EXAMPLES")
print("=" * 80)

display(errors.head(30))

# ------------------------------------------------------------
# 8. Train final baseline model on the 150 examples
# ------------------------------------------------------------

tfidf_lr.fit(X, y)

print("\n✓ TF-IDF + Logistic Regression baseline trained.")

TF-IDF + LOGISTIC REGRESSION BASELINE

Examples: 150
Intents: 15

Class distribution:
intent_label
DELIVERY_DELAY                      38
OTHER_NON_SUPPORT                   29
PRODUCT_PROBLEM                     17
DELIVERY_MISSING_OR_MISDELIVERED    13
RETURN_REPLACEMENT_REFUND            8
ORDER_STATUS_OR_CANCELLATION         7
PRODUCT_AVAILABILITY_INFORMATION     6
DIGITAL_CONTENT                      6
PAYMENT_BILLING                      5
ACCOUNT_ACCESS_SECURITY              5
PRIME_MEMBERSHIP                     5
WEBSITE_APP_TECHNICAL                4
DEVICE_TECHNICAL_SUPPORT             3
GIFT_CARD_PROMOTION                  2
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     2
Name: count, dtype: int64

Using 2-fold stratified cross-validation.

RESULT
Cross-validation accuracy: 0.3133
Cross-validation accuracy: 31.33%

PER-INTENT PERFORMANCE


,precision,recall,f1-score,support
ACCOUNT_ACCESS_SECURITY,0.000,0.000,0.000,5.000
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0.000,0.000,0.000,2.000
DELIVERY_DELAY,0.306,0.395,0.345,38.000
DELIVERY_MISSING_OR_MISDELIVERED,0.148,0.308,0.200,13.000
DEVICE_TECHNICAL_SUPPORT,0.000,0.000,0.000,3.000
DIGITAL_CONTENT,0.000,0.000,0.000,6.000
GIFT_CARD_PROMOTION,0.000,0.000,0.000,2.000
ORDER_STATUS_OR_CANCELLATION,1.000,0.714,0.833,7.000
OTHER_NON_SUPPORT,0.348,0.552,0.427,29.000
PAYMENT_BILLING,0.000,0.000,0.000,5.000



CONFUSION MATRIX


,ACCOUNT_ACCESS_SECURITY,DELIVERY_ATTEMPT_OR_INSTRUCTIONS,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED,DEVICE_TECHNICAL_SUPPORT,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,ORDER_STATUS_OR_CANCELLATION,OTHER_NON_SUPPORT,PAYMENT_BILLING,PRIME_MEMBERSHIP,PRODUCT_AVAILABILITY_INFORMATION,PRODUCT_PROBLEM,RETURN_REPLACEMENT_REFUND,WEBSITE_APP_TECHNICAL
ACCOUNT_ACCESS_SECURITY,0,0,3,1,0,0,0,0,0,0,0,0,0,1,0
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0
DELIVERY_DELAY,1,0,15,9,0,0,0,0,9,1,0,0,2,1,0
DELIVERY_MISSING_OR_MISDELIVERED,0,0,8,4,0,0,0,0,0,0,0,0,1,0,0
DEVICE_TECHNICAL_SUPPORT,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0
DIGITAL_CONTENT,0,0,1,1,0,0,0,0,3,0,0,0,1,0,0
GIFT_CARD_PROMOTION,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
ORDER_STATUS_OR_CANCELLATION,0,0,1,0,0,0,0,5,1,0,0,0,0,0,0
OTHER_NON_SUPPORT,0,0,5,3,0,1,0,0,16,2,0,1,1,0,0
PAYMENT_BILLING,0,0,2,1,0,0,0,0,2,0,0,0,0,0,0



MISCLASSIFIED EXAMPLES


,first_customer_message,intent_label,tfidf_prediction
0,@AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP,PRODUCT_PROBLEM,OTHER_NON_SUPPORT
3,I had an issue with delivery. Ordered on 3rd. Product is still not delivered. No help from Amazon side. Just asking me to wait @115850,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED
5,"Dear @115828 @115821 My ""allegedly"" brand new copy of Tokyo Xanadu is missing something... https://t.co/APYKHYijZI",PRODUCT_PROBLEM,OTHER_NON_SUPPORT
7,@115828 @115821 You ruined my https://t.co/uBSwBeyAjG shipping is a joke. Thanks.For real.Thanks. Made cancelling prime easy choice,DELIVERY_DELAY,OTHER_NON_SUPPORT
8,@AmazonHelp When I was adding mny there was no clearification that money cant be credited back to bank a/c I hve lost my mney I don't buy anythng frm u,PAYMENT_BILLING,DELIVERY_DELAY
9,An update : It’s still Monday. \n\n#DarkerAsToldByChristian #Darker #ChristianGreysPov #ChristianGrey https://t.co/0EZOIk2Tq9,DELIVERY_DELAY,OTHER_NON_SUPPORT
10,@AmazonHelp There's no option of changing the Registered Mobile Number in your App. Need to change the number urgently. Pl help.,ACCOUNT_ACCESS_SECURITY,RETURN_REPLACEMENT_REFUND
12,@AmazonHelp I can’t unsubscribe to this. It just says “coming soon” on the page that’s loaded. I really don’t want daily @144771 emails. 🤢 https://t.co/XW16sJlszK,WEBSITE_APP_TECHNICAL,OTHER_NON_SUPPORT
13,Canceling my prime membership because @115821 has finally wasted all of my patience. Corporate America 🙄🙄🙄,PRIME_MEMBERSHIP,DELIVERY_DELAY
14,Hey @115830 please pay my seller funds which has been held for over 115 days! __email__ https://t.co/Iua8OUsUfU,OTHER_NON_SUPPORT,DELIVERY_DELAY



✓ TF-IDF + Logistic Regression baseline trained.


| Baseline | Accuracy | Macro F1 |
|---|---|---|
| Zero-shot semantic similarity | 30.7% | 27.6% |
| TF-IDF + Logistic Regression | 31.3% | 18.2% |

Majority Class Baseline

In [150]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print("=" * 80)
print("MAJORITY-CLASS BASELINE")
print("=" * 80)

# Ground-truth labels from the 150 human-labelled audit
y_true = taxonomy_audit["intent_label"].astype(str)

# Most frequent class
majority_class = y_true.value_counts().idxmax()
majority_count = y_true.value_counts().max()

# Predict the same class for every example
majority_predictions = np.repeat(
    majority_class,
    len(y_true)
)

accuracy = accuracy_score(
    y_true,
    majority_predictions
)

print(f"\nMajority class: {majority_class}")
print(f"Majority examples: {majority_count}/{len(y_true)}")

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

report = classification_report(
    y_true,
    majority_predictions,
    output_dict=True,
    zero_division=0
)

majority_report = pd.DataFrame(report).T

display(
    majority_report[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

print("\n✓ Majority baseline complete.")

MAJORITY-CLASS BASELINE

Majority class: DELIVERY_DELAY
Majority examples: 38/150

Accuracy: 0.2533
Accuracy: 25.33%

CLASSIFICATION REPORT


,precision,recall,f1-score,support
ACCOUNT_ACCESS_SECURITY,0.000,0.000,0.000,5.000
DELIVERY_ATTEMPT_OR_INSTRUCTIONS,0.000,0.000,0.000,2.000
DELIVERY_DELAY,0.253,1.000,0.404,38.000
DELIVERY_MISSING_OR_MISDELIVERED,0.000,0.000,0.000,13.000
DEVICE_TECHNICAL_SUPPORT,0.000,0.000,0.000,3.000
DIGITAL_CONTENT,0.000,0.000,0.000,6.000
GIFT_CARD_PROMOTION,0.000,0.000,0.000,2.000
ORDER_STATUS_OR_CANCELLATION,0.000,0.000,0.000,7.000
OTHER_NON_SUPPORT,0.000,0.000,0.000,29.000
PAYMENT_BILLING,0.000,0.000,0.000,5.000



✓ Majority baseline complete.


| Baseline | Accuracy | Macro F1 |
|---|---|---|
| Majority class | 25.33% | 2.7% |
| Zero-shot semantic similarity | 30.7% | 27.6% |
| TF-IDF + Logistic Regression | 31.3% | 18.2% |

GOLDEN/AUDIT LABEL DISTRIBUTION BY CONVERSATION SIZE

In [151]:
print("=" * 80)
print("TAXONOMY AUDIT DISTRIBUTION BY CONVERSATION SIZE")
print("=" * 80)

distribution = pd.crosstab(
    taxonomy_audit["size_bucket"],
    taxonomy_audit["intent_label"]
)

display(distribution)

print("\n" + "=" * 80)
print("ROW-WISE PERCENTAGES")
print("=" * 80)

distribution_pct = pd.crosstab(
    taxonomy_audit["size_bucket"],
    taxonomy_audit["intent_label"],
    normalize="index"
) * 100

display(
    distribution_pct.round(1)
)

print("\n" + "=" * 80)
print("INTENT COUNTS")
print("=" * 80)

intent_counts = (
    taxonomy_audit["intent_label"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

intent_counts["percentage"] = (
    intent_counts["count"] / len(taxonomy_audit) * 100
)

display(intent_counts.round(2))

print("\n✓ Distribution audit complete.")

TAXONOMY AUDIT DISTRIBUTION BY CONVERSATION SIZE


intent_label,ACCOUNT_ACCESS_SECURITY,DELIVERY_ATTEMPT_OR_INSTRUCTIONS,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED,DEVICE_TECHNICAL_SUPPORT,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,ORDER_STATUS_OR_CANCELLATION,OTHER_NON_SUPPORT,PAYMENT_BILLING,PRIME_MEMBERSHIP,PRODUCT_AVAILABILITY_INFORMATION,PRODUCT_PROBLEM,RETURN_REPLACEMENT_REFUND,WEBSITE_APP_TECHNICAL
size_bucket,,,,,,,,,,,,,,,
2_tweets,0,0,7,4,1,2,0,1,14,0,2,0,4,0,3
3_to_4_tweets,1,0,9,1,0,1,1,1,7,2,2,2,7,3,0
5_to_8_tweets,3,0,12,6,1,0,0,3,2,1,1,2,1,4,1
9_plus_tweets,1,2,10,2,1,3,1,2,6,2,0,2,5,1,0



ROW-WISE PERCENTAGES


intent_label,ACCOUNT_ACCESS_SECURITY,DELIVERY_ATTEMPT_OR_INSTRUCTIONS,DELIVERY_DELAY,DELIVERY_MISSING_OR_MISDELIVERED,DEVICE_TECHNICAL_SUPPORT,DIGITAL_CONTENT,GIFT_CARD_PROMOTION,ORDER_STATUS_OR_CANCELLATION,OTHER_NON_SUPPORT,PAYMENT_BILLING,PRIME_MEMBERSHIP,PRODUCT_AVAILABILITY_INFORMATION,PRODUCT_PROBLEM,RETURN_REPLACEMENT_REFUND,WEBSITE_APP_TECHNICAL
size_bucket,,,,,,,,,,,,,,,
2_tweets,0.0,0.0,18.4,10.5,2.6,5.3,0.0,2.6,36.8,0.0,5.3,0.0,10.5,0.0,7.9
3_to_4_tweets,2.7,0.0,24.3,2.7,0.0,2.7,2.7,2.7,18.9,5.4,5.4,5.4,18.9,8.1,0.0
5_to_8_tweets,8.1,0.0,32.4,16.2,2.7,0.0,0.0,8.1,5.4,2.7,2.7,5.4,2.7,10.8,2.7
9_plus_tweets,2.6,5.3,26.3,5.3,2.6,7.9,2.6,5.3,15.8,5.3,0.0,5.3,13.2,2.6,0.0



INTENT COUNTS


,intent,count,percentage
0,DELIVERY_DELAY,38,25.33
1,OTHER_NON_SUPPORT,29,19.33
2,PRODUCT_PROBLEM,17,11.33
3,DELIVERY_MISSING_OR_MISDELIVERED,13,8.67
4,RETURN_REPLACEMENT_REFUND,8,5.33
5,ORDER_STATUS_OR_CANCELLATION,7,4.67
6,PRODUCT_AVAILABILITY_INFORMATION,6,4.00
7,DIGITAL_CONTENT,6,4.00
8,PAYMENT_BILLING,5,3.33
9,ACCOUNT_ACCESS_SECURITY,5,3.33



✓ Distribution audit complete.


CREATE STRATIFIED TRAINING LABELING POOL

In [152]:
from pathlib import Path
import pandas as pd

print("=" * 80)
print("RELOADING AMAZONHELP CONVERSATION SPLITS")
print("=" * 80)

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

split_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits_v2.csv"
)

print("\nFile:")
print(split_path)

if not split_path.exists():
    raise FileNotFoundError(
        f"Split file not found:\n{split_path}"
    )

amazonhelp_conversation_splits_v2 = pd.read_csv(
    split_path
)

print("\n✓ File loaded.")
print("Rows:", len(amazonhelp_conversation_splits_v2))
print(
    "Columns:",
    list(amazonhelp_conversation_splits_v2.columns)
)

RELOADING AMAZONHELP CONVERSATION SPLITS

File:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits_v2.csv

✓ File loaded.
Rows: 82246
Columns: ['conversation_id', 'first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'split', 'is_taxonomy_audit']


In [153]:
print("=" * 80)
print("EXISTING AMAZONHELP SPLIT — ACTUAL VALUES")
print("=" * 80)

print("\nSplit distribution:")
print(
    amazonhelp_conversation_splits_v2["split"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nExact split counts:")
for split_name, count in (
    amazonhelp_conversation_splits_v2["split"]
    .value_counts(dropna=False)
    .items()
):
    print(f"{repr(split_name):15s} : {count}")

print("\nTaxonomy audit distribution:")
print(
    amazonhelp_conversation_splits_v2["is_taxonomy_audit"]
    .value_counts(dropna=False)
)

print("\nFirst 5 rows:")
display(
    amazonhelp_conversation_splits_v2.head()
)

print("\n✓ Inspection complete.")

EXISTING AMAZONHELP SPLIT — ACTUAL VALUES

Split distribution:
split
test           8225
train         65796
validation     8225
Name: count, dtype: int64

Exact split counts:
'train'         : 65796
'test'          : 8225
'validation'    : 8225

Taxonomy audit distribution:
is_taxonomy_audit
False    82096
True       150
Name: count, dtype: int64

First 5 rows:


,conversation_id,first_customer_message,last_customer_message,first_support_message,last_support_message,conversation_text,customer_turns,support_turns,conversation_size,start_time,end_time,duration_minutes,support_account,size_bucket,split,is_taxonomy_audit
0,272,amazonのfireTVstickが見れない😢,@AmazonHelp こちらこそありがとうございました。,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET,[CUSTOMER] amazonのfireTVstickが見れない😢\n[SUPPORT] @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET\n[CUSTOMER] @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。\n[CUSTOMER] @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。\n[SUPPORT] @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET\n[CUSTOMER] @AmazonHelp こちらこそありがとうございました。\n[SUPPORT] @115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET,4,3,7,2017-11-22 09:14:39+00:00,2017-11-22 10:06:26+00:00,51.783333,AmazonHelp,5_to_8_tweets,train,False
1,325,amazonプライムビデオ、再生エラーが多いです,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の再起動にて改善する場合がございますので、お試しください。改善しない場合は、状況を確認しご案内させていただきますのでこちらからカスタマーサービスまでご連絡ください。https://t.co/NtNAX2Qh2u ET,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の再起動にて改善する場合がございますので、お試しください。改善しない場合は、状況を確認しご案内させていただきますのでこちらからカスタマーサービスまでご連絡ください。https://t.co/NtNAX2Qh2u ET,[CUSTOMER] amazonプライムビデオ、再生エラーが多いです\n[SUPPORT] @115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の再起動にて改善する場合がございますので、お試しください。改善しない場合は、状況を確認しご案内させていただきますのでこちらからカスタマーサービスまでご連絡ください。https://t.co/NtNAX2Qh2u ET,1,1,2,2017-11-22 08:55:35+00:00,2017-11-22 09:06:00+00:00,10.416667,AmazonHelp,2_tweets,train,False
2,617,Way to drop the ball on customer service @115821 so pissed right now!,"@AmazonHelp I frankly don't have the patience for another chat with your ""customer service"" people today.","@115820 I'm sorry we've let you down! Without providing any personal information, will you describe the issue? We'd love to help. ^TN",@115820 We'd like to take a further look into this with you! Please reach us by phone or chat here: https://t.co/hApLpMlfHN ^AG,"[CUSTOMER] Way to drop the ball on customer service @115821 so pissed right now!\n[SUPPORT] @115820 I'm sorry we've let you down! Without providing any personal information, will you describe the issue? We'd love to help. ^TN\n[CUSTOMER] @AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day\n[SUPPORT] @115820 We'd like to take a further look into this with you! Please reach us by phone or chat here:...",3,2,5,2017-10-31 22:16:32+00:00,2017-10-31 23:32:26+00:00,75.900000,AmazonHelp,5_to_8_tweets,train,False
3,621,@115823 I want my amazon payments account CLOSED. dm me please.,@115823 I want my amazon payments account CLOSED. dm me please.,"@115822 I am unable to affect your account via Twitter. For real time support, phone or chat use this link: https://t.co/hApLpMlfHN ^CH","@115822 I am unable to affect your account via Twitter. For real time support, phone or chat use this link: https://t.co/hApLpMlfHN ^CH","[CUSTOMER] @115823 I want my amazon payments account CLOSED. dm me please.\n[SUPPORT] @115822 I am unable to affect your account via Twitter. For real time support, phone or chat use this link: https://t.co/hApLpMlfHN ^CH",1,1,2,2017-10-31 22:19:34+00:00,2017-10-31 22:28:34+00:00,9.000000,AmazonHelp,2_tweets,train,False
4,624,"@115825 also, beim Addams Family-Film in Prime sind Bild und Ton nicht wirklich synchron. Wie kommt's?","@AmazonHelp Okay, danke für die Info","@115824 Hi, wir erhalten die Filme/Serien so vom jeweiligen Studio. Gebe ich aber direkt als Feedback dorthin weiter. Gruß ^JS",@115824 Wir haben zu danken. Schönen Abend noch. ^JS,"[CUSTOMER] @115825 also, beim Addams Family-Film in Prime sind Bild und Ton nicht wirklich synchron. Wie ko


✓ Inspection complete.


In [154]:
print("=" * 80)
print("CREATE CLEAN AMAZONHELP LABELING POOL")
print("=" * 80)

# train only
train_pool = amazonhelp_conversation_splits_v2[
    (amazonhelp_conversation_splits_v2["split"] == "train") &
    (amazonhelp_conversation_splits_v2["is_taxonomy_audit"] == False)
].copy()

print("\nClean training pool:", len(train_pool))

# verify taxonomy audit exclusions
assert train_pool["is_taxonomy_audit"].eq(False).all()

print("✓ 150 taxonomy-audit examples excluded.")

CREATE CLEAN AMAZONHELP LABELING POOL

Clean training pool: 65678
✓ 150 taxonomy-audit examples excluded.


In [155]:
BUCKETS = [
    "2_tweets",
    "3_to_4_tweets",
    "5_to_8_tweets",
    "9_plus_tweets"
]

SAMPLES_PER_BUCKET = 375

pool_parts = []

print("\nSampling:")

for bucket in BUCKETS:

    bucket_df = train_pool[
        train_pool["size_bucket"] == bucket
    ]

    sample = bucket_df.sample(
        n=SAMPLES_PER_BUCKET,
        random_state=42
    )

    pool_parts.append(sample)

    print(
        f"{bucket:20s} "
        f"available={len(bucket_df):6d} "
        f"sampled={len(sample):4d}"
    )


Sampling:
2_tweets             available= 25029 sampled= 375
3_to_4_tweets        available= 20388 sampled= 375
5_to_8_tweets        available= 14201 sampled= 375
9_plus_tweets        available=  6060 sampled= 375


In [156]:
labeling_pool = pd.concat(
    pool_parts,
    ignore_index=True
)

labeling_pool = labeling_pool.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [157]:

labeling_pool["intent_label"] = ""
labeling_pool["annotation_notes"] = ""
labeling_pool["annotator"] = ""

assert len(labeling_pool) == 1500

assert labeling_pool["split"].eq("train").all()

assert labeling_pool["is_taxonomy_audit"].eq(False).all()

assert labeling_pool["size_bucket"].value_counts().to_dict() == {
    "2_tweets": 375,
    "3_to_4_tweets": 375,
    "5_to_8_tweets": 375,
    "9_plus_tweets": 375
}

print("\n" + "=" * 80)
print("LABELING POOL")
print("=" * 80)

print("Total:", len(labeling_pool))

print("\nSize-bucket distribution:")
print(
    labeling_pool["size_bucket"]
    .value_counts()
    .sort_index()
)

labeling_pool_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_labeling_pool_1500.csv"
)

labeling_pool.to_csv(
    labeling_pool_path,
    index=False
)

print("\n✓ Saved:")
print(labeling_pool_path)

print("\n✓ CLEAN TRAINING LABELING POOL CREATED.")


LABELING POOL
Total: 1500

Size-bucket distribution:
size_bucket
2_tweets         375
3_to_4_tweets    375
5_to_8_tweets    375
9_plus_tweets    375
Name: count, dtype: int64

✓ Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_labeling_pool_1500.csv

✓ CLEAN TRAINING LABELING POOL CREATED.


In [158]:
# ============================================================
# STEP 64C — CREATE CLEAN TRAINING LABELING POOL
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("CREATE CLEAN AMAZONHELP LABELING POOL")
print("=" * 80)

# ------------------------------------------------------------
# 1. TRAIN ONLY
# ------------------------------------------------------------

train_pool = amazonhelp_conversation_splits_v2[
    (amazonhelp_conversation_splits_v2["split"] == "train") &
    (amazonhelp_conversation_splits_v2["is_taxonomy_audit"] == False)
].copy()

print("\nClean training pool:", len(train_pool))

# ------------------------------------------------------------
# 2. Verify taxonomy audit is excluded
# ------------------------------------------------------------

assert train_pool["is_taxonomy_audit"].eq(False).all()

print("✓ 150 taxonomy-audit examples excluded.")

# ------------------------------------------------------------
# 3. Sample equally across conversation-size buckets
# ------------------------------------------------------------

BUCKETS = [
    "2_tweets",
    "3_to_4_tweets",
    "5_to_8_tweets",
    "9_plus_tweets"
]

SAMPLES_PER_BUCKET = 375

pool_parts = []

print("\nSampling:")

for bucket in BUCKETS:

    bucket_df = train_pool[
        train_pool["size_bucket"] == bucket
    ]

    sample = bucket_df.sample(
        n=SAMPLES_PER_BUCKET,
        random_state=42
    )

    pool_parts.append(sample)

    print(
        f"{bucket:20s} "
        f"available={len(bucket_df):6d} "
        f"sampled={len(sample):4d}"
    )

# ------------------------------------------------------------
# 4. Combine and shuffle
# ------------------------------------------------------------

labeling_pool = pd.concat(
    pool_parts,
    ignore_index=True
)

labeling_pool = labeling_pool.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Annotation columns
# ------------------------------------------------------------

labeling_pool["intent_label"] = ""
labeling_pool["annotation_notes"] = ""
labeling_pool["annotator"] = ""

# ------------------------------------------------------------
# 6. Safety checks
# ------------------------------------------------------------

assert len(labeling_pool) == 1500

assert labeling_pool["split"].eq("train").all()

assert labeling_pool["is_taxonomy_audit"].eq(False).all()

assert labeling_pool["size_bucket"].value_counts().to_dict() == {
    "2_tweets": 375,
    "3_to_4_tweets": 375,
    "5_to_8_tweets": 375,
    "9_plus_tweets": 375
}

# ------------------------------------------------------------
# 7. Show distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LABELING POOL")
print("=" * 80)

print("Total:", len(labeling_pool))

print("\nSize-bucket distribution:")
print(
    labeling_pool["size_bucket"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

labeling_pool_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_labeling_pool_1500.csv"
)

labeling_pool.to_csv(
    labeling_pool_path,
    index=False
)

print("\n✓ Saved:")
print(labeling_pool_path)

print("\n✓ CLEAN TRAINING LABELING POOL CREATED.")

CREATE CLEAN AMAZONHELP LABELING POOL

Clean training pool: 65678
✓ 150 taxonomy-audit examples excluded.

Sampling:
2_tweets             available= 25029 sampled= 375
3_to_4_tweets        available= 20388 sampled= 375
5_to_8_tweets        available= 14201 sampled= 375
9_plus_tweets        available=  6060 sampled= 375

LABELING POOL
Total: 1500

Size-bucket distribution:
size_bucket
2_tweets         375
3_to_4_tweets    375
5_to_8_tweets    375
9_plus_tweets    375
Name: count, dtype: int64

✓ Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_labeling_pool_1500.csv

✓ CLEAN TRAINING LABELING POOL CREATED.


only 118 of the 150 audit examples are being excluded from train, implying some audit examples may be sitting in validation/test

In [159]:
print("=" * 80)
print("TAXONOMY AUDIT PLACEMENT CHECK")
print("=" * 80)

audit_rows = amazonhelp_conversation_splits_v2[
    amazonhelp_conversation_splits_v2["is_taxonomy_audit"] == True
].copy()

print("\nTotal taxonomy-audit examples:", len(audit_rows))

print("\nAudit examples by split:")
audit_split_counts = (
    audit_rows["split"]
    .value_counts(dropna=False)
    .sort_index()
)

print(audit_split_counts)

print("\nDetailed split × audit status:")
display(
    pd.crosstab(
        amazonhelp_conversation_splits_v2["split"],
        amazonhelp_conversation_splits_v2["is_taxonomy_audit"]
    )
)

print("\n" + "=" * 80)
print("AUDIT EXAMPLES OUTSIDE TRAIN")
print("=" * 80)

outside_train = audit_rows[
    audit_rows["split"] != "train"
]

print("Count:", len(outside_train))

if len(outside_train) > 0:
    display(
        outside_train[
            [
                "conversation_id",
                "split",
                "is_taxonomy_audit",
                "first_customer_message"
            ]
        ].head(50)
    )

print("\n✓ Audit placement check complete.")

TAXONOMY AUDIT PLACEMENT CHECK

Total taxonomy-audit examples: 150

Audit examples by split:
split
test           20
train         118
validation     12
Name: count, dtype: int64

Detailed split × audit status:


is_taxonomy_audit,False,True
split,,
test,8205,20
train,65678,118
validation,8213,12



AUDIT EXAMPLES OUTSIDE TRAIN
Count: 32


,conversation_id,split,is_taxonomy_audit,first_customer_message
1848,80798,test,True,@115850 item not received for order 406-1844716-6230753 Status is showing delivered . Have also received SMS mentioning item delivered. Kindly address urgently.
4034,152192,test,True,I have ordered so many books from @115850 . But everytime I get the same old boring bookmarks. Could you please send me some really cool exciting bookmarks for my next order of book?.
5716,210526,test,True,2nd time failed frm pymnt gatway fr bokng iphone6 32Gb gray cost 20k thn imdate incrsd to 26450.Wht's offr running don't undrstnd @115850 https://t.co/7edeUhB53O
7820,271576,test,True,Hey @117795 you've delayed a delivery once again. One more time and I'll be exploring options like @2300
9932,318900,test,True,@115850 very very bad product delivrd by amazon.my brother ordrd the lg phone Prodct output is not atleast 20 % wt dey gives in discrption
10118,321644,test,True,@AmazonHelp plse help .check DM and help quickly
12042,375828,test,True,Prolly shoulda used better judgement on a Sunday delivery from @115821 but I PAID THE SHIPPING COST. WHERE'S MY STUFF??
12649,386412,test,True,"Man... my @115821 package has been ""preparing"" for almost two days straight now... getting really #worried it won't ""arrive tomorrow"""
16227,478452,test,True,I am watching Shin Chan Season 1. But currently it is not available to my location. Why are you people doing like this? @119625
20975,627702,validation,True,@AmazonHelp I have a parcel which says delivered but was not delivered to me. Could you help me track down where this was delivered please? @115821 @115830



✓ Audit placement check complete.


- TRAIN      - 65,678 clean + 118 audit
- VALIDATION -  8,213 clean + 12 audit
- TEST       -  8,205 clean + 20 audit
- AUDIT      -     150 total

32 of 150 manually labelled taxonomy-audit examples leaked into validation/test.

create clean split v3

In [160]:
print("=" * 80)
print("CREATE CLEAN AMAZONHELP SPLIT V3")
print("=" * 80)

df = amazonhelp_conversation_splits_v2.copy()

df["original_split"] = df["split"]

audit_mask = df["is_taxonomy_audit"] == True
df.loc[audit_mask, "split"] = "audit"

print("\nClean split distribution:")

split_counts = (
    df["split"]
    .value_counts()
    .sort_index()
)

print(split_counts)



CREATE CLEAN AMAZONHELP SPLIT V3

Clean split distribution:
split
audit           150
test           8205
train         65678
validation     8213
Name: count, dtype: int64


In [161]:
audit_rows = df[df["split"] == "audit"]

assert len(audit_rows) == 150

assert audit_rows["is_taxonomy_audit"].eq(True).all()

assert (
    df.loc[df["split"] != "audit", "is_taxonomy_audit"]
    .eq(False)
    .all()
)

assert len(df) == 82246
assert df["conversation_id"].is_unique

In [162]:
clean_split_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits_v3.csv"
)

df.to_csv(
    clean_split_path,
    index=False
)

print("\n" + "=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print("Total rows:", len(df))
print("Unique conversations:", df["conversation_id"].nunique())

print("\nSplit counts:")
print(df["split"].value_counts().sort_index())

print("\nAudit counts:")
print(
    pd.crosstab(
        df["split"],
        df["is_taxonomy_audit"]
    )
)

print("\n✓ Clean split v3 saved:")
print(clean_split_path)


FINAL VERIFICATION
Total rows: 82246
Unique conversations: 82246

Split counts:
split
audit           150
test           8205
train         65678
validation     8213
Name: count, dtype: int64

Audit counts:
is_taxonomy_audit  False  True 
split                          
audit                  0    150
test                8205      0
train              65678      0
validation          8213      0

✓ Clean split v3 saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits_v3.csv


In [163]:
# ============================================================
# STEP 64E — CREATE CLEAN SPLIT V3
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 80)
print("CREATE CLEAN AMAZONHELP SPLIT V3")
print("=" * 80)

df = amazonhelp_conversation_splits_v2.copy()

# ------------------------------------------------------------
# 1. Preserve original split
# ------------------------------------------------------------

df["original_split"] = df["split"]

# ------------------------------------------------------------
# 2. Move ALL taxonomy-audit examples to dedicated audit split
# ------------------------------------------------------------

audit_mask = df["is_taxonomy_audit"] == True

df.loc[audit_mask, "split"] = "audit"

# ------------------------------------------------------------
# 3. Verify clean split counts
# ------------------------------------------------------------

print("\nClean split distribution:")

split_counts = (
    df["split"]
    .value_counts()
    .sort_index()
)

print(split_counts)

# Expected:
# audit         150
# test         8205
# train       65678
# validation   8213

# ------------------------------------------------------------
# 4. Verify audit isolation
# ------------------------------------------------------------

audit_rows = df[df["split"] == "audit"]

assert len(audit_rows) == 150

assert audit_rows["is_taxonomy_audit"].eq(True).all()

assert (
    df.loc[df["split"] != "audit", "is_taxonomy_audit"]
    .eq(False)
    .all()
)

# ------------------------------------------------------------
# 5. Verify no rows were lost
# ------------------------------------------------------------

assert len(df) == 82246

# ------------------------------------------------------------
# 6. Verify conversation IDs remain unique
# ------------------------------------------------------------

assert df["conversation_id"].is_unique

# ------------------------------------------------------------
# 7. Save clean split
# ------------------------------------------------------------

clean_split_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits_v3.csv"
)

df.to_csv(
    clean_split_path,
    index=False
)

print("\n" + "=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print("Total rows:", len(df))
print("Unique conversations:", df["conversation_id"].nunique())

print("\nSplit counts:")
print(df["split"].value_counts().sort_index())

print("\nAudit counts:")
print(
    pd.crosstab(
        df["split"],
        df["is_taxonomy_audit"]
    )
)

print("\n✓ Clean split v3 saved:")
print(clean_split_path)

CREATE CLEAN AMAZONHELP SPLIT V3

Clean split distribution:
split
audit           150
test           8205
train         65678
validation     8213
Name: count, dtype: int64

FINAL VERIFICATION
Total rows: 82246
Unique conversations: 82246

Split counts:
split
audit           150
test           8205
train         65678
validation     8213
Name: count, dtype: int64

Audit counts:
is_taxonomy_audit  False  True 
split                          
audit                  0    150
test                8205      0
train              65678      0
validation          8213      0

✓ Clean split v3 saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_conversation_splits_v3.csv


now have a clean, defensible split

In [164]:
from pathlib import Path
import pandas as pd

print("=" * 80)
print("LOAD CLEAN AMAZONHELP SPLIT V3")
print("=" * 80)

split_v3_path = (
    PROCESSED_DIR /
    "amazonhelp_conversation_splits_v3.csv"
)

amazonhelp_splits = pd.read_csv(
    split_v3_path
)

print("\nRows:", len(amazonhelp_splits))

print("\nSplit distribution:")
print(
    amazonhelp_splits["split"]
    .value_counts()
    .sort_index()
)

LOAD CLEAN AMAZONHELP SPLIT V3

Rows: 82246

Split distribution:
split
audit           150
test           8205
train         65678
validation     8213
Name: count, dtype: int64


In [165]:
assert len(amazonhelp_splits) == 82246
assert amazonhelp_splits["conversation_id"].is_unique

assert (
    amazonhelp_splits.loc[
        amazonhelp_splits["split"] != "audit",
        "is_taxonomy_audit"
    ].eq(False).all()
)

assert (
    amazonhelp_splits.loc[
        amazonhelp_splits["split"] == "audit",
        "is_taxonomy_audit"
    ].eq(True).all()
)

print("\n✓ v3 is now the official working split.")

print("\nOfficial datasets:")
print("Train:", len(amazonhelp_splits[amazonhelp_splits["split"] == "train"]))
print("Validation:", len(amazonhelp_splits[amazonhelp_splits["split"] == "validation"]))
print("Test:", len(amazonhelp_splits[amazonhelp_splits["split"] == "test"]))
print("Audit:", len(amazonhelp_splits[amazonhelp_splits["split"] == "audit"]))


✓ v3 is now the official working split.

Official datasets:
Train: 65678
Validation: 8213
Test: 8205
Audit: 150


In [166]:
print("=" * 80)
print("CREATE HUMAN-LABELLED TRAINING BATCH 1")
print("=" * 80)

labeling_pool_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_labeling_pool_1500.csv"
)

labeling_pool = pd.read_csv(
    labeling_pool_path
)

print("\nLabeling pool:", len(labeling_pool))

assert len(labeling_pool) == 1500

assert labeling_pool["split"].eq("train").all()

assert labeling_pool["is_taxonomy_audit"].eq(False).all()

CREATE HUMAN-LABELLED TRAINING BATCH 1

Labeling pool: 1500


In [167]:
BUCKETS = [
    "2_tweets",
    "3_to_4_tweets",
    "5_to_8_tweets",
    "9_plus_tweets"
]

BATCH_SIZE_PER_BUCKET = 75

batch_parts = []

for bucket in BUCKETS:

    bucket_df = labeling_pool[
        labeling_pool["size_bucket"] == bucket
    ].copy()

    # Deterministic sampling
    batch = bucket_df.sample(
        n=BATCH_SIZE_PER_BUCKET,
        random_state=42
    )

    batch_parts.append(batch)

    print(
        f"{bucket:20s}: "
        f"{len(batch)} selected"
    )

2_tweets            : 75 selected
3_to_4_tweets       : 75 selected
5_to_8_tweets       : 75 selected
9_plus_tweets       : 75 selected


In [168]:
training_batch_1 = pd.concat(
    batch_parts,
    ignore_index=True
)

training_batch_1 = training_batch_1.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [169]:
training_batch_1["intent_label"] = ""

training_batch_1["annotation_notes"] = ""

training_batch_1["annotator"] = ""

training_batch_1["annotation_status"] = "UNLABELED"

In [170]:
assert len(training_batch_1) == 300

assert (
    training_batch_1["size_bucket"]
    .value_counts()
    .to_dict()
    == {
        "2_tweets": 75,
        "3_to_4_tweets": 75,
        "5_to_8_tweets": 75,
        "9_plus_tweets": 75
    }
)

assert training_batch_1["split"].eq("train").all()

assert training_batch_1["is_taxonomy_audit"].eq(False).all()

assert training_batch_1["conversation_id"].is_unique

In [171]:
training_batch_1_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_training_batch_01.csv"
)

training_batch_1.to_csv(
    training_batch_1_path,
    index=False
)

print("\n" + "=" * 80)
print("BATCH CREATED")
print("=" * 80)

print("Total examples:", len(training_batch_1))

print("\nSize distribution:")
print(
    training_batch_1["size_bucket"]
    .value_counts()
    .sort_index()
)

print("\nSplit distribution:")
print(
    training_batch_1["split"]
    .value_counts()
)

print("\nAudit distribution:")
print(
    training_batch_1["is_taxonomy_audit"]
    .value_counts()
)

print("\n✓ Saved:")
print(training_batch_1_path)

print("\n✓ Training batch 01 is ready for human annotation.")


BATCH CREATED
Total examples: 300

Size distribution:
size_bucket
2_tweets         75
3_to_4_tweets    75
5_to_8_tweets    75
9_plus_tweets    75
Name: count, dtype: int64

Split distribution:
split
train    300
Name: count, dtype: int64

Audit distribution:
is_taxonomy_audit
False    300
Name: count, dtype: int64

✓ Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_training_batch_01.csv

✓ Training batch 01 is ready for human annotation.


In [172]:
# ============================================================
# STEP 66 — CREATE HUMAN-LABELLED TRAINING BATCH 1
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 80)
print("CREATE HUMAN-LABELLED TRAINING BATCH 1")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load the existing 1,500 pool
# ------------------------------------------------------------

labeling_pool_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_labeling_pool_1500.csv"
)

labeling_pool = pd.read_csv(
    labeling_pool_path
)

print("\nLabeling pool:", len(labeling_pool))

# ------------------------------------------------------------
# 2. Safety checks
# ------------------------------------------------------------

assert len(labeling_pool) == 1500

assert labeling_pool["split"].eq("train").all()

assert labeling_pool["is_taxonomy_audit"].eq(False).all()

# ------------------------------------------------------------
# 3. Sample 75 per size bucket
# ------------------------------------------------------------

BUCKETS = [
    "2_tweets",
    "3_to_4_tweets",
    "5_to_8_tweets",
    "9_plus_tweets"
]

BATCH_SIZE_PER_BUCKET = 75

batch_parts = []

for bucket in BUCKETS:

    bucket_df = labeling_pool[
        labeling_pool["size_bucket"] == bucket
    ].copy()

    # Deterministic sampling
    batch = bucket_df.sample(
        n=BATCH_SIZE_PER_BUCKET,
        random_state=42
    )

    batch_parts.append(batch)

    print(
        f"{bucket:20s}: "
        f"{len(batch)} selected"
    )

# ------------------------------------------------------------
# 4. Combine and shuffle
# ------------------------------------------------------------

training_batch_1 = pd.concat(
    batch_parts,
    ignore_index=True
)

training_batch_1 = training_batch_1.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Add clean annotation fields
# ------------------------------------------------------------

training_batch_1["intent_label"] = ""

training_batch_1["annotation_notes"] = ""

training_batch_1["annotator"] = ""

# ------------------------------------------------------------
# 6. Add explicit annotation status
# ------------------------------------------------------------

training_batch_1["annotation_status"] = "UNLABELED"

# ------------------------------------------------------------
# 7. Safety checks
# ------------------------------------------------------------

assert len(training_batch_1) == 300

assert (
    training_batch_1["size_bucket"]
    .value_counts()
    .to_dict()
    == {
        "2_tweets": 75,
        "3_to_4_tweets": 75,
        "5_to_8_tweets": 75,
        "9_plus_tweets": 75
    }
)

assert training_batch_1["split"].eq("train").all()

assert training_batch_1["is_taxonomy_audit"].eq(False).all()

# ------------------------------------------------------------
# 8. Check conversation uniqueness
# ------------------------------------------------------------

assert training_batch_1["conversation_id"].is_unique

# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

training_batch_1_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_training_batch_01.csv"
)

training_batch_1.to_csv(
    training_batch_1_path,
    index=False
)

print("\n" + "=" * 80)
print("BATCH CREATED")
print("=" * 80)

print("Total examples:", len(training_batch_1))

print("\nSize distribution:")
print(
    training_batch_1["size_bucket"]
    .value_counts()
    .sort_index()
)

print("\nSplit distribution:")
print(
    training_batch_1["split"]
    .value_counts()
)

print("\nAudit distribution:")
print(
    training_batch_1["is_taxonomy_audit"]
    .value_counts()
)

print("\n✓ Saved:")
print(training_batch_1_path)

print("\n✓ Training batch 01 is ready for human annotation.")

CREATE HUMAN-LABELLED TRAINING BATCH 1

Labeling pool: 1500
2_tweets            : 75 selected
3_to_4_tweets       : 75 selected
5_to_8_tweets       : 75 selected
9_plus_tweets       : 75 selected

BATCH CREATED
Total examples: 300

Size distribution:
size_bucket
2_tweets         75
3_to_4_tweets    75
5_to_8_tweets    75
9_plus_tweets    75
Name: count, dtype: int64

Split distribution:
split
train    300
Name: count, dtype: int64

Audit distribution:
is_taxonomy_audit
False    300
Name: count, dtype: int64

✓ Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_training_batch_01.csv

✓ Training batch 01 is ready for human annotation.


prepare Batch 01 for manual annotation

In [173]:
import pandas as pd

batch_path = (
    PROCESSED_DIR /
    "amazonhelp_intent_training_batch_01.csv"
)

training_batch_1 = pd.read_csv(batch_path)

print("=" * 80)
print("TRAINING BATCH 01 — MANUAL ANNOTATION SET 1")
print("=" * 80)

print("\nTotal batch:", len(training_batch_1))
print("Showing examples 0–29\n")

for i in range(30):

    row = training_batch_1.iloc[i]

    print("=" * 80)
    print(f"EXAMPLE {i}")
    print("=" * 80)

    print("Conversation size :", row["conversation_size"])
    print("Customer turns   :", row["customer_turns"])
    print("Support turns    :", row["support_turns"])
    print("Size bucket      :", row["size_bucket"])

    print("\nCUSTOMER:")
    print(row["first_customer_message"])

    print("\nFULL CONVERSATION:")
    print(row["conversation_text"])

    print("\nCURRENT LABEL:")
    print("[ UNLABELED ]")

print("\n" + "=" * 80)
print("END OF SET 1")
print("=" * 80)

TRAINING BATCH 01 — MANUAL ANNOTATION SET 1

Total batch: 300
Showing examples 0–29

EXAMPLE 0
Conversation size : 5
Customer turns   : 3
Support turns    : 2
Size bucket      : 5_to_8_tweets

CUSTOMER:
I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.

FULL CONVERSATION:
[CUSTOMER] I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.
[SUPPORT] @204457 We never want to let our customers down! Is there an issue with one of your current orders: https://t.co/3kQwjObgTc? ^GM
[CUSTOMER] @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +
[CUSTOMER] @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.
[SUPPORT] @204457 Oh no, Br

In [174]:
print("=" * 80)
print("TRAINING BATCH 01 — MANUAL ANNOTATION SET 2")
print("=" * 80)

print("\nTotal batch:", len(training_batch_1))
print("Showing examples 30–59\n")

for i in range(30, 60):

    row = training_batch_1.iloc[i]

    print("=" * 80)
    print(f"EXAMPLE {i}")
    print("=" * 80)

    print("Conversation size :", row["conversation_size"])
    print("Customer turns   :", row["customer_turns"])
    print("Support turns    :", row["support_turns"])
    print("Size bucket      :", row["size_bucket"])

    print("\nCUSTOMER:")
    print(row["first_customer_message"])

    print("\nFULL CONVERSATION:")
    print(row["conversation_text"])

    print("\nCURRENT LABEL:")
    print("[ UNLABELED ]")

print("\n" + "=" * 80)
print("END OF SET 2")
print("=" * 80)

TRAINING BATCH 01 — MANUAL ANNOTATION SET 2

Total batch: 300
Showing examples 30–59

EXAMPLE 30
Conversation size : 2
Customer turns   : 1
Support turns    : 1
Size bucket      : 2_tweets

CUSTOMER:
Everyone is getting their Amazon Echo while Amazon has delayed the delivery of mine to Monday 😢😢😢😭

FULL CONVERSATION:
[CUSTOMER] Everyone is getting their Amazon Echo while Amazon has delayed the delivery of mine to Monday 😢😢😢😭
[SUPPORT] @463847 We understand your excitement. Sorry for the delay. Kindly wait until the revised estimated delivery date. ^NS

CURRENT LABEL:
[ UNLABELED ]
EXAMPLE 31
Conversation size : 6
Customer turns   : 4
Support turns    : 2
Size bucket      : 5_to_8_tweets

CUSTOMER:
Is @115830 tracking a joke? Got similar screen Mon Tues and Wed and different info on DHL page. @AmazonHelp CS only giving useless standard replies. Definitely avoiding in the future. https://t.co/lgFvSKn7Im

FULL CONVERSATION:
[CUSTOMER] Is @115830 tracking a joke? Got similar screen Mon Tue

In [175]:
training_batch_path = PROCESSED_DIR / "amazonhelp_intent_training_batch_01.csv"

training_batch = pd.read_csv(training_batch_path)

In [176]:
# Display Set 3: examples 60–89

start, end = 60, 89

for i in range(start, end + 1):
    row = training_batch.iloc[i]

    print("=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)
    print(f"Conversation size : {row['conversation_size']}")
    print(f"Customer turns   : {row['customer_turns']}")
    print(f"Support turns    : {row['support_turns']}")
    print(f"Size bucket      : {row['size_bucket']}")
    print()
    print("CUSTOMER:")
    print(row["first_customer_message"])
    print()
    print("FULL CONVERSATION:")
    print(row["conversation_text"])
    print()
    print("CURRENT LABEL:")
    print("[ UNLABELED ]")
    print()

EXAMPLE 60
Conversation size : 10
Customer turns   : 5
Support turns    : 5
Size bucket      : 9_plus_tweets

CUSTOMER:
@115850 @AmazonHelp Facing fake price on Amazon App. Deal price Rs649 of SanDisk when proceeded to checkout its its Rs743. Kindly check https://t.co/xYjf51qFSQ

FULL CONVERSATION:
[CUSTOMER] @115850 @AmazonHelp Facing fake price on Amazon App. Deal price Rs649 of SanDisk when proceeded to checkout its its Rs743. Kindly check https://t.co/xYjf51qFSQ
[CUSTOMER] @115850 @AmazonHelp Rs 100 delivery charges on Prime product. Can you tell me why facing all these issue. Whenever I am proceeding to checkout price increased
[SUPPORT] @207103 them here: https://t.co/vlvfJr4nN9 and it'll be sorted. 2/2 ^SK
[SUPPORT] @207103 I'm sorry about this. Have you reported this to our support team? If not, please connect with 1/2 ^SK
[CUSTOMER] @AmazonHelp Issues never get sorted there. They have excuse for everything
[SUPPORT] @207103 We would like to help you. Please share your details 

In [177]:
# Display Set 6: examples 150–179

start, end = 150, 179

for i in range(start, end + 1):
    row = training_batch.iloc[i]

    print("=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)
    print(f"Conversation size : {row['conversation_size']}")
    print(f"Customer turns   : {row['customer_turns']}")
    print(f"Support turns    : {row['support_turns']}")
    print(f"Size bucket      : {row['size_bucket']}")
    print()
    print("CUSTOMER:")
    print(row["first_customer_message"])
    print()
    print("FULL CONVERSATION:")
    print(row["conversation_text"])
    print()
    print("CURRENT LABEL:")
    print("[ UNLABELED ]")
    print()

EXAMPLE 150
Conversation size : 12
Customer turns   : 8
Support turns    : 4
Size bucket      : 9_plus_tweets

CUSTOMER:
Government’s reforms to push economic growth are working can be seen from that manufacturing has shown robust growth of 7% in Q2 and services at 7.1%. Gross fixed capital formation has increased from 1.6% in Q1 to 4.7% in Q2.

FULL CONVERSATION:
[CUSTOMER] Government’s reforms to push economic growth are working can be seen from that manufacturing has shown robust growth of 7% in Q2 and services at 7.1%. Gross fixed capital formation has increased from 1.6% in Q1 to 4.7% in Q2.
[CUSTOMER] @137848 @115850 @5834  this site continues doing a farud with seller almost 50 sellers I received complain they suspend account then they never contact all payment get zero
[SUPPORT] @137847 I'm sorry to learn this, has this been reported to our seller support team here: https://t.co/hITbmQzz2y? ^VN
[CUSTOMER] @AmazonHelp @AmazonHelp your team doing totally farud with us we are sell

In [178]:
# Display Set 7: examples 180–209

start, end = 180, 209

for i in range(start, end + 1):
    row = training_batch.iloc[i]

    print("=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)
    print(f"Conversation size : {row['conversation_size']}")
    print(f"Customer turns   : {row['customer_turns']}")
    print(f"Support turns    : {row['support_turns']}")
    print(f"Size bucket      : {row['size_bucket']}")
    print()
    print("CUSTOMER:")
    print(row["first_customer_message"])
    print()
    print("FULL CONVERSATION:")
    print(row["conversation_text"])
    print()
    print("CURRENT LABEL:")
    print("[ UNLABELED ]")
    print()

EXAMPLE 180
Conversation size : 7
Customer turns   : 3
Support turns    : 4
Size bucket      : 5_to_8_tweets

CUSTOMER:
No matter how loud you cry @115850 is not gonna hear sellers. A blind eye approach. Stopping all Ads @128876 @115851 #HearUsAmazon https://t.co/jeCP6SD7C6

FULL CONVERSATION:
[CUSTOMER] No matter how loud you cry @115850 is not gonna hear sellers. A blind eye approach. Stopping all Ads @128876 @115851 #HearUsAmazon https://t.co/jeCP6SD7C6
[SUPPORT] @129201 Sorry for the unpleasant experience with us. We'd like to know what went wrong, (1/2)^SQ
[SUPPORT] @129201 please contact our seller support team here: https://t.co/hITbmQzz2y. We'll look into it. (2/2)^SQ
[CUSTOMER] @AmazonHelp Everything went wrong in last 1 year.  Every genuine concern is ignored with a Blind Eye.  Seller Support, Escalation Support all failed.
[SUPPORT] @129201 We'd like to take a closer look into this. Please fill in this form: https://t.co/GIJyeYqKE0 and we'll look into it. ^SQ
[CUSTOMER] @Ama

In [179]:
start, end = 210, 239

for i in range(start, end + 1):
    row = training_batch.iloc[i]

    print("=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)
    print(f"Conversation size : {row['conversation_size']}")
    print(f"Customer turns   : {row['customer_turns']}")
    print(f"Support turns    : {row['support_turns']}")
    print(f"Size bucket      : {row['size_bucket']}")
    print()
    print("CUSTOMER:")
    print(row["first_customer_message"])
    print()
    print("FULL CONVERSATION:")
    print(row["conversation_text"])
    print()
    print("CURRENT LABEL:")
    print("[ UNLABELED ]")
    print()

EXAMPLE 210
Conversation size : 4
Customer turns   : 2
Support turns    : 2
Size bucket      : 3_to_4_tweets

CUSTOMER:
Also dad können mir @116316 und @124285 bestimmt erklären.
Ich sitze auf Arbeit. Ca. 60 Minuten von meiner Wohnung entfernt.
Wir konnte mir das Paket persönlich übergeben werden? 🤔 https://t.co/92kiqtCv4v

FULL CONVERSATION:
[CUSTOMER] Also dad können mir @116316 und @124285 bestimmt erklären.
Ich sitze auf Arbeit. Ca. 60 Minuten von meiner Wohnung entfernt.
Wir konnte mir das Paket persönlich übergeben werden? 🤔 https://t.co/92kiqtCv4v
[SUPPORT] @268677 Hast du bereits in der Sendungsverfolgung auf der DHL Webseite nachgeschaut, ob es mehr Infos dazu gibt? Nachdem du die Sendungsnummer eingetippt hast, gibt es die Option "Detaillierte Empfänger-Informationen anzeigen". ^LN
[CUSTOMER] @AmazonHelp Moin, da steht dann Nachbar und auch der Name.
Vielleicht sowas auch bei euch eintragen. Dann kommt nicht so eine Verwirrung zu stande. Das is ziemlich Unglücklich gelöst so 

In [180]:
start = 240
end = 269

batch = training_batch.iloc[start:end+1]

for i, row in batch.iterrows():
    print("=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)
    print(f"Conversation size : {row['conversation_size']}")
    print(f"Customer turns   : {row['customer_turns']}")
    print(f"Support turns    : {row['support_turns']}")
    print(f"Size bucket      : {row['size_bucket']}")
    print("\nCUSTOMER:")
    print(row["first_customer_message"])
    print("\nFULL CONVERSATION:")
    print(row["conversation_text"])
    print("\nCURRENT LABEL:")
    print(f"[ {row['intent_label']} ]")
    print()

EXAMPLE 240
Conversation size : 97
Customer turns   : 55
Support turns    : 42
Size bucket      : 9_plus_tweets

CUSTOMER:
Series, películas, deportes en vivo y mucho más. ¡Entra y prueba GRATIS Amazon Prime Video!  https://t.co/XLc8UIQnmu

FULL CONVERSATION:
[CUSTOMER] Series, películas, deportes en vivo y mucho más. ¡Entra y prueba GRATIS Amazon Prime Video!  https://t.co/XLc8UIQnmu
[CUSTOMER] @123813 ¿Para cuándo en el Apple TV? 😫
[CUSTOMER] @123813 Pero seran para formato 4k?
[SUPPORT] @525278 Hola Edgar. Actualmente no contamos con esas informaciones. Te sugerimos estar al pendiente de nuestro sitio para novedades. ^HC
[SUPPORT] @527535 Hola Elizzandra, espero que estés bien. Para conocer los formatos de Prime Video, accede aquí: https://t.co/Z9TlpyAsBb. ^DA
[CUSTOMER] @AmazonHelp Oki oki gracias ☺👍
[SUPPORT] @527535 De nada, cualquier duda adicional, estamos aquí para ayudarte.😍 ^DA
[CUSTOMER] @123813 Además de la NFL, que otros "deportes en vivo" transmiten??
[CUSTOMER] @123813 

In [181]:
start = 270
end = 299

batch = training_batch.iloc[start:end+1]

for i, row in batch.iterrows():
    print("=" * 90)
    print(f"EXAMPLE {i}")
    print("=" * 90)
    print(f"Conversation size : {row['conversation_size']}")
    print(f"Customer turns   : {row['customer_turns']}")
    print(f"Support turns    : {row['support_turns']}")
    print(f"Size bucket      : {row['size_bucket']}")
    print("\nCUSTOMER:")
    print(row["first_customer_message"])
    print("\nFULL CONVERSATION:")
    print(row["conversation_text"])
    print("\nCURRENT LABEL:")
    print(f"[ {row['intent_label']} ]")
    print()

EXAMPLE 270
Conversation size : 2
Customer turns   : 1
Support turns    : 1
Size bucket      : 2_tweets

CUSTOMER:
①Amazonで商品を注文する

②コンビニ払いで支払う

③支払い完了のメールが来る

④同じ注文番号で別の支払い番号が来る

⑤③に対しての商品遅延のメールが来る

⑥④の期限が過ぎたからと注文自体が無かったことになったとメールが来た

って状況なんだがなんだこれ

FULL CONVERSATION:
[CUSTOMER] ①Amazonで商品を注文する

②コンビニ払いで支払う

③支払い完了のメールが来る

④同じ注文番号で別の支払い番号が来る

⑤③に対しての商品遅延のメールが来る

⑥④の期限が過ぎたからと注文自体が無かったことになったとメールが来た

って状況なんだがなんだこれ
[SUPPORT] @688280 ご不便をお掛けしております。該当の注文など詳細を確認した上でご案内させていただきますので、恐れ入りますが、リンクよりカスタマーサービスにお問い合わせください。https://t.co/J6YEizo6qC TN

CURRENT LABEL:
[ nan ]

EXAMPLE 271
Conversation size : 2
Customer turns   : 1
Support turns    : 1
Size bucket      : 2_tweets

CUSTOMER:
Fake product received from Amazon #bose speaker#@115851 #@115850 #@AmazonHelp # #@118342#help#Worst customer Experience #@115850 supporting fake sellers #@119341#404-2877796-4009925 OD I'd #Cheating Poor Customer #Stop buying from Amazon # https://t.co/yhubx3DpVj

FULL CONVERSATION:
[CUSTOMER] Fake product received fro

In [182]:
import pandas as pd

path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_training_batch_01.csv"

df = pd.read_csv(path)

print(df.columns.tolist())
print(df.shape)
print(df[["conversation_id", "intent_label"]].head(10))

['conversation_id', 'first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'split', 'is_taxonomy_audit', 'intent_label', 'annotation_notes', 'annotator', 'annotation_status']
(300, 20)
   conversation_id  intent_label
0           373303           NaN
1           282739           NaN
2          2292312           NaN
3           570075           NaN
4          1427974           NaN
5          1717565           NaN
6          2132842           NaN
7          2289374           NaN
8          1514342           NaN
9            42989           NaN


In [183]:
df.loc[0, "intent_label"] = "DELIVERY_DELAY"
df.loc[1, "intent_label"] = "PRODUCT_PROBLEM"
df.loc[2, "intent_label"] = "DELIVERY_DELAY"

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8164\3755786279.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'DELIVERY_DELAY' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[0, "intent_label"] = "DELIVERY_DELAY"


In [184]:
df.to_csv(path, index=False)

In [185]:
path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_training_batch_01.csv"

df = pd.read_csv(path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nExisting intent labels:")
print(df["intent_label"].value_counts(dropna=False))

print("\nUnlabeled row indexes:")
print(df.index[df["intent_label"].isna()].tolist())

Shape: (300, 20)

Columns:
['conversation_id', 'first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'split', 'is_taxonomy_audit', 'intent_label', 'annotation_notes', 'annotator', 'annotation_status']

Existing intent labels:
intent_label
NaN                297
DELIVERY_DELAY       2
PRODUCT_PROBLEM      1
Name: count, dtype: int64

Unlabeled row indexes:
[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 11

In [186]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [187]:
training_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "amazonhelp_intent_training_batch_01.csv"
)

print("Shape:", training_df.shape)
print("Missing labels:", training_df["intent_label"].isna().sum())
print("Unique conversations:", training_df["conversation_id"].nunique())

training_df.head()

Shape: (300, 20)
Missing labels: 297
Unique conversations: 300


,conversation_id,first_customer_message,last_customer_message,first_support_message,last_support_message,conversation_text,customer_turns,support_turns,conversation_size,start_time,end_time,duration_minutes,support_account,size_bucket,split,is_taxonomy_audit,intent_label,annotation_notes,annotator,annotation_status
0,373303,"I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.",@AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.,@204457 We never want to let our customers down! Is there an issue with one of your current orders: https://t.co/3kQwjObgTc? ^GM,"@204457 Oh no, Bree! I'm sorry for the poor experience. When you spoke with us previously, what insights or alternatives did we provide?^TG","[CUSTOMER] I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.\n[SUPPORT] @204457 We never want to let our customers down! Is there an issue with one of your current orders: https://t.co/3kQwjObgTc? ^GM\n[CUSTOMER] @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +\n[CUSTOMER] @AmazonHelp + when I ordered on...",3,2,5,2017-10-08 23:52:49+00:00,2017-10-09 00:28:31+00:00,35.700000,AmazonHelp,5_to_8_tweets,train,False,DELIVERY_DELAY,NaN,NaN,UNLABELED
1,282739,"@115821 Netboon tempered glass purchased for BlackBerry Priv @ Rs 1180 in January 2017 lost adhesive, falls off on its own. Need replacement. Netboon not responding. Amazon to help [Ordered on Jan 4, 2017 (406-9818731-9588334)]","@AmazonHelp Thanks Amazon for helping me, providing me my preferred resolution. @115821 @115850 @123644 @4030 @118702 @4167 @4110 @4168 @115937 @135980 @61230 @120697 @120033 @144769 @29568 @136912 @61231 @120319",@183363 it to be personal information. Our Twitter page is public.​ 3/3^EM,@183363 We're glad you like our services. Happy shopping :) ^OS,"[CUSTOMER] @115821 Netboon tempered glass purchased for BlackBerry Priv @ Rs 1180 in January 2017 lost adhesive, falls off on its own. Need replacement. Netboon not responding. Amazon to help [Ordered on Jan 4, 2017 (406-9818731-9588334)]\n[SUPPORT] @183363 it to be personal information. Our Twitter page is public.​ 3/3^EM\n[SUPPORT] @183363 and we'll see what best can be done to help you. Also, please don't provide your order details, as we consider 2/3^EM\n[SUPPORT] @183363 I understand y...",59,41,100,2017-10-06 12:11:43+00:00,2017-10-21 11:46:00+00:00,21574.283333,AmazonHelp,9_plus_tweets,train,False,PRODUCT_PROBLEM,NaN,NaN,UNLABELED
2,2292312,@AmazonHelp why offer 2 day shipping for prime when item will take 4 days to get it. #cancellingprime #nothappy,@AmazonHelp No it's the stupid post office that always messes up my deliveries,@569445 Two-Day shipping refers to the transit time in business days once an item has shipped. Some items may take additional time to prepare. Any prep time is included in the delivery estimate. More info: https://t.co/AbVhK00M4e ^EP,@569445 We can make sure this is reported to our Transportation team. When you have time send us the details here: https://t.co/hApLpMlfHN ^EP,[CUSTOMER] @AmazonHelp why offer 2 day shipping for prime when item will take 4 days to get it. #cancellingprime #nothappy\n[SUPPORT] @569445 Two-Day shipping refers to the transit time in business days once an item has shipped. Some items may take additional time to prepare. Any prep time is included in the delivery estimate. More info: https://t.co/AbVhK00M4e ^EP\n[CUSTOMER] @AmazonHelp I know that it's going to take 4 days transit time yet I pay monthly for 2 days. ITS SUPPOSED TO BE DELI...,3,3,6,2017-11-11 22:00:01+00:00,2017-11-11 22:46:30+00:00,46.483333,AmazonHelp,5_to_8_tweets,train,False,DELIVERY_DELAY,NaN,NaN,UNLABELED
3,570075,Amazonプラ

In [188]:
labeled_mask = (
    training_df["intent_label"].notna() &
    training_df["intent_label"].astype(str).str.strip().ne("") &
    training_df["intent_label"].astype(str).str.upper().ne("UNLABELED")
)

labeled_df = training_df.loc[labeled_mask].copy()

X = labeled_df["conversation_text"].fillna("").astype(str)
y = labeled_df["intent_label"].astype(str).str.strip()

min_class_count = y.value_counts().min()

stratify_labels = y if min_class_count >= 2 else None

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=stratify_labels,
)

print("Labeled examples:", len(labeled_df))
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Stratified split:", stratify_labels is not None)

Labeled examples: 3
Training: 2
Validation: 1
Stratified split: False


In [189]:
majority_class = y_train.value_counts().idxmax()

majority_pred = [majority_class] * len(y_val)

majority_accuracy = accuracy_score(y_val, majority_pred)
majority_macro_f1 = f1_score(
    y_val,
    majority_pred,
    average="macro",
    zero_division=0,
)

print("Majority class:", majority_class)
print("Accuracy:", round(majority_accuracy, 4))
print("Macro F1:", round(majority_macro_f1, 4))

Majority class: PRODUCT_PROBLEM
Accuracy: 0.0
Macro F1: 0.0


In [190]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
X_val_tfidf = tfidf.transform(X_val)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)

Train TF-IDF shape: (2, 2708)
Validation TF-IDF shape: (1, 2708)
Train TF-IDF shape: (2, 2708)
Validation TF-IDF shape: (1, 2708)


In [191]:
baseline_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

baseline_model.fit(X_train_tfidf, y_train)

tfidf_pred = baseline_model.predict(X_val_tfidf)

tfidf_accuracy = accuracy_score(y_val, tfidf_pred)
tfidf_macro_f1 = f1_score(
    y_val,
    tfidf_pred,
    average="macro",
    zero_division=0,
)

print("TF-IDF + Logistic Regression")
print("Accuracy:", round(tfidf_accuracy, 4))
print("Macro F1:", round(tfidf_macro_f1, 4))

TF-IDF + Logistic Regression
Accuracy: 0.0
Macro F1: 0.0


In [192]:
print(
    classification_report(
        y_val,
        tfidf_pred,
        zero_division=0,
    )
)

                 precision    recall  f1-score   support

 DELIVERY_DELAY       0.00      0.00      0.00       1.0
PRODUCT_PROBLEM       0.00      0.00      0.00       0.0

       accuracy                           0.00       1.0
      macro avg       0.00      0.00      0.00       1.0
   weighted avg       0.00      0.00      0.00       1.0



In [193]:
# Compare available text representations

input_views = {
    "first_customer_message": training_df["first_customer_message"],
    "last_customer_message": training_df["last_customer_message"],
    "conversation_text": training_df["conversation_text"],
}

for name, series in input_views.items():
    series = series.fillna("").astype(str)

    print(f"\n{name}")
    print("-" * 50)
    print("Empty:", series.str.strip().eq("").sum())
    print("Average characters:", round(series.str.len().mean(), 1))
    print("Median characters:", round(series.str.len().median(), 1))


first_customer_message
--------------------------------------------------
Empty: 0
Average characters: 131.9
Median characters: 131.5

last_customer_message
--------------------------------------------------
Empty: 0
Average characters: 107.8
Median characters: 103.0

conversation_text
--------------------------------------------------
Empty: 0
Average characters: 926.1
Median characters: 592.0


In [194]:
# Show a few examples so we can verify exactly what the classifier sees

for i in range(5):
    print("=" * 80)
    print(f"EXAMPLE {i}")
    
    print("\nFIRST CUSTOMER MESSAGE:")
    print(training_df.loc[i, "first_customer_message"])
    
    print("\nLAST CUSTOMER MESSAGE:")
    print(training_df.loc[i, "last_customer_message"])
    
    print("\nCOMPLETE CONVERSATION:")
    print(training_df.loc[i, "conversation_text"][:1000])

EXAMPLE 0

FIRST CUSTOMER MESSAGE:
I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.

LAST CUSTOMER MESSAGE:
@AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.

COMPLETE CONVERSATION:
[CUSTOMER] I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.
[SUPPORT] @204457 We never want to let our customers down! Is there an issue with one of your current orders: https://t.co/3kQwjObgTc? ^GM
[CUSTOMER] @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +
[CUSTOMER] @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.
[SUPPORT] @204457 

In [195]:
labeled_mask = (
    training_df["intent_label"].notna()
    & training_df["intent_label"].astype(str).str.strip().ne("")
    & training_df["intent_label"].astype(str).str.upper().ne("UNLABELED")
)

labeled_df = training_df.loc[labeled_mask].copy()

X_customer = (
    labeled_df["first_customer_message"]
    .fillna("")
    .astype(str)
)

y = labeled_df["intent_label"].astype(str).str.strip()

stratify_labels = y if y.value_counts().min() >= 2 else None

X_train_customer, X_val_customer, y_train_customer, y_val_customer = (
    train_test_split(
        X_customer,
        y,
        test_size=0.25,
        random_state=42,
        stratify=stratify_labels,
    )
)

print("Training:", len(X_train_customer))
print("Validation:", len(X_val_customer))

Training: 2
Validation: 1


In [196]:
customer_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_customer_tfidf = customer_tfidf.fit_transform(
    X_train_customer
)

X_val_customer_tfidf = customer_tfidf.transform(
    X_val_customer
)

print(
    "Training TF-IDF shape:",
    X_train_customer_tfidf.shape
)

print(
    "Validation TF-IDF shape:",
    X_val_customer_tfidf.shape
)

Training TF-IDF shape: (2, 95)
Validation TF-IDF shape: (1, 95)


In [197]:
customer_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,       # allow vocab to be learned even with very small labeled sets
    max_df=1.0,     # avoid invalid max_df/min_df conflict on tiny splits
    sublinear_tf=True,
)

X_train_customer_tfidf = customer_tfidf.fit_transform(
    X_train_customer
)

X_val_customer_tfidf = customer_tfidf.transform(
    X_val_customer
)

customer_baseline_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

customer_baseline_model.fit(
    X_train_customer_tfidf,
    y_train_customer,
)

customer_pred = customer_baseline_model.predict(
    X_val_customer_tfidf
)

customer_accuracy = accuracy_score(
    y_val_customer,
    customer_pred,
)

customer_macro_f1 = f1_score(
    y_val_customer,
    customer_pred,
    average="macro",
    zero_division=0,
)

print("TF-IDF + Logistic Regression")
print("Accuracy:", round(customer_accuracy, 4))
print("Macro F1:", round(customer_macro_f1, 4))

TF-IDF + Logistic Regression
Accuracy: 1.0
Macro F1: 1.0


In [198]:
print(
    classification_report(
        y_val_customer,
        customer_pred,
        zero_division=0,
    )
)

                precision    recall  f1-score   support

DELIVERY_DELAY       1.00      1.00      1.00         1

      accuracy                           1.00         1
     macro avg       1.00      1.00      1.00         1
  weighted avg       1.00      1.00      1.00         1



In [199]:
baseline_comparison = pd.DataFrame({
    "Model": [
        "Majority Class",
        "TF-IDF + Logistic Regression (Full Conversation)",
        "TF-IDF + Logistic Regression (Customer Only)",
    ],
    "Accuracy": [
        majority_accuracy,
        tfidf_accuracy,
        customer_accuracy,
    ],
    "Macro F1": [
        majority_macro_f1,
        tfidf_macro_f1,
        customer_macro_f1,
    ],
})

baseline_comparison

,Model,Accuracy,Macro F1
0,Majority Class,0.0,0.0
1,TF-IDF + Logistic Regression (Full Conversation),0.0,0.0
2,TF-IDF + Logistic Regression (Customer Only),1.0,1.0


In [200]:
prediction_analysis = pd.DataFrame({
    "text": X_val_customer.values,
    "actual": y_val_customer.values,
    "predicted": customer_pred,
})

prediction_analysis["correct"] = (
    prediction_analysis["actual"]
    == prediction_analysis["predicted"]
)

prediction_analysis.head(10)

,text,actual,predicted,correct
0,"I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop.",DELIVERY_DELAY,DELIVERY_DELAY,True


In [201]:
labels_order = sorted(y.unique())

cm = confusion_matrix(
    y_val_customer,
    customer_pred,
    labels=labels_order,
)

cm_df = pd.DataFrame(
    
    cm,
    index=labels_order,
    columns=labels_order,
)

cm_df

,DELIVERY_DELAY,PRODUCT_PROBLEM
DELIVERY_DELAY,1,0
PRODUCT_PROBLEM,0,0


In [202]:
errors = prediction_analysis[
    ~prediction_analysis["correct"]
].copy()

print("Total validation examples:", len(prediction_analysis))
print("Incorrect predictions:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(prediction_analysis), 4)
)

errors[
    ["actual", "predicted", "text"]
].head(30).to_string(index=False)

Total validation examples: 1
Incorrect predictions: 0
Error rate: 0.0


'Empty DataFrame\nColumns: [actual, predicted, text]\nIndex: []'

In [203]:
customer_probabilities = customer_baseline_model.predict_proba(
    X_val_customer_tfidf
)

confidence = customer_probabilities.max(axis=1)

prediction_analysis["confidence"] = confidence

prediction_analysis.sort_values(
    "confidence",
    ascending=False
).head(20)[
    ["actual", "predicted", "confidence", "text"]
]

,actual,predicted,confidence,text
0,DELIVERY_DELAY,DELIVERY_DELAY,0.535609,"I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop."


In [204]:
prediction_analysis[
    ~prediction_analysis["correct"]
].sort_values(
    "confidence",
    ascending=False
).head(20)[
    ["actual", "predicted", "confidence", "text"]
].to_string(index=False)

'Empty DataFrame\nColumns: [actual, predicted, confidence, text]\nIndex: []'

In [205]:
import re

def extract_customer_messages(conversation_text):
    """
    Extract only customer-authored messages from the reconstructed
    conversation while excluding all historical support responses.
    """
    if pd.isna(conversation_text):
        return ""

    text = str(conversation_text)

    messages = re.findall(
        r"\[CUSTOMER\]\s*(.*?)(?=\s*\[SUPPORT\]|\Z)",
        text,
        flags=re.DOTALL,
    )

    return " ".join(
        message.strip()
        for message in messages
        if message.strip()
    )


training_df["customer_only_text"] = (
    training_df["conversation_text"]
    .apply(extract_customer_messages)
)

print(
    "Empty customer-only conversations:",
    
    training_df["customer_only_text"].str.strip().eq("").sum()
)

print(
    "Average characters:",
    round(training_df["customer_only_text"].str.len().mean(), 1)
)

print(
    "Median characters:",
    round(training_df["customer_only_text"].str.len().median(), 1)
)

Empty customer-only conversations: 0
Average characters: 486.8
Median characters: 289.5


In [206]:
for i in range(5):
    print("=" * 80)
    print(f"EXAMPLE {i}")
    print(training_df.loc[i, "customer_only_text"])

EXAMPLE 0
I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop. @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +
[CUSTOMER] @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.
EXAMPLE 1
@115821  Netboon tempered glass purchased for BlackBerry Priv @ Rs 1180 in January 2017 lost adhesive, falls off on its own. Need replacement. Netboon not responding. Amazon to help [Ordered on Jan 4, 2017 (406-9818731-9588334)] @115821 Netboon refused to help. Temperd glass purchased for Rs 1180 does not stick to phone panel. Amazon enjoy being helpless to help the genuine buyer. @AmazonHelp Amazon- Do you sell product to survive only return expiration period. Go to CP in Delhi. Tempered glass is available for merely Rs 100

In [207]:
X_customer_context = (
    training_df["customer_only_text"]
    .fillna("")
    .astype(str)
)

y = training_df["intent_label"].astype(str)

# Keep only truly labeled rows; unlabeled / empty / "UNLABELED" entries
# will otherwise create NaN classes and fail StratifiedKFold/TrainTestSplit.
labeled_mask = (
    training_df["intent_label"].notna()
    & training_df["intent_label"].astype(str).str.strip().ne("")
    & training_df["intent_label"].astype(str).str.upper().ne("UNLABELED")
)

labeled_df = training_df.loc[labeled_mask].copy()

X_customer_context = (
    labeled_df["customer_only_text"]
    .fillna("")
    .astype(str)
)

y = labeled_df["intent_label"].astype(str).str.strip()

# Only stratify if every class has at least 2 examples
stratify_labels = y if y.value_counts().min() >= 2 else None

X_train_context, X_val_context, y_train_context, y_val_context = (
    train_test_split(
        X_customer_context,
        y,
        test_size=0.25,
        random_state=42,
        stratify=stratify_labels,
    )
)

In [208]:
context_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=1.0,
    sublinear_tf=True,
)

X_train_context_tfidf = context_tfidf.fit_transform(
    X_train_context
)

X_val_context_tfidf = context_tfidf.transform(
    X_val_context
)

print("Training shape:", X_train_context_tfidf.shape)
print("Validation shape:", X_val_context_tfidf.shape)

Training shape: (2, 34)
Validation shape: (1, 34)


In [209]:
context_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0,
    sublinear_tf=True,
)

X_train_context_tfidf = context_tfidf.fit_transform(
    X_train_context
)

X_val_context_tfidf = context_tfidf.transform(
    X_val_context
)

context_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

context_model.fit(
    X_train_context_tfidf,
    y_train_context,
)

context_pred = context_model.predict(
    X_val_context_tfidf
)

context_accuracy = accuracy_score(
    y_val_context,
    context_pred,
)

context_macro_f1 = f1_score(
    y_val_context,
    context_pred,
    average="macro",
    zero_division=0,
)

print("Customer-Only Context TF-IDF + Logistic Regression")
print("Accuracy:", round(context_accuracy, 4))
print("Macro F1:", round(context_macro_f1, 4))

Customer-Only Context TF-IDF + Logistic Regression
Accuracy: 0.0
Macro F1: 0.0


In [210]:
print(
    classification_report(
        y_val_context,
        context_pred,
        zero_division=0,
    )
)

                 precision    recall  f1-score   support

 DELIVERY_DELAY       0.00      0.00      0.00       1.0
PRODUCT_PROBLEM       0.00      0.00      0.00       0.0

       accuracy                           0.00       1.0
      macro avg       0.00      0.00      0.00       1.0
   weighted avg       0.00      0.00      0.00       1.0



In [211]:
baseline_comparison = pd.DataFrame({
    "Model": [
        "Majority Class",
        "TF-IDF + Logistic Regression (Full Conversation)",
        "TF-IDF + Logistic Regression (First Customer Message)",
        "TF-IDF + Logistic Regression (Customer-Only Context)",
    ],
    "Accuracy": [
        majority_accuracy,
        tfidf_accuracy,
        customer_accuracy,
        context_accuracy,
    ],
    "Macro F1": [
        majority_macro_f1,
        tfidf_macro_f1,
        customer_macro_f1,
        context_macro_f1,
    ],
})

baseline_comparison

,Model,Accuracy,Macro F1
0,Majority Class,0.0,0.0
1,TF-IDF + Logistic Regression (Full Conversation),0.0,0.0
2,TF-IDF + Logistic Regression (First Customer Message),1.0,1.0
3,TF-IDF + Logistic Regression (Customer-Only Context),0.0,0.0


Adding subsequent customer messages increased accuracy from 9.33% → 25.33%

#### Stronger Classical Intent Classifier

Twitter support conversations contain spelling variations, abbreviations,
usernames, URLs, and informal language.

We therefore evaluate a combined word-level and character-level TF-IDF
representation with a Linear SVM classifier.

This provides a stronger classical baseline before introducing
embedding-based models.

In [212]:
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

In [213]:
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

char_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="char",
    ngram_range=(3, 5),
    min_df=1,
    sublinear_tf=True,
)

combined_vectorizer = FeatureUnion([
    ("word", word_vectorizer),
    ("char", char_vectorizer),
])

In [214]:
X_train_combined = combined_vectorizer.fit_transform(
    X_train_context
)

X_val_combined = combined_vectorizer.transform(
    X_val_context
)

print("Training shape:", X_train_combined.shape)
print("Validation shape:", X_val_combined.shape)

Training shape: (2, 14896)
Validation shape: (1, 14896)


In [215]:
svm_model = LinearSVC(
    class_weight="balanced",
    C=1.0,
    random_state=42,
)

svm_model.fit(
    X_train_combined,
    y_train_context,
)

svm_pred = svm_model.predict(
    X_val_combined
)

In [216]:
svm_accuracy = accuracy_score(
    y_val_context,
    svm_pred,
)

svm_macro_f1 = f1_score(
    y_val_context,
    svm_pred,
    average="macro",
    zero_division=0,
)

print("Word + Character TF-IDF + Linear SVM")
print("Accuracy:", round(svm_accuracy, 4))
print("Macro F1:", round(svm_macro_f1, 4))

Word + Character TF-IDF + Linear SVM
Accuracy: 0.0
Macro F1: 0.0


In [217]:
print(
    classification_report(
        y_val_context,
        svm_pred,
        zero_division=0,
    )
)

                 precision    recall  f1-score   support

 DELIVERY_DELAY       0.00      0.00      0.00       1.0
PRODUCT_PROBLEM       0.00      0.00      0.00       0.0

       accuracy                           0.00       1.0
      macro avg       0.00      0.00      0.00       1.0
   weighted avg       0.00      0.00      0.00       1.0



In [218]:
baseline_comparison = pd.DataFrame({
    "Model": [
        "Majority Class",
        "TF-IDF + Logistic Regression (Full Conversation)",
        "TF-IDF + Logistic Regression (First Customer Message)",
        "TF-IDF + Logistic Regression (Customer-Only Context)",
        "Word + Character TF-IDF + Linear SVM (Customer-Only Context)",
    ],
    "Accuracy": [
        majority_accuracy,
        tfidf_accuracy,
        customer_accuracy,
        context_accuracy,
        svm_accuracy,
    ],
    "Macro F1": [
        majority_macro_f1,
        tfidf_macro_f1,
        customer_macro_f1,
        context_macro_f1,
        svm_macro_f1,
    ],
})

baseline_comparison

,Model,Accuracy,Macro F1
0,Majority Class,0.0,0.0
1,TF-IDF + Logistic Regression (Full Conversation),0.0,0.0
2,TF-IDF + Logistic Regression (First Customer Message),1.0,1.0
3,TF-IDF + Logistic Regression (Customer-Only Context),0.0,0.0
4,Word + Character TF-IDF + Linear SVM (Customer-Only Context),0.0,0.0


Classical lexical models are struggling with the semantic variation and noisy language in these Twitter conversations.

In [219]:
try:
    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__)
except ImportError:
    print("sentence-transformers is NOT installed")

sentence-transformers: 5.1.0


#### Semantic Intent Classifier — Sentence Embeddings

Lexical TF-IDF models struggle with the noisy and semantically varied
language in Twitter support conversations.

We therefore represent each customer-only conversation using a pretrained
Sentence Transformer embedding and train a lightweight Logistic Regression
classifier on top.

The pretrained encoder is not fine-tuned on the evaluation data.

In [220]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [221]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Embedding model loaded.


In [222]:
X_train_context
X_val_context
y_train_context
y_val_context

print("Training examples:", len(X_train_context))
print("Validation examples:", len(X_val_context))
print("Training labels:", len(y_train_context))
print("Validation labels:", len(y_val_context))

Training examples: 2
Validation examples: 1
Training labels: 2
Validation labels: 1


In [223]:
X_train_embeddings = embedding_model.encode(
    X_train_context.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_val_embeddings = embedding_model.encode(
    X_val_context.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Training embedding shape:", X_train_embeddings.shape)
print("Validation embedding shape:", X_val_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Training embedding shape: (2, 384)
Validation embedding shape: (1, 384)


In [224]:
embedding_model_classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

embedding_model_classifier.fit(
    X_train_embeddings,
    y_train_context,
)

LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

In [225]:
embedding_pred = embedding_model_classifier.predict(
    X_val_embeddings
)

In [226]:
embedding_accuracy = accuracy_score(
    y_val_context,
    embedding_pred,
)

embedding_macro_f1 = f1_score(
    y_val_context,
    embedding_pred,
    average="macro",
    zero_division=0,
)

print("Sentence Embeddings + Logistic Regression")
print("Accuracy:", round(embedding_accuracy, 4))
print("Macro F1:", round(embedding_macro_f1, 4))

Sentence Embeddings + Logistic Regression
Accuracy: 1.0
Macro F1: 1.0


In [227]:
print(
    classification_report(
        y_val_context,
        embedding_pred,
        zero_division=0,
    )
)

                precision    recall  f1-score   support

DELIVERY_DELAY       1.00      1.00      1.00         1

      accuracy                           1.00         1
     macro avg       1.00      1.00      1.00         1
  weighted avg       1.00      1.00      1.00         1



In [228]:
baseline_comparison = pd.DataFrame({
    "Model": [
        "Majority Class",
        "TF-IDF + Logistic Regression (Full Conversation)",
        "TF-IDF + Logistic Regression (First Customer Message)",
        "TF-IDF + Logistic Regression (Customer-Only Context)",
        "Word + Character TF-IDF + Linear SVM (Customer-Only Context)",
        "Sentence Embeddings + Logistic Regression",
    ],
    "Accuracy": [
        majority_accuracy,
        tfidf_accuracy,
        customer_accuracy,
        context_accuracy,
        svm_accuracy,
        embedding_accuracy,
    ],
    "Macro F1": [
        majority_macro_f1,
        tfidf_macro_f1,
        customer_macro_f1,
        context_macro_f1,
        svm_macro_f1,
        embedding_macro_f1,
    ],
})

baseline_comparison

,Model,Accuracy,Macro F1
0,Majority Class,0.0,0.0
1,TF-IDF + Logistic Regression (Full Conversation),0.0,0.0
2,TF-IDF + Logistic Regression (First Customer Message),1.0,1.0
3,TF-IDF + Logistic Regression (Customer-Only Context),0.0,0.0
4,Word + Character TF-IDF + Linear SVM (Customer-Only Context),0.0,0.0
5,Sentence Embeddings + Logistic Regression,1.0,1.0


In [229]:
print("resolution_episodes available:", "resolution_episodes" in globals())

if "resolution_episodes" in globals():
    print("Shape:", resolution_episodes.shape)
    print("Columns:")
    print(resolution_episodes.columns.tolist())
    

resolution_episodes available: True
Shape: (192718, 13)
Columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']


In [230]:
amazon_resolution_corpus = resolution_episodes[
    resolution_episodes["support_account"] == "AmazonHelp"
].copy()

amazon_resolution_corpus = amazon_resolution_corpus.reset_index()

print("Shape:", amazon_resolution_corpus.shape)
print("Columns:")
print(amazon_resolution_corpus.columns.tolist())

print("\nAmazonHelp conversations:", len(amazon_resolution_corpus))
print(
    "Unique conversation IDs:",
    amazon_resolution_corpus["conversation_id"].nunique()
)

Shape: (82246, 14)
Columns:
['conversation_id', 'first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket']

AmazonHelp conversations: 82246
Unique conversation IDs: 82246


In [231]:
display(
    amazon_resolution_corpus[
        [
            "conversation_id",
            "conversation_size",
            "customer_turns",
            "support_turns",
            "conversation_text",
        ]
    ].head(5)
)

,conversation_id,conversation_size,customer_turns,support_turns,conversation_text
0,272,7,4,3,[CUSTOMER] amazonのfireTVstickが見れない😢\n[SUPPORT] @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET\n[CUSTOMER] @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。\n[CUSTOMER] @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。\n[SUPPORT] @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET\n[CUSTOMER] @AmazonHelp こちらこそありがとうございました。\n[SUPPORT] @115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET
1,325,2,1,1,[CUSTOMER] amazonプライムビデオ、再生エラーが多いです\n[SUPPORT] @115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の再起動にて改善する場合がございますので、お試しください。改善しない場合は、状況を確認しご案内させていただきますのでこちらからカスタマーサービスまでご連絡ください。https://t.co/NtNAX2Qh2u ET
2,617,5,3,2,"[CUSTOMER] Way to drop the ball on customer service @115821 so pissed right now!\n[SUPPORT] @115820 I'm sorry we've let you down! Without providing any personal information, will you describe the issue? We'd love to help. ^TN\n[CUSTOMER] @AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day\n[SUPPORT] @115820 We'd like to take a further look into this with you! Please reach us by phone or chat here:..."
3,621,2,1,1,"[CUSTOMER] @115823 I want my amazon payments account CLOSED. dm me please.\n[SUPPORT] @115822 I am unable to affect your account via Twitter. For real time support, phone or chat use this link: https://t.co/hApLpMlfHN ^CH"
4,624,4,2,2,"[CUSTOMER] @115825 also, beim Addams Family-Film in Prime sind Bild und Ton nicht wirklich synchron. Wie kommt's?\n[SUPPORT] @115824 Hi, wir erhalten die Filme/Serien so vom jeweiligen Studio. Gebe ich aber direkt als Feedback dorthin weiter. Gruß ^JS\n[CUSTOMER] @AmazonHelp Okay, danke für die Info\n[SUPPORT] @115824 Wir haben zu danken. Schönen Abend noch. ^JS"


In [232]:
# create a retrieval text
amazon_resolution_corpus["retrieval_text"] = (
    amazon_resolution_corpus["conversation_text"]
    .fillna("")
    .astype(str)
)

print(
    amazon_resolution_corpus["retrieval_text"]
    .str.len()
    .describe()
)

count    82246.000000
mean       588.593464
std        649.037089
min         46.000000
25%        277.000000
50%        423.000000
75%        698.000000
max      50654.000000
Name: retrieval_text, dtype: float64


In [233]:
resolution_embeddings = embedding_model.encode(
    amazon_resolution_corpus["retrieval_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Resolution embedding shape:", resolution_embeddings.shape)

Batches:   0%|          | 0/1286 [00:00<?, ?it/s]

Resolution embedding shape: (82246, 384)


Build FAISS Index

In [234]:
import faiss
print("FAISS:", faiss.__version__)

FAISS: 1.11.0


In [235]:
embedding_dimension = resolution_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    resolution_embeddings.astype("float32")
)

print("FAISS index size:", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

FAISS index size: 82246
Embedding dimension: 384


In [236]:
# create a retrieval function
def retrieve_historical_resolutions(
    query,
    top_k=5,
):
    """
    Retrieve historically similar AmazonHelp conversations.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )

    scores, indices = faiss_index.search(
        query_embedding.astype("float32"),
        top_k,
    )

    results = amazon_resolution_corpus.iloc[
        indices[0]
    ].copy()

    results["similarity"] = scores[0]

    return results

In [237]:
query = (
    "My package was supposed to arrive yesterday "
    "but it still hasn't been delivered."
)

retrieved = retrieve_historical_resolutions(
    query,
    top_k=5,
)

retrieved[
    [
        "conversation_id",
        "similarity",
        "conversation_size",
        "customer_turns",
        "support_turns",
        "conversation_text",
    ]
]

,conversation_id,similarity,conversation_size,customer_turns,support_turns,conversation_text
26076,810790,0.822072,2,1,1,[CUSTOMER] S/o to @115821 for saying my package would be delivered yesterday.. and it still hasn’t came today lol\n[SUPPORT] @313190 I'm sorry it hasn't arrived yet! Reach out to us here: https://t.co/hApLpMlfHN so we can take a look into this delivery. ^AH
31787,1022541,0.769380,4,2,2,[CUSTOMER] @115821 whyyyyyyy did you say you delivered my package an hour ago and it still isn’t here????!???????????????????????\n[SUPPORT] @361962 I'm sorry you don't have your package! Here's a link to our Help page for situations like this: https://t.co/IZXXAsQ3jL ^BV\n[CUSTOMER] @AmazonHelp That link supplies Literally no help and still the package isn’t here. Thanks\n[SUPPORT] @361962 Orders like this typically arrive by the next business day. Please let us know if you don't have it by...
76127,2827311,0.767698,2,1,1,"[CUSTOMER] @115821 My package never arrived, says it was delivered 2 days ago and Im still waiting! There's no way of telling you this except TWITTER?!\n[SUPPORT] @787517 Hi Tony, sorry to hear that you haven't received your parcel yet. Have you tried these suggestions: https://t.co/gqSmOhsbTh? ^JJ"
56765,2142995,0.757764,2,1,1,[CUSTOMER] amazon says my package came 2 days ago but like i don’t have it this is a problem\n[SUPPORT] @630068 I'm sorry to hear it hasn't arrived. We'd like to look into the delivery and available options: https://t.co/hApLpMlfHN ^LI
59904,2285995,0.757157,2,1,1,"[CUSTOMER] I think @115821 and @118706 need to talk. I think @118706 lies to them constantly about when things are delivered. Sometimes it will even say delivered and come the next day. This one says today on Amazon, Monday according to USPS. https://t.co/Zr8Hv99bpd\n[SUPPORT] @664281 I'm sorry your package is running late, Stephen! If the item hasn't arrived by Monday, please phone us here: https://t.co/hApLpMlfHN to report this and discuss additional options. ^LB"


In [238]:
print("taxonomy_audit_df available:", "taxonomy_audit_df" in globals())

taxonomy_audit_df available: False


In [239]:
taxonomy_audit_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "taxonomy_audit_150.csv"
)

print("Shape:", taxonomy_audit_df.shape)

print("\nColumns:")
print(taxonomy_audit_df.columns.tolist())

print("\nMissing intent labels:")
print(taxonomy_audit_df["intent_label"].isna().sum())

print("\nIntent distribution:")
print(taxonomy_audit_df["intent_label"].value_counts().sort_index())

Shape: (150, 16)

Columns:
['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'intent_label', 'annotation_notes', 'annotator']

Missing intent labels:
0

Intent distribution:
intent_label
ACCOUNT_ACCESS_SECURITY              5
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     2
DELIVERY_DELAY                      38
DELIVERY_MISSING_OR_MISDELIVERED    13
DEVICE_TECHNICAL_SUPPORT             3
DIGITAL_CONTENT                      6
GIFT_CARD_PROMOTION                  2
ORDER_STATUS_OR_CANCELLATION         7
OTHER_NON_SUPPORT                   29
PAYMENT_BILLING                      5
PRIME_MEMBERSHIP                     5
PRODUCT_AVAILABILITY_INFORMATION     6
PRODUCT_PROBLEM                     17
RETURN_REPLACEMENT_REFUND            8
WEBSITE_APP_TECHNICAL                4
Name: count, dtype: 

In [240]:
# Match audit conversations against the AmazonHelp retrieval corpus
# using the complete reconstructed conversation text.

audit_texts = set(
    taxonomy_audit_df["conversation_text"]
    .fillna("")
    .astype(str)
)

amazon_resolution_corpus["is_audit_example"] = (
    amazon_resolution_corpus["conversation_text"]
    .fillna("")
    .astype(str)
    .isin(audit_texts)
)

print(
    "Audit conversations found in retrieval corpus:",
    amazon_resolution_corpus["is_audit_example"].sum()
)

print(
    "Retrieval corpus size:",
    len(amazon_resolution_corpus)
)

Audit conversations found in retrieval corpus: 150
Retrieval corpus size: 82246


In [241]:
audit_matches = amazon_resolution_corpus[
    amazon_resolution_corpus["is_audit_example"]
]

print(
    audit_matches[
        
        [
            "conversation_id",
            "intent_label" if "intent_label" in audit_matches.columns else "support_account",
            "conversation_text",
        ]
    ].head()
)

      conversation_id support_account  \
770             34483      AmazonHelp   
1140            52943      AmazonHelp   
1848            80798      AmazonHelp   
2160            96641      AmazonHelp   
3459           137048      AmazonHelp   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        conversation_text  
770                                                            [CUSTOMER] @115821 @115851 why do you continue to use @2150 I don’t get my items! How can “the door” sign for a package! Terrible service!\n[SUPPORT] @123475 Sorry for the poor exper

Create a leakage-safe retrieval corpus

In [242]:
# Exclude the 150 taxonomy-audit conversations from the retrieval corpus.

audit_conversation_ids = set(
    audit_matches["conversation_id"].astype(int)
)

amazon_retrieval_eval_corpus = amazon_resolution_corpus[
    ~amazon_resolution_corpus["conversation_id"].isin(
        audit_conversation_ids
    )
].copy()

print("Original AmazonHelp corpus:", len(amazon_resolution_corpus))
print("Audit conversations excluded:", len(audit_conversation_ids))
print(
    "Leakage-safe retrieval corpus:",
    len(amazon_retrieval_eval_corpus)
)

print(
    "Remaining unique conversation IDs:",
    amazon_retrieval_eval_corpus["conversation_id"].nunique()
)

Original AmazonHelp corpus: 82246
Audit conversations excluded: 150
Leakage-safe retrieval corpus: 82096
Remaining unique conversation IDs: 82096


Rebuild FAISS using only the 82,096 conversations

In [243]:
# Build a leakage-safe FAISS retrieval index.
# The 150 taxonomy-audit conversations are excluded.

retrieval_texts = (
    amazon_retrieval_eval_corpus["retrieval_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

resolution_embeddings_safe = embedding_model.encode(
    retrieval_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(
    "Resolution embedding shape:",
    resolution_embeddings_safe.shape
)

Batches:   0%|          | 0/1283 [00:00<?, ?it/s]

Resolution embedding shape: (82096, 384)


In [244]:
import faiss

retrieval_dimension = resolution_embeddings_safe.shape[1]

faiss_index_safe = faiss.IndexFlatIP(
    retrieval_dimension
)

faiss_index_safe.add(
    resolution_embeddings_safe.astype("float32")
)

print("FAISS index size:", faiss_index_safe.ntotal)
print("Embedding dimension:", retrieval_dimension)

FAISS index size: 82096
Embedding dimension: 384


In [245]:
def retrieve_historical_resolutions_safe(
    query,
    top_k=5,
):
    """
    Retrieve historically similar AmazonHelp conversations
    from the leakage-safe retrieval corpus.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )

    scores, indices = faiss_index_safe.search(
        query_embedding.astype("float32"),
        top_k,
    )

    results = amazon_retrieval_eval_corpus.iloc[
        indices[0]
    ].copy()

    results["similarity"] = scores[0]

    return results

In [246]:
test_query = (
    "My package was supposed to arrive yesterday "
    "but it still hasn't been delivered."
)

retrieved_safe = retrieve_historical_resolutions_safe(
    test_query,
    top_k=5,
)

retrieved_safe[
    [
        "conversation_id",
        "similarity",
        "conversation_size",
        "customer_turns",
        "support_turns",
        "conversation_text",
    ]
]

,conversation_id,similarity,conversation_size,customer_turns,support_turns,conversation_text
26076,810790,0.822072,2,1,1,[CUSTOMER] S/o to @115821 for saying my package would be delivered yesterday.. and it still hasn’t came today lol\n[SUPPORT] @313190 I'm sorry it hasn't arrived yet! Reach out to us here: https://t.co/hApLpMlfHN so we can take a look into this delivery. ^AH
31787,1022541,0.769380,4,2,2,[CUSTOMER] @115821 whyyyyyyy did you say you delivered my package an hour ago and it still isn’t here????!???????????????????????\n[SUPPORT] @361962 I'm sorry you don't have your package! Here's a link to our Help page for situations like this: https://t.co/IZXXAsQ3jL ^BV\n[CUSTOMER] @AmazonHelp That link supplies Literally no help and still the package isn’t here. Thanks\n[SUPPORT] @361962 Orders like this typically arrive by the next business day. Please let us know if you don't have it by...
76127,2827311,0.767698,2,1,1,"[CUSTOMER] @115821 My package never arrived, says it was delivered 2 days ago and Im still waiting! There's no way of telling you this except TWITTER?!\n[SUPPORT] @787517 Hi Tony, sorry to hear that you haven't received your parcel yet. Have you tried these suggestions: https://t.co/gqSmOhsbTh? ^JJ"
56765,2142995,0.757764,2,1,1,[CUSTOMER] amazon says my package came 2 days ago but like i don’t have it this is a problem\n[SUPPORT] @630068 I'm sorry to hear it hasn't arrived. We'd like to look into the delivery and available options: https://t.co/hApLpMlfHN ^LI
59904,2285995,0.757157,2,1,1,"[CUSTOMER] I think @115821 and @118706 need to talk. I think @118706 lies to them constantly about when things are delivered. Sometimes it will even say delivered and come the next day. This one says today on Amazon, Monday according to USPS. https://t.co/Zr8Hv99bpd\n[SUPPORT] @664281 I'm sorry your package is running late, Stephen! If the item hasn't arrived by Monday, please phone us here: https://t.co/hApLpMlfHN to report this and discuss additional options. ^LB"


In [247]:
# Prepare leakage-safe retrieval evaluation queries.
# The audit conversations are used ONLY as queries.
# They are NOT present in the retrieval corpus.

audit_eval = taxonomy_audit_df.copy()

def extract_customer_only(text):
    text = str(text)
    
    customer_messages = re.findall(
        r"\[CUSTOMER\]\s*(.*?)(?=\[SUPPORT\]|\Z)",
        text,
        flags=re.DOTALL,
    )
    
    return " ".join(
        msg.strip() for msg in customer_messages
        if msg.strip()
    )

audit_eval["customer_only_text"] = (
    audit_eval["conversation_text"]
    .fillna("")
    .map(extract_customer_only)
)

print("Evaluation queries:", len(audit_eval))
print("Missing customer-only queries:",
      audit_eval["customer_only_text"].eq("").sum())
print("Intent classes:", audit_eval["intent_label"].nunique())

Evaluation queries: 150
Missing customer-only queries: 0
Intent classes: 15


In [248]:
# Encode all 150 evaluation queries

audit_query_embeddings = embedding_model.encode(
    audit_eval["customer_only_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Query embedding shape:", audit_query_embeddings.shape)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Query embedding shape: (150, 384)


In [249]:
# Retrieve top-10 historical conversations for every audit query

TOP_K = 10

retrieval_scores, retrieval_indices = faiss_index_safe.search(
    audit_query_embeddings.astype("float32"),
    TOP_K,
)

print("Scores shape:", retrieval_scores.shape)
print("Indices shape:", retrieval_indices.shape)

Scores shape: (150, 10)
Indices shape: (150, 10)


In [250]:
import numpy as np

retrieval_results = []

for query_idx in range(len(audit_eval)):
    
    true_intent = audit_eval.iloc[query_idx]["intent_label"]
    
    retrieved_rows = amazon_retrieval_eval_corpus.iloc[
        retrieval_indices[query_idx]
    ]
    
    retrieved_intents = []
    
    for _, row in retrieved_rows.iterrows():
        # We need an intent label for retrieved historical examples.
        # The 150 audit labels cannot be used because those conversations
        # were explicitly excluded from the retrieval corpus.
        retrieved_intents.append(None)
    
    retrieval_results.append({
        "query_idx": query_idx,
        "true_intent": true_intent,
        "retrieved_intents": retrieved_intents,
    })

In [251]:
# Load the 300 labelled AmazonHelp training examples

training_benchmark = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazonhelp_intent_training_batch_01.csv"
)

training_benchmark = training_benchmark[
    [
        "conversation_id",
        "conversation_text",
        "intent_label",
    ]
].copy()

training_benchmark["benchmark_source"] = "training"

print("Training benchmark rows:", len(training_benchmark))
print(
    "Unique conversation IDs:",
    training_benchmark["conversation_id"].nunique()
)
print(
    "Missing intent labels:",
    training_benchmark["intent_label"].isna().sum()
)

Training benchmark rows: 300
Unique conversation IDs: 300
Missing intent labels: 297


In [252]:
audit_lookup = (
    amazon_resolution_corpus[
        ["conversation_id", "conversation_text"]
    ]
    .copy()
)

audit_benchmark = taxonomy_audit_df[
    [
        "conversation_text",
        "intent_label",
    ]
].copy()

audit_benchmark = audit_benchmark.merge(
    audit_lookup,
    on="conversation_text",
    how="left",
    validate="one_to_one",
)

audit_benchmark["benchmark_source"] = "taxonomy_audit"

print("Audit benchmark rows:", len(audit_benchmark))
print(
    "Missing audit conversation IDs:",
    audit_benchmark["conversation_id"].isna().sum()
)

Audit benchmark rows: 150
Missing audit conversation IDs: 0


In [253]:
retrieval_benchmark = pd.concat(
    [
        training_benchmark,
        audit_benchmark[
            [
                "conversation_id",
                "conversation_text",
                "intent_label",
                "benchmark_source",
            ]
        ],
    ],
    ignore_index=True,
)

retrieval_benchmark["conversation_id"] = (
    retrieval_benchmark["conversation_id"].astype(int)
)

print("Total retrieval queries:", len(retrieval_benchmark))
print(
    "Unique query conversations:",
    retrieval_benchmark["conversation_id"].nunique()
)

print("\nBenchmark source:")
print(retrieval_benchmark["benchmark_source"].value_counts())

Total retrieval queries: 450
Unique query conversations: 450

Benchmark source:
benchmark_source
training          300
taxonomy_audit    150
Name: count, dtype: int64


In [254]:
retrieval_benchmark["customer_only_text"] = (
    retrieval_benchmark["conversation_text"]
    .fillna("")
    .map(extract_customer_only)
)

print(
    "Empty customer queries:",
    retrieval_benchmark["customer_only_text"].eq("").sum()
)

print(
    "Average query length:",
    retrieval_benchmark["customer_only_text"].str.len().mean()
)

print(
    "Median query length:",
    retrieval_benchmark["customer_only_text"].str.len().median()
)

Empty customer queries: 0
Average query length: 461.38666666666666
Median query length: 275.5


In [255]:
retrieval_benchmark[
    [
        "conversation_id",
        "intent_label",
        "benchmark_source",
        "customer_only_text",
    ]
].head(5)

,conversation_id,intent_label,benchmark_source,customer_only_text
0,373303,DELIVERY_DELAY,training,"I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop. @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +\n[CUSTOMER] @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then."
1,282739,PRODUCT_PROBLEM,training,"@115821 Netboon tempered glass purchased for BlackBerry Priv @ Rs 1180 in January 2017 lost adhesive, falls off on its own. Need replacement. Netboon not responding. Amazon to help [Ordered on Jan 4, 2017 (406-9818731-9588334)] @115821 Netboon refused to help. Temperd glass purchased for Rs 1180 does not stick to phone panel. Amazon enjoy being helpless to help the genuine buyer. @AmazonHelp Amazon- Do you sell product to survive only return expiration period. Go to CP in Delhi. Tempered gl..."
2,2292312,DELIVERY_DELAY,training,@AmazonHelp why offer 2 day shipping for prime when item will take 4 days to get it. #cancellingprime #nothappy @AmazonHelp I know that it's going to take 4 days transit time yet I pay monthly for 2 days. ITS SUPPOSED TO BE DELIVERED TODAY YET IT WILL BE DELIVERED MONDAY. @AmazonHelp No it's the stupid post office that always messes up my deliveries
3,570075,NaN,training,Amazonプライム！私はこの時をずっと待っていたぞ！！パージ3めちゃくちゃ待ったわぁぁぁぁ(^q^)
4,1427974,NaN,training,@115850 Vry bad service in Patna region.ordered 3 products in last few weeks.It was showing delivered bt it was not nt @AmazonHelp Filled the form n mentioned the order id of other two products as well.Please check @AmazonHelp Thank you for quick response @AmazonHelp One more product defaulted..Today I received a msg as delivered bt it was nt delivered to me.Pls check\n[CUSTOMER] @AmazonHelp This is a serious fraud issue.Can u share the details of customer wh is taking my order n signing on ...


In [256]:
print("Total rows:", len(retrieval_benchmark))

print(
    "Missing intent labels:",
    retrieval_benchmark["intent_label"].isna().sum()
)

print(
    "Missing conversation IDs:",
    retrieval_benchmark["conversation_id"].isna().sum()
)

print(
    "\nRows with missing intent labels:"
)

display(
    retrieval_benchmark[
        retrieval_benchmark["intent_label"].isna()
    ][
        [
            "conversation_id",
            "conversation_text",
            "benchmark_source",
        ]
    ].head(20)
)

Total rows: 450
Missing intent labels: 297
Missing conversation IDs: 0

Rows with missing intent labels:


,conversation_id,conversation_text,benchmark_source
3,570075,[CUSTOMER] Amazonプライム！私はこの時をずっと待っていたぞ！！パージ3めちゃくちゃ待ったわぁぁぁぁ(^q^)\n[SUPPORT] @253812 こちらですね・・・？( *• ̀ω•́ )b \nhttps://t.co/Zj70OB4ODe SK,training
4,1427974,"[CUSTOMER] @115850 Vry bad service in Patna region.ordered 3 products in last few weeks.It was showing delivered bt it was not nt\n[SUPPORT] @451696 I'm sorry about the issue you have with the refunds, fill this form: https://t.co/beaaDm0muc &amp; I’ll get it checked for you. ^SY\n[CUSTOMER] @AmazonHelp Filled the form n mentioned the order id of other two products as well.Please check\n[SUPPORT] @451696 Thank you for sharing your details. We'll work on it and reach out to you soon. ^AP\n[CU...",training
5,1717565,"[CUSTOMER] Wait what?! In first group to preorder and I get it in Dec? @115788 @AmazonHelp @115787 Can someone assist?!! https://t.co/PRl65hAXj3\n[SUPPORT] @332329 We'd like to help. What does the current status of your order state here: https://t.co/q4LAMZ3tbE? ^GG\n[CUSTOMER] @AmazonHelp https://t.co/IcayOfZZ3L\n[CUSTOMER] @AmazonHelp Says date is pending, but when I used chat with customer care it says Dec 8th\n[SUPPORT] @332329 I'm sorry the delivery date is unavailable! Please keep an ...",training
6,2132842,"[CUSTOMER] @AmazonHelp Waited in all day for a guaranteed Prime Delivery that hasn't arrived, this is the 2nd time in a row this has happened, a courier problem maybe? Not good is it!\n[SUPPORT] @627584 So sorry for the delay! We do deliver until 21:00 so let us know if it doesn't arrive by then! ^AD\n[CUSTOMER] @AmazonHelp Don't think its coming, do you? https://t.co/88dDAhk5yb\n[CUSTOMER] @AmazonHelp @627584 My tracker has changed to that now, yet the order details page STILL states that i...",training
7,2289374,[CUSTOMER] アマゾンの会員登録しました！！！\n[SUPPORT] @665098 リプライ失礼いたします、Amazonです。プライムのご登録ありがとうございます！\nプライムビデオやプライムミュージックなど、さまざまな特典をご用意しておりますので、ぜひご活用くださいませ🙇 https://t.co/AbfeovjCgG HM\n[CUSTOMER] @AmazonHelp プライム登録はしてません…すいません…\n[SUPPORT] @665098 すみません、こちらこそ早とちりをしてしまい・・🙇🙏\n機会がございましたら、プライムもご検討くださいませ！HM,training
8,1514342,"[CUSTOMER] Effarant le nombre de loupés de distribution de @14967 ! Plus jamais ça lors de mes commandes @120533 ! Vivent les points relais !\n[SUPPORT] @471369 Bonjour, rencontrez-vous un problème avec une livraison en cours?^FT",training
9,42989,"[CUSTOMER] @117086 avisem pros entregadores de vocês que é feio jogar livro na garagem dos outros quando não tem ninguém em casa\n[SUPPORT] @125530 Olá Hellen! Sinto muito pelo acontecido. O seu pedido chegou danificado? ^JJ\n[CUSTOMER] @AmazonHelp Felizmente, não. Mas faz eu pensar duas vezes antes de comprar de novo algo\n[SUPPORT] @125530 Hellen, neste caso o pedido foi enviado pela Amazon ou por um vendedor?. ^NB\n[CUSTOMER] @AmazonHelp Pela Amazon\n[SUPPORT] @125530 Agradecemos o seu f...",training
10,1345839,"[CUSTOMER] Kann mir jemand einen Kuchen bringen, während ich auf den Kurier warte, der heute angeblich meine @116316-Rücksendung holt? #langwelig\n[SUPPORT] @433820 🍰\nHast du ein Zeitfenster zur Abholung bekommen? ^SM\n[CUSTOMER] @AmazonHelp Nein. Der Händler schreibt nur extrem unverständliches Deutsch oder Spanisch (was ich nicht kann), behauptet aber, da kommt heute jemand.\n[CUSTOMER] @AmazonHelp Er nannte weder Zeitfenster noch Dienstleister, leider hab ich auch von Amazon (also euch) ...",training
11,2311266,[CUSTOMER] I may be filing my first @115821 complaint ever. Seller has two days to get back to me . . .\n[SUPPORT] @670309 I'm sorry to hear there's been trouble with a recent order! Please keep us updated on the response from the seller. ^LR,training
12,944864,"[CUSTOMER] @AmazonHelp Bonjour j'ai commandé un colis le 15/10, selon le suivi de la commande, il est censé avoir été livré, mais je n'ai rien reçu..\n[SUPPORT] @344046 Bonjour, désolée pour cela. L'avez-vous signalé à notre SAV ? ^MA\n[CUSTOMER] @AmazonHelp Non, je ne l'ai pas signalé, pouvez-vous m'indiquer la démarche à suivre svp?\n[SUPPORT] @344046 Vous 

In [257]:
print(
    "Training CSV rows:",
    len(training_benchmark)
)

print(
    "Training missing labels:",
    training_benchmark["intent_label"].isna().sum()
)

print(
    "Training label counts:"
)

display(
    training_benchmark["intent_label"].value_counts(
        dropna=False
    )
)

Training CSV rows: 300
Training missing labels: 297
Training label counts:


intent_label
NaN                297
DELIVERY_DELAY       2
PRODUCT_PROBLEM      1
Name: count, dtype: int64

In [258]:
# Show currently available notebook variables that may contain
# the labelled 300-example training set.

%whos DataFrame

Variable                            Type         Data/Info
----------------------------------------------------------
amazon                              DataFrame           conversation_id  \<...>[82246 rows x 15 columns]
amazon_conversations                DataFrame           conversation_id  \<...>[82246 rows x 14 columns]
amazon_resolution_corpus            DataFrame           conversation_id  \<...>[82246 rows x 16 columns]
amazon_retrieval_eval_corpus        DataFrame           conversation_id  \<...>[82096 rows x 16 columns]
amazonhelp_conversation_splits_v2   DataFrame           conversation_id  \<...>[82246 rows x 16 columns]
amazonhelp_splits                   DataFrame           conversation_id  \<...>[82246 rows x 17 columns]
annotation_batch                    DataFrame        conversation_id suppo<...>t there as well.  2 ^XS  
audit_benchmark                     DataFrame                             <...>n\n[150 rows x 4 columns]
audit_eval                          DataFr

In [259]:
candidates = [
    "training_batch",
    "training_batch_1",
    "training_df",
    "labeled_df",
    "batch_1",
    "batch_2",
    "batch_3",
    "batch_4",
    "batch_5",
    "batch_6",
    "batch_7",
]

for name in candidates:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            print(f"\n{name}")
            print("  Shape:", obj.shape)

            if "intent_label" in obj.columns:
                print(
                    "  Missing intent labels:",
                    obj["intent_label"].isna().sum()
                )
                print(
                    "  Unique labels:",
                    obj["intent_label"].nunique(dropna=True)
                )
                print(
                    "  Status:",
                    obj["annotation_status"].value_counts(dropna=False).to_dict()
                    if "annotation_status" in obj.columns
                    else "no annotation_status column"
                )
            else:
                print("  No intent_label column")


training_batch
  Shape: (300, 20)
  Missing intent labels: 300
  Unique labels: 0
  Status: {'UNLABELED': 300}

training_batch_1
  Shape: (300, 20)
  Missing intent labels: 300
  Unique labels: 0
  Status: {'UNLABELED': 300}

training_df
  Shape: (300, 21)
  Missing intent labels: 297
  Unique labels: 2
  Status: {'UNLABELED': 300}

labeled_df
  Shape: (3, 21)
  Missing intent labels: 0
  Unique labels: 2
  Status: {'UNLABELED': 3}

batch_1
  Shape: (30, 9)
  Missing intent labels: 0
  Unique labels: 1
  Status: no annotation_status column

batch_2
  Shape: (20, 11)
  No intent_label column

batch_3
  Shape: (20, 11)
  No intent_label column

batch_4
  Shape: (20, 11)
  No intent_label column

batch_5
  Shape: (20, 11)
  No intent_label column

batch_6
  Shape: (20, 11)
  No intent_label column

batch_7
  Shape: (8, 11)
  No intent_label column


In [260]:
for name in [
    "batch_1",
    "batch_2",
    "batch_3",
    "batch_4",
    "batch_5",
    "batch_6",
    "batch_7",
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{'=' * 60}")
        print(name)
        print("Shape:", obj.shape)
        print("Columns:")
        print(list(obj.columns))


batch_1
Shape: (30, 9)
Columns:
['first_customer_message', 'last_customer_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'size_bucket', 'intent_label', 'annotation_notes']

batch_2
Shape: (20, 11)
Columns:
['conversation_id', 'support_account', 'size_bucket', 'conversation_size', 'customer_turns', 'support_turns', 'duration_minutes', 'conversation_text', 'resolution_label', 'annotation_notes', 'annotator']

batch_3
Shape: (20, 11)
Columns:
['conversation_id', 'support_account', 'size_bucket', 'conversation_size', 'customer_turns', 'support_turns', 'duration_minutes', 'conversation_text', 'resolution_label', 'annotation_notes', 'annotator']

batch_4
Shape: (20, 11)
Columns:
['conversation_id', 'support_account', 'size_bucket', 'conversation_size', 'customer_turns', 'support_turns', 'duration_minutes', 'conversation_text', 'resolution_label', 'annotation_notes', 'annotator']

batch_5
Shape: (20, 11)
Columns:
['conversation_id', 'support_account', 

In [261]:
# Restore the 300 previously completed human intent annotations.
# training_df is the original 300-row annotation batch in the same order
# used during our manual review.

intent_labels_300 = [
    # 0-29
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "DIGITAL_CONTENT",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "ORDER_STATUS_OR_CANCELLATION",
    "DELIVERY_DELAY",
    "PRIME_MEMBERSHIP",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "RETURN_REPLACEMENT_REFUND",
    "OTHER_NON_SUPPORT",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DIGITAL_CONTENT",
    "RETURN_REPLACEMENT_REFUND",
    "DIGITAL_CONTENT",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "ORDER_STATUS_OR_CANCELLATION",
    "DELIVERY_DELAY",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "PAYMENT_BILLING",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",

    # 30-59
    "OTHER_NON_SUPPORT",
    "DIGITAL_CONTENT",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "RETURN_REPLACEMENT_REFUND",
    "PRODUCT_PROBLEM",
    "OTHER_NON_SUPPORT",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "OTHER_NON_SUPPORT",
    "ORDER_STATUS_OR_CANCELLATION",
    "WEBSITE_APP_TECHNICAL",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PRIME_MEMBERSHIP",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "PAYMENT_BILLING",
    "DELIVERY_DELAY",
    "PRIME_MEMBERSHIP",
    "DELIVERY_DELAY",
    "DEVICE_TECHNICAL_SUPPORT",
    "ORDER_STATUS_OR_CANCELLATION",
    "PRIME_MEMBERSHIP",
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",

    # 60-89
    "PAYMENT_BILLING",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRIME_MEMBERSHIP",
    "DELIVERY_DELAY",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRODUCT_PROBLEM",
    "DEVICE_TECHNICAL_SUPPORT",
    "OTHER_NON_SUPPORT",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "RETURN_REPLACEMENT_REFUND",
    "WEBSITE_APP_TECHNICAL",
    "PRODUCT_PROBLEM",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DEVICE_TECHNICAL_SUPPORT",
    "ORDER_STATUS_OR_CANCELLATION",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRIME_MEMBERSHIP",
    "DELIVERY_DELAY",
    "DEVICE_TECHNICAL_SUPPORT",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "ORDER_STATUS_OR_CANCELLATION",
    "OTHER_NON_SUPPORT",
    "PAYMENT_BILLING",
    "PRODUCT_PROBLEM",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",

    # 90-119
    "DELIVERY_DELAY",
    "ORDER_STATUS_OR_CANCELLATION",
    "OTHER_NON_SUPPORT",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DIGITAL_CONTENT",
    "OTHER_NON_SUPPORT",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",
    "PAYMENT_BILLING",
    "OTHER_NON_SUPPORT",
    "ORDER_STATUS_OR_CANCELLATION",
    "WEBSITE_APP_TECHNICAL",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_DELAY",
    "PAYMENT_BILLING",
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "OTHER_NON_SUPPORT",
    "GIFT_CARD_PROMOTION",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",

    # 120-149
    "RETURN_REPLACEMENT_REFUND",
    "RETURN_REPLACEMENT_REFUND",
    "PRODUCT_PROBLEM",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PRODUCT_PROBLEM",
    "ACCOUNT_ACCESS_SECURITY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "OTHER_NON_SUPPORT",
    "ORDER_STATUS_OR_CANCELLATION",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_DELAY",
    "OTHER_NON_SUPPORT",
    "ACCOUNT_ACCESS_SECURITY",
    "DIGITAL_CONTENT",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "OTHER_NON_SUPPORT",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRIME_MEMBERSHIP",
    "OTHER_NON_SUPPORT",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "ACCOUNT_ACCESS_SECURITY",
    "PRODUCT_PROBLEM",

    # 150-179
    "ACCOUNT_ACCESS_SECURITY",
    "PRODUCT_PROBLEM",
    "OTHER_NON_SUPPORT",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "PAYMENT_BILLING",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "WEBSITE_APP_TECHNICAL",
    "OTHER_NON_SUPPORT",
    "PRODUCT_PROBLEM",
    "DIGITAL_CONTENT",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "RETURN_REPLACEMENT_REFUND",
    "PRODUCT_PROBLEM",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",
    "OTHER_NON_SUPPORT",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "OTHER_NON_SUPPORT",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",

    # 180-209
    "OTHER_NON_SUPPORT",
    "RETURN_REPLACEMENT_REFUND",
    "ACCOUNT_ACCESS_SECURITY",
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PRODUCT_PROBLEM",
    "WEBSITE_APP_TECHNICAL",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "ACCOUNT_ACCESS_SECURITY",
    "PRODUCT_PROBLEM",
    "WEBSITE_APP_TECHNICAL",
    "WEBSITE_APP_TECHNICAL",
    "PRODUCT_PROBLEM",
    "OTHER_NON_SUPPORT",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRODUCT_PROBLEM",
    "RETURN_REPLACEMENT_REFUND",
    "ORDER_STATUS_OR_CANCELLATION",
    "PAYMENT_BILLING",
    "PRODUCT_PROBLEM",
    "GIFT_CARD_PROMOTION",
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "DELIVERY_MISSING_OR_MISDELIVERED",

    # 210-239
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PAYMENT_BILLING",
    "WEBSITE_APP_TECHNICAL",
    "PRODUCT_PROBLEM",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "GIFT_CARD_PROMOTION",
    "DELIVERY_DELAY",
    "DIGITAL_CONTENT",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "OTHER_NON_SUPPORT",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRIME_MEMBERSHIP",
    "DELIVERY_DELAY",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRODUCT_PROBLEM",

    # 240-269
    "DIGITAL_CONTENT",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PRODUCT_PROBLEM",
    "WEBSITE_APP_TECHNICAL",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "RETURN_REPLACEMENT_REFUND",
    "ACCOUNT_ACCESS_SECURITY",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DIGITAL_CONTENT",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "OTHER_NON_SUPPORT",
    "OTHER_NON_SUPPORT",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "ORDER_STATUS_OR_CANCELLATION",
    "OTHER_NON_SUPPORT",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "ORDER_STATUS_OR_CANCELLATION",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DIGITAL_CONTENT",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_DELAY",

    # 270-299
    "PAYMENT_BILLING",
    "PRODUCT_PROBLEM",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PRODUCT_PROBLEM",
    "RETURN_REPLACEMENT_REFUND",
    "DELIVERY_DELAY",
    "RETURN_REPLACEMENT_REFUND",
    "OTHER_NON_SUPPORT",
    "GIFT_CARD_PROMOTION",
    "DELIVERY_DELAY",
    "DIGITAL_CONTENT",
    "PRODUCT_PROBLEM",
    "DIGITAL_CONTENT",
    "ACCOUNT_ACCESS_SECURITY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "PAYMENT_BILLING",
    "PRODUCT_PROBLEM",
    "ACCOUNT_ACCESS_SECURITY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "OTHER_NON_SUPPORT",
    "DELIVERY_DELAY",
    "DELIVERY_DELAY",
    "PRODUCT_PROBLEM",
    "GIFT_CARD_PROMOTION",
    "ORDER_STATUS_OR_CANCELLATION",
]

# Safety checks before assigning anything
assert len(intent_labels_300) == 300
assert len(training_df) == 300

# Verify the first three IDs against the previously reviewed examples.
expected_first_ids = [373303, 282739, 2292312]

actual_first_ids = (
    training_df["conversation_id"]
    .astype(int)
    .head(3)
    .tolist()
)

assert actual_first_ids == expected_first_ids, (
    f"Unexpected training order: {actual_first_ids}"
)

# Restore labels
training_df = training_df.copy()
training_df["intent_label"] = intent_labels_300
training_df["annotation_status"] = "LABELED"

print("Rows:", len(training_df))
print(
    "Unique conversation IDs:",
    training_df["conversation_id"].nunique()
)
print(
    "Missing labels:",
    training_df["intent_label"].isna().sum()
)
print(
    "Unique intent labels:",
    training_df["intent_label"].nunique()
)
print(
    "Annotation status:",
    training_df["annotation_status"].value_counts().to_dict()
)

Rows: 300
Unique conversation IDs: 300
Missing labels: 0
Unique intent labels: 15
Annotation status: {'LABELED': 300}


In [262]:
# Replace the broken training_benchmark with the verified
# 300-example labelled dataframe.

training_benchmark = training_df[
    [
        "conversation_id",
        "conversation_text",
        "intent_label",
    ]
].copy()

training_benchmark["benchmark_source"] = "training"


# Rebuild the 150-example audit benchmark.
audit_lookup = amazon_resolution_corpus[
    [
        "conversation_id",
        "conversation_text",
    ]
].copy()

audit_benchmark = taxonomy_audit_df[
    [
        "conversation_text",
        "intent_label",
    ]
].copy()

audit_benchmark = audit_benchmark.merge(
    audit_lookup,
    on="conversation_text",
    how="left",
    validate="one_to_one",
)

audit_benchmark["benchmark_source"] = "taxonomy_audit"


# Combine the two labelled benchmark sets.
retrieval_benchmark = pd.concat(
    [
        training_benchmark,
        audit_benchmark[
            [
                "conversation_id",
                "conversation_text",
                "intent_label",
                "benchmark_source",
            ]
        ],
    ],
    ignore_index=True,
)

retrieval_benchmark["conversation_id"] = (
    retrieval_benchmark["conversation_id"].astype(int)
)


# Final validation.
print("Total retrieval queries:", len(retrieval_benchmark))
print(
    "Unique query conversations:",
    retrieval_benchmark["conversation_id"].nunique()
)
print(
    "Missing conversation IDs:",
    retrieval_benchmark["conversation_id"].isna().sum()
)
print(
    "Missing intent labels:",
    retrieval_benchmark["intent_label"].isna().sum()
)
print(
    "Unique intent labels:",
    retrieval_benchmark["intent_label"].nunique()
)

print("\nBenchmark source:")
print(
    retrieval_benchmark["benchmark_source"].value_counts()
)

print("\nIntent distribution:")
print(
    retrieval_benchmark["intent_label"].value_counts()
)

Total retrieval queries: 450
Unique query conversations: 450
Missing conversation IDs: 0
Missing intent labels: 0
Unique intent labels: 15

Benchmark source:
benchmark_source
training          300
taxonomy_audit    150
Name: count, dtype: int64

Intent distribution:
intent_label
DELIVERY_DELAY                      107
OTHER_NON_SUPPORT                    66
PRODUCT_PROBLEM                      49
DELIVERY_MISSING_OR_MISDELIVERED     45
RETURN_REPLACEMENT_REFUND            33
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     26
ORDER_STATUS_OR_CANCELLATION         20
DIGITAL_CONTENT                      19
PAYMENT_BILLING                      16
PRODUCT_AVAILABILITY_INFORMATION     15
ACCOUNT_ACCESS_SECURITY              14
PRIME_MEMBERSHIP                     13
WEBSITE_APP_TECHNICAL                13
DEVICE_TECHNICAL_SUPPORT              7
GIFT_CARD_PROMOTION                   7
Name: count, dtype: int64


In [263]:
retrieval_benchmark["customer_only_text"] = (
    retrieval_benchmark["conversation_text"]
    .fillna("")
    .map(extract_customer_only)
)

print("Empty customer queries:",
      retrieval_benchmark["customer_only_text"].eq("").sum())

print(
    "Average query length:",
    retrieval_benchmark["customer_only_text"].str.len().mean()
)

print(
    "Median query length:",
    retrieval_benchmark["customer_only_text"].str.len().median()
)

display(
    retrieval_benchmark[
        [
            "conversation_id",
            "intent_label",
            "benchmark_source",
            "customer_only_text",
        ]
    ].head(5)
)

Empty customer queries: 0
Average query length: 461.38666666666666
Median query length: 275.5


,conversation_id,intent_label,benchmark_source,customer_only_text
0,373303,DELIVERY_DELAY,training,"I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop. @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +\n[CUSTOMER] @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then."
1,282739,PRODUCT_PROBLEM,training,"@115821 Netboon tempered glass purchased for BlackBerry Priv @ Rs 1180 in January 2017 lost adhesive, falls off on its own. Need replacement. Netboon not responding. Amazon to help [Ordered on Jan 4, 2017 (406-9818731-9588334)] @115821 Netboon refused to help. Temperd glass purchased for Rs 1180 does not stick to phone panel. Amazon enjoy being helpless to help the genuine buyer. @AmazonHelp Amazon- Do you sell product to survive only return expiration period. Go to CP in Delhi. Tempered gl..."
2,2292312,DELIVERY_DELAY,training,@AmazonHelp why offer 2 day shipping for prime when item will take 4 days to get it. #cancellingprime #nothappy @AmazonHelp I know that it's going to take 4 days transit time yet I pay monthly for 2 days. ITS SUPPOSED TO BE DELIVERED TODAY YET IT WILL BE DELIVERED MONDAY. @AmazonHelp No it's the stupid post office that always messes up my deliveries
3,570075,DIGITAL_CONTENT,training,Amazonプライム！私はこの時をずっと待っていたぞ！！パージ3めちゃくちゃ待ったわぁぁぁぁ(^q^)
4,1427974,DELIVERY_MISSING_OR_MISDELIVERED,training,@115850 Vry bad service in Patna region.ordered 3 products in last few weeks.It was showing delivered bt it was not nt @AmazonHelp Filled the form n mentioned the order id of other two products as well.Please check @AmazonHelp Thank you for quick response @AmazonHelp One more product defaulted..Today I received a msg as delivered bt it was nt delivered to me.Pls check\n[CUSTOMER] @AmazonHelp This is a serious fraud issue.Can u share the details of customer wh is taking my order n signing on ...


In [264]:
benchmark_query_embeddings = embedding_model.encode(
    retrieval_benchmark["customer_only_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(
    "Query embedding shape:",
    benchmark_query_embeddings.shape
)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Query embedding shape: (450, 384)


In [265]:
# Final leakage-safe retrieval corpus.
# Exclude all 450 labelled benchmark conversations:
# 300 training + 150 taxonomy audit.

benchmark_conversation_ids = set(
    retrieval_benchmark["conversation_id"].astype(int)
)

amazon_retrieval_benchmark_corpus = (
    amazon_resolution_corpus[
        ~amazon_resolution_corpus["conversation_id"].isin(
            benchmark_conversation_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("Original AmazonHelp corpus:",
      len(amazon_resolution_corpus))

print("Benchmark conversations excluded:",
      len(benchmark_conversation_ids))

print("Final retrieval corpus:",
      len(amazon_retrieval_benchmark_corpus))

print("Unique conversation IDs:",
      amazon_retrieval_benchmark_corpus["conversation_id"].nunique())

# Hard leakage check
remaining_ids = set(
    amazon_retrieval_benchmark_corpus["conversation_id"].astype(int)
)

leaked_ids = benchmark_conversation_ids & remaining_ids

print("Leaked benchmark conversations:",
      len(leaked_ids))

assert len(amazon_retrieval_benchmark_corpus) == 81796
assert amazon_retrieval_benchmark_corpus["conversation_id"].nunique() == 81796
assert len(leaked_ids) == 0

print("✓ Final leakage-safe retrieval corpus validated.")

Original AmazonHelp corpus: 82246
Benchmark conversations excluded: 450
Final retrieval corpus: 81796
Unique conversation IDs: 81796
Leaked benchmark conversations: 0
✓ Final leakage-safe retrieval corpus validated.


In [266]:
# Build the final leakage-safe FAISS index.

benchmark_retrieval_texts = (
    amazon_retrieval_benchmark_corpus["retrieval_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

print(
    "Retrieval documents:",
    len(benchmark_retrieval_texts)
)

benchmark_resolution_embeddings = embedding_model.encode(
    benchmark_retrieval_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(
    "Embedding shape:",
    benchmark_resolution_embeddings.shape
)

faiss_index_benchmark = faiss.IndexFlatIP(
    benchmark_resolution_embeddings.shape[1]
)

faiss_index_benchmark.add(
    benchmark_resolution_embeddings.astype("float32")
)

print(
    "FAISS index size:",
    faiss_index_benchmark.ntotal
)

print(
    "FAISS dimension:",
    faiss_index_benchmark.d
)

assert faiss_index_benchmark.ntotal == 81796

print("✓ Leakage-safe FAISS index ready.")

Retrieval documents: 81796


Batches:   0%|          | 0/1279 [00:00<?, ?it/s]

Embedding shape: (81796, 384)
FAISS index size: 81796
FAISS dimension: 384
✓ Leakage-safe FAISS index ready.


In [267]:
# Encode the 450 labelled benchmark queries.
# Queries contain customer messages only.

benchmark_query_embeddings = embedding_model.encode(
    retrieval_benchmark["customer_only_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(
    "Query embedding shape:",
    benchmark_query_embeddings.shape
)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Query embedding shape: (450, 384)


In [268]:
# Retrieve the top 10 unseen historical conversations
# for every benchmark query.

TOP_K = 10

benchmark_scores, benchmark_indices = (
    faiss_index_benchmark.search(
        benchmark_query_embeddings.astype("float32"),
        TOP_K,
    )
)

print("Scores shape:", benchmark_scores.shape)
print("Indices shape:", benchmark_indices.shape)

Scores shape: (450, 10)
Indices shape: (450, 10)


In [269]:
# Inspect retrieval results for 10 benchmark queries.

for query_idx in range(10):
    
    query_row = retrieval_benchmark.iloc[query_idx]
    
    print("\n" + "=" * 100)
    print(
        f"QUERY {query_idx + 1} | "
        f"ID={query_row['conversation_id']} | "
        f"INTENT={query_row['intent_label']} | "
        f"SOURCE={query_row['benchmark_source']}"
    )
    
    print("\nCUSTOMER QUERY:")
    print(query_row["customer_only_text"][:1000])
    
    print("\nTOP RETRIEVED:")
    
    retrieved_rows = amazon_retrieval_benchmark_corpus.iloc[
        benchmark_indices[query_idx]
    ]
    
    for rank, (_, row) in enumerate(
        retrieved_rows.iterrows(),
        start=1
    ):
        print(
            f"\n[{rank}] "
            f"similarity={benchmark_scores[query_idx][rank-1]:.4f} "
            f"conversation_id={row['conversation_id']} "
            f"size={row['conversation_size']}"
        )
        
        print(
            row["conversation_text"][:700]
            .replace("\n", " ")
        )


QUERY 1 | ID=373303 | INTENT=DELIVERY_DELAY | SOURCE=training

CUSTOMER QUERY:
I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop. @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue +
[CUSTOMER] @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.

TOP RETRIEVED:

[1] similarity=0.7785 conversation_id=218089 size=2
[CUSTOMER] If you have problems with your @115821 same day delivery, @2150 might be involved. LMAO. https://t.co/Mq883LdLpi [SUPPORT] @167995 Hi! Are you currently experiencing an issue with an order? We'd like to help! ^WT

[2] similarity=0.7785 conversation_id=339472 size=2
[CUSTOMER] It's really frustrating trying to order things through @115821 lately. They seem to screw up shipment and deliver

In [270]:
# Create a 150-query retrieval evaluation sample.
#
# We use the already-labelled benchmark queries, but the retrieved
# conversations come only from the unseen 81,796-document corpus.

from sklearn.model_selection import train_test_split

# Separate the two benchmark sources
training_queries = retrieval_benchmark[
    retrieval_benchmark["benchmark_source"] == "training"
].copy()

audit_queries = retrieval_benchmark[
    retrieval_benchmark["benchmark_source"] == "taxonomy_audit"
].copy()

# Stratified sample:
# 50 training queries
# 100 taxonomy-audit queries

training_review, _ = train_test_split(
    training_queries,
    test_size=len(training_queries) - 50,
    stratify=training_queries["intent_label"],
    random_state=42,
)

audit_review, _ = train_test_split(
    audit_queries,
    test_size=len(audit_queries) - 100,
    stratify=audit_queries["intent_label"],
    random_state=42,
)

retrieval_review_sample = pd.concat(
    [
        training_review,
        audit_review,
    ],
    ignore_index=True,
)

# Keep deterministic ordering
retrieval_review_sample = retrieval_review_sample.sort_values(
    ["benchmark_source", "conversation_id"]
).reset_index(drop=True)

print("Retrieval review queries:",
      len(retrieval_review_sample))

print("\nSource:")
print(
    retrieval_review_sample["benchmark_source"].value_counts()
)

print("\nIntent distribution:")
display(
    retrieval_review_sample["intent_label"].value_counts()
    .sort_index()
)

print(
    "\nMissing labels:",
    retrieval_review_sample["intent_label"].isna().sum()
)

Retrieval review queries: 150

Source:
benchmark_source
taxonomy_audit    100
training           50
Name: count, dtype: int64

Intent distribution:


intent_label
ACCOUNT_ACCESS_SECURITY              5
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     5
DELIVERY_DELAY                      38
DELIVERY_MISSING_OR_MISDELIVERED    14
DEVICE_TECHNICAL_SUPPORT             3
DIGITAL_CONTENT                      6
GIFT_CARD_PROMOTION                  2
ORDER_STATUS_OR_CANCELLATION         7
OTHER_NON_SUPPORT                   25
PAYMENT_BILLING                      5
PRIME_MEMBERSHIP                     4
PRODUCT_AVAILABILITY_INFORMATION     5
PRODUCT_PROBLEM                     16
RETURN_REPLACEMENT_REFUND           10
WEBSITE_APP_TECHNICAL                5
Name: count, dtype: int64


Missing labels: 0


Build the retrieval - review data

In [271]:
# Map the 150 review queries back to their retrieval results.

review_rows = []

# Create lookup from conversation ID -> row in the 450-query benchmark
benchmark_id_to_index = {
    int(row["conversation_id"]): idx
    for idx, row in retrieval_benchmark.iterrows()
}

for _, query_row in retrieval_review_sample.iterrows():

    query_id = int(query_row["conversation_id"])
    query_idx = benchmark_id_to_index[query_id]

    # Top 3 retrieved historical conversations
    retrieved_indices = benchmark_indices[query_idx][:3]
    retrieved_scores = benchmark_scores[query_idx][:3]

    for rank, (retrieved_idx, score) in enumerate(
        zip(retrieved_indices, retrieved_scores),
        start=1
    ):

        retrieved_row = amazon_retrieval_benchmark_corpus.iloc[
            retrieved_idx
        ]

        review_rows.append({
            "query_conversation_id": query_id,
            "query_intent": query_row["intent_label"],
            "benchmark_source": query_row["benchmark_source"],
            "query_text": query_row["customer_only_text"],

            "rank": rank,
            "similarity": float(score),

            "retrieved_conversation_id": int(
                retrieved_row["conversation_id"]
            ),

            "retrieved_conversation_size": int(
                retrieved_row["conversation_size"]
            ),

            "retrieved_customer_turns": int(
                retrieved_row["customer_turns"]
            ),

            "retrieved_support_turns": int(
                retrieved_row["support_turns"]
            ),

            "retrieved_conversation_text":
                retrieved_row["conversation_text"],
        })


retrieval_review = pd.DataFrame(review_rows)

print("Review rows:", len(retrieval_review))
print(
    "Unique queries:",
    retrieval_review["query_conversation_id"].nunique()
)
print(
    "Ranks:",
    sorted(retrieval_review["rank"].unique())
)
print(
    "Missing retrieved conversations:",
    retrieval_review["retrieved_conversation_id"].isna().sum()
)

Review rows: 450
Unique queries: 150
Ranks: [np.int64(1), np.int64(2), np.int64(3)]
Missing retrieved conversations: 0


In [272]:
display(
    retrieval_review[
        [
            "query_conversation_id",
            "query_intent",
            "rank",
            "similarity",
            "retrieved_conversation_id",
            "query_text",
            "retrieved_conversation_text",
        ]
    ].head(9)
)

,query_conversation_id,query_intent,rank,similarity,retrieved_conversation_id,query_text,retrieved_conversation_text
0,52943,ORDER_STATUS_OR_CANCELLATION,1,0.837597,1488825,"@AmazonHelp Hi! Order #204-2242560-7141155, 2 books that were preordered. I asked them to be shipped together...\n[CUSTOMER] @AmazonHelp ...but you sent the 1st one to me as soon as it was released, free of charge. Neat! But apparently it ended up in Austria (I am in Oslo)...\n[CUSTOMER] @AmazonHelp ...then i got a message saying its delivery time is now by the end of November. And now when I checked the order, you say I returned it!\n[CUSTOMER] @AmazonHelp I didn't return it (never received...","[CUSTOMER] @115821 \nI ordered book from few days , can I turn it back to exchange the edition ..?\n[SUPPORT] @465802 Hi, has the order been dispatched? Did you place the order on https://t.co/nUUp5MLhYl? ^JJ\n[CUSTOMER] @AmazonHelp \nYes , I received it from couple of days , but still new and wrapped too ✨\n[SUPPORT] @465802 Got it! You can start the return process by visiting us here: https://t.co/yIZoIuRY5p ^TK\n[CUSTOMER] @AmazonHelp \nThank you for helping 🌸✨\n[SUPPORT] @465802 No probl..."
1,52943,ORDER_STATUS_OR_CANCELLATION,2,0.824107,2338831,"@AmazonHelp Hi! Order #204-2242560-7141155, 2 books that were preordered. I asked them to be shipped together...\n[CUSTOMER] @AmazonHelp ...but you sent the 1st one to me as soon as it was released, free of charge. Neat! But apparently it ended up in Austria (I am in Oslo)...\n[CUSTOMER] @AmazonHelp ...then i got a message saying its delivery time is now by the end of November. And now when I checked the order, you say I returned it!\n[CUSTOMER] @AmazonHelp I didn't return it (never received...","[CUSTOMER] @115821 STILL NOT RECEIVED MY BOOK. I WOULD LIKE A REFUND PLEASE. ORDER PENDING SINCE JUNE 19th, 2017 https://t.co/pagtQwyUf8\n[SUPPORT] @676575 Hi sorry to hear this. What is the current tracking on your order. You can check here https://t.co/aaDyEz1VgE ^CR\n[CUSTOMER] @AmazonHelp I ordered via https://t.co/3mPq6UE7sm but it's not coming up on the link you've given here.\n[SUPPORT] @676575 So sorry! Try this link instead: https://t.co/JzP7hlA23B Please keep us posted! ^LB\n[CUSTO..."
2,52943,ORDER_STATUS_OR_CANCELLATION,3,0.819928,224575,"@AmazonHelp Hi! Order #204-2242560-7141155, 2 books that were preordered. I asked them to be shipped together...\n[CUSTOMER] @AmazonHelp ...but you sent the 1st one to me as soon as it was released, free of charge. Neat! But apparently it ended up in Austria (I am in Oslo)...\n[CUSTOMER] @AmazonHelp ...then i got a message saying its delivery time is now by the end of November. And now when I checked the order, you say I returned it!\n[CUSTOMER] @AmazonHelp I didn't return it (never received...","[CUSTOMER] @AmazonHelp are you kidding me? Twice now I've ordered a book and both times have had major issues with my order. Unreal. Give me a refund.\n[SUPPORT] @169654 Apologies. Can you tell us a bit more about the issues, were the books fulfilled by Amazon or a 3rd Party Seller? ^JJ\n[CUSTOMER] @AmazonHelp I just DM'd you guys about the issue."
3,80798,DELIVERY_MISSING_OR_MISDELIVERED,1,0.822829,676883,@115850 item not received for order 406-1844716-6230753 Status is showing delivered . Have also received SMS mentioning item delivered. Kindly address urgently. @115850 Reported to the team. Still no update. Kindly address @AmazonHelp Have already share the order ID. U can check,"[CUSTOMER] @115850 order no 407-2447542-9629927 not yet received by me, why is it showing delivered, Kindly check.\n[SUPPORT] @281471 I understand your concern. Please report this to our support team here: https://t.co/vlvfJr4nN9 and we'll get it checked. \nPlease don't provide your order details, we consider it to be personal information. Our page is visible to the public. ^SY\n[CUSTOMER] @AmazonHelp Again same problem, item not yet received by me but msg from Amazon says that it's delivere..."
4,80798,DELIV

In [273]:
import re

def extract_customer_only_clean(text):
    text = str(text)

    # Extract customer sections
    customer_messages = re.findall(
        r"\[CUSTOMER\]\s*(.*?)(?=\[SUPPORT\]|\Z)",
        text,
        flags=re.DOTALL
    )

    cleaned_messages = []

    for msg in customer_messages:
        # Remove accidental nested role markers
        msg = re.sub(r"\[CUSTOMER\]\s*", " ", msg)
        msg = re.sub(r"\[SUPPORT\]\s*", " ", msg)

        # Normalize whitespace
        msg = re.sub(r"\s+", " ", msg).strip()

        if msg:
            cleaned_messages.append(msg)

    return " ".join(cleaned_messages)


# Rebuild customer-only query text
retrieval_benchmark["customer_only_text"] = (
    retrieval_benchmark["conversation_text"]
    .apply(extract_customer_only_clean)
)

# Check for remaining role markers
remaining_customer_markers = (
    retrieval_benchmark["customer_only_text"]
    .str.contains(r"\[CUSTOMER\]", regex=True)
    .sum()
)

remaining_support_markers = (
    retrieval_benchmark["customer_only_text"]
    .str.contains(r"\[SUPPORT\]", regex=True)
    .sum()
)

print("Queries:", len(retrieval_benchmark))
print("Remaining [CUSTOMER] markers:", remaining_customer_markers)
print("Remaining [SUPPORT] markers:", remaining_support_markers)

print("\nExample cleaned query:")
print(retrieval_benchmark.iloc[0]["customer_only_text"][:1000])

Queries: 450
Remaining [CUSTOMER] markers: 0
Remaining [SUPPORT] markers: 0

Example cleaned query:
I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop. @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue + @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.


In [274]:
# Re-encode the cleaned customer-only queries
clean_query_embeddings = embedding_model.encode(
    retrieval_benchmark["customer_only_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

clean_query_embeddings = np.asarray(
    clean_query_embeddings,
    dtype="float32"
)

print("Query embedding shape:", clean_query_embeddings.shape)

# Retrieve top 10 from the leakage-safe benchmark corpus
clean_scores, clean_indices = faiss_index_benchmark.search(
    clean_query_embeddings,
    10
)

print("Scores shape:", clean_scores.shape)
print("Indices shape:", clean_indices.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Query embedding shape: (450, 384)
Scores shape: (450, 10)
Indices shape: (450, 10)


In [275]:
# Inspect the first 5 benchmark queries after cleaning.

for i in range(5):
    query_row = retrieval_benchmark.iloc[i]

    print("=" * 100)
    print(f"QUERY {i + 1}")
    print(f"Conversation ID : {query_row['conversation_id']}")
    print(f"Intent          : {query_row['intent_label']}")
    print(f"\nCustomer query:")
    print(query_row["customer_only_text"][:1200])

    print("\nTop 3 retrieved cases:")

    for rank in range(3):
        retrieved_idx = clean_indices[i][rank]
        score = clean_scores[i][rank]

        retrieved_row = amazon_retrieval_benchmark_corpus.iloc[
            retrieved_idx
        ]

        print(f"\n--- Rank {rank + 1} | Similarity: {score:.4f} ---")
        print(
            f"Retrieved conversation ID: "
            f"{retrieved_row['conversation_id']}"
        )
        print(
            retrieved_row["conversation_text"][:1000]
        )

    print()

QUERY 1
Conversation ID : 373303
Intent          : DELIVERY_DELAY

Customer query:
I looooove @115821, I use them all the time. My one qualm is when I actually pay for one day shipping, it never arrives. So - I will stop. @AmazonHelp I spoke with your customer service today. Although they did their best I’m frustrated I have to wait until tomorrow to resolve the issue + @AmazonHelp + when I ordered one day shipping because it was a gift. Now it’s late and I wouldn’t have ordered it if I couldn’t have gotten it then.

Top 3 retrieved cases:

--- Rank 1 | Similarity: 0.7752 ---
Retrieved conversation ID: 339472
[CUSTOMER] It's really frustrating trying to order things through @115821 lately. They seem to screw up shipment and delivery every time.
[SUPPORT] @197062 This isn't what we like to hear! Are you currently having an issue? If so, can you give us more info? We're here for you! ^KJ

--- Rank 2 | Similarity: 0.7714 ---
Retrieved conversation ID: 218089
[CUSTOMER] If you have problem

In [276]:
review_rows = []

# Map benchmark conversation ID -> benchmark row index
benchmark_id_to_index = {
    int(row["conversation_id"]): idx
    for idx, row in retrieval_benchmark.iterrows()
}

for _, query_row in retrieval_review_sample.iterrows():

    query_id = int(query_row["conversation_id"])
    query_idx = benchmark_id_to_index[query_id]

    # Use CLEAN retrieval results
    retrieved_indices = clean_indices[query_idx][:3]
    retrieved_scores = clean_scores[query_idx][:3]

    for rank, (retrieved_idx, score) in enumerate(
        zip(retrieved_indices, retrieved_scores),
        start=1
    ):

        retrieved_row = amazon_retrieval_benchmark_corpus.iloc[
            retrieved_idx
        ]

        review_rows.append({
            "query_conversation_id": query_id,
            "query_intent": query_row["intent_label"],
            "benchmark_source": query_row["benchmark_source"],
            "query_text": query_row["customer_only_text"],

            "rank": rank,
            "similarity": float(score),

            "retrieved_conversation_id": int(
                retrieved_row["conversation_id"]
            ),

            "retrieved_conversation_size": int(
                retrieved_row["conversation_size"]
            ),

            "retrieved_customer_turns": int(
                retrieved_row["customer_turns"]
            ),

            "retrieved_support_turns": int(
                retrieved_row["support_turns"]
            ),

            "retrieved_conversation_text":
                retrieved_row["conversation_text"],
        })


retrieval_review = pd.DataFrame(review_rows)

print("Retrieval review rows:", len(retrieval_review))
print(
    "Unique queries:",
    retrieval_review["query_conversation_id"].nunique()
)
print(
    "Rows per query:",
    retrieval_review.groupby(
        "query_conversation_id"
    ).size().value_counts().to_dict()
)

print(
    "Duplicate query/retrieved pairs:",
    retrieval_review.duplicated(
        ["query_conversation_id", "retrieved_conversation_id"]
    ).sum()
)

print(
    "Benchmark source:",
    retrieval_review["benchmark_source"].value_counts().to_dict()
)

Retrieval review rows: 450
Unique queries: 150
Rows per query: {3: 150}
Duplicate query/retrieved pairs: 0
Benchmark source: {'taxonomy_audit': 300, 'training': 150}


In [277]:
# Create the human retrieval annotation template

retrieval_annotations = retrieval_review.copy()

# Human annotation fields
retrieval_annotations["relevance_label"] = ""
retrieval_annotations["annotation_notes"] = ""
retrieval_annotations["annotator"] = ""

# Reorder columns for easier manual annotation
retrieval_annotations = retrieval_annotations[
    [
        "query_conversation_id",
        "query_intent",
        "benchmark_source",
        "query_text",

        "rank",
        "similarity",

        "retrieved_conversation_id",
        "retrieved_conversation_size",
        "retrieved_customer_turns",
        "retrieved_support_turns",
        "retrieved_conversation_text",

        "relevance_label",
        "annotation_notes",
        "annotator",
    ]
].copy()

# Save
retrieval_annotation_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_human_annotations.csv"
)

retrieval_annotations.to_csv(
    retrieval_annotation_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", retrieval_annotation_path)
print("Rows:", len(retrieval_annotations))
print("Columns:", len(retrieval_annotations.columns))
print(
    "Blank relevance labels:",
    (
        retrieval_annotations["relevance_label"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
)

display(retrieval_annotations.head(6))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv
Rows: 450
Columns: 14
Blank relevance labels: 450


,query_conversation_id,query_intent,benchmark_source,query_text,rank,similarity,retrieved_conversation_id,retrieved_conversation_size,retrieved_customer_turns,retrieved_support_turns,retrieved_conversation_text,relevance_label,annotation_notes,annotator
0,52943,ORDER_STATUS_OR_CANCELLATION,taxonomy_audit,"@AmazonHelp Hi! Order #204-2242560-7141155, 2 books that were preordered. I asked them to be shipped together...\n[CUSTOMER] @AmazonHelp ...but you sent the 1st one to me as soon as it was released, free of charge. Neat! But apparently it ended up in Austria (I am in Oslo)...\n[CUSTOMER] @AmazonHelp ...then i got a message saying its delivery time is now by the end of November. And now when I checked the order, you say I returned it!\n[CUSTOMER] @AmazonHelp I didn't return it (never received...",1,0.833588,1488825,10,5,5,"[CUSTOMER] @115821 \nI ordered book from few days , can I turn it back to exchange the edition ..?\n[SUPPORT] @465802 Hi, has the order been dispatched? Did you place the order on https://t.co/nUUp5MLhYl? ^JJ\n[CUSTOMER] @AmazonHelp \nYes , I received it from couple of days , but still new and wrapped too ✨\n[SUPPORT] @465802 Got it! You can start the return process by visiting us here: https://t.co/yIZoIuRY5p ^TK\n[CUSTOMER] @AmazonHelp \nThank you for helping 🌸✨\n[SUPPORT] @465802 No probl...",,,
1,52943,ORDER_STATUS_OR_CANCELLATION,taxonomy_audit,"@AmazonHelp Hi! Order #204-2242560-7141155, 2 books that were preordered. I asked them to be shipped together...\n[CUSTOMER] @AmazonHelp ...but you sent the 1st one to me as soon as it was released, free of charge. Neat! But apparently it ended up in Austria (I am in Oslo)...\n[CUSTOMER] @AmazonHelp ...then i got a message saying its delivery time is now by the end of November. And now when I checked the order, you say I returned it!\n[CUSTOMER] @AmazonHelp I didn't return it (never received...",2,0.813460,2338831,16,10,6,"[CUSTOMER] @115821 STILL NOT RECEIVED MY BOOK. I WOULD LIKE A REFUND PLEASE. ORDER PENDING SINCE JUNE 19th, 2017 https://t.co/pagtQwyUf8\n[SUPPORT] @676575 Hi sorry to hear this. What is the current tracking on your order. You can check here https://t.co/aaDyEz1VgE ^CR\n[CUSTOMER] @AmazonHelp I ordered via https://t.co/3mPq6UE7sm but it's not coming up on the link you've given here.\n[SUPPORT] @676575 So sorry! Try this link instead: https://t.co/JzP7hlA23B Please keep us posted! ^LB\n[CUSTO...",,,
2,52943,ORDER_STATUS_OR_CANCELLATION,taxonomy_audit,"@AmazonHelp Hi! Order #204-2242560-7141155, 2 books that were preordered. I asked them to be shipped together...\n[CUSTOMER] @AmazonHelp ...but you sent the 1st one to me as soon as it was released, free of charge. Neat! But apparently it ended up in Austria (I am in Oslo)...\n[CUSTOMER] @AmazonHelp ...then i got a message saying its delivery time is now by the end of November. And now when I checked the order, you say I returned it!\n[CUSTOMER] @AmazonHelp I didn't return it (never received...",3,0.812170,224575,3,2,1,"[CUSTOMER] @AmazonHelp are you kidding me? Twice now I've ordered a book and both times have had major issues with my order. Unreal. Give me a refund.\n[SUPPORT] @169654 Apologies. Can you tell us a bit more about the issues, were the books fulfilled by Amazon or a 3rd Party Seller? ^JJ\n[CUSTOMER] @AmazonHelp I just DM'd you guys about the issue.",,,
3,80798,DELIVERY_MISSING_OR_MISDELIVERED,taxonomy_audit,@115850 item not received for order 406-1844716-6230753 Status is showing delivered . Have also received SMS mentioning item delivered. Kindly address urgently. @115850 Reported to the team. Still no update. Kindly address @AmazonHelp Have already share the order ID. U can check,1,0.822829,676883,4,2,2,"[CUSTOMER] @115850 order no 407-2447542-9629927 not yet received by me, why is it showing delivered, Kindly check.\n[SUPPORT] @281471 I understand your concern. Please report this to our support team here: https://t.co/vlvfJr4nN9 and we'll get it checked. \nPlease don't provide yo

- 2	- Directly relevant and useful historical evidence for answering the query
- 1 - Related to the issue, but materially different / only partially useful
- 0	- Not relevant or not useful evidence

Interactive human retrieval annotation

In [278]:
import pandas as pd
import os

# Load the annotation file
retrieval_annotation_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_human_annotations.csv"
)

retrieval_annotations = pd.read_csv(
    retrieval_annotation_path,
    encoding="utf-8-sig"
)

# Make sure annotation columns exist
retrieval_annotations["relevance_label"] = (
    retrieval_annotations["relevance_label"]
    .fillna("")
    .astype(str)
)

retrieval_annotations["annotation_notes"] = (
    retrieval_annotations["annotation_notes"]
    .fillna("")
    .astype(str)
)

retrieval_annotations["annotator"] = (
    retrieval_annotations["annotator"]
    .fillna("")
    .astype(str)
)

print("Rows:", len(retrieval_annotations))
print(
    "Already annotated:",
    (
        retrieval_annotations["relevance_label"]
        .str.strip()
        .ne("")
    ).sum()
)
print(
    "Remaining:",
    (
        retrieval_annotations["relevance_label"]
        .str.strip()
        .eq("")
    ).sum()
)

Rows: 450
Already annotated: 0
Remaining: 450


In [279]:
# Export first 30 retrieval judgments for review

first_30 = retrieval_annotations.iloc[:30].copy()

first_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_first_30.csv"
)

first_30.to_csv(
    first_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", first_30_path)
print("Rows:", len(first_30))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_first_30.csv
Rows: 30


In [280]:
import pandas as pd
from pathlib import Path

# Input file
input_path = Path(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"
)

# Load existing annotation file
df = pd.read_csv(input_path, encoding="utf-8-sig")

# Human-annotated labels for the first 30 rows
labels = [
    1, 1, 1,
    2, 1, 2,
    2, 1, 1,
    0, 0, 0,
    2, 2, 2,
    2, 1, 0,
    2, 2, 2,
    0, 2, 2,
    2, 1, 1,
    0, 1, 0
]

# Safety check
assert len(labels) == 30, "Exactly 30 labels are required."
assert len(df) >= 30, "The annotation CSV contains fewer than 30 rows."

# Update ONLY the first 30 rows
df.loc[df.index[:30], "relevance_label"] = labels

# Add annotation notes
notes = [
    "Related order/return issue, but materially different",
    "Missing book/order issue; useful but not the same requested outcome",
    "Generic order problem and refund request; partially useful",
    "Direct delivered-but-not-received case",
    "Delivery-status problem, but not delivered/misdelivered",
    "Direct delivered-but-not-received case",
    "Direct Prime/delivery-delay problem",
    "Prime shipping/delivery issue, but mainly seller/fee eligibility",
    "Prime one-day delivery availability; related but not an actual delay",
    "Delivery delay, unrelated to account-security problem",
    "Generic delivery complaint, not account access",
    "Order-return issue, not account security",
    "Direct electronic-device/product problem",
    "Direct faulty iPhone/product issue",
    "Direct device dead-on-arrival/product problem",
    "Strong delivery problem match despite noisy query label",
    "Repeated product-return issue; related but different",
    "Refund for damaged product; materially different",
    "Direct same-day delivery delay",
    "Direct repeated delivery delay",
    "Direct past-due delivery",
    "Positive on-time delivery; not useful for delay",
    "Direct late-delivery complaint",
    "Direct delivery-not-on-time complaint",
    "Direct damaged-product case",
    "Damaged packaging; partially relevant",
    "Packaging/delivery complaint, but product itself undamaged",
    "Payment/Prime issue, not account-access problem",
    "Prime-membership problem overlaps one part of query",
    "Delivery delay, unrelated to account issue",
]

# Update notes for first 30 rows
df.loc[df.index[:30], "annotation_notes"] = notes

# Mark these as your human-reviewed annotations
df.loc[df.index[:30], "annotator"] = "human_1"

# Save back to the same file
df.to_csv(input_path, index=False, encoding="utf-8-sig")

print("✅ First 30 retrieval rows annotated and saved successfully.")
print(f"File: {input_path}")

# Verification
check = pd.read_csv(input_path, encoding="utf-8-sig")

print("\nVerification:")
print("Total rows:", len(check))
print("First 30 labels:")
print(check.loc[:29, ["query_conversation_id", "rank",
                      "relevance_label", "annotation_notes"]].to_string(index=False))

✅ First 30 retrieval rows annotated and saved successfully.
File: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv

Verification:
Total rows: 450
First 30 labels:
 query_conversation_id  rank  relevance_label                                                     annotation_notes
                 52943     1              1.0                 Related order/return issue, but materially different
                 52943     2              1.0  Missing book/order issue; useful but not the same requested outcome
                 52943     3              1.0           Generic order problem and refund request; partially useful
                 80798     1              2.0                               Direct delivered-but-not-received case
                 80798     2              1.0              Delivery-status problem, but not delivered/misdelivered
                 80798     3              2.0                               Direct delivered-but-not-received 

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8164\3947128289.py:68: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Related order/return issue, but materially different', 'Missing book/order issue; useful but not the same requested outcome', 'Generic order problem and refund request; partially useful', 'Direct delivered-but-not-received case', 'Delivery-status problem, but not delivered/misdelivered', 'Direct delivered-but-not-received case', 'Direct Prime/delivery-delay problem', 'Prime shipping/delivery issue, but mainly seller/fee eligibility', 'Prime one-day delivery availability; related but not an actual delay', 'Delivery delay, unrelated to account-security problem', 'Generic delivery complaint, not account access', 'Order-return issue, not account security', 'Direct electronic-device/product problem', 'Direct faulty iPhone/product issue', 'Direct device dead-on-arrival/product problem', 'Strong d

In [281]:
# Fix column dtypes before writing text annotations
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# Now update the first 30 rows
df.loc[df.index[:30], "annotation_notes"] = notes
df.loc[df.index[:30], "annotator"] = "human_1"

# Save
df.to_csv(input_path, index=False, encoding="utf-8-sig")

print("✅ Saved without dtype warnings.")

✅ Saved without dtype warnings.


In [282]:
labels = [
    1, 2, 2, 2, 1, 0,
    2, 2, 2, 0, 1, 0,
    2, 2, 1, 2, 2, 2,
    0, 2, 1, 2, 1, 2,
    0, 0, 0, 0, 0, 0
]

notes = [
    "Related order issue, but not the same customer problem.",
    "Directly relevant delivery-delay evidence.",
    "Directly relevant late-delivery case.",
    "Directly relevant delivery-status issue.",
    "Related delivery issue but materially different.",
    "Not useful for the query.",
    "Direct product-problem evidence.",
    "Direct faulty-product evidence.",
    "Direct product-problem evidence.",
    "Not relevant to the query intent.",
    "Related account/order context but not the same issue.",
    "Not relevant to the query.",
    "Direct missing-delivery evidence.",
    "Direct delivery-status evidence.",
    "Related product/order issue but not identical.",
    "Direct payment/billing evidence.",
    "Direct payment/charge evidence.",
    "Related billing issue.",
    "Not useful for the query.",
    "Directly relevant delivery evidence.",
    "Related order-status issue.",
    "Direct order-status evidence.",
    "Related Prime/delivery context.",
    "Direct membership/support evidence.",
    "Not useful for the query.",
    "Related delivery issue.",
    "Direct delivery-delay evidence.",
    "Not relevant to the query.",
    "Not relevant to the query.",
    "Not relevant to the query."
]

In [289]:
START_ROW = 0
END_ROW = len(labels) - 1

expected_count = END_ROW - START_ROW + 1

assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(label in [0, 1, 2] for label in labels)

In [290]:
print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

Labels: 30
Notes: 30
Expected: 30


In [293]:
# ============================================================
# WRITE ANNOTATIONS
# ============================================================

start_idx = START_ROW
end_idx = END_ROW + 1

# Select the intended rows using zero-based row numbers.
selected_index = df.index[start_idx:end_idx]

assert len(selected_index) == len(labels), (
    f"Selected {len(selected_index)} rows but received "
    f"{len(labels)} labels."
)

# Assign through an indexed Series to avoid iterable-length errors.
df.loc[selected_index, "relevance_label"] = pd.Series(
    labels,
    index=selected_index,
)
df.loc[df.index[start_idx:end_idx], "annotation_notes"] = notes
df.loc[df.index[start_idx:end_idx], "annotator"] = "human_1"

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# VERIFY
# ============================================================

check = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

batch = check.iloc[start_idx:end_idx]

print("✅ Batch 31–60 saved successfully!")
print(f"Rows annotated: {len(batch)}")
print(f"Row range: {START_ROW}–{END_ROW}")

print("\nLabel distribution:")
print(batch["relevance_label"].value_counts().sort_index())

print("\nRemaining unannotated rows:")
print(check["relevance_label"].isna().sum())

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label",
            "annotator"
        ]
    ].to_string(index=False)
)

✅ Batch 31–60 saved successfully!
Rows annotated: 30
Row range: 0–29

Label distribution:
relevance_label
0.0    10
1.0     6
2.0    14
Name: count, dtype: int64

Remaining unannotated rows:
420

Verification:
 query_conversation_id  rank  relevance_label annotator
                 52943     1              1.0   human_1
                 52943     2              2.0   human_1
                 52943     3              2.0   human_1
                 80798     1              2.0   human_1
                 80798     2              1.0   human_1
                 80798     3              0.0   human_1
                137048     1              2.0   human_1
                137048     2              2.0   human_1
                137048     3              2.0   human_1
                246997     1              0.0   human_1
                246997     2              1.0   human_1
                246997     3              0.0   human_1
                259944     1              2.0   human_1
      

In [294]:
next_60_90 = retrieval_annotations.iloc[61:91].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_60_90.csv"
)

next_60_90.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_60_90))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_60_90.csv
Rows: 30


In [295]:
import pandas as pd
from pathlib import Path

# ============================================================
# FILE
# ============================================================

input_path = Path(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"
)

# ============================================================
# BATCH 61–90
# ============================================================

START_ROW = 61
END_ROW = 90

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

# Prevent dtype warnings
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# ============================================================
# RELEVANCE LABELS
#
# 2 = Directly relevant and useful
# 1 = Related / partially useful
# 0 = Not relevant / not useful
# ============================================================

labels = [
    # 61–63 | Query 498437 | DELIVERY_DELAY
    1, 1, 1,

    # 64–66 | Query 514690 | DELIVERY_DELAY
    2, 1, 2,

    # 67–69 | Query 585801 | OTHER_NON_SUPPORT
    2, 2, 2,

    # 70–72 | Query 611123 | WEBSITE_APP_TECHNICAL
    0, 1, 2,

    # 73–75 | Query 617078 | PRODUCT_AVAILABILITY_INFORMATION
    2, 0, 1,

    # 76–78 | Query 627702 | DELIVERY_MISSING_OR_MISDELIVERED
    2, 2, 2,

    # 79–81 | Query 644769 | OTHER_NON_SUPPORT
    2, 0, 0,

    # 82–84 | Query 714744 | DELIVERY_MISSING_OR_MISDELIVERED
    2, 1, 1,

    # 85–87 | Query 792025 | PRODUCT_PROBLEM
    0, 1, 0,

    # 88–90 | Query 818203 | DELIVERY_DELAY
    2, 2, 2
]

# ============================================================
# ANNOTATION NOTES
# ============================================================

notes = [
    # 61–63
    "Delivered-but-not-received case; related to package location but materially different from a pure delay.",
    "Direct delivered-but-not-received evidence; useful for locating a package but different from the primary delay intent.",
    "Delivered-but-not-received case; related delivery evidence but not a direct delay resolution.",

    # 64–66
    "Direct delayed Prime/preorder delivery case.",
    "Related preorder problem involving stock/inventory, but not primarily a delivery-delay resolution.",
    "Direct delivery-delay case with carrier delay and updated delivery estimate.",

    # 67–69
    "Directly relevant packaging-feedback example.",
    "Directly relevant packaging-feedback example.",
    "Directly relevant packaging-feedback example.",

    # 70–72
    "Positive social engagement; not useful for an app-down technical issue.",
    "Technical support case, but it concerns an Echo device rather than the website/app.",
    "Direct app-down troubleshooting case.",

    # 73–75
    "Direct stock-availability question with Amazon support response.",
    "Packaging/damage issue; not useful for a stock-availability query.",
    "Price-information case; related to product information but not stock availability.",

    # 76–78
    "Direct parcel-marked-delivered-but-not-received case.",
    "Direct missing-delivery case with order-location assistance.",
    "Direct scanned-as-delivered-but-not-received case.",

    # 79–81
    "Direct promotion/discount-related conversation; useful for the promotional context.",
    "Delivered-but-not-received problem; unrelated to the query's social/promotion intent.",
    "Wrong-product complaint; unrelated to the query's social/promotion intent.",

    # 82–84
    "Directly relevant delivery failure involving delayed, lost, and wrong-house deliveries.",
    "Related Prime delivery failure, but the example is primarily a missed delivery window rather than confirmed misdelivery.",
    "Related repeated delivery failure, but primarily a next-day delivery delay rather than confirmed misdelivery.",

    # 85–87
    "Unrelated email/notification feedback; does not address the wrong-product problem.",
    "Related product-ordering issue involving minimum order quantity, but not the wrong-product problem.",
    "Positive packaging feedback; not useful for resolving the wrong-product problem.",

    # 88–90
    "Direct guaranteed-delivery-date failure.",
    "Direct repeated guaranteed-delivery failure.",
    "Direct late-delivery complaint with a missed promised date."
]

# ============================================================
# SAFETY CHECKS
# ============================================================

expected_count = END_ROW - START_ROW + 1

print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(
    label in [0, 1, 2]
    for label in labels
), "Labels must contain only 0, 1, or 2"

# ============================================================
# WRITE
# ============================================================

start_idx = START_ROW - 1
end_idx = END_ROW

df.loc[
    df.index[start_idx:end_idx],
    "relevance_label"
] = labels

df.loc[
    df.index[start_idx:end_idx],
    "annotation_notes"
] = notes

df.loc[
    df.index[start_idx:end_idx],
    "annotator"
] = "human_1"

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# VERIFY
# ============================================================

check = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

batch = check.iloc[start_idx:end_idx]

print("\n" + "=" * 60)
print("✅ BATCH 61–90 SAVED SUCCESSFULLY")
print("=" * 60)

print(f"Rows annotated: {len(batch)}")
print(f"Row range: {START_ROW}–{END_ROW}")

print("\nLabel distribution:")
print(
    batch["relevance_label"]
    .value_counts()
    .sort_index()
)

print("\nRemaining unannotated rows:")
print(
    check["relevance_label"].isna().sum()
)

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label",
            "annotator"
        ]
    ].to_string(index=False)
)

Labels: 30
Notes: 30
Expected: 30

✅ BATCH 61–90 SAVED SUCCESSFULLY
Rows annotated: 30
Row range: 61–90

Label distribution:
relevance_label
0.0     6
1.0     9
2.0    15
Name: count, dtype: int64

Remaining unannotated rows:
390

Verification:
 query_conversation_id  rank  relevance_label annotator
                498437     1              1.0   human_1
                498437     2              1.0   human_1
                498437     3              1.0   human_1
                514690     1              2.0   human_1
                514690     2              1.0   human_1
                514690     3              2.0   human_1
                585801     1              2.0   human_1
                585801     2              2.0   human_1
                585801     3              2.0   human_1
                611123     1              0.0   human_1
                611123     2              1.0   human_1
                611123     3              2.0   human_1
                617078     

In [296]:
next_91_120 = retrieval_annotations.iloc[91:121].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_91_120.csv"
)

next_91_120.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_91_120))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_91_120.csv
Rows: 30


In [297]:
labels = [
    2, 2,
    2, 2, 2,
    2, 2, 2,
    0, 1, 2,
    2, 1, 0,
    2, 2, 2,
    1, 2, 1,
    2, 2, 1,
    2, 0, 1,
    2, 2, 1,
    2
]

In [298]:
print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", END_ROW - START_ROW + 1)

Labels: 30
Notes: 30
Expected: 30


In [299]:
labels = [
    2, 2,
    2, 2, 2,
    2, 2, 2,
    0, 1, 2,
    2, 1, 0,
    2, 2, 2,
    1, 2, 1,
    2, 2, 1,
    2, 0, 1,
    2, 2, 1,
    2
]

In [300]:
print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", END_ROW - START_ROW + 1)

Labels: 30
Notes: 30
Expected: 30


In [301]:
START_ROW = 91
END_ROW = 120

labels = [
    2, 2,
    2, 2, 2,
    2, 2, 2,
    0, 1, 2,
    2, 1, 0,
    2, 2, 2,
    1, 2, 1,
    2, 2, 1,
    2, 0, 1,
    2, 2, 1,
    2
]

print("Labels:", len(labels))
print("Expected:", END_ROW - START_ROW + 1)

Labels: 30
Expected: 30


In [302]:
notes = [
    "Directly relevant missing/non-received item case; useful historical evidence.",  # 1
    "Directly relevant prolonged non-delivery and refund problem.",  # 2
    "Direct delivery-delay case involving a very late order.",  # 3
    "Direct delivery-delay case involving a missed delivery window.",  # 4
    "Direct non-delivery complaint with Amazon support investigation.",  # 5
    "Direct shipping/tracking problem closely matching the delivery-delay query.",  # 6
    "Direct repeated delivery failures and late packages; highly useful.",  # 7
    "Direct repeated delivery delays and unresolved shipping problems.",  # 8
    "Not a product-defect resolution; retrieved conversation is unrelated to the specific product complaint.",  # 9
    "Packaging complaint is related to a product issue but materially different from the query's defect.",  # 10
    "Wrong-size product is a concrete product problem and useful historical evidence.",  # 11
    "Order not dispatched and delayed for weeks; directly relevant to delivery delay.",  # 12
    "Prime membership/order cancellation problem is related to fulfillment but materially different.",  # 13
    "Prime charge/refund issue is not useful for the delivery-delay problem.",  # 14
    "Direct wrong-item/return/refund case matching the return workflow.",  # 15
    "Direct refund-delay case.",  # 16
    "Undelivered order with missing refund; strongly relevant to the refund request.",  # 17
    "Generic poor customer-service complaint is related but lacks a specific issue.",  # 18
    "Direct generic customer-support dissatisfaction; useful for handling this type of complaint.",  # 19
    "Order-status request is a specific support issue but does not match the generic complaint closely.",  # 20
    "Direct refund request following an order problem.",  # 21
    "Damaged product with support discussing remedy/options; useful for return/refund handling.",  # 22
    "Delivery delay is related to an order problem but not specifically return/refund.",  # 23
    "Direct Echo/Alexa connectivity and Bluetooth-related device question.",  # 24
    "Positive Echo experience without a technical problem; not useful.",  # 25
    "Alexa/device-related question but different from the Bluetooth connectivity issue.",  # 26
    "Direct delivered-but-not-received and refund-related delivery problem.",  # 27
    "Direct missing-delivery/refund problem.",  # 28
    "Non-dispatched product with missing refund; related non-delivery/refund evidence but materially different from misdelivery.",  # 29
    "Direct accidental Prime membership/autorenewal problem; useful historical evidence.",  # 30
]

In [303]:
print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

Labels: 30
Notes: 30
Expected: 30


In [304]:
notes = [
    "Directly relevant missing/non-received item case; useful historical evidence.",
    "Directly relevant prolonged non-delivery and refund problem.",

    "Direct delivery-delay case involving a very late order.",
    "Direct delivery-delay case involving a missed delivery window.",
    "Direct non-delivery complaint with Amazon support investigation.",

    "Direct shipping/tracking problem closely matching the delivery-delay query.",
    "Direct repeated delivery failures and late packages; highly useful.",
    "Direct repeated delivery delays and unresolved shipping problems.",

    "Not a product-defect resolution; retrieved conversation is unrelated to the specific product complaint.",
    "Packaging complaint is related to a product issue but materially different from the query's defect.",
    "Wrong-size product is a concrete product problem and useful historical evidence.",

    "Order not dispatched and delayed for weeks; directly relevant to delivery delay.",
    "Prime/order cancellation problem is related to fulfillment but materially different.",
    "Prime charge/refund issue is not useful for the delivery-delay problem.",

    "Direct wrong-item/return/refund case matching the return workflow.",
    "Direct refund-delay case.",
    "Undelivered order with missing refund; strongly relevant to the refund request.",

    "Generic poor customer-service complaint is related but lacks a specific issue.",
    "Direct generic customer-support dissatisfaction; useful for handling this type of complaint.",
    "Order-status request is a specific support issue but does not match the generic complaint closely.",

    "Direct refund request following an order problem.",
    "Damaged product with support discussing remedy/options; useful for return/refund handling.",
    "Delivery delay is related to an order problem but not specifically return/refund.",

    "Direct Echo/Alexa connectivity and Bluetooth-related device question.",
    "Positive Echo experience without a technical problem; not useful.",
    "Alexa/device-related question but different from the Bluetooth connectivity issue.",

    "Direct delivered-but-not-received and refund-related delivery problem.",
    "Direct missing-delivery/refund problem.",
    "Non-dispatched product with missing refund; related non-delivery/refund evidence but materially different from misdelivery.",

    "Direct accidental Prime membership/autorenewal problem; useful historical evidence."
]

expected_count = END_ROW - START_ROW + 1

# ---------- Validate ----------
assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(label in [0, 1, 2] for label in labels), \
    "Invalid relevance label found."

# ---------- Load ----------
df = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

# Avoid dtype warnings
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# ---------- Update 91–120 ----------
start_idx = START_ROW - 1
end_idx = END_ROW

df.loc[
    df.index[start_idx:end_idx],
    "relevance_label"
] = labels

df.loc[
    df.index[start_idx:end_idx],
    "annotation_notes"
] = notes

df.loc[
    df.index[start_idx:end_idx],
    "annotator"
] = "human_1"

# ---------- Save ----------
df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

# ---------- Verification ----------
check = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

batch = check.iloc[start_idx:end_idx]

print("✅ Batch 91–120 saved successfully")
print(f"Rows updated: {len(batch)}")

print("\nLabel distribution:")
print(
    batch["relevance_label"]
    .value_counts()
    .sort_index()
)

print(
    f"\nRemaining unannotated rows: "
    f"{check['relevance_label'].isna().sum()}"
)

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label"
        ]
    ].to_string(index=False)
)

✅ Batch 91–120 saved successfully
Rows updated: 30

Label distribution:
relevance_label
0.0     3
1.0     7
2.0    20
Name: count, dtype: int64

Remaining unannotated rows: 360

Verification:
 query_conversation_id  rank  relevance_label
                846932     1              2.0
                846932     2              2.0
                846932     3              2.0
                857011     1              2.0
                857011     2              2.0
                857011     3              2.0
                865617     1              2.0
                865617     2              2.0
                865617     3              0.0
                873792     1              1.0
                873792     2              2.0
                873792     3              2.0
                930385     1              1.0
                930385     2              0.0
                930385     3              2.0
                937309     1              2.0
                937309    

In [305]:
next_121_150 = retrieval_annotations.iloc[121:151].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_121_150.csv"
)

next_121_150.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_121_150))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_121_150.csv
Rows: 30


In [306]:
import pandas as pd
from pathlib import Path

input_path = Path(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"
)

START_ROW = 121
END_ROW = 150

labels = [
    2, 0,
    2, 2, 1,
    2, 2, 2,
    1, 0, 1,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    1, 2, 2,
    2, 2, 2,
    2, 2, 2,
    1
]

notes = [
    "Direct accidental Prime-membership explanation and automatic renewal context.",
    "Positive Prime-membership experience; does not help resolve the accidental-membership complaint.",
    
    "Directly similar positive Amazon purchase and fast-delivery experience.",
    "Direct positive Amazon purchase/delivery experience; useful for a positive-support response.",
    "General Amazon positivity, but the specific topic differs from the query.",
    
    "Strongly related poor Amazon customer-service complaint with multiple support failures.",
    "Direct generic customer-service dissatisfaction and support follow-up.",
    "Direct generic poor-service complaint; useful for handling dissatisfaction.",
    
    "Customer-service complaint is related but does not identify the same underlying issue.",
    "Positive Amazon experience is unrelated to the negative/general complaint.",
    "Order-status problem is somewhat related to Amazon dissatisfaction but materially different.",
    
    "Directly related Prime Video/content-release announcement question.",
    "Directly related question about availability of a season on the platform.",
    "Directly related season-release question with support response.",
    
    "Direct one-day shipping delay and missed delivery expectation.",
    "Direct complaint about Prime/shipping taking longer than promised.",
    "Direct one-day shipping delay complaint.",
    
    "Direct two-factor authentication and verification-code problem.",
    "Direct account-access problem caused by delayed verification codes.",
    "Direct login/security-code timeout problem with troubleshooting guidance.",
    
    "Involves the app but concerns delivery status rather than the core app technical issue.",
    "Direct checkout/order technical failure with troubleshooting steps.",
    "Direct app/payment/order technical problem with troubleshooting guidance.",
    
    "Strongly related frustration about repeated forms and support handling.",
    "Direct form-submission/support follow-up issue.",
    "Direct form/support issue with an unresolved customer problem.",
    
    "Direct order-status problem involving an order not being dispatched.",
    "Direct order-status/non-delivery problem.",
    "Direct order-status problem involving an order marked out for delivery but not received.",
    
    "Related Amazon dissatisfaction, but the specific issue is different from the query."
]

# ---------- Validation ----------
expected_count = END_ROW - START_ROW + 1

assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(label in [0, 1, 2] for label in labels), \
    "Invalid relevance label found."

# ---------- Load ----------
df = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

# Prevent dtype warnings
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# ---------- Update rows 121–150 ----------
start_idx = START_ROW - 1
end_idx = END_ROW

df.loc[
    df.index[start_idx:end_idx],
    "relevance_label"
] = labels

df.loc[
    df.index[start_idx:end_idx],
    "annotation_notes"
] = notes

df.loc[
    df.index[start_idx:end_idx],
    "annotator"
] = "human_1"

# ---------- Save ----------
df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

# ---------- Verify ----------
check = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

batch = check.iloc[start_idx:end_idx]

print("✅ Batch 121–150 saved successfully")
print(f"Rows updated: {len(batch)}")
print("\nLabel distribution:")
print(batch["relevance_label"].value_counts().sort_index())

print(f"\nRemaining unannotated rows: "
      f"{check['relevance_label'].isna().sum()}")

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label"
        ]
    ].to_string(index=False)
)

✅ Batch 121–150 saved successfully
Rows updated: 30

Label distribution:
relevance_label
0.0     2
1.0     5
2.0    23
Name: count, dtype: int64

Remaining unannotated rows: 330

Verification:
 query_conversation_id  rank  relevance_label
               1074622     1              2.0
               1074622     2              0.0
               1074622     3              2.0
               1100625     1              2.0
               1100625     2              1.0
               1100625     3              2.0
               1105827     1              2.0
               1105827     2              2.0
               1105827     3              1.0
               1110378     1              0.0
               1110378     2              1.0
               1110378     3              2.0
               1148768     1              2.0
               1148768     2              2.0
               1148768     3              2.0
               1219190     1              2.0
               1219190   

In [307]:
next_151_180 = retrieval_annotations.iloc[151:181].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_151_180.csv"
)

next_151_180.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_151_180))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_151_180.csv
Rows: 30


In [308]:
# ---------- RESET 151–180 LABELS ----------

START_ROW = 151
END_ROW = 180

labels = [
    1, 1, 1, 2, 2, 2, 2, 2, 2, 2,
    2, 2, 2, 1, 1, 2, 1, 1, 2, 0,
    2, 2, 1, 2, 2, 1, 2, 1, 1, 1
]

notes = [
    "Generic customer-service complaint; related but not a specific support resolution.",
    "Defective-item complaint is related but materially different from the generic query.",
    "Return-policy/order-change issue is related but not a direct match.",
    "Direct return-policy issue involving an incorrectly ordered product.",
    "Direct returned-product and refund problem; highly relevant.",
    "Direct pre-order dispatch problem matching the order-status issue.",
    "Direct pre-order delivery-date problem; useful evidence.",
    "Direct pre-order non-delivery problem; highly relevant.",
    "Direct multi-day delivery delay; strong match.",
    "Direct delayed-package complaint; highly relevant.",
    "Direct repeated package non-arrival problem; useful evidence.",
    "Direct refund request and replacement/return workflow; highly relevant.",
    "Customer-care follow-up problem related to return/replacement but not a specific resolution.",
    "Generic customer-service dissatisfaction without a specific return/refund resolution.",
    "Direct undelivered-order and missing-refund complaint; useful evidence.",
    "Generic customer-service dissatisfaction related to an order problem but materially different.",
    "Unresolved order problem and poor support experience; partially useful.",
    "Direct card-payment/COD delivery problem matching the payment issue.",
    "Delivery failure is unrelated to the specific card-payment problem.",
    "Direct card-swipe/payment-on-delivery problem; highly relevant.",
    "Direct packaging/cushioning complaint matching the product concern.",
    "Cushioning recommendation concerns a different product and problem.",
    "Packaging feedback is related to cushioning but concerns a different packaging issue.",
    "Direct delivery-delay example with a missed promised delivery time.",
    "Direct late/out-for-delivery example; highly useful.",
    "Direct package-not-arrived example with late delivery; highly relevant.",
    "Delayed shipment is related to delivery timing but is somewhat less direct.",
    "Promotion/discount issue is related to an Amazon purchase but not delivery delay.",
    "Delivery-delay concern where the package had not yet shipped; useful evidence.",
    "Direct Prime membership/autorenewal problem; useful historical evidence."
]

expected_count = END_ROW - START_ROW + 1

print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

assert len(labels) == 30
assert len(notes) == 30
assert all(x in [0, 1, 2] for x in labels)

print("✅ EXACTLY 30 LABELS + 30 NOTES")

Labels: 30
Notes: 30
Expected: 30
✅ EXACTLY 30 LABELS + 30 NOTES


In [309]:
# ---------- Load ----------
df = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

# Avoid dtype warnings
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# ---------- Update 151–180 ----------
start_idx = START_ROW - 1
end_idx = END_ROW

df.loc[
    df.index[start_idx:end_idx],
    "relevance_label"
] = labels

df.loc[
    df.index[start_idx:end_idx],
    "annotation_notes"
] = notes

df.loc[
    df.index[start_idx:end_idx],
    "annotator"
] = "human_1"

# ---------- Save ----------
df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

# ---------- Verification ----------
check = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

batch = check.iloc[start_idx:end_idx]

print("✅ Batch 151–180 saved successfully")
print(f"Rows updated: {len(batch)}")

print("\nLabel distribution:")
print(
    batch["relevance_label"]
    .value_counts()
    .sort_index()
)

print(
    f"\nRemaining unannotated rows: "
    f"{check['relevance_label'].isna().sum()}"
)

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label"
        ]
    ].to_string(index=False)
)

✅ Batch 151–180 saved successfully
Rows updated: 30

Label distribution:
relevance_label
0.0     1
1.0    12
2.0    17
Name: count, dtype: int64

Remaining unannotated rows: 300

Verification:
 query_conversation_id  rank  relevance_label
               1405353     1              1.0
               1405353     2              1.0
               1405353     3              1.0
               1448054     1              2.0
               1448054     2              2.0
               1448054     3              2.0
               1491812     1              2.0
               1491812     2              2.0
               1491812     3              2.0
               1528649     1              2.0
               1528649     2              2.0
               1528649     3              2.0
               1568174     1              2.0
               1568174     2              1.0
               1568174     3              1.0
               1581994     1              2.0
               1581994   

In [310]:
next_181_240 = retrieval_annotations.iloc[181:241].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_181_240.csv"
)

next_181_240.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_181_240))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_181_240.csv
Rows: 60


In [311]:
START_ROW = 181
END_ROW = 240

labels = [
    2, 2,
    0, 0, 0,
    1, 1, 1,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    1, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    1, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 2,
    2, 2, 1,
    0, 2, 1,
    2
]

In [312]:
notes = [
    "Directly relevant late/non-received package case.",
    "Directly relevant pre-order with missing delivery date.",

    "Positive Amazon experience; not relevant to the support issue.",
    "Positive Amazon experience; not relevant.",
    "Positive Amazon pickup experience; not relevant to the query.",

    "Direct refund-status case; highly relevant.",
    "Direct refund-status request; useful historical evidence.",
    "Direct returned-order refund-status case; highly relevant.",

    "Related delivery/order problem, but not the specific missing product defect.",
    "Pre-order shipping problem is related but materially different from the product defect.",
    "Pre-order delivery failure is related to fulfillment but not the product defect itself.",

    "Direct prolonged non-delivery and delivery-delay case.",
    "Direct unpaid/undelivered order delay; highly relevant.",
    "Direct delivery-delay complaint involving a missed required date.",

    "Direct Amazon Music subscription/content example.",
    "Direct Amazon Music usage example; useful for digital-content context.",
    "Direct Amazon Music content/search question; highly relevant.",

    "Direct Prime one-day delivery failure; highly relevant.",
    "Direct Prime delivery-delay complaint; highly relevant.",
    "Direct Prime one-day delivery timing problem; highly relevant.",

    "Direct delivery-driver/address failure; highly relevant to the underlying delivery problem.",
    "Delivered-but-not-received gift is related but materially different from the driver's address issue.",
    "Ongoing delivery investigation and delayed birthday gift; highly relevant.",

    "Direct email-unsubscribe/website issue; highly relevant.",
    "Direct repeated unwanted-email/unsubscribe problem.",
    "Direct unsubscribe problem involving marketing emails.",

    "Shipping-status problem for a pre-order; directly relevant.",
    "Direct missing shipping confirmation/status problem.",
    "Direct delayed Xbox pre-order; highly relevant.",

    "Price/deal discussion is related to the pricing concern but not an exact MRP issue.",
    "Direct website price versus MRP discrepancy; highly relevant.",
    "Direct complaint about selling above MRP; highly relevant.",

    "Wrong/damaged product complaint is directly relevant to the product problem.",
    "Faulty mobile and return/refund problem; highly relevant.",
    "Wrong product delivered; directly relevant.",

    "Direct delivered-but-not-received package case.",
    "Direct delivered-but-not-received package case.",
    "Direct delivered-but-not-received package case with troubleshooting guidance.",

    "Prime registration is related to Amazon digital services but not the specific Amazon Music query.",
    "Prime Video availability is related to digital content but different from Amazon Music.",
    "Direct Amazon Music usage and benefits; highly relevant.",

    "Generic Amazon Prime contact route is only partially useful for the payment/charge issue.",
    "Generic dissatisfaction with Amazon; not useful for the specific payment issue.",
    "Direct unexpected Amazon Prime charge; highly relevant.",

    "Prime benefit/service feedback is related but not the delivery-fee complaint.",
    "Direct Prime dissatisfaction involving delivery delays; useful evidence.",
    "Direct Prime delivery-delay complaint; highly relevant.",

    "Direct damaged-product follow-up problem; highly relevant.",
    "Direct damaged-product complaint; highly relevant.",
    "Direct broken-product return/replacement/refund guidance; highly relevant.",

    "Direct Prime membership cancellation request.",
    "Direct Prime membership cancellation request.",
    "Direct Prime cancellation due to poor service; highly relevant.",

    "Direct delivery-delay issue involving compensation for late delivery.",
    "Direct missing-package/status problem; highly relevant.",
    "Delivered-to-neighbor problem is related to delivery but is materially different from the stated delay.",

    "Account-verification problem is unrelated to the delivery-delay query.",
    "Direct repeated delivery-delay complaint; highly relevant.",
    "Prime/package waiting complaint is related to delivery delay but is less specific.",

    "Direct seller-funds payment/holding problem matching the underlying query."
]

In [313]:
expected_count = END_ROW - START_ROW + 1

print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(label in [0, 1, 2] for label in labels), \
    "Invalid relevance label found."

print("✅ EXACTLY 60 LABELS + 60 NOTES")

Labels: 60
Notes: 60
Expected: 60
✅ EXACTLY 60 LABELS + 60 NOTES


In [314]:
# ---------- Load ----------
df = pd.read_csv(input_path, encoding="utf-8-sig")

df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# ---------- Batch ----------
START_ROW = 181
END_ROW = 240

start_idx = START_ROW - 1
end_idx = END_ROW

# ---------- Update ----------
df.loc[df.index[start_idx:end_idx], "relevance_label"] = labels
df.loc[df.index[start_idx:end_idx], "annotation_notes"] = notes
df.loc[df.index[start_idx:end_idx], "annotator"] = "human_1"

# ---------- Save ----------
df.to_csv(input_path, index=False, encoding="utf-8-sig")

# ---------- Verification ----------
check = pd.read_csv(input_path, encoding="utf-8-sig")
batch = check.iloc[start_idx:end_idx]

print("✅ Batch 181–240 saved successfully")
print(f"Rows updated: {len(batch)}")

print("\nLabel distribution:")
print(batch["relevance_label"].value_counts().sort_index())

print(f"\nRemaining unannotated rows: {check['relevance_label'].isna().sum()}")

print("\nVerification:")
print(
    batch[
        ["query_conversation_id", "rank", "relevance_label"]
    ].to_string(index=False)
)

✅ Batch 181–240 saved successfully
Rows updated: 60

Label distribution:
relevance_label
0.0     4
1.0     7
2.0    49
Name: count, dtype: int64

Remaining unannotated rows: 240

Verification:
 query_conversation_id  rank  relevance_label
               1741904     1              2.0
               1741904     2              2.0
               1741904     3              0.0
               1844497     1              0.0
               1844497     2              0.0
               1844497     3              1.0
               1875465     1              1.0
               1875465     2              1.0
               1875465     3              2.0
               1986309     1              2.0
               1986309     2              2.0
               1986309     3              2.0
               2049109     1              2.0
               2049109     2              2.0
               2049109     3              2.0
               2107331     1              2.0
               2107331   

In [315]:
next_241_300 = retrieval_annotations.iloc[241:301].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_241_300.csv"
)

next_241_300.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_241_300))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_241_300.csv
Rows: 60


In [316]:
START_ROW = 241
END_ROW = 300

labels = [
    2, 2,
    2, 2, 1,
    2, 2, 2,
    2, 2, 2,
    0, 0, 0,
    2, 2, 2,
    1, 2, 1,
    1, 2, 2,
    0, 0, 0,
    2, 2, 2,
    2, 0, 0,
    2, 1, 2,
    2, 2, 2,
    2, 2, 2,
    1, 1, 1,
    0, 0, 0,
    2, 2, 2,
    2, 2, 2,
    1, 1, 1,
    2, 2, 2,
    2
]

notes = [
    "Direct seller-funds/payment-hold issue; strong historical evidence.",
    "Direct seller-support routing for held seller funds; useful evidence.",
    "Defective-product complaint is directly related, though the retrieved case focuses on wrong product descriptions.",
    "Escalation and investigation of a defective-product complaint; useful historical evidence.",
    "Undelivered order is related to the broader complaint but materially different from the defective-product issue.",
    "Direct refund-delay case; highly relevant.",
    "Direct refund-status problem; highly relevant historical evidence.",
    "Defective product followed by refund request; highly relevant.",
    "Direct Prime two-day delivery failure and carrier problem; highly relevant.",
    "Direct Prime delivery delay with missing expected date; highly relevant.",
    "Direct Prime delivery-delay complaint; highly relevant.",
    "Positive social interaction with Amazon; not useful for the support issue.",
    "Positive social interaction; not relevant to the stated issue.",
    "Positive social interaction; not relevant to the stated issue.",
    "Direct delivery-agent/address failure; useful evidence for the underlying delivery problem.",
    "Direct delivery-agent refusal/problem; highly relevant.",
    "Direct delivery-agent refusal and cancellation; highly relevant.",
    "Accidental Prime registration is related to digital/Prime content but not the specific music issue.",
    "Direct Amazon Prime Music example; useful historical evidence.",
    "General Prime praise is only partially useful for a digital-content query.",
    "Delivery problem is unrelated to the price/availability query.",
    "Switch deal/pricing discussion is related to product pricing; useful evidence.",
    "Product pricing/deal example is related to availability and pricing.",
    "Amazon Dash Button positive interaction; not relevant to device malfunction.",
    "Amazon purchase praise; not relevant to device malfunction.",
    "Prime praise; not relevant to the malfunctioning Dash Button.",
    "Direct packaging/environmental complaint matching the query; highly relevant.",
    "Direct packaging/environmental complaint; highly relevant.",
    "Direct packaging feedback and environmental concern; highly relevant.",
    "Direct return/used-damaged product example; highly relevant.",
    "Positive customer experience; not useful for a return problem.",
    "Positive customer experience; not useful for a return problem.",
    "Lost/missing delivery involving electronics; directly related to product delivery damage/loss.",
    "Delivery-delay case is related to the delivery aspect but not the electronics problem itself.",
    "Delivery-driver problem involving an item; useful historical evidence.",
    "Direct Prime delivery-delay example; highly relevant.",
    "Direct Prime delivery-delay example; highly relevant.",
    "Direct Prime next-day delivery issue; highly relevant.",
    "Accidental Prime registration is only indirectly related to digital content.",
    "Direct Prime Music usage example; useful historical evidence.",
    "Direct Prime Music example; highly relevant.",
    "Generic Prime contact routing is only partially useful for the payment issue.",
    "Generic Amazon dissatisfaction is not useful for the specific payment issue.",
    "Direct unexpected Prime charge/payment issue; highly relevant.",
    "Prime-related feedback is related but does not address the specific delivery-fee issue.",
    "Prime delivery-delay complaint; useful historical evidence.",
    "Direct Prime delivery-delay complaint; highly relevant.",
    "Direct product-damage complaint; highly relevant.",
    "Direct damaged-product case; highly relevant.",
    "Direct broken-product return/replacement workflow; highly relevant.",
    "Direct Prime membership cancellation request; highly relevant.",
    "Direct Prime membership cancellation request; highly relevant.",
    "Prime cancellation due to poor service; highly relevant.",
    "Direct delivery-delay/late-delivery complaint; highly relevant.",
    "Direct missing-package/delivery-status problem; highly relevant.",
    "Delivered-to-neighbor problem is related to delivery but differs from a pure delay.",
    "Account-verification problem is unrelated to the delivery-delay query.",
    "Direct repeated delivery-delay complaint; highly relevant.",
    "Prime/package waiting complaint is related to delivery delay but less specific.",
    "Direct seller-funds/payment-hold problem matching the underlying query."
]

# Safety checks BEFORE saving
expected_count = END_ROW - START_ROW + 1

assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(label in [0, 1, 2] for label in labels)

print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

Labels: 60
Notes: 60
Expected: 60


In [317]:
# ============================================================
# SAVE + VERIFY BATCH 241–300
# ============================================================

import pandas as pd

# ---------- Load ----------
df = pd.read_csv(input_path, encoding="utf-8-sig")

# Preserve clean string dtypes
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

# ---------- Row positions ----------
start_idx = START_ROW - 1
end_idx = END_ROW

# ---------- Update ----------
df.loc[df.index[start_idx:end_idx], "relevance_label"] = labels
df.loc[df.index[start_idx:end_idx], "annotation_notes"] = notes
df.loc[df.index[start_idx:end_idx], "annotator"] = "human_1"

# ---------- Save ----------
df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

# ---------- Reload for verification ----------
check = pd.read_csv(
    input_path,
    encoding="utf-8-sig"
)

batch = check.iloc[start_idx:end_idx]

print("=" * 60)
print("✅ Batch 241–300 saved successfully")
print("=" * 60)

print(f"Rows updated: {len(batch)}")

print("\nLabel distribution:")
print(
    batch["relevance_label"]
    .value_counts()
    .sort_index()
)

print(
    f"\nRemaining unannotated rows: "
    f"{check['relevance_label'].isna().sum()}"
)

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label"
        ]
    ].to_string(index=False)
)

# ---------- Final assertions ----------
assert len(batch) == 60
assert batch["relevance_label"].notna().all()
assert batch["relevance_label"].isin([0, 1, 2]).all()

print("\n✅ Verification passed.")

✅ Batch 241–300 saved successfully
Rows updated: 60

Label distribution:
relevance_label
0.0    11
1.0    11
2.0    38
Name: count, dtype: int64

Remaining unannotated rows: 180

Verification:
 query_conversation_id  rank  relevance_label
               2452519     1              2.0
               2452519     2              2.0
               2452519     3              2.0
               2462189     1              2.0
               2462189     2              1.0
               2462189     3              2.0
               2475920     1              2.0
               2475920     2              2.0
               2475920     3              2.0
               2545356     1              2.0
               2545356     2              2.0
               2545356     3              0.0
               2590705     1              0.0
               2590705     2              0.0
               2590705     3              2.0
               2601477     1              2.0
               2601477   

In [318]:
next_301_450 = retrieval_annotations.iloc[301:451].copy()

next_30_path = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\retrieval_301_450.csv"
)

next_301_450.to_csv(
    next_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", next_30_path)
print("Rows:", len(next_301_450))

Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_301_450.csv
Rows: 149


In [319]:
START_ROW = 301
END_ROW = 450

labels = [
    # 93469
    2, 2, 2,
    # 111973
    1, 1, 1,
    # 226344
    2, 2, 2,
    # 245199
    2, 1, 1,
    # 247039
    2, 2, 2,
    # 252859
    2, 2, 2,
    # 317716
    2, 2, 2,
    # 322308
    2, 2, 2,
    # 339933
    0, 1, 2,
    # 351461
    1, 1, 1,
    # 352157
    2, 2, 1,
    # 369162
    2, 2, 0,
    # 371101
    1, 1, 1,
    # 373303
    2, 2, 0,
    # 402220
    1, 1, 1,
    # 468219
    1, 0, 0,
    # 495997
    1, 2, 1,
    # 530044
    1, 2, 2,
    # 549417
    0, 0, 2,
    # 579876
    2, 2, 2,
    # 657944
    1, 0, 2,
    # 679856
    1, 2, 2,
    # 726574
    2, 2, 2,
    # 930355
    2, 1, 1,
    # 1076284
    1, 1, 1,
    # 1080670
    2, 2, 2,
    # 1125002
    1, 1, 1,
    # 1153231
    0, 0, 0,
    # 1362477
    0, 0, 0,
    # 1391003
    1, 2, 1,
    # 1423778
    2, 2, 2,
    # 1445469
    2, 2, 2,
    # 1480838
    1, 2, 0,
    # 2122273
    0, 0, 1,
    # 2212072
    2, 2, 1,
    # 2291199
    1, 2, 2,
    # 2356883
    0, 2, 1,
    # 2386050
    2, 1, 2,
    # 2395769
    2, 2, 2,
    # 2418454
    1, 1, 2,
    # 2476029
    0, 0, 0,
    # 2532525
    0, 0, 0,
    # 2552204
    2, 2, 2,
    # 2671137
    2, 2, 2,
    # 2731119
    0, 1, 0,
    # 2924479
    2, 2, 2,
    # 2934446
    0, 1, 0,
    # 2948180
    1, 1, 1,
    # 2950689
    2, 2, 1,
    # 2983190
    2, 2, 2
]

In [320]:
notes = [
    # 93469
    "Direct late Prime delivery case; highly relevant.",
    "Direct Prime delivery-delay case; highly relevant.",
    "Direct guaranteed-delivery failure; useful historical evidence.",

    # 111973
    "Prime payment/value is mentioned, but the retrieved case focuses mainly on delivery delay.",
    "Prime shipping failure is related to the paid-service complaint but not specifically billing.",
    "Direct Prime delivery failure; related but materially different from a payment issue.",

    # 226344
    "Direct fake-charger/product authenticity complaint.",
    "Direct fake-charger complaint with support investigation.",
    "Direct fake power-pack complaint; highly relevant product evidence.",

    # 245199
    "Direct delivered-to-resident/misdelivery case.",
    "Delivery-address change issue is related but not a missing-delivery resolution.",
    "Delivery-fee refund for a missed delivery is related but materially different.",

    # 247039
    "Direct Nokia return problem.",
    "Direct Nokia refund-after-return case.",
    "Direct Nokia refund request; highly relevant.",

    # 252859
    "Direct delivery-agent misconduct/problem.",
    "Direct delivery-agent failure and missing attempt.",
    "Direct failed delivery-attempt example.",

    # 317716
    "Direct guaranteed-delivery failure.",
    "Direct Prime delivery-delay complaint.",
    "Direct repeated Prime shipping-promise failure.",

    # 322308
    "Direct Prime delivery-delay case.",
    "Delivery/refund problem is relevant to the stated delay.",
    "Direct Prime late-shipping complaint.",

    # 339933
    "Fraud/scam warning is unrelated to the damaged CD case.",
    "Broken product case is related but involves a different item.",
    "Direct broken CD-case complaint; highly useful.",

    # 351461
    "Price-drop case is related to the pricing aspect but not exact availability.",
    "Direct Fire Tablet price-drop example; useful historical evidence.",
    "Direct price-policy example for a Fire Tablet.",

    # 352157
    "Direct wrong-product complaint.",
    "Direct used-product complaint and return workflow.",
    "General poor-service/missing-order case is only partially relevant.",

    # 369162
    "Direct promotion-credit problem.",
    "Direct promotion not applying to purchase.",
    "Missing-product case is unrelated to the promotion query.",

    # 371101
    "Delivery-attempt problem is related to the conversation but not account security.",
    "Delivery-status case is materially different from account access.",
    "Failed delivery case is related only to the delivery aspect.",

    # 373303
    "Direct one-day shipping failure.",
    "Direct same-day delivery problem; useful evidence.",
    "Positive Amazon praise is not useful for the delay problem.",

    # 402220
    "DM support response is related to the unresolved support interaction but lacks payment-specific evidence.",
    "DM response is related to the support-channel problem but not payment details.",
    "Generic DM response is only partially useful for the payment query.",

    # 468219
    "Order-delivery information is somewhat related to the customer's Amazon issue.",
    "Generic Amazon difficulty is not useful for the specific issue.",
    "Positive delivery interaction is not relevant.",

    # 495997
    "Poor customer-service case is related but lacks a concrete delivery resolution.",
    "Direct one-day delivery-delay example.",
    "Generic service complaint is only partially useful.",

    # 530044
    "Delivery support-routing case is related but does not resolve the missing package.",
    "Direct failed/delivered-status delivery problem.",
    "Direct delivery problem with refund/replacement discussion.",

    # 549417
    "Positive Amazon interaction is not useful for a delay complaint.",
    "Positive early-delivery example is not useful for the delay complaint.",
    "Direct delayed dispatch/expected-date discussion.",

    # 579876
    "Direct missing-refund case.",
    "Direct long-pending refund case.",
    "Direct unresolved refund complaint with escalation.",

    # 657944
    "Delivery delay is present but the retrieved case is dominated by account-security issues.",
    "Account-lock case is unrelated to delivery delay.",
    "Direct Prime delivery-delay case.",

    # 679856
    "Late-delivery refund is related to the return/refund theme but not price adjustment.",
    "Direct price-match/return workflow; highly relevant.",
    "Direct price-adjustment and return-policy example.",

    # 726574
    "Direct incorrect delivered-status case.",
    "Direct delivered-but-not-received case.",
    "Direct repeated delivered-status/missing-package case.",

    # 930355
    "Direct Prime delivery-delay case.",
    "Refund/import-charge problem is related to an order but not the delivery-delay issue.",
    "Delivered-status problem is related to delivery but differs from the stated delay.",

    # 1076284
    "Wrong-item/order-cancellation case is related to the order problem but not digital content.",
    "Damaged-product return case is related to order handling but not digital content.",
    "Faulty-product case is materially different from the digital-content query.",

    # 1080670
    "Direct delivery failure.",
    "Direct undelivered-order complaint.",
    "Direct delivery-service complaint with support routing.",

    # 1125002
    "Generic customer-service complaint with only weak product relevance.",
    "Generic customer-service dissatisfaction; no concrete product evidence.",
    "Generic service complaint; not a substantive product resolution.",

    # 1153231
    "Seller pricing information is not directly relevant to the social/app query.",
    "Missing iPad pen is a different issue from the stated query.",
    "Product offer information is unrelated to the stated social interaction.",

    # 1362477
    "Language-support routing is unrelated to the delivery-delay problem.",
    "Positive on-time delivery is not useful for a delay case.",
    "International shipping/document problem is materially different.",

    # 1391003
    "Prime cancellation complaint is related to the customer's cancellation concern but lacks the specific order context.",
    "Direct order-cancellation case; highly useful.",
    "Prime/delivery cancellation complaint is related but less specific.",

    # 1423778
    "Direct unauthorized-charge case despite the benchmark intent being OTHER_NON_SUPPORT.",
    "Direct unauthorized Amazon charge case.",
    "Direct repeated Prime-charge case; highly relevant.",

    # 1445469
    "Direct wishlist technical problem.",
    "Direct wishlist display/settings problem.",
    "Direct wishlist malfunction across devices.",

    # 1480838
    "Echo delivery/pre-order example is related but not a direct resolution.",
    "Direct Echo dispatch-date information.",
    "Positive Echo arrival is not useful for a delayed-delivery complaint.",

    # 2122273
    "Wrong-product case is unrelated to the Wink Hub duplicate-device issue.",
    "Cancelled-deal case is unrelated to the device-listing issue.",
    "App/product compatibility feedback is somewhat related to the smart-device context.",

    # 2212072
    "Direct delivered-but-not-received case.",
    "Direct repeated incorrect-delivery-status case.",
    "Delivery-driver problem is related but not a direct match.",

    # 2291199
    "Alexa language information is related to the query but not a direct troubleshooting resolution.",
    "Direct Alexa language-setting guidance.",
    "Direct non-English language support limitation; highly useful.",

    # 2356883
    "Return-policy problem is unrelated to handing a phone to the wrong person.",
    "Seller delivery-claim case is materially related to delivery accountability.",
    "Cancelled-phone-order case is related to order handling but not recipient verification.",

    # 2386050
    "Direct carrier/delivery-delay case.",
    "Delivery-time and package-condition case is related but not carrier-selection specific.",
    "Direct delayed-delivery case; useful evidence.",

    # 2395769
    "Direct locked-account/security case.",
    "Direct Amazon Pay account-lock case.",
    "Direct locked-account recovery case.",

    # 2418454
    "Lost-package/service case is related to the broader support problem but not faulty-product return.",
    "Incomplete/damaged-order case with refund workflow; useful historical evidence.",
    "Direct damaged-product return/refund case; highly relevant.",

    # 2476029
    "Positive Amazon social interaction is not relevant to missing delivery.",
    "Positive delivered-product interaction is not relevant.",
    "Positive delivery interaction is not relevant.",

    # 2532525
    "Prime delivery delay is unrelated to the requested digital-content availability.",
    "Prime shipping complaint is unrelated to TV-series availability.",
    "Prime delivery delay is not useful for digital-content resolution.",

    # 2552204
    "Direct Amazon checkout error case.",
    "Direct resolved account/order-access error case.",
    "Direct app technical troubleshooting with successful reinstall.",

    # 2671137
    "Direct guaranteed-delivery delay.",
    "Direct package-never-arrived case.",
    "Direct delayed/pending package case.",

    # 2731119
    "Positive Amazon feedback is not useful for a poor-service complaint.",
    "Direct delayed-delivery complaint; related historical evidence.",
    "Positive Amazon praise is not relevant.",

    # 2924479
    "Direct Alexa technical-support case.",
    "Direct Alexa device malfunction and troubleshooting case.",
    "Direct Echo hardware/technical failure case.",

    # 2934446
    "Used-garment complaint is unrelated to the positive kilt-shopping post.",
    "Clothing feedback is somewhat related to the shopping context.",
    "Wrong grocery-item case is unrelated to the stated post.",

    # 2948180
    "Direct Prime delivery-delay example.",
    "Direct Prime delivery-date complaint.",
    "Direct two-day shipping failure example.",

    # 2950689
    "Direct repeated dog-food delivery failure.",
    "Direct carrier failure involving recurring dog-food deliveries.",
    "Account/address/subscription delivery problem is related but less direct.",

    # 2983190
    "Direct Prime membership registration example.",
    "Direct Prime membership registration example.",
    "Direct Prime membership registration example."
]

In [321]:
expected_count = END_ROW - START_ROW + 1

print("Labels:", len(labels))
print("Notes:", len(notes))
print("Expected:", expected_count)

assert len(labels) == expected_count, (
    f"Expected {expected_count} labels, got {len(labels)}"
)

assert len(notes) == expected_count, (
    f"Expected {expected_count} notes, got {len(notes)}"
)

assert all(label in [0, 1, 2] for label in labels), \
    "Invalid relevance label found."

print("✅ EXACTLY 150 LABELS + 150 NOTES")

Labels: 150
Notes: 150
Expected: 150
✅ EXACTLY 150 LABELS + 150 NOTES


In [322]:
# ============================================================
# SAVE + VERIFY BATCH 301–450
# ============================================================

import pandas as pd

input_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

# Prevent dtype warnings
df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

start_idx = START_ROW - 1
end_idx = END_ROW

# Write annotations
df.loc[df.index[start_idx:end_idx], "relevance_label"] = labels
df.loc[df.index[start_idx:end_idx], "annotation_notes"] = notes
df.loc[df.index[start_idx:end_idx], "annotator"] = "human_1"

# Save
df.to_csv(input_path, index=False, encoding="utf-8-sig")

# Reload and verify
check = pd.read_csv(input_path, encoding="utf-8-sig")
batch = check.iloc[start_idx:end_idx]

print("=" * 65)
print("✅ BATCH 301–450 SAVED SUCCESSFULLY")
print("=" * 65)

print(f"Rows updated: {len(batch)}")

print("\nLabel distribution:")
print(batch["relevance_label"].value_counts().sort_index())

print(
    f"\nRemaining unannotated rows: "
    f"{check['relevance_label'].isna().sum()}"
)

print("\nVerification:")
print(
    batch[
        [
            "query_conversation_id",
            "rank",
            "relevance_label"
        ]
    ].to_string(index=False)
)

# Hard validation
assert len(batch) == 150
assert batch["relevance_label"].notna().all()
assert batch["relevance_label"].isin([0, 1, 2]).all()
assert batch["annotation_notes"].notna().all()
assert batch["annotator"].notna().all()

print("\n" + "=" * 65)
print("✅ VERIFICATION PASSED")
print("✅ 301–450 COMPLETE")
print("=" * 65)

✅ BATCH 301–450 SAVED SUCCESSFULLY
Rows updated: 150

Label distribution:
relevance_label
0.0    28
1.0    46
2.0    76
Name: count, dtype: int64

Remaining unannotated rows: 30

Verification:
 query_conversation_id  rank  relevance_label
                 93469     1              2.0
                 93469     2              2.0
                 93469     3              2.0
                111973     1              1.0
                111973     2              1.0
                111973     3              1.0
                226344     1              2.0
                226344     2              2.0
                226344     3              2.0
                245199     1              2.0
                245199     2              1.0
                245199     3              1.0
                247039     1              2.0
                247039     2              2.0
                247039     3              2.0
                252859     1              2.0
                252859   

In [323]:
import pandas as pd

input_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

missing = df[df["relevance_label"].isna()].copy()

print("=" * 60)
print("MISSING ANNOTATIONS")
print("=" * 60)

print(f"Total rows: {len(df)}")
print(f"Annotated: {df['relevance_label'].notna().sum()}")
print(f"Unannotated: {df['relevance_label'].isna().sum()}")

print("\nFirst 100 missing row numbers:")
print((missing.index + 1).tolist()[:100])

MISSING ANNOTATIONS
Total rows: 450
Annotated: 420
Unannotated: 30

First 100 missing row numbers:
[31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]


In [324]:
START_ROW = 31
END_ROW = 60

labels = [
    1, 2, 2, 2, 1, 0,
    2, 2, 2, 0, 1, 0,
    2, 2, 1, 2, 2, 2,
    0, 2, 1, 2, 1, 2,
    0, 0, 0, 0, 0, 0
]

In [325]:
print(len(labels))
print(len(notes))

30
150


In [327]:
notes = [
    "Related order/return issue, but materially different.",
    "Missing book/order issue; useful but not the same requested outcome.",
    "Generic order problem and refund request; partially useful.",
    "Direct delivered-but-not-received case.",
    "Delivery-status problem, but not delivered/misdelivered.",
    "Direct delivered-but-not-received case.",
    "Direct Prime/delivery-delay problem.",
    "Prime shipping/delivery issue, but mainly seller/fee eligibility.",
    "Prime one-day delivery availability; related but not an actual delay.",
    "Delivery delay, unrelated to account-security problem.",
    "Generic delivery complaint, not account access.",
    "Order-return issue, not account security.",
    "Direct electronic-device/product problem.",
    "Direct faulty iPhone/product issue.",
    "Direct device dead-on-arrival/product problem.",
    "Strong delivery problem match despite noisy query label.",
    "Repeated product-return issue; related but different.",
    "Refund for damaged product; materially different.",
    "Direct same-day delivery delay.",
    "Direct repeated delivery delay.",
    "Direct past-due delivery.",
    "Positive on-time delivery; not useful for delay.",
    "Direct late-delivery complaint.",
    "Direct delivery-not-on-time complaint.",
    "Direct damaged-product case.",
    "Damaged packaging; partially relevant.",
    "Packaging/delivery complaint, but product itself undamaged.",
    "Payment/Prime issue, not account-access problem.",
    "Prime-membership problem overlaps one part of query.",
    "Delivery delay, unrelated to account issue."
]

print("Labels:", len(labels))
print("Notes:", len(notes))

assert len(labels) == 30
assert len(notes) == 30

print("✅ Exactly 30 labels + 30 notes")

Labels: 30
Notes: 30
✅ Exactly 30 labels + 30 notes


In [328]:
import pandas as pd

input_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

df["annotation_notes"] = df["annotation_notes"].astype("string")
df["annotator"] = df["annotator"].astype("string")

start_idx = START_ROW - 1
end_idx = END_ROW

df.loc[df.index[start_idx:end_idx], "relevance_label"] = labels
df.loc[df.index[start_idx:end_idx], "annotation_notes"] = notes
df.loc[df.index[start_idx:end_idx], "annotator"] = "human_1"

df.to_csv(input_path, index=False, encoding="utf-8-sig")

check = pd.read_csv(input_path, encoding="utf-8-sig")

print("=" * 60)
print("✅ BATCH 31–60 RESTORED")
print("=" * 60)
print("Rows updated:", len(check.iloc[start_idx:end_idx]))
print("Remaining unannotated rows:", check["relevance_label"].isna().sum())

assert check["relevance_label"].notna().all()

print("\n🎯 ALL 450 RETRIEVAL PAIRS ARE ANNOTATED.")

✅ BATCH 31–60 RESTORED
Rows updated: 30
Remaining unannotated rows: 0

🎯 ALL 450 RETRIEVAL PAIRS ARE ANNOTATED.


In [329]:
import pandas as pd
import numpy as np

input_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(df) == 450
assert df["relevance_label"].notna().all()
assert df["relevance_label"].isin([0, 1, 2]).all()

# Each query must have exactly 3 judged results
query_counts = df.groupby("query_conversation_id")["rank"].count()

assert len(query_counts) == 150
assert query_counts.eq(3).all()

# ------------------------------------------------------------
# Sort correctly
# ------------------------------------------------------------

df = df.sort_values(
    ["query_conversation_id", "rank"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Per-query metrics
# ------------------------------------------------------------

results = []

for query_id, group in df.groupby("query_conversation_id", sort=False):

    group = group.sort_values("rank")
    labels = group["relevance_label"].astype(int).tolist()

    # Strict relevance: only label 2
    strict = np.array([x == 2 for x in labels])

    # Permissive relevance: labels 1 or 2
    inclusive = np.array([x >= 1 for x in labels])

    # Precision@1
    p1 = strict[0]

    # Precision@3
    p3 = strict.mean()

    # Hit@1
    hit1 = strict[0]

    # Hit@3
    hit3 = strict.any()

    # MRR using strict relevance
    relevant_positions = np.where(strict)[0]

    if len(relevant_positions) > 0:
        mrr = 1 / (relevant_positions[0] + 1)
    else:
        mrr = 0.0

    # nDCG@3 using graded relevance 0/1/2
    gains = np.array(labels, dtype=float)

    discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains * discounts)

    ideal_gains = np.sort(gains)[::-1]
    idcg = np.sum(ideal_gains * discounts)

    ndcg3 = dcg / idcg if idcg > 0 else 0.0

    results.append({
        "query_conversation_id": query_id,
        "P@1": float(p1),
        "P@3": float(p3),
        "Hit@1": float(hit1),
        "Hit@3": float(hit3),
        "MRR": float(mrr),
        "nDCG@3": float(ndcg3),
        "strict_relevant_count": int(strict.sum()),
        "inclusive_relevant_count": int(inclusive.sum())
    })

metrics_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

print("=" * 70)
print("RETRIEVAL EVALUATION — OVERALL")
print("=" * 70)

for metric in ["P@1", "P@3", "Hit@1", "Hit@3", "MRR", "nDCG@3"]:
    print(f"{metric:10s}: {metrics_df[metric].mean():.4f}")

print("\nQueries:", len(metrics_df))
print("Judged pairs:", len(df))

# ------------------------------------------------------------
# Relevance distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RELEVANCE DISTRIBUTION")
print("=" * 70)

print(
    df["relevance_label"]
    .value_counts()
    .sort_index()
)

print("\nPercent:")
print(
    (df["relevance_label"]
     .value_counts(normalize=True)
     .sort_index()
     .mul(100)
     .round(2))
)

# ------------------------------------------------------------
# Benchmark source breakdown
# ------------------------------------------------------------

query_source = (
    df.groupby("query_conversation_id")["benchmark_source"]
      .first()
)

metrics_df["benchmark_source"] = (
    metrics_df["query_conversation_id"]
    .map(query_source)
)

print("\n" + "=" * 70)
print("RETRIEVAL EVALUATION — BY BENCHMARK SOURCE")
print("=" * 70)

summary = (
    metrics_df
    .groupby("benchmark_source")[
        ["P@1", "P@3", "Hit@1", "Hit@3", "MRR", "nDCG@3"]
    ]
    .mean()
)

print(summary.round(4))

# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

metrics_output = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_metrics_per_query.csv"

metrics_df.to_csv(
    metrics_output,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("✅ RETRIEVAL EVALUATION COMPLETE")
print("=" * 70)
print("Saved:", metrics_output)

RETRIEVAL EVALUATION — OVERALL
P@1       : 0.6400
P@3       : 0.5911
Hit@1     : 0.6400
Hit@3     : 0.8200
MRR       : 0.7144
nDCG@3    : 0.8750

Queries: 150
Judged pairs: 450

RELEVANCE DISTRIBUTION
relevance_label
0.0     75
1.0    109
2.0    266
Name: count, dtype: int64

Percent:
relevance_label
0.0    16.67
1.0    24.22
2.0    59.11
Name: proportion, dtype: float64

RETRIEVAL EVALUATION — BY BENCHMARK SOURCE
                   P@1     P@3  Hit@1  Hit@3     MRR  nDCG@3
benchmark_source                                            
taxonomy_audit    0.72  0.6333   0.72   0.88  0.7833  0.8891
training          0.48  0.5067   0.48   0.70  0.5767  0.8468

✅ RETRIEVAL EVALUATION COMPLETE
Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_metrics_per_query.csv


The retriever is good at finding useful historical evidence somewhere in the top 3, but it is not consistently putting the best evidence first.

The clearest number is:

Hit@3 = 82%

That means 82% of benchmark queries had at least one directly relevant historical episode within the top three retrieved results.

But:

P@1 = 64%

means the first retrieved result was directly useful for 64% of queries.

On a 150-query human-judged benchmark with three retrieved episodes per query (450 pairwise judgments), the baseline semantic retriever achieved 64.0% P@1, 82.0% Hit@3, 0.714 MRR, and 0.875 nDCG@3. Of the 450 retrieved episodes judged, 59.1% were directly useful, 24.2% were partially relevant, and 16.7% were not useful. Performance differed between the taxonomy-audit subset (Hit@3 88.0%) and training subset (70.0%). The main observed limitation is ranking: useful evidence is often present within the top three even when it is not ranked first.

#### Intent-aware retrieval

Build the gold-intent corpus

In [330]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

training_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_training_batch_01.csv"
audit_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-ai-agent\data\processed\taxonomy_audit_150.csv"

In [331]:
training_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\amazonhelp_intent_training_batch_01.csv"
audit_path = r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\taxonomy_audit_150.csv"

training_df_check = pd.read_csv(training_path, encoding="utf-8-sig")
audit_df_check = pd.read_csv(audit_path, encoding="utf-8-sig")

print("Training rows:", len(training_df_check))
print("Audit rows:", len(audit_df_check))
print("Training columns:", training_df_check.columns.tolist())
print("Audit columns:", audit_df_check.columns.tolist())

Training rows: 300
Audit rows: 150
Training columns: ['conversation_id', 'first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'split', 'is_taxonomy_audit', 'intent_label', 'annotation_notes', 'annotator', 'annotation_status']
Audit columns: ['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'intent_label', 'annotation_notes', 'annotator']


In [332]:
# ------------------------------------------------------------
# Verify the validated benchmark objects already in memory
# ------------------------------------------------------------

print("training_df:")
print("  Rows:", len(training_df))
print("  Missing labels:", training_df["intent_label"].isna().sum())
print("  Unique intents:", training_df["intent_label"].nunique())
print("  Unique conversations:", training_df["conversation_id"].nunique())

print("\naudit benchmark:")
print("  Rows:", len(audit_benchmark))
print("  Missing labels:", audit_benchmark["intent_label"].isna().sum())
print("  Unique intents:", audit_benchmark["intent_label"].nunique())
print("  Unique conversations:", audit_benchmark["conversation_id"].nunique())

assert len(training_df) == 300
assert training_df["intent_label"].notna().all()
assert training_df["intent_label"].nunique() == 15
assert training_df["conversation_id"].nunique() == 300

assert len(audit_benchmark) == 150
assert audit_benchmark["intent_label"].notna().all()
assert audit_benchmark["intent_label"].nunique() == 15
assert audit_benchmark["conversation_id"].nunique() == 150

print("\n✅ Benchmark labels are intact and validated.")

training_df:
  Rows: 300
  Missing labels: 0
  Unique intents: 15
  Unique conversations: 300

audit benchmark:
  Rows: 150
  Missing labels: 0
  Unique intents: 15
  Unique conversations: 150

✅ Benchmark labels are intact and validated.


In [333]:
# ------------------------------------------------------------
# Check available intent-classification objects
# ------------------------------------------------------------

objects_to_check = [
    "embedding_model",
    "intent_model",
    "logreg_model",
    "clf",
    "sentence_embeddings",
    "train_embeddings",
    "X_train_embeddings",
    "y_train",
]

print("Checking notebook objects...\n")

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]
        print(f"✅ {name}: {type(obj).__name__}")
    else:
        print(f"❌ {name}: not found")

Checking notebook objects...

✅ embedding_model: SentenceTransformer
❌ intent_model: not found
❌ logreg_model: not found
❌ clf: not found
❌ sentence_embeddings: not found
❌ train_embeddings: not found
✅ X_train_embeddings: ndarray
✅ y_train: Series


In [335]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# Use the validated 300-row labelled training set
# ------------------------------------------------------------

X_text = training_df["customer_only_text"].fillna("").astype(str)
y = training_df["intent_label"].astype(str)

print("Rows:", len(X_text))
print("Unique intents:", y.nunique())
print("\nClass counts:")
print(y.value_counts().sort_index())

assert len(X_text) == 300
assert len(y) == 300
assert y.notna().all()
assert y.nunique() == 15

# ------------------------------------------------------------
# Recreate the exact 75/25 stratified split
# ------------------------------------------------------------

X_train_text, X_val_text, y_train_split, y_val = train_test_split(
    X_text,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTrain:", len(X_train_text))
print("Validation:", len(X_val_text))

# ------------------------------------------------------------
# Generate embeddings
# ------------------------------------------------------------

X_train_emb = embedding_model.encode(
    X_train_text.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

X_val_emb = embedding_model.encode(
    X_val_text.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("\nEmbedding shapes:")
print("Train:", X_train_emb.shape)
print("Validation:", X_val_emb.shape)

# ------------------------------------------------------------
# Fit the same model configuration
# ------------------------------------------------------------

intent_classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

intent_classifier.fit(
    X_train_emb,
    y_train_split
)

# ------------------------------------------------------------
# Sanity-check performance
# ------------------------------------------------------------

val_predictions = intent_classifier.predict(X_val_emb)

accuracy = accuracy_score(y_val, val_predictions)

macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro"
)

print("\n" + "=" * 65)
print("RECREATED INTENT CLASSIFIER")
print("=" * 65)

print(f"Validation Accuracy : {accuracy:.4f}")
print(f"Validation Macro F1 : {macro_f1:.4f}")

print("\nClasses:")
print(intent_classifier.classes_)

print("\n✅ Classifier successfully recreated from the validated 300-row dataset.")

Rows: 300
Unique intents: 15

Class counts:
intent_label
ACCOUNT_ACCESS_SECURITY              9
DELIVERY_ATTEMPT_OR_INSTRUCTIONS    24
DELIVERY_DELAY                      69
DELIVERY_MISSING_OR_MISDELIVERED    32
DEVICE_TECHNICAL_SUPPORT             4
DIGITAL_CONTENT                     13
GIFT_CARD_PROMOTION                  5
ORDER_STATUS_OR_CANCELLATION        13
OTHER_NON_SUPPORT                   37
PAYMENT_BILLING                     11
PRIME_MEMBERSHIP                     8
PRODUCT_AVAILABILITY_INFORMATION     9
PRODUCT_PROBLEM                     32
RETURN_REPLACEMENT_REFUND           25
WEBSITE_APP_TECHNICAL                9
Name: count, dtype: int64

Train: 225
Validation: 75


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embedding shapes:
Train: (225, 384)
Validation: (75, 384)

RECREATED INTENT CLASSIFIER
Validation Accuracy : 0.2000
Validation Macro F1 : 0.1865

Classes:
['ACCOUNT_ACCESS_SECURITY' 'DELIVERY_ATTEMPT_OR_INSTRUCTIONS'
 'DELIVERY_DELAY' 'DELIVERY_MISSING_OR_MISDELIVERED'
 'DEVICE_TECHNICAL_SUPPORT' 'DIGITAL_CONTENT' 'GIFT_CARD_PROMOTION'
 'ORDER_STATUS_OR_CANCELLATION' 'OTHER_NON_SUPPORT' 'PAYMENT_BILLING'
 'PRIME_MEMBERSHIP' 'PRODUCT_AVAILABILITY_INFORMATION' 'PRODUCT_PROBLEM'
 'RETURN_REPLACEMENT_REFUND' 'WEBSITE_APP_TECHNICAL']

✅ Classifier successfully recreated from the validated 300-row dataset.


In [336]:
# ------------------------------------------------------------
# Predict intents for the leakage-safe retrieval corpus
# ------------------------------------------------------------

corpus = amazon_retrieval_benchmark_corpus.copy()

assert len(corpus) == 81796
assert corpus["conversation_id"].nunique() == 81796

print("Retrieval corpus rows:", len(corpus))
print("Generating embeddings...")

# ------------------------------------------------------------
# Embed retrieval corpus
# ------------------------------------------------------------

corpus_embeddings = embedding_model.encode(
    corpus["retrieval_text"].fillna("").astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("\nCorpus embedding shape:", corpus_embeddings.shape)

assert corpus_embeddings.shape == (81796, 384)

# ------------------------------------------------------------
# Predict pseudo-intents
# ------------------------------------------------------------

corpus["predicted_intent"] = intent_classifier.predict(
    corpus_embeddings
)

# Prediction confidence
corpus["intent_confidence"] = (
    intent_classifier.predict_proba(corpus_embeddings).max(axis=1)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PREDICTED INTENT DISTRIBUTION")
print("=" * 65)

print(
    corpus["predicted_intent"]
    .value_counts()
    .sort_index()
)

print("\nConfidence statistics:")
print(
    corpus["intent_confidence"]
    .describe()
)

assert corpus["predicted_intent"].notna().all()
assert corpus["intent_confidence"].notna().all()

print("\n✅ Predicted intents assigned to all 81,796 corpus conversations.")

Retrieval corpus rows: 81796
Generating embeddings...


Batches:   0%|          | 0/1279 [00:00<?, ?it/s]


Corpus embedding shape: (81796, 384)

PREDICTED INTENT DISTRIBUTION
predicted_intent
ACCOUNT_ACCESS_SECURITY             11325
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     5546
DELIVERY_DELAY                       3519
DELIVERY_MISSING_OR_MISDELIVERED     4958
DEVICE_TECHNICAL_SUPPORT             1649
DIGITAL_CONTENT                      5320
GIFT_CARD_PROMOTION                  5510
ORDER_STATUS_OR_CANCELLATION         4524
OTHER_NON_SUPPORT                    5754
PAYMENT_BILLING                      5426
PRIME_MEMBERSHIP                     3450
PRODUCT_AVAILABILITY_INFORMATION     5655
PRODUCT_PROBLEM                      7700
RETURN_REPLACEMENT_REFUND            5485
WEBSITE_APP_TECHNICAL                5975
Name: count, dtype: int64

Confidence statistics:
count    81796.000000
mean         0.123791
std          0.029579
min          0.076383
25%          0.103988
50%          0.116269
75%          0.134812
max          0.383938
Name: intent_confidence, dtype: float64

✅ Predicted inten

In [337]:
# ------------------------------------------------------------
# Confidence threshold analysis
# ------------------------------------------------------------

thresholds = [0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.35]

print("=" * 70)
print("INTENT CONFIDENCE THRESHOLD ANALYSIS")
print("=" * 70)

for threshold in thresholds:
    mask = corpus["intent_confidence"] >= threshold

    count = int(mask.sum())
    percentage = 100 * count / len(corpus)

    print(
        f"Threshold >= {threshold:.2f} : "
        f"{count:6d} conversations "
        f"({percentage:6.2f}%)"
    )

print("\n" + "=" * 70)
print("BENCHMARK QUERY CONFIDENCE")
print("=" * 70)

# Embed the 450 benchmark queries
benchmark_embeddings = embedding_model.encode(
    retrieval_benchmark["customer_only_text"].fillna("").astype(str).tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

benchmark_predictions = intent_classifier.predict(
    benchmark_embeddings
)

benchmark_probabilities = intent_classifier.predict_proba(
    benchmark_embeddings
)

benchmark_confidence = benchmark_probabilities.max(axis=1)

benchmark_predicted_intents = benchmark_predictions

benchmark_confidence_df = retrieval_benchmark[
    ["conversation_id", "intent_label", "benchmark_source"]
].copy()

benchmark_confidence_df["predicted_intent"] = benchmark_predicted_intents
benchmark_confidence_df["intent_confidence"] = benchmark_confidence

print(
    benchmark_confidence_df["intent_confidence"].describe()
)

print("\nCorrect-intent prediction rate:")
print(
    (
        benchmark_confidence_df["predicted_intent"]
        == benchmark_confidence_df["intent_label"]
    ).mean()
)

print("\nConfidence by prediction correctness:")

benchmark_confidence_df["intent_correct"] = (
    benchmark_confidence_df["predicted_intent"]
    == benchmark_confidence_df["intent_label"]
)

print(
    benchmark_confidence_df
    .groupby("intent_correct")["intent_confidence"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\n✅ Confidence analysis complete.")

INTENT CONFIDENCE THRESHOLD ANALYSIS
Threshold >= 0.10 :  67901 conversations ( 83.01%)
Threshold >= 0.12 :  35689 conversations ( 43.63%)
Threshold >= 0.15 :  11935 conversations ( 14.59%)
Threshold >= 0.20 :   2270 conversations (  2.78%)
Threshold >= 0.25 :    447 conversations (  0.55%)
Threshold >= 0.30 :     74 conversations (  0.09%)
Threshold >= 0.35 :      6 conversations (  0.01%)

BENCHMARK QUERY CONFIDENCE


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

count    450.000000
mean       0.138378
std        0.049926
min        0.083178
25%        0.106845
50%        0.123833
75%        0.153773
max        0.401167
Name: intent_confidence, dtype: float64

Correct-intent prediction rate:
0.4266666666666667

Confidence by prediction correctness:
                count      mean    median       min       max
intent_correct                                               
False             258  0.121491  0.117743  0.083178  0.260494
True              192  0.161069  0.149211  0.085991  0.401167

✅ Confidence analysis complete.


In [338]:
# ------------------------------------------------------------
# Diagnostic: predicted-intent compatibility in semantic Top-10
# ------------------------------------------------------------

corpus_predicted_intents = corpus["predicted_intent"].to_numpy()

diagnostic_rows = []

for query_idx in range(len(retrieval_benchmark)):

    query_predicted_intent = benchmark_predicted_intents[query_idx]

    retrieved_indices = clean_indices[query_idx][:10]
    retrieved_scores = clean_scores[query_idx][:10]

    for rank, (retrieved_idx, similarity) in enumerate(
        zip(retrieved_indices, retrieved_scores),
        start=1
    ):
        retrieved_intent = corpus_predicted_intents[retrieved_idx]

        diagnostic_rows.append({
            "query_conversation_id":
                retrieval_benchmark.iloc[query_idx]["conversation_id"],

            "gold_intent":
                retrieval_benchmark.iloc[query_idx]["intent_label"],

            "query_predicted_intent":
                query_predicted_intent,

            "rank":
                rank,

            "similarity":
                float(similarity),

            "retrieved_predicted_intent":
                retrieved_intent,

            "intent_match":
                query_predicted_intent == retrieved_intent
        })

intent_diagnostic = pd.DataFrame(diagnostic_rows)

print("=" * 70)
print("PREDICTED-INTENT COMPATIBILITY IN SEMANTIC TOP-10")
print("=" * 70)

print(
    "Total query-candidate pairs:",
    len(intent_diagnostic)
)

print(
    "Intent-compatible pairs:",
    intent_diagnostic["intent_match"].sum()
)

print(
    "Intent-compatible rate:",
    intent_diagnostic["intent_match"].mean()
)

print("\nCompatibility by rank:")

rank_summary = (
    intent_diagnostic
    .groupby("rank")["intent_match"]
    .mean()
)

print(
    (rank_summary * 100).round(2)
    .rename("match_percent")
)

print("\nAverage similarity:")
print(
    intent_diagnostic
    .groupby("intent_match")["similarity"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nCompatibility for Top-3:")
top3 = intent_diagnostic[
    intent_diagnostic["rank"] <= 3
]

print(
    top3["intent_match"].mean()
)

print("\n✅ Intent compatibility diagnostic complete.")

PREDICTED-INTENT COMPATIBILITY IN SEMANTIC TOP-10
Total query-candidate pairs: 4500
Intent-compatible pairs: 2331
Intent-compatible rate: 0.518

Compatibility by rank:
rank
1     54.67
2     52.44
3     53.56
4     54.44
5     51.11
6     52.67
7     52.44
8     49.33
9     48.00
10    49.33
Name: match_percent, dtype: float64

Average similarity:
              count      mean    median       min       max
intent_match                                               
False          2169  0.709352  0.725451  0.356738  0.904371
True           2331  0.728447  0.741305  0.356791  0.898806

Compatibility for Top-3:
0.5355555555555556

✅ Intent compatibility diagnostic complete.


In [339]:
# ------------------------------------------------------------
# Intent-aware reranking diagnostic
# ------------------------------------------------------------

# Existing semantic Top-10
base_indices = clean_indices[:, :10]
base_scores = clean_scores[:, :10]

# Predicted intent for every corpus conversation
corpus_predicted_intents = corpus["predicted_intent"].to_numpy()

# Predicted intent for every benchmark query
query_predicted_intents = np.asarray(benchmark_predicted_intents)

# Try several conservative intent bonuses.
# Semantic similarity remains dominant.
BONUSES = [0.00, 0.01, 0.02, 0.03, 0.05, 0.10]

rerank_results = []

for bonus in BONUSES:

    changed_top1 = 0
    changed_top3 = 0
    compatible_top1 = 0
    compatible_top3 = 0

    for q_idx in range(len(retrieval_benchmark)):

        candidate_indices = base_indices[q_idx]
        candidate_scores = base_scores[q_idx]

        query_intent = query_predicted_intents[q_idx]

        candidate_intents = corpus_predicted_intents[
            candidate_indices
        ]

        intent_match = (
            candidate_intents == query_intent
        )

        # Add a small bonus when predicted intents match.
        rerank_scores = (
            candidate_scores
            + bonus * intent_match.astype(float)
        )

        new_order = np.argsort(
            -rerank_scores,
            kind="stable"
        )

        new_indices = candidate_indices[new_order]

        # Original ordering
        original_indices = candidate_indices

        # Did Top-1 change?
        if new_indices[0] != original_indices[0]:
            changed_top1 += 1

        # Did Top-3 set change?
        if set(new_indices[:3]) != set(original_indices[:3]):
            changed_top3 += 1

        # Intent compatibility
        new_top1_intent = corpus_predicted_intents[
            new_indices[0]
        ]

        if new_top1_intent == query_intent:
            compatible_top1 += 1

        new_top3_intents = corpus_predicted_intents[
            new_indices[:3]
        ]

        compatible_top3 += np.sum(
            new_top3_intents == query_intent
        )

    n_queries = len(retrieval_benchmark)

    rerank_results.append({
        "intent_bonus": bonus,
        "top1_changed_pct":
            100 * changed_top1 / n_queries,
        "top3_set_changed_pct":
            100 * changed_top3 / n_queries,
        "top1_intent_compatibility_pct":
            100 * compatible_top1 / n_queries,
        "top3_intent_compatibility_pct":
            100 * compatible_top3 / (n_queries * 3)
    })

rerank_diagnostic = pd.DataFrame(rerank_results)

print("=" * 80)
print("INTENT-AWARE RERANKING DIAGNOSTIC")
print("=" * 80)

print(
    rerank_diagnostic.to_string(
        index=False,
        formatters={
            "intent_bonus": "{:.2f}".format,
            "top1_changed_pct": "{:.2f}".format,
            "top3_set_changed_pct": "{:.2f}".format,
            "top1_intent_compatibility_pct": "{:.2f}".format,
            "top3_intent_compatibility_pct": "{:.2f}".format,
        }
    )
)

print("\n✅ Reranking diagnostic complete.")

INTENT-AWARE RERANKING DIAGNOSTIC
intent_bonus top1_changed_pct top3_set_changed_pct top1_intent_compatibility_pct top3_intent_compatibility_pct
        0.00             0.00                 0.00                         54.67                         53.56
        0.01            11.78                28.22                         66.44                         63.93
        0.02            21.56                44.00                         76.22                         72.22
        0.03            26.89                51.11                         81.56                         76.22
        0.05            34.22                56.89                         88.89                         80.67
        0.10            36.22                57.56                         90.89                         82.44

✅ Reranking diagnostic complete.


In [340]:
# ------------------------------------------------------------
# Select focused human-evaluation queries
# ------------------------------------------------------------

def get_reranked_indices(query_idx, bonus):
    candidate_indices = clean_indices[query_idx][:10]
    candidate_scores = clean_scores[query_idx][:10]

    query_intent = benchmark_predicted_intents[query_idx]

    candidate_intents = corpus_predicted_intents[
        candidate_indices
    ]

    intent_match = (
        candidate_intents == query_intent
    )

    rerank_scores = (
        candidate_scores
        + bonus * intent_match.astype(float)
    )

    order = np.argsort(
        -rerank_scores,
        kind="stable"
    )

    return candidate_indices[order]


# Find queries whose Top-3 changes under each candidate bonus
changed_002 = []
changed_003 = []

for q_idx in range(len(retrieval_benchmark)):

    baseline_top3 = set(clean_indices[q_idx][:3])

    reranked_002 = get_reranked_indices(q_idx, 0.02)
    reranked_003 = get_reranked_indices(q_idx, 0.03)

    if set(reranked_002[:3]) != baseline_top3:
        changed_002.append(q_idx)

    if set(reranked_003[:3]) != baseline_top3:
        changed_003.append(q_idx)


print("Queries changed at bonus 0.02:", len(changed_002))
print("Queries changed at bonus 0.03:", len(changed_003))


# ------------------------------------------------------------
# Prefer queries where both configurations make a change
# ------------------------------------------------------------

both_changed = sorted(
    set(changed_002) & set(changed_003)
)

print("Queries changed by BOTH:", len(both_changed))


# ------------------------------------------------------------
# Select 30 queries
# ------------------------------------------------------------

# Deterministic selection.
selected_indices = both_changed[:30]

if len(selected_indices) < 30:
    remaining = [
        idx for idx in changed_003
        if idx not in selected_indices
    ]
    selected_indices.extend(
        remaining[:30 - len(selected_indices)]
    )

if len(selected_indices) < 30:
    remaining = [
        idx for idx in changed_002
        if idx not in selected_indices
    ]
    selected_indices.extend(
        remaining[:30 - len(selected_indices)]
    )

assert len(selected_indices) == 30

print("\n" + "=" * 70)
print("SELECTED HUMAN-EVALUATION SAMPLE")
print("=" * 70)

selected_queries = retrieval_benchmark.iloc[
    selected_indices
].copy()

print("Queries selected:", len(selected_queries))

print(
    selected_queries[
        [
            "conversation_id",
            "intent_label",
            "benchmark_source"
        ]
    ].to_string(index=False)
)

print("\n✅ 30-query reranking evaluation sample selected.")

Queries changed at bonus 0.02: 198
Queries changed at bonus 0.03: 230
Queries changed by BOTH: 198

SELECTED HUMAN-EVALUATION SAMPLE
Queries selected: 30
 conversation_id                     intent_label benchmark_source
          373303                   DELIVERY_DELAY         training
         1427974 DELIVERY_MISSING_OR_MISDELIVERED         training
         2132842                   DELIVERY_DELAY         training
         1514342 DELIVERY_ATTEMPT_OR_INSTRUCTIONS         training
         1165345                  DIGITAL_CONTENT         training
         2061685                  PRODUCT_PROBLEM         training
         1171343                   DELIVERY_DELAY         training
         1125079     ORDER_STATUS_OR_CANCELLATION         training
         2178028                   DELIVERY_DELAY         training
         2883351 PRODUCT_AVAILABILITY_INFORMATION         training
         2512863                   DELIVERY_DELAY         training
         1039533        RETURN_REPLACEMENT

In [341]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

rerank_review_path = Path(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\intent_reranking_human_review.csv"
)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def build_reranked_indices(query_idx, bonus):

    candidate_indices = clean_indices[query_idx][:10]
    candidate_scores = clean_scores[query_idx][:10]

    query_intent = benchmark_predicted_intents[query_idx]

    candidate_intents = corpus_predicted_intents[
        candidate_indices
    ]

    intent_match = (
        candidate_intents == query_intent
    )

    rerank_scores = (
        candidate_scores
        + bonus * intent_match.astype(float)
    )

    order = np.argsort(
        -rerank_scores,
        kind="stable"
    )

    return (
        candidate_indices[order],
        rerank_scores[order]
    )


# ------------------------------------------------------------
# Build review rows
# ------------------------------------------------------------

review_rows = []

for query_idx in selected_indices:

    query_row = retrieval_benchmark.iloc[query_idx]

    query_id = int(query_row["conversation_id"])

    query_text = str(
        query_row["customer_only_text"]
    )

    gold_intent = str(
        query_row["intent_label"]
    )

    predicted_intent = str(
        benchmark_predicted_intents[query_idx]
    )

    confidence = float(
        benchmark_confidence[query_idx]
    )

    # --------------------------------------------------------
    # Three systems
    # --------------------------------------------------------

    systems = []

    # Baseline
    systems.append(
        ("baseline", 0.00)
    )

    # Intent reranking
    systems.append(
        ("intent_rerank_002", 0.02)
    )

    systems.append(
        ("intent_rerank_003", 0.03)
    )

    for system_name, bonus in systems:

        if bonus == 0:
            retrieved_indices = clean_indices[
                query_idx
            ][:3]

            retrieved_scores = clean_scores[
                query_idx
            ][:3]

        else:
            reranked_indices, reranked_scores = (
                build_reranked_indices(
                    query_idx,
                    bonus
                )
            )

            retrieved_indices = reranked_indices[:3]
            retrieved_scores = reranked_scores[:3]

        for rank, (retrieved_idx, score) in enumerate(
            zip(
                retrieved_indices,
                retrieved_scores
            ),
            start=1
        ):

            retrieved_row = corpus.iloc[
                retrieved_idx
            ]

            retrieved_predicted_intent = str(
                retrieved_row["predicted_intent"]
            )

            review_rows.append({

                "query_conversation_id":
                    query_id,

                "query_gold_intent":
                    gold_intent,

                "query_predicted_intent":
                    predicted_intent,

                "query_intent_confidence":
                    confidence,

                "query_text":
                    query_text,

                "system":
                    system_name,

                "intent_bonus":
                    bonus,

                "rank":
                    rank,

                "retrieved_conversation_id":
                    int(
                        retrieved_row[
                            "conversation_id"
                        ]
                    ),

                "similarity_or_rerank_score":
                    float(score),

                "retrieved_predicted_intent":
                    retrieved_predicted_intent,

                "intent_match":
                    (
                        predicted_intent
                        ==
                        retrieved_predicted_intent
                    ),

                "retrieved_conversation_text":
                    str(
                        retrieved_row[
                            "conversation_text"
                        ]
                    ),

                "relevance_label":
                    pd.NA,

                "annotation_notes":
                    pd.NA,

                "annotator":
                    pd.NA
            })


rerank_review = pd.DataFrame(
    review_rows
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("INTENT-RERANKING HUMAN REVIEW DATASET")
print("=" * 70)

print("Rows:", len(rerank_review))
print(
    "Queries:",
    rerank_review[
        "query_conversation_id"
    ].nunique()
)

print(
    "Systems:",
    rerank_review["system"].value_counts()
)

print(
    "Rows per query/system:",
    rerank_review.groupby(
        [
            "query_conversation_id",
            "system"
        ]
    ).size().value_counts()
)

assert len(rerank_review) == 270
assert (
    rerank_review[
        "query_conversation_id"
    ].nunique()
    == 30
)

assert (
    rerank_review["system"].nunique()
    == 3
)

assert (
    rerank_review.groupby(
        [
            "query_conversation_id",
            "system"
        ]
    ).size().eq(3).all()
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

rerank_review.to_csv(
    rerank_review_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("✅ HUMAN REVIEW FILE CREATED")
print("=" * 70)

print("Saved:", rerank_review_path)
print("Total judgments required:", 270)
print("Queries:", 30)
print("Systems:", 3)
print("Results per system:", 3)

INTENT-RERANKING HUMAN REVIEW DATASET
Rows: 270
Queries: 30
Systems: system
baseline             90
intent_rerank_002    90
intent_rerank_003    90
Name: count, dtype: int64
Rows per query/system: 3    90
Name: count, dtype: int64

✅ HUMAN REVIEW FILE CREATED
Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\intent_reranking_human_review.csv
Total judgments required: 270
Queries: 30
Systems: 3
Results per system: 3


In [343]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# PATHS
# ============================================================

rerank_path = Path(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\intent_reranking_human_review.csv"
)

human_path = Path(
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\retrieval_human_annotations.csv"
)

# ============================================================
# LOAD ONLY THE 270 + EXISTING 450-JUDGMENT FILES
# ============================================================

rerank = pd.read_csv(
    rerank_path,
    encoding="utf-8-sig"
)

human = pd.read_csv(
    human_path,
    encoding="utf-8-sig"
)

assert len(rerank) == 270
assert len(human) == 450

# ============================================================
# CLEAN TEXT
# ============================================================

def clean_text(x):
    x = str(x).lower()
    x = re.sub(r"https?://\S+", " ", x)
    x = re.sub(r"[@#]\w+", " ", x)
    x = re.sub(r"\[customer\]|\[support\]", " ", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

rerank["query_clean"] = rerank["query_text"].map(clean_text)
rerank["retrieved_clean"] = (
    rerank["retrieved_conversation_text"].map(clean_text)
)

# ============================================================
# REUSE EXISTING HUMAN JUDGMENTS WHEN EXACT PAIR EXISTS
# ============================================================

human_lookup = {}

for _, row in human.iterrows():

    key = (
        int(row["query_conversation_id"]),
        int(row["retrieved_conversation_id"])
    )

    label = row["relevance_label"]

    if pd.notna(label):
        human_lookup[key] = int(label)

# ============================================================
# LIGHTWEIGHT TF-IDF ON ONLY 270 ROWS
# ============================================================

all_text = pd.concat([
    rerank["query_clean"],
    rerank["retrieved_clean"]
]).fillna("")

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=15000,
    sublinear_tf=True
)

matrix = vectorizer.fit_transform(all_text)

n = len(rerank)

query_matrix = matrix[:n]
retrieved_matrix = matrix[n:]

similarities = cosine_similarity(
    query_matrix,
    retrieved_matrix
).diagonal()

rerank["text_similarity"] = similarities

# ============================================================
# ASSISTANT RELEVANCE ASSESSMENT
# ============================================================

labels = []
notes = []
sources = []

for i, row in rerank.iterrows():

    query_id = int(row["query_conversation_id"])
    retrieved_id = int(row["retrieved_conversation_id"])

    key = (query_id, retrieved_id)

    # --------------------------------------------------------
    # 1. Exact pair already judged by human
    # --------------------------------------------------------

    if key in human_lookup:

        label = human_lookup[key]

        labels.append(label)
        notes.append(
            "Reused existing human relevance judgment for the identical query/retrieved conversation pair."
        )
        sources.append("existing_human_judgment")

        continue

    # --------------------------------------------------------
    # 2. New pair: lightweight assistant assessment
    # --------------------------------------------------------

    sim = float(row["text_similarity"])

    gold_intent = str(row["query_gold_intent"])
    retrieved_intent = str(
        row["retrieved_predicted_intent"]
    )

    intent_match = (
        gold_intent == retrieved_intent
    )

    # Strong direct semantic evidence
    if sim >= 0.32 and intent_match:
        label = 2
        note = (
            "Assistant assessment: strong textual overlap with "
            "intent-compatible historical evidence."
        )

    # Moderate direct evidence
    elif sim >= 0.22 and intent_match:
        label = 2
        note = (
            "Assistant assessment: materially related historical "
            "case with matching predicted intent."
        )

    # Strong semantic relation despite predicted-intent mismatch
    elif sim >= 0.38:
        label = 1
        note = (
            "Assistant assessment: related historical case, but "
            "predicted intent differs from the query."
        )

    # Moderate relation
    elif sim >= 0.20:
        label = 1
        note = (
            "Assistant assessment: partially related historical "
            "evidence but not a direct match."
        )

    # Weak / unrelated
    else:
        label = 0
        note = (
            "Assistant assessment: insufficient evidence that "
            "the historical case would materially help answer the query."
        )

    labels.append(label)
    notes.append(note)
    sources.append("assistant_assessment")

# ============================================================
# WRITE RESULTS
# ============================================================

rerank["relevance_label"] = labels
rerank["annotation_notes"] = notes
rerank["annotator"] = "assistant_review"
rerank["assessment_source"] = sources

# Remove temporary columns if desired
# Keep text_similarity because it is useful for analysis.
rerank.to_csv(
    rerank_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# VALIDATION
# ============================================================

check = pd.read_csv(
    rerank_path,
    encoding="utf-8-sig"
)

print("=" * 70)
print("✅ RERANKING RELEVANCE PASS COMPLETE")
print("=" * 70)

print("Rows:", len(check))
print(
    "Annotated:",
    check["relevance_label"].notna().sum()
)
print(
    "Remaining:",
    check["relevance_label"].isna().sum()
)

print("\nRelevance distribution:")
print(
    check["relevance_label"]
    .value_counts()
    .sort_index()
)

print("\nAssessment source:")
print(
    check["assessment_source"]
    .value_counts()
)

print("\nSystem distribution:")
print(
    check["system"]
    .value_counts()
)

assert len(check) == 270
assert check["relevance_label"].notna().all()
assert check["relevance_label"].isin([0, 1, 2]).all()

print("\n🎯 All 270 reranking results have relevance assessments.")
print("⚠️ These are assistant assessments, not additional human judgments.")

✅ RERANKING RELEVANCE PASS COMPLETE
Rows: 270
Annotated: 270
Remaining: 0

Relevance distribution:
relevance_label
0    250
1      5
2     15
Name: count, dtype: int64

Assessment source:
assessment_source
assistant_assessment       245
existing_human_judgment     25
Name: count, dtype: int64

System distribution:
system
baseline             90
intent_rerank_002    90
intent_rerank_003    90
Name: count, dtype: int64

🎯 All 270 reranking results have relevance assessments.
⚠️ These are assistant assessments, not additional human judgments.


Intent-aware reranking: Adding a small intent-consistency bonus substantially increased predicted-intent compatibility in retrieved results (Top-1: 54.67% baseline → 76.22% with a 0.02 bonus → 81.56% with a 0.03 bonus). However, because the intent classifier itself achieved only 18.65% Macro F1 on the development validation split, this improvement is treated as a diagnostic rather than evidence of improved retrieval relevance. The primary retrieval evaluation therefore remains the independently human-judged 150-query benchmark.

#### Create the deterministic policy

In [344]:
import re
import pandas as pd
import numpy as np

# ============================================================
# ESCALATION POLICY — V1
# ============================================================

HIGH_RISK_INTENTS = {
    "ACCOUNT_ACCESS_SECURITY",
}

INVESTIGATION_INTENTS = {
    "PAYMENT_BILLING",
    "RETURN_REPLACEMENT_REFUND",
}

ACTIONABLE_INTENTS = {
    "DELIVERY_DELAY",
    "DELIVERY_MISSING_OR_MISDELIVERED",
    "DELIVERY_ATTEMPT_OR_INSTRUCTIONS",
    "ORDER_STATUS_OR_CANCELLATION",
    "PRODUCT_PROBLEM",
    "PRIME_MEMBERSHIP",
    "GIFT_CARD_PROMOTION",
    "PRODUCT_AVAILABILITY_INFORMATION",
    "DIGITAL_CONTENT",
    "DEVICE_TECHNICAL_SUPPORT",
    "WEBSITE_APP_TECHNICAL",
}


def escalation_decision(
    intent,
    retrieval_score,
    retrieved_resolution_text,
    customer_text
):
    """
    Deterministic escalation policy.

    Returns:
        decision: AUTO_HANDLE / ESCALATE
        reason: explicit human-readable reason
        policy_rule: machine-readable rule identifier
    """

    intent = str(intent)
    evidence = str(retrieved_resolution_text).lower()
    customer = str(customer_text).lower()

    # --------------------------------------------------------
    # Rule 1 — Security/account risk
    # --------------------------------------------------------

    if intent in HIGH_RISK_INTENTS:

        return (
            "ESCALATE",
            "Account or security issue requires account-specific handling.",
            "HIGH_RISK_ACCOUNT_SECURITY"
        )

    # --------------------------------------------------------
    # Rule 2 — Insufficient retrieval evidence
    # --------------------------------------------------------

    if retrieval_score < 0.65:

        return (
            "ESCALATE",
            "Historical retrieval evidence is not sufficiently strong for confident handling.",
            "LOW_RETRIEVAL_CONFIDENCE"
        )

    # --------------------------------------------------------
    # Rule 3 — Payment/billing investigation
    # --------------------------------------------------------

    if intent == "PAYMENT_BILLING":

        return (
            "ESCALATE",
            "Payment or billing issue may require account or transaction-specific investigation.",
            "PAYMENT_INVESTIGATION"
        )

    # --------------------------------------------------------
    # Rule 4 — Return/refund with weak evidence
    # --------------------------------------------------------

    if intent == "RETURN_REPLACEMENT_REFUND":

        resolution_terms = [
            "refund",
            "replacement",
            "return",
            "credited",
            "refunded",
            "replacement order"
        ]

        has_resolution_signal = any(
            term in evidence
            for term in resolution_terms
        )

        if not has_resolution_signal:

            return (
                "ESCALATE",
                "Return, replacement, or refund issue lacks clear historical resolution evidence.",
                "RETURN_REFUND_INSUFFICIENT_EVIDENCE"
            )

    # --------------------------------------------------------
    # Rule 5 — Explicit unresolved language
    # --------------------------------------------------------

    unresolved_terms = [
        "still not",
        "still hasn't",
        "still have not",
        "not resolved",
        "again",
        "again and again",
        "multiple times",
        "same problem",
        "no response",
        "nothing happened",
        "not fixed",
        "didn't help",
        "hasn't been resolved"
    ]

    if any(term in customer for term in unresolved_terms):

        return (
            "ESCALATE",
            "Customer indicates a repeated or unresolved problem.",
            "UNRESOLVED_OR_REPEATED"
        )

    # --------------------------------------------------------
    # Rule 6 — Non-support / insufficiently identifiable
    # --------------------------------------------------------

    if intent == "OTHER_NON_SUPPORT":

        return (
            "ESCALATE",
            "Customer intent is not sufficiently identifiable for automated support handling.",
            "NON_SUPPORT_OR_AMBIGUOUS"
        )

    # --------------------------------------------------------
    # Rule 7 — Default auto-handle
    # --------------------------------------------------------

    if intent in ACTIONABLE_INTENTS:

        return (
            "AUTO_HANDLE",
            "Intent is actionable and sufficient historical evidence is available.",
            "ACTIONABLE_WITH_EVIDENCE"
        )

    # --------------------------------------------------------
    # Safety fallback
    # --------------------------------------------------------

    return (
        "ESCALATE",
        "No explicit auto-handling rule applies.",
        "DEFAULT_ESCALATION"
    )


print("✅ Escalation policy V1 defined.")

✅ Escalation policy V1 defined.


In [345]:
# sanity check thhe policy

test_cases = [
    {
        "intent": "DELIVERY_DELAY",
        "score": 0.82,
        "evidence": "Support provided delivery tracking and next delivery steps.",
        "customer": "My package is late."
    },
    {
        "intent": "ACCOUNT_ACCESS_SECURITY",
        "score": 0.90,
        "evidence": "General account help information.",
        "customer": "Someone accessed my account."
    },
    {
        "intent": "PAYMENT_BILLING",
        "score": 0.88,
        "evidence": "General payment information.",
        "customer": "I was charged twice."
    },
    {
        "intent": "DELIVERY_DELAY",
        "score": 0.82,
        "evidence": "Delivery information.",
        "customer": "My package is still not delivered after contacting support twice."
    },
    {
        "intent": "OTHER_NON_SUPPORT",
        "score": 0.90,
        "evidence": "Generic Amazon conversation.",
        "customer": "Just wanted to say thanks!"
    },
]

print("=" * 80)
print("ESCALATION POLICY SANITY CHECK")
print("=" * 80)

for i, case in enumerate(test_cases, start=1):

    decision, reason, rule = escalation_decision(
        intent=case["intent"],
        retrieval_score=case["score"],
        retrieved_resolution_text=case["evidence"],
        customer_text=case["customer"]
    )

    print(f"\nCase {i}")
    print("Decision :", decision)
    print("Rule     :", rule)
    print("Reason   :", reason)

ESCALATION POLICY SANITY CHECK

Case 1
Decision : AUTO_HANDLE
Rule     : ACTIONABLE_WITH_EVIDENCE
Reason   : Intent is actionable and sufficient historical evidence is available.

Case 2
Decision : ESCALATE
Rule     : HIGH_RISK_ACCOUNT_SECURITY
Reason   : Account or security issue requires account-specific handling.

Case 3
Decision : ESCALATE
Rule     : PAYMENT_INVESTIGATION
Reason   : Payment or billing issue may require account or transaction-specific investigation.

Case 4
Decision : ESCALATE
Rule     : UNRESOLVED_OR_REPEATED
Reason   : Customer indicates a repeated or unresolved problem.

Case 5
Decision : ESCALATE
Rule     : NON_SUPPORT_OR_AMBIGUOUS
Reason   : Customer intent is not sufficiently identifiable for automated support handling.


#### Build the escalation benchmark

In [346]:
# ============================================================
# STEP 16 — BUILD ESCALATION BENCHMARK
# ============================================================

import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# Start from the existing 450-query benchmark
# ------------------------------------------------------------

escalation_benchmark = retrieval_benchmark.copy()

# ------------------------------------------------------------
# Attach baseline top-1 retrieval score
# ------------------------------------------------------------

benchmark_id_to_idx = {
    int(row["conversation_id"]): idx
    for idx, row in escalation_benchmark.iterrows()
}

top1_scores = []

for _, row in escalation_benchmark.iterrows():
    idx = benchmark_id_to_idx[int(row["conversation_id"])]
    top1_scores.append(float(clean_scores[idx][0]))

escalation_benchmark["top1_retrieval_score"] = top1_scores

# ------------------------------------------------------------
# Predicted intent
# ------------------------------------------------------------

escalation_benchmark["predicted_intent"] = (
    benchmark_predicted_intents
)

escalation_benchmark["intent_confidence"] = (
    benchmark_confidence
)

# ------------------------------------------------------------
# Customer text
# ------------------------------------------------------------

def extract_customer_text_clean(text):
    text = str(text)

    messages = re.findall(
        r"\[CUSTOMER\]\s*(.*?)(?=\[SUPPORT\]|\Z)",
        text,
        flags=re.DOTALL
    )

    cleaned = []

    for msg in messages:
        msg = re.sub(
            r"\[CUSTOMER\]\s*",
            " ",
            msg
        )
        msg = re.sub(
            r"\[SUPPORT\]\s*",
            " ",
            msg
        )
        msg = re.sub(
            r"\s+",
            " ",
            msg
        ).strip()

        if msg:
            cleaned.append(msg)

    return " ".join(cleaned)


escalation_benchmark["customer_text"] = (
    escalation_benchmark["conversation_text"]
    .fillna("")
    .map(extract_customer_text_clean)
)

# ------------------------------------------------------------
# Explicit unresolved / repeated signals
# ------------------------------------------------------------

unresolved_patterns = [
    r"\bstill not\b",
    r"\bstill hasn't\b",
    r"\bstill have not\b",
    r"\bnot resolved\b",
    r"\bagain\b",
    r"\bagain and again\b",
    r"\bmultiple times\b",
    r"\bsame problem\b",
    r"\bno response\b",
    r"\bnothing happened\b",
    r"\bnot fixed\b",
    r"\bdidn't help\b",
    r"\bhasn't been resolved\b"
]

def has_unresolved_signal(text):
    text = str(text).lower()
    return any(
        re.search(pattern, text)
        for pattern in unresolved_patterns
    )


escalation_benchmark["unresolved_signal"] = (
    escalation_benchmark["customer_text"]
    .map(has_unresolved_signal)
)

# ------------------------------------------------------------
# Target escalation label
# ------------------------------------------------------------

HIGH_RISK_INTENTS = {
    "ACCOUNT_ACCESS_SECURITY",
    "PAYMENT_BILLING",
    "OTHER_NON_SUPPORT"
}

def assign_target_escalation(row):

    intent = row["intent_label"]

    if intent in HIGH_RISK_INTENTS:
        return 1

    if row["unresolved_signal"]:
        return 1

    if row["top1_retrieval_score"] < 0.65:
        return 1

    return 0


escalation_benchmark["target_escalation"] = (
    escalation_benchmark.apply(
        assign_target_escalation,
        axis=1
    )
)

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

print("=" * 70)
print("ESCALATION BENCHMARK")
print("=" * 70)

print("Rows:", len(escalation_benchmark))

print("\nTarget decisions:")
print(
    escalation_benchmark[
        "target_escalation"
    ]
    .map({
        0: "AUTO_HANDLE",
        1: "ESCALATE"
    })
    .value_counts()
)

print("\nTarget escalation rate:")
print(
    escalation_benchmark[
        "target_escalation"
    ].mean()
)

print("\nBy intent:")
print(
    pd.crosstab(
        escalation_benchmark["intent_label"],
        escalation_benchmark["target_escalation"]
    )
)

assert len(escalation_benchmark) == 450
assert escalation_benchmark[
    "target_escalation"
].isin([0, 1]).all()

print("\n✅ Escalation benchmark created.")

ESCALATION BENCHMARK
Rows: 450

Target decisions:
target_escalation
AUTO_HANDLE    283
ESCALATE       167
Name: count, dtype: int64

Target escalation rate:
0.3711111111111111

By intent:
target_escalation                  0   1
intent_label                            
ACCOUNT_ACCESS_SECURITY            0  14
DELIVERY_ATTEMPT_OR_INSTRUCTIONS  20   6
DELIVERY_DELAY                    85  22
DELIVERY_MISSING_OR_MISDELIVERED  39   6
DEVICE_TECHNICAL_SUPPORT           7   0
DIGITAL_CONTENT                   15   4
GIFT_CARD_PROMOTION                6   1
ORDER_STATUS_OR_CANCELLATION      19   1
OTHER_NON_SUPPORT                  0  66
PAYMENT_BILLING                    0  16
PRIME_MEMBERSHIP                  13   0
PRODUCT_AVAILABILITY_INFORMATION  11   4
PRODUCT_PROBLEM                   33  16
RETURN_REPLACEMENT_REFUND         27   6
WEBSITE_APP_TECHNICAL              8   5

✅ Escalation benchmark created.


In [347]:
# ============================================================
#  APPLY ESCALATION POLICY V1
# ============================================================

policy_results = []

for _, row in escalation_benchmark.iterrows():

    # Use the top-1 retrieved historical conversation
    query_idx = benchmark_id_to_idx[
        int(row["conversation_id"])
    ]

    top1_retrieved_idx = clean_indices[
        query_idx
    ][0]

    top1_retrieved_row = (
        amazon_retrieval_benchmark_corpus
        .iloc[top1_retrieved_idx]
    )

    decision, reason, rule = escalation_decision(
        intent=row["predicted_intent"],
        retrieval_score=row["top1_retrieval_score"],
        retrieved_resolution_text=(
            top1_retrieved_row["conversation_text"]
        ),
        customer_text=row["customer_text"]
    )

    policy_results.append({
        "conversation_id":
            int(row["conversation_id"]),

        "predicted_intent":
            row["predicted_intent"],

        "gold_intent":
            row["intent_label"],

        "intent_confidence":
            row["intent_confidence"],

        "top1_retrieval_score":
            row["top1_retrieval_score"],

        "target_escalation":
            int(row["target_escalation"]),

        "policy_decision":
            decision,

        "policy_escalation":
            int(decision == "ESCALATE"),

        "policy_rule":
            rule,

        "policy_reason":
            reason
    })

policy_results_df = pd.DataFrame(policy_results)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(policy_results_df) == 450
assert policy_results_df["policy_decision"].isin(
    ["AUTO_HANDLE", "ESCALATE"]
).all()

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

print("=" * 70)
print("ESCALATION POLICY V1 — PREDICTIONS")
print("=" * 70)

print("\nPolicy decisions:")
print(
    policy_results_df["policy_decision"]
    .value_counts()
)

print("\nPolicy escalation rate:")
print(
    policy_results_df["policy_escalation"].mean()
)

print("\nPolicy rules:")
print(
    policy_results_df["policy_rule"]
    .value_counts()
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

y_true = policy_results_df[
    "target_escalation"
].astype(int)

y_pred = policy_results_df[
    "policy_escalation"
].astype(int)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

print("\n" + "=" * 70)
print("ESCALATION CONFUSION MATRIX")
print("=" * 70)

print(
    "                 Pred AUTO   Pred ESCALATE"
)
print(
    f"True AUTO       {cm[0,0]:10d}   {cm[0,1]:13d}"
)
print(
    f"True ESCALATE   {cm[1,0]:10d}   {cm[1,1]:13d}"
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

false_auto = (
    (y_true == 1) &
    (y_pred == 0)
).sum()

false_escalation = (
    (y_true == 0) &
    (y_pred == 1)
).sum()

false_auto_rate = (
    false_auto / (y_true == 1).sum()
)

false_escalation_rate = (
    false_escalation / (y_true == 0).sum()
)

print("\n" + "=" * 70)
print("ESCALATION METRICS")
print("=" * 70)

print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1                     : {f1:.4f}")
print(f"False-auto count       : {false_auto}")
print(f"False-auto rate        : {false_auto_rate:.4f}")
print(f"False-escalation count : {false_escalation}")
print(f"False-escalation rate  : {false_escalation_rate:.4f}")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

policy_output = (
    r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent"
    r"\data\processed\escalation_policy_results.csv"
)

policy_results_df.to_csv(
    policy_output,
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ Escalation policy evaluation complete.")
print("Saved:", policy_output)

ESCALATION POLICY V1 — PREDICTIONS

Policy decisions:
policy_decision
AUTO_HANDLE    239
ESCALATE       211
Name: count, dtype: int64

Policy escalation rate:
0.4688888888888889

Policy rules:
policy_rule
ACTIONABLE_WITH_EVIDENCE               239
HIGH_RISK_ACCOUNT_SECURITY             120
UNRESOLVED_OR_REPEATED                  32
LOW_RETRIEVAL_CONFIDENCE                30
DEFAULT_ESCALATION                      16
RETURN_REFUND_INSUFFICIENT_EVIDENCE     13
Name: count, dtype: int64

ESCALATION CONFUSION MATRIX
                 Pred AUTO   Pred ESCALATE
True AUTO              209              74
True ESCALATE           30             137

ESCALATION METRICS
Precision              : 0.6493
Recall                 : 0.8204
F1                     : 0.7249
False-auto count       : 30
False-auto rate        : 0.1796
False-escalation count : 74
False-escalation rate  : 0.2615

✅ Escalation policy evaluation complete.
Saved: d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\escalation_pol

Escalation policy: An explicit rule-based escalation layer was evaluated on 450 benchmark conversations using project-defined escalation criteria. V1 achieved 82.04% recall, 64.93% precision, and 72.49% F1. It produced 30 false-auto decisions (17.96% of escalation targets) and 74 false-escalations (26.15% of auto-handle targets). The policy therefore prioritizes detecting cases requiring escalation while retaining 53.11% of benchmark cases for automated handling.

In [348]:
# ============================================================
# HELD-OUT RESPONSE GENERATION BENCHMARK
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent")
PROCESSED = PROJECT_ROOT / "data" / "processed"

# ------------------------------------------------------------
# 1. Load held-out taxonomy audit
# ------------------------------------------------------------

audit_path = PROCESSED / "taxonomy_audit_150.csv"

response_benchmark = pd.read_csv(audit_path)

print("=" * 70)
print("HELD-OUT RESPONSE GENERATION BENCHMARK")
print("=" * 70)

print("Rows:", len(response_benchmark))
print("Columns:", response_benchmark.columns.tolist())

assert len(response_benchmark) == 150, (
    f"Expected 150 held-out examples, got {len(response_benchmark)}"
)

assert response_benchmark["intent_label"].notna().all(), \
    "Missing intent labels found."

print("\nIntent distribution:")
print(response_benchmark["intent_label"].value_counts().sort_index())


# ------------------------------------------------------------
# 2. Create stable benchmark IDs
# ------------------------------------------------------------

response_benchmark = response_benchmark.reset_index(drop=True)

response_benchmark["benchmark_id"] = (
    "AUDIT_" + (response_benchmark.index + 1).astype(str).str.zfill(3)
)

# ------------------------------------------------------------
# 3. Customer-facing query
# ------------------------------------------------------------

def choose_customer_query(row):
    """
    Prefer the latest customer message because it represents
    the customer's most recent stated issue.
    Fall back to the first customer message when necessary.
    """
    last_msg = str(row.get("last_customer_message", "")).strip()

    if last_msg and last_msg.lower() != "nan":
        return last_msg

    first_msg = str(row.get("first_customer_message", "")).strip()

    return first_msg


response_benchmark["customer_query"] = response_benchmark.apply(
    choose_customer_query,
    axis=1
)

assert response_benchmark["customer_query"].str.len().gt(0).all(), \
    "Empty customer queries found."


# ------------------------------------------------------------
# 4. Preserve the original conversation as reference
# ------------------------------------------------------------

response_benchmark["historical_conversation"] = (
    response_benchmark["conversation_text"]
    .fillna("")
    .astype(str)
)

# ------------------------------------------------------------
# 5. Save benchmark
# ------------------------------------------------------------

benchmark_path = (
    PROCESSED / "response_generation_benchmark_150.csv"
)

response_benchmark.to_csv(
    benchmark_path,
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("BENCHMARK CREATED")
print("=" * 70)

print("Rows:", len(response_benchmark))
print("Unique benchmark IDs:",
      response_benchmark["benchmark_id"].nunique())

print("Saved:")
print(benchmark_path)

HELD-OUT RESPONSE GENERATION BENCHMARK
Rows: 150
Columns: ['first_customer_message', 'last_customer_message', 'first_support_message', 'last_support_message', 'conversation_text', 'customer_turns', 'support_turns', 'conversation_size', 'start_time', 'end_time', 'duration_minutes', 'support_account', 'size_bucket', 'intent_label', 'annotation_notes', 'annotator']

Intent distribution:
intent_label
ACCOUNT_ACCESS_SECURITY              5
DELIVERY_ATTEMPT_OR_INSTRUCTIONS     2
DELIVERY_DELAY                      38
DELIVERY_MISSING_OR_MISDELIVERED    13
DEVICE_TECHNICAL_SUPPORT             3
DIGITAL_CONTENT                      6
GIFT_CARD_PROMOTION                  2
ORDER_STATUS_OR_CANCELLATION         7
OTHER_NON_SUPPORT                   29
PAYMENT_BILLING                      5
PRIME_MEMBERSHIP                     5
PRODUCT_AVAILABILITY_INFORMATION     6
PRODUCT_PROBLEM                     17
RETURN_REPLACEMENT_REFUND            8
WEBSITE_APP_TECHNICAL                4
Name: count, dt

In [349]:
# ============================================================
#  RESPONSE BENCHMARK LEAKAGE CHECK
# ============================================================

audit_ids = set()

# taxonomy_audit_150 does not currently contain conversation_id
# so recover the IDs through the benchmark mapping if available.

print("Response benchmark rows:", len(response_benchmark))

# Check whether the benchmark conversations appear in the
# leakage-safe retrieval corpus.

if "amazon_retrieval_benchmark_corpus" in globals():

    corpus_ids = set(
        amazon_retrieval_benchmark_corpus.index.astype(str)
    )

    print("Retrieval corpus conversations:",
          len(corpus_ids))

    print(
        "Note: taxonomy_audit_150.csv does not contain "
        "conversation_id, so direct ID leakage verification "
        "requires the original audit-to-conversation mapping."
    )

else:
    print(
        "Leakage-safe retrieval corpus is not currently loaded. "
        "Do not rebuild the corpus; load the existing benchmark corpus."
    )

Response benchmark rows: 150
Retrieval corpus conversations: 81796
Note: taxonomy_audit_150.csv does not contain conversation_id, so direct ID leakage verification requires the original audit-to-conversation mapping.


In [351]:
# ============================================================
# MAP HELD-OUT AUDIT QUERIES TO EXISTING
#             450-QUERY RETRIEVAL BENCHMARK
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent")
PROCESSED = PROJECT_ROOT / "data" / "processed"

print("=" * 70)
print("MAPPING 150 AUDIT QUERIES TO RETRIEVAL BENCHMARK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify retrieval benchmark exists
# ------------------------------------------------------------

if "retrieval_benchmark" not in globals():
    raise RuntimeError(
        "retrieval_benchmark is not loaded. "
        "Load the existing 450-query retrieval benchmark. "
        "Do NOT rebuild the retrieval embeddings."
    )

print("Retrieval benchmark rows:", len(retrieval_benchmark))

assert len(retrieval_benchmark) == 450, \
    "Expected exactly 450 retrieval benchmark queries."


# ------------------------------------------------------------
# 2. Inspect columns
# ------------------------------------------------------------

print("\nRetrieval benchmark columns:")
print(retrieval_benchmark.columns.tolist())


# ------------------------------------------------------------
# 3. Normalize conversation text
# ------------------------------------------------------------

def normalize_text(x):
    return (
        str(x)
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )


response_benchmark["_match_text"] = (
    response_benchmark["conversation_text"]
    .fillna("")
    .map(normalize_text)
)

retrieval_benchmark["_match_text"] = (
    retrieval_benchmark["conversation_text"]
    .fillna("")
    .map(normalize_text)
)


# ------------------------------------------------------------
# 4. Match audit conversations against the ORIGINAL
#    450-query benchmark, NOT the leakage-safe corpus
# ------------------------------------------------------------

retrieval_text_to_rows = {}

for idx, txt in zip(
    retrieval_benchmark.index,
    retrieval_benchmark["_match_text"]
):
    retrieval_text_to_rows.setdefault(txt, []).append(idx)


response_benchmark["retrieval_benchmark_rows"] = (
    response_benchmark["_match_text"].map(
        lambda x: retrieval_text_to_rows.get(x, [])
    )
)

match_counts = response_benchmark[
    "retrieval_benchmark_rows"
].map(len)


print("\nMatching results:")
print("Exact unique matches:",
      (match_counts == 1).sum())

print("No match:",
      (match_counts == 0).sum())

print("Multiple matches:",
      (match_counts > 1).sum())


# ------------------------------------------------------------
# 5. Validate
# ------------------------------------------------------------

if (match_counts == 0).any():

    print("\nUNMATCHED AUDIT EXAMPLES:")
    print(
        response_benchmark.loc[
            match_counts == 0,
            ["benchmark_id", "conversation_text"]
        ].head(10).to_string(index=False)
    )

    raise RuntimeError(
        "Some audit conversations could not be mapped to the "
        "450-query retrieval benchmark."
    )


if (match_counts > 1).any():

    raise RuntimeError(
        "Some audit conversations matched multiple retrieval "
        "benchmark rows. Do not guess the mapping."
    )


# ------------------------------------------------------------
# 6. Extract the unique retrieval-benchmark row
# ------------------------------------------------------------

response_benchmark["retrieval_benchmark_index"] = (
    response_benchmark["retrieval_benchmark_rows"]
    .map(lambda x: x[0])
)


# ------------------------------------------------------------
# 7. Verify one-to-one mapping
# ------------------------------------------------------------

unique_query_rows = (
    response_benchmark["retrieval_benchmark_index"]
    .nunique()
)

print("\nUnique retrieval benchmark rows matched:",
      unique_query_rows)

assert unique_query_rows == 150, (
    f"Expected 150 unique retrieval queries, "
    f"got {unique_query_rows}"
)


# ------------------------------------------------------------
# 8. Save mapping checkpoint
# ------------------------------------------------------------

mapping_path = (
    PROCESSED /
    "response_generation_audit_retrieval_mapping.csv"
)

response_benchmark[
    [
        "benchmark_id",
        "retrieval_benchmark_index",
        "intent_label",
        "customer_query"
    ]
].to_csv(
    mapping_path,
    index=False,
    encoding="utf-8"
)


print("\n" + "=" * 70)
print("MAPPING SUCCESSFUL")
print("=" * 70)

print("Audit examples:", len(response_benchmark))
print("Unique retrieval queries:", unique_query_rows)

print("\nSaved:")
print(mapping_path)

MAPPING 150 AUDIT QUERIES TO RETRIEVAL BENCHMARK
Retrieval benchmark rows: 450

Retrieval benchmark columns:
['conversation_id', 'conversation_text', 'intent_label', 'benchmark_source', 'customer_only_text']

Matching results:
Exact unique matches: 150
No match: 0
Multiple matches: 0

Unique retrieval benchmark rows matched: 150

MAPPING SUCCESSFUL
Audit examples: 150
Unique retrieval queries: 150

Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\response_generation_audit_retrieval_mapping.csv


In [355]:
# ============================================================
#  ATTACH EXISTING TOP-3 RETRIEVAL RESULTS
#             TO THE 150 HELD-OUT AUDIT QUERIES
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent")
PROCESSED = PROJECT_ROOT / "data" / "processed"

print("=" * 70)
print("ATTACHING TOP-3 RETRIEVAL EVIDENCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate required objects
# ------------------------------------------------------------

required = [
    "response_benchmark",
    "retrieval_benchmark",
    "clean_scores",
    "clean_indices",
    "amazon_retrieval_benchmark_corpus"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        "Load the existing objects; do NOT rebuild retrieval."
    )

assert len(response_benchmark) == 150
assert len(retrieval_benchmark) == 450


# ------------------------------------------------------------
# 2. IMPORTANT:
#    response_benchmark contains the audit examples.
#    The mapping created in Step 45C tells us which row in
#    retrieval_benchmark corresponds to each audit example.
# ------------------------------------------------------------

if "retrieval_benchmark_index" not in response_benchmark.columns:
    raise RuntimeError(
        "retrieval_benchmark_index is missing from response_benchmark. "
        "Run the successful Step 45C mapping cell first."
    )

assert response_benchmark[
    "retrieval_benchmark_index"
].notna().all()

generation_inputs = response_benchmark.copy()


# ------------------------------------------------------------
# 3. Recover the actual source conversation ID from the
#    ORIGINAL retrieval benchmark.
# ------------------------------------------------------------

generation_inputs["conversation_id"] = (
    generation_inputs["retrieval_benchmark_index"]
    .map(
        retrieval_benchmark["conversation_id"].to_dict()
    )
)

if generation_inputs["conversation_id"].isna().any():
    raise RuntimeError(
        "Could not recover conversation IDs from the "
        "retrieval benchmark."
    )


# ------------------------------------------------------------
# 4. Retrieval corpus
# ------------------------------------------------------------

corpus = amazon_retrieval_benchmark_corpus

required_corpus_columns = [
    "conversation_text",
    "first_customer_message",
    "last_customer_message",
    "first_support_message",
    "last_support_message",
    "support_account",
    "conversation_size"
]

missing_corpus_columns = [
    c for c in required_corpus_columns
    if c not in corpus.columns
]

if missing_corpus_columns:
    raise RuntimeError(
        f"Missing corpus columns: {missing_corpus_columns}"
    )


# ------------------------------------------------------------
# 5. Function to obtain an existing retrieval result
# ------------------------------------------------------------

def get_retrieval_result(benchmark_row_index, rank):

    score = float(
        clean_scores[benchmark_row_index][rank]
    )

    corpus_position = int(
        clean_indices[benchmark_row_index][rank]
    )

    retrieved = corpus.iloc[corpus_position]

    return {
        "conversation_id": corpus.index[corpus_position],
        "score": score,
        "conversation_text": retrieved["conversation_text"],
        "first_customer_message": retrieved["first_customer_message"],
        "last_customer_message": retrieved["last_customer_message"],
        "first_support_message": retrieved["first_support_message"],
        "last_support_message": retrieved["last_support_message"],
        "support_account": retrieved["support_account"],
        "conversation_size": retrieved["conversation_size"]
    }


# ------------------------------------------------------------
# 6. Attach Top-1 / Top-2 / Top-3
# ------------------------------------------------------------

for rank in range(3):

    generation_inputs[
        f"retrieval_{rank+1}_conversation_id"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_score"
    ] = np.nan

    generation_inputs[
        f"retrieval_{rank+1}_conversation_text"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_first_customer"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_last_customer"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_first_support"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_last_support"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_support_account"
    ] = None

    generation_inputs[
        f"retrieval_{rank+1}_conversation_size"
    ] = np.nan


for row_idx in generation_inputs.index:

    benchmark_row = int(
        generation_inputs.loc[
            row_idx,
            "retrieval_benchmark_index"
        ]
    )

    for rank in range(3):

        result = get_retrieval_result(
            benchmark_row,
            rank
        )

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_conversation_id"
        ] = result["conversation_id"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_score"
        ] = result["score"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_conversation_text"
        ] = result["conversation_text"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_first_customer"
        ] = result["first_customer_message"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_last_customer"
        ] = result["last_customer_message"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_first_support"
        ] = result["first_support_message"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_last_support"
        ] = result["last_support_message"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_support_account"
        ] = result["support_account"]

        generation_inputs.loc[
            row_idx,
            f"retrieval_{rank+1}_conversation_size"
        ] = result["conversation_size"]


# ------------------------------------------------------------
# 7. Validate all retrieval results
# ------------------------------------------------------------

for rank in range(1, 4):

    score_col = f"retrieval_{rank}_score"
    id_col = f"retrieval_{rank}_conversation_id"

    assert generation_inputs[score_col].notna().all(), \
        f"Missing scores at rank {rank}"

    assert generation_inputs[id_col].notna().all(), \
        f"Missing IDs at rank {rank}"


# ------------------------------------------------------------
# 8. Self-retrieval leakage check
# ------------------------------------------------------------

leaked_rows = []

for idx, row in generation_inputs.iterrows():

    source_id = str(row["conversation_id"])

    retrieved_ids = {
        str(row[f"retrieval_{r}_conversation_id"])
        for r in range(1, 4)
    }

    if source_id in retrieved_ids:
        leaked_rows.append(
            row["benchmark_id"]
        )


print("\nLeakage check:")
print(
    "Held-out audit conversations:",
    generation_inputs["conversation_id"].nunique()
)
print(
    "Self-retrieval leaks:",
    len(leaked_rows)
)

assert len(leaked_rows) == 0, (
    f"Found {len(leaked_rows)} self-retrieval leaks."
)


# ------------------------------------------------------------
# 9. Retrieval score statistics
# ------------------------------------------------------------

print("\nRetrieval score statistics:")

for rank in range(1, 4):

    scores = generation_inputs[
        f"retrieval_{rank}_score"
    ]

    print(
        f"Rank {rank}: "
        f"mean={scores.mean():.4f}, "
        f"median={scores.median():.4f}, "
        f"min={scores.min():.4f}, "
        f"max={scores.max():.4f}"
    )


# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

output_path = (
    PROCESSED /
    "response_generation_inputs_150.csv"
)

generation_inputs.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("TOP-3 RETRIEVAL ATTACHMENT COMPLETE")
print("=" * 70)

print("Rows:", len(generation_inputs))
print(
    "Unique source conversation IDs:",
    generation_inputs["conversation_id"].nunique()
)
print("Columns:", len(generation_inputs.columns))

print("\nSaved:")
print(output_path)

ATTACHING TOP-3 RETRIEVAL EVIDENCE

Leakage check:
Held-out audit conversations: 150
Self-retrieval leaks: 0

Retrieval score statistics:
Rank 1: mean=0.7586, median=0.7660, min=0.5609, max=0.9044
Rank 2: mean=0.7432, median=0.7546, min=0.5316, max=0.9028
Rank 3: mean=0.7341, median=0.7444, min=0.4720, max=0.8811

TOP-3 RETRIEVAL ATTACHMENT COMPLETE
Rows: 150
Unique source conversation IDs: 150
Columns: 51

Saved:
d:\BECAME_DEVELOPER\hiver-sde-ai-agent\data\processed\response_generation_inputs_150.csv


In [358]:
# ============================================================
# APPLY ESCALATION POLICY V1
#             TO 150 HELD-OUT AUDIT EXAMPLES
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent")
PROCESSED = PROJECT_ROOT / "data" / "processed"

print("=" * 70)
print("APPLYING ESCALATION POLICY V1")
print("=" * 70)


# ------------------------------------------------------------
# 1. Validate required objects
# ------------------------------------------------------------

if "generation_inputs" not in globals():
    raise RuntimeError(
        "generation_inputs is not loaded. "
        "Run Step 45D first."
    )

if "escalation_decision" not in globals():
    raise RuntimeError(
        "Existing escalation_decision() function is not loaded."
    )

assert len(generation_inputs) == 150


# ------------------------------------------------------------
# 2. Apply the EXISTING V1 policy exactly
#
# IMPORTANT:
# escalation_decision() returns:
#
#     decision
#     reason_text
#     rule_name
#
# We use those outputs directly.
# ------------------------------------------------------------

policy_inputs = generation_inputs.copy()

policy_decisions = []
policy_reason_texts = []
policy_rules = []


for _, row in policy_inputs.iterrows():

    intent = row["intent_label"]

    retrieval_score = float(
        row["retrieval_1_score"]
    )

    evidence = str(
        row["retrieval_1_conversation_text"]
    )

    customer_text = str(
        row["customer_query"]
    )

    decision, reason_text, rule_name = escalation_decision(
        intent=intent,
        retrieval_score=retrieval_score,
        retrieved_resolution_text=evidence,
        customer_text=customer_text
    )

    policy_decisions.append(decision)
    policy_reason_texts.append(reason_text)
    policy_rules.append(rule_name)


policy_inputs["policy_decision"] = policy_decisions
policy_inputs["policy_reason_text"] = policy_reason_texts
policy_inputs["policy_rule"] = policy_rules


# ------------------------------------------------------------
# 3. Validate policy decisions
# ------------------------------------------------------------

valid_decisions = {
    "AUTO_HANDLE",
    "ESCALATE"
}

invalid_decisions = set(
    policy_inputs["policy_decision"].unique()
) - valid_decisions

assert not invalid_decisions, (
    f"Unexpected policy decisions: {invalid_decisions}"
)


# ------------------------------------------------------------
# 4. Validate that every row has a rule and reason
# ------------------------------------------------------------

assert policy_inputs[
    "policy_reason_text"
].notna().all()

assert policy_inputs[
    "policy_rule"
].notna().all()


# ------------------------------------------------------------
# 5. Display policy decisions
# ------------------------------------------------------------

print("\nPolicy decisions:")
print(
    policy_inputs[
        "policy_decision"
    ].value_counts()
)


# ------------------------------------------------------------
# 6. Escalation rate
# ------------------------------------------------------------

escalation_rate = (
    policy_inputs[
        "policy_decision"
    ]
    .eq("ESCALATE")
    .mean()
)

print("\nPolicy escalation rate:")
print(f"{escalation_rate:.4f}")


# ------------------------------------------------------------
# 7. Rule distribution
# ------------------------------------------------------------

print("\nPolicy rules:")
print(
    policy_inputs[
        "policy_rule"
    ].value_counts()
)


# ------------------------------------------------------------
# 8. Intent × policy
# ------------------------------------------------------------

print("\nIntent × policy decision:")

intent_policy = pd.crosstab(
    policy_inputs["intent_label"],
    policy_inputs["policy_decision"]
)

print(intent_policy)


# ------------------------------------------------------------
# 9. Show sample decisions
# ------------------------------------------------------------

print("\nSample policy decisions:")

print(
    policy_inputs[
        [
            "benchmark_id",
            "intent_label",
            "retrieval_1_score",
            "policy_decision",
            "policy_rule",
            "policy_reason_text"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

output_path = (
    PROCESSED /
    "response_generation_policy_inputs_150.csv"
)

policy_inputs.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)


print("\n" + "=" * 70)
print("ESCALATION POLICY APPLICATION COMPLETE")
print("=" * 70)

print("Rows:", len(policy_inputs))
print("Columns:", len(policy_inputs.columns))

print("\nSaved:")
print(output_path)

APPLYING ESCALATION POLICY V1

Policy decisions:
policy_decision
AUTO_HANDLE    90
ESCALATE       60
Name: count, dtype: int64

Policy escalation rate:
0.4000

Policy rules:
policy_rule
ACTIONABLE_WITH_EVIDENCE               90
HIGH_RISK_ACCOUNT_SECURITY             39
LOW_RETRIEVAL_CONFIDENCE                8
DEFAULT_ESCALATION                      5
UNRESOLVED_OR_REPEATED                  5
RETURN_REFUND_INSUFFICIENT_EVIDENCE     3
Name: count, dtype: int64

Intent × policy decision:
policy_decision                   AUTO_HANDLE  ESCALATE
intent_label                                           
ACCOUNT_ACCESS_SECURITY                     0         5
DELIVERY_ATTEMPT_OR_INSTRUCTIONS            1         1
DELIVERY_DELAY                             35         3
DELIVERY_MISSING_OR_MISDELIVERED           10         3
DEVICE_TECHNICAL_SUPPORT                    3         0
DIGITAL_CONTENT                             6         0
GIFT_CARD_PROMOTION                         2         0
ORDER

In [359]:
# ============================================================
# BUILD GROUNDED RESPONSE GENERATION PROMPTS
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent")
PROCESSED = PROJECT_ROOT / "data" / "processed"

print("=" * 70)
print("BUILDING RESPONSE GENERATION PROMPTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate
# ------------------------------------------------------------

if "policy_inputs" not in globals():
    raise RuntimeError(
        "policy_inputs is not loaded. Run Step 45E first."
    )

assert len(policy_inputs) == 150


# ------------------------------------------------------------
# 2. Prompt builder
# ------------------------------------------------------------

def build_generation_prompt(row):

    evidence_blocks = []

    for rank in range(1, 4):

        evidence = str(
            row[f"retrieval_{rank}_conversation_text"]
        ).strip()

        score = float(
            row[f"retrieval_{rank}_score"]
        )

        evidence_blocks.append(
            f"""
--- HISTORICAL EPISODE {rank} ---
Retrieval similarity: {score:.4f}

{evidence}
"""
        )

    evidence_text = "\n".join(evidence_blocks)

    prompt = f"""
You are an AI customer-support response assistant.

Your task is to draft a concise customer-facing reply
for the customer query below.

Use the historical support conversations only as
evidence/examples of how similar cases were handled.

IMPORTANT RULES:

1. Address the customer's actual issue.
2. Do not invent Amazon policies, refunds, credits,
   delivery dates, replacement promises, or account actions.
3. Historical conversations are examples of past support
   interactions, not guaranteed current policy.
4. Do not claim that you personally performed an action.
5. Do not expose intent labels, retrieval scores,
   internal rules, or internal reasoning.
6. If escalation is required, clearly communicate that
   the case needs further support/account-specific review.
7. Do not claim that escalation has already happened.
8. If historical evidence is insufficient, do not guess.
9. Keep the response concise and professional.
10. Do not copy a historical reply verbatim unless it is
    necessary; adapt the useful information to this customer.

CUSTOMER QUERY:
{row["customer_query"]}

INTENT:
{row["intent_label"]}

ESCALATION DECISION:
{row["policy_decision"]}

ESCALATION REASON:
{row["policy_reason_text"]}

HISTORICAL SUPPORT EVIDENCE:
{evidence_text}

Write ONLY the final customer-facing support reply.
""".strip()

    return prompt


# ------------------------------------------------------------
# 3. Generate prompts
# ------------------------------------------------------------

policy_inputs["generation_prompt"] = (
    policy_inputs.apply(
        build_generation_prompt,
        axis=1
    )
)


# ------------------------------------------------------------
# 4. Validate
# ------------------------------------------------------------

assert policy_inputs[
    "generation_prompt"
].notna().all()

assert policy_inputs[
    "generation_prompt"
].str.len().gt(500).all()


# ------------------------------------------------------------
# 5. Preview
# ------------------------------------------------------------

print("\nPrompt preview:")
print("-" * 70)

print(
    policy_inputs.iloc[0][
        [
            "benchmark_id",
            "customer_query",
            "intent_label",
            "policy_decision",
            "policy_rule",
            "generation_prompt"
        ]
    ].to_string()
)


# ------------------------------------------------------------
# 6. Save prompts
# ------------------------------------------------------------

output_path = (
    PROCESSED /
    "response_generation_prompts_150.csv"
)

policy_inputs.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("GENERATION PROMPTS CREATED")
print("=" * 70)

print("Rows:", len(policy_inputs))

print("\nSaved:")
print(output_path)

BUILDING RESPONSE GENERATION PROMPTS

Prompt preview:
----------------------------------------------------------------------
benchmark_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   AUDIT_001
customer_query                                                                                                                                                                                                                                                                                                                                                    

In [362]:
# ============================================================
# USE FULL CUSTOMER CONTEXT FOR GENERATION
# ============================================================

def extract_customer_messages(conversation_text):
    """
    Keep only customer messages from the historical episode.
    """
    lines = str(conversation_text).splitlines()

    customer_messages = []

    for line in lines:
        line = line.strip()

        if line.startswith("[CUSTOMER]"):
            customer_messages.append(
                line.replace("[CUSTOMER]", "", 1).strip()
            )

    return "\n".join(customer_messages)


policy_inputs["generation_context"] = (
    policy_inputs["conversation_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

assert policy_inputs[
    "generation_context"
].str.len().gt(0).all()

print("=" * 70)
print("GENERATION CONTEXT CHECK")
print("=" * 70)

print(
    policy_inputs.iloc[0][
        [
            "benchmark_id",
            "generation_context",
            "intent_label",
            "policy_decision",
            "policy_rule"
        ]
    ].to_string()
)

GENERATION CONTEXT CHECK
benchmark_id                                                                                                                                                                                                                                                                                                                               AUDIT_001
generation_context    [CUSTOMER] @AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm https://t.co/0DIvZDSRIP\n[SUPPORT] @347202 I'm sorry for the condition of the parcel. Please contact us directly to explore options: https://t.co/JzP7hlA23B ^JF\n[CUSTOMER] @AmazonHelp Thank you
intent_label                                                                                                                                                                                                                                                                                         

In [363]:
# ============================================================
# REBUILD GROUNDED GENERATION PROMPTS
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"d:\BECAME_DEVELOPER\hiver-sde-ai-agent")
PROCESSED = PROJECT_ROOT / "data" / "processed"

print("=" * 70)
print("REBUILDING GROUNDED RESPONSE GENERATION PROMPTS")
print("=" * 70)


def build_generation_prompt(row):

    evidence_blocks = []

    for rank in range(1, 4):

        evidence = str(
            row[f"retrieval_{rank}_conversation_text"]
        ).strip()

        score = float(
            row[f"retrieval_{rank}_score"]
        )

        evidence_blocks.append(
            f"""
--- HISTORICAL EPISODE {rank} ---
Retrieval similarity: {score:.4f}

{evidence}
"""
        )

    evidence_text = "\n".join(evidence_blocks)

    prompt = f"""
You are an AI customer-support response assistant.

Draft a concise customer-facing support reply based on
the customer's full conversation context and the historical
support evidence provided below.

Use the full conversation to understand the customer's
current support state.

If the latest customer message is only an acknowledgement
such as "thank you", do not invent a new support request.

If the issue appears already addressed, respond appropriately
rather than repeating unsupported troubleshooting.

IMPORTANT RULES:

1. Address the customer's actual support situation.
2. Use historical conversations only as evidence/examples.
3. Do not invent Amazon policies, refunds, credits,
   delivery dates, replacement promises, or account actions.
4. Historical conversations are examples of past support
   interactions, not guaranteed current policy.
5. Never claim that you personally performed an action.
6. Never expose internal intent labels, retrieval scores,
   policy rules, or reasoning.
7. If escalation is required, communicate that the case
   needs further support or account-specific review.
8. Do not claim that escalation has already happened.
9. If evidence is insufficient, acknowledge that rather
   than guessing.
10. Keep the reply concise, professional, and helpful.
11. Do not copy historical replies verbatim unless necessary.
12. Do not fabricate details that are absent from the evidence.

CUSTOMER CONVERSATION:
{row["generation_context"]}

INTENT:
{row["intent_label"]}

ESCALATION DECISION:
{row["policy_decision"]}

ESCALATION REASON:
{row["policy_reason_text"]}

HISTORICAL SUPPORT EVIDENCE:
{evidence_text}

Write ONLY the final customer-facing support reply.
""".strip()

    return prompt


policy_inputs["generation_prompt"] = (
    policy_inputs.apply(
        build_generation_prompt,
        axis=1
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(policy_inputs) == 150

assert policy_inputs[
    "generation_prompt"
].notna().all()

assert policy_inputs[
    "generation_prompt"
].str.len().gt(500).all()


# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

print("\nPrompt preview — AUDIT_001")
print("-" * 70)

print(
    policy_inputs.iloc[0][
        [
            "benchmark_id",
            "generation_context",
            "intent_label",
            "policy_decision",
            "policy_rule",
            "policy_reason_text",
            "generation_prompt"
        ]
    ].to_string()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_path = (
    PROCESSED /
    "response_generation_prompts_150.csv"
)

policy_inputs.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("GENERATION PROMPTS REBUILT")
print("=" * 70)

print("Rows:", len(policy_inputs))
print("Prompt column:", "generation_prompt")

print("\nSaved:")
print(output_path)

REBUILDING GROUNDED RESPONSE GENERATION PROMPTS

Prompt preview — AUDIT_001
----------------------------------------------------------------------
benchmark_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    AUDIT_001
generation_context                                                                                                                                                                                         [CUSTOMER] @AmazonHelp please see the attached photos of the state of the ruined parcel which was left outside in the storm http

In [365]:
!pip install -U google-genai

  Using cached google_genai-2.24.0-py3-none-any.whl.metadata (56 kB)
  Using cached google_auth-2.58.0-py3-none-any.whl.metadata (6.0 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached websockets-16.1.1-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pydantic_core-2.46.5-cp311-cp311-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached google_genai-2.24.0-py3-none-any.whl (1.1 MB)
Using cached google_auth-2.58.0-py3-none-any.whl (262 kB)
Using cached pydantic-2.13.5-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.5-cp311-cp311-win_amd64.whl (2.0 MB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using cached websockets-16.1.1-cp311-cp311-win_amd64.whl (180 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)

   ---------------------------------------- 0/7 [websocke

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.8.1 requires numpy<2, but you have numpy 2.1.3 which is incompatible.
mlflow 2.8.1 requires pyarrow<15,>=4.0.0, but you have pyarrow 24.0.0 which is incompatible.
tensorflow-intel 2.15.0 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.1.3 which is incompatible.
torchaudio 2.8.0+cpu requires torch==2.8.0, but you have torch 2.4.1 which is incompatible.
torchvision 0.16.0 requires torch==2.1.0, but you have torch 2.4.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [368]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


#### Export the final Intent model

In [377]:
from pathlib import Path
import joblib
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression


# ============================================================
# FINAL SERVING INTENT MODEL
# ============================================================

PROJECT_ROOT = Path(
    r"D:\BECAME_DEVELOPER\hiver-sde-ai-agent"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "backend"
    / "artifacts"
    / "intent_model"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Use the validated 300-example training dataframe
# ------------------------------------------------------------

serving_df = training_df.copy()

print("Rows:", len(serving_df))
print(
    "Missing labels:",
    serving_df["intent_label"].isna().sum()
)


assert len(serving_df) == 300

assert (
    serving_df["intent_label"]
    .notna()
    .all()
), "Some validated labels are missing."


# ------------------------------------------------------------
# 2. Build training text
# ------------------------------------------------------------

X_text = (
    serving_df["conversation_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

y = (
    serving_df["intent_label"]
    .astype(str)
    .to_numpy()
)


# ------------------------------------------------------------
# 3. Same embedding model selected during evaluation
# ------------------------------------------------------------

embedding_model_name = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)

print(
    "Loading:",
    embedding_model_name
)

embedder = SentenceTransformer(
    embedding_model_name
)


# ------------------------------------------------------------
# 4. Generate embeddings
# ------------------------------------------------------------

X_embeddings = embedder.encode(
    X_text,
    normalize_embeddings=True,
    show_progress_bar=True,
)


X_embeddings = np.asarray(
    X_embeddings,
    dtype="float32"
)


print(
    "Embedding shape:",
    X_embeddings.shape
)


# ------------------------------------------------------------
# 5. Same selected classifier
# ------------------------------------------------------------

serving_classifier = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
)


serving_classifier.fit(
    X_embeddings,
    y,
)


# ------------------------------------------------------------
# 6. Save classifier
# ------------------------------------------------------------

classifier_path = (
    MODEL_DIR
    / "classifier.joblib"
)

joblib.dump(
    serving_classifier,
    classifier_path
)


print(
    "\nSaved:",
    classifier_path
)

print(
    "\nClasses:"
)

print(
    serving_classifier.classes_
)

Rows: 300
Missing labels: 0
Loading: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding shape: (300, 384)

Saved: D:\BECAME_DEVELOPER\hiver-sde-ai-agent\backend\artifacts\intent_model\classifier.joblib

Classes:
['ACCOUNT_ACCESS_SECURITY' 'DELIVERY_ATTEMPT_OR_INSTRUCTIONS'
 'DELIVERY_DELAY' 'DELIVERY_MISSING_OR_MISDELIVERED'
 'DEVICE_TECHNICAL_SUPPORT' 'DIGITAL_CONTENT' 'GIFT_CARD_PROMOTION'
 'ORDER_STATUS_OR_CANCELLATION' 'OTHER_NON_SUPPORT' 'PAYMENT_BILLING'
 'PRIME_MEMBERSHIP' 'PRODUCT_AVAILABILITY_INFORMATION' 'PRODUCT_PROBLEM'
 'RETURN_REPLACEMENT_REFUND' 'WEBSITE_APP_TECHNICAL']


#### Build the serving retrieval corpus

In [378]:
from pathlib import Path
import pandas as pd


PROJECT_ROOT = Path(
    r"D:\BECAME_DEVELOPER\hiver-sde-ai-agent"
)

CORPUS_DIR = (
    PROJECT_ROOT
    / "backend"
    / "artifacts"
    / "corpus"
)

CORPUS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Start from the full AmazonHelp corpus
# ------------------------------------------------------------

serving_corpus = amazon_resolution_corpus.copy()

print(
    "Original AmazonHelp corpus:",
    len(serving_corpus)
)


# ------------------------------------------------------------
# Remove benchmark conversations
# ------------------------------------------------------------

benchmark_ids = set(
    retrieval_benchmark["conversation_id"]
    .astype(str)
)


serving_corpus[
    "conversation_id"
] = (
    serving_corpus[
        "conversation_id"
    ]
    .astype(str)
)


serving_corpus = serving_corpus[
    ~serving_corpus[
        "conversation_id"
    ].isin(
        benchmark_ids
    )
].copy()


print(
    "Leakage-safe corpus:",
    len(serving_corpus)
)


# ------------------------------------------------------------
# Build retrieval text
# ------------------------------------------------------------

serving_corpus[
    "retrieval_text"
] = (
    serving_corpus[
        "conversation_text"
    ]
    .fillna("")
    .astype(str)
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

corpus_path = (
    CORPUS_DIR
    / "amazonhelp_corpus.csv"
)


serving_corpus.to_csv(
    corpus_path,
    index=False
)


print(
    "\nSaved:",
    corpus_path
)

Original AmazonHelp corpus: 82246
Leakage-safe corpus: 81796

Saved: D:\BECAME_DEVELOPER\hiver-sde-ai-agent\backend\artifacts\corpus\amazonhelp_corpus.csv
